In [ ]:
import os
import shutil
import torch

print("PyTorch version:", torch.__version__)
print("GPU available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("GPU not detected. Enable a GPU runtime if available.")

total, used, free = shutil.disk_usage("/content")

print(f"Total storage: {total / (1024**3):.2f} GB")
print(f"Used storage:  {used / (1024**3):.2f} GB")
print(f"Free storage:  {free / (1024**3):.2f} GB")


In [ ]:
import os

folders = [
    "/content/marine_project/data",
    "/content/marine_project/models",
    "/content/marine_project/results",
    "/content/marine_project/outputs"
]

for folder in folders:
    os.makedirs(folder, exist_ok=True)

print("Project folders created successfully.")

for folder in folders:
    print(folder)


In [ ]:
import torch
import shutil

print("PyTorch version:", torch.__version__)
print("GPU available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("GPU is not available. We can still test a small model on CPU.")

total, used, free = shutil.disk_usage("/content")
print(f"Free storage: {free / (1024**3):.2f} GB")

In [ ]:

import os
import torch
import torchvision
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from pathlib import Path

# Project directories
PROJECT_DIR = Path("/content/marine_project")
DATA_DIR = PROJECT_DIR / "data"
MODEL_DIR = PROJECT_DIR / "models"
RESULTS_DIR = PROJECT_DIR / "results"
OUTPUT_DIR = PROJECT_DIR / "outputs"

for folder in [DATA_DIR, MODEL_DIR, RESULTS_DIR, OUTPUT_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

# Verify GPU
print("PyTorch:", torch.__version__)
print("Torchvision:", torchvision.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU name:", torch.cuda.get_device_name(0))
    test_tensor = torch.randn(1000, 1000, device="cuda")
    print("GPU tensor test:", test_tensor.mean().item())
else:
    print("WARNING: GPU unavailable. Do not start GPU training yet.")

print("Project directory:", PROJECT_DIR)
print("Data directory:", DATA_DIR)
print("Model directory:", MODEL_DIR)
print("Results directory:", RESULTS_DIR)
print("Outputs directory:", OUTPUT_DIR)

print("Setup verification completed.")


In [ ]:

from google.colab import drive

drive.mount('/content/drive')


In [ ]:

from pathlib import Path

DRIVE_DATA_DIR = Path(
    "/content/drive/MyDrive/Marine_Project/dataset_zips"
)

print("Folder exists:", DRIVE_DATA_DIR.exists())

if DRIVE_DATA_DIR.exists():
    for file in sorted(DRIVE_DATA_DIR.iterdir()):
        if file.is_file():
            size_gb = file.stat().st_size / (1024 ** 3)
            print(f"{file.name} — {size_gb:.2f} GB")
else:
    print("Check the folder name and path in Google Drive.")


In [ ]:

from pathlib import Path
from zipfile import ZipFile
from collections import Counter

DRIVE_DATA_DIR = Path(
    "/content/drive/MyDrive/Marine_Project/dataset_zips"
)

image_extensions = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff", ".webp"}

zip_files = sorted(DRIVE_DATA_DIR.glob("*.zip"))

if not zip_files:
    print("No ZIP files found. Check your Google Drive folder.")
else:
    for zip_path in zip_files:
        print("\n" + "=" * 65)
        print("Archive:", zip_path.name)

        try:
            with ZipFile(zip_path, "r") as z:
                members = [m for m in z.infolist() if not m.is_dir()]
                images = [
                    m for m in members
                    if Path(m.filename).suffix.lower() in image_extensions
                ]

                print("Total files:", len(members))
                print("Image files:", len(images))
                print("Corrupt archive:", z.testzip())

                print("Example image paths:")
                for item in images[:5]:
                    print(" ", item.filename)

        except Exception as e:
            print("Could not inspect archive:", repr(e))


In [ ]:

from pathlib import Path
from zipfile import ZipFile
from collections import Counter
import shutil
import random
from PIL import Image
import matplotlib.pyplot as plt

DRIVE_DATA_DIR = Path(
    "/content/drive/MyDrive/Marine_Project/dataset_zips"
)
TEST_DIR = Path("/content/marine_project/data/test")
OUTPUT_DIR = Path("/content/marine_project/outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

classes = ["Amphibolis", "Background", "Halophila", "Posidonia"]
extensions = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff", ".webp"}

# Extract only the official testing archives.
for class_name in classes:
    zip_path = DRIVE_DATA_DIR / f"ECU-MSS-2 {class_name}_testing.zip"
    class_dir = TEST_DIR / class_name

    if not zip_path.exists():
        raise FileNotFoundError(f"Missing archive: {zip_path}")

    class_dir.mkdir(parents=True, exist_ok=True)

    with ZipFile(zip_path, "r") as archive:
        for member in archive.infolist():
            if member.is_dir():
                continue

            source_path = Path(member.filename)

            # Accept image files from the expected class folder only.
            if (source_path.parts[0] != class_name or
                source_path.suffix.lower() not in extensions):
                continue

            destination = class_dir / source_path.name

            # Copy only files directly into the expected class directory.
            with archive.open(member) as src, open(destination, "wb") as dst:
                shutil.copyfileobj(src, dst)

    print(f"{class_name}: extracted")

# Verify actual extracted counts.
print("\nExtracted image counts:")
total = 0

for class_name in classes:
    image_files = [
        p for p in (TEST_DIR / class_name).iterdir()
        if p.is_file() and p.suffix.lower() in extensions
    ]
    print(f"{class_name}: {len(image_files)}")
    total += len(image_files)

print("TOTAL TEST IMAGES:", total)

if total != 1040:
    print("WARNING: Expected 1,040 images. Check the counts above.")
else:
    print("Test-set count verification passed.")

# Create a four-class visual preview.
random.seed(42)
fig, axes = plt.subplots(1, 4, figsize=(16, 5))

for ax, class_name in zip(axes, classes):
    candidates = [
        p for p in (TEST_DIR / class_name).iterdir()
        if p.is_file() and p.suffix.lower() in extensions
    ]

    if not candidates:
        ax.set_title(f"{class_name}: no images")
        ax.axis("off")
        continue

    image_path = random.choice(candidates)

    with Image.open(image_path) as img:
        ax.imshow(img.convert("RGB"))

    ax.set_title(class_name)
    ax.axis("off")

fig.suptitle("ECU-MSS-2 Test Dataset: Example Images", fontsize=14)
plt.tight_layout()

preview_path = OUTPUT_DIR / "dataset_class_preview.png"
plt.savefig(preview_path, dpi=200, bbox_inches="tight")
plt.show()

print("\nPreview saved to:", preview_path)


In [ ]:

from pathlib import Path

DRIVE_ROOT = Path("/content/drive/MyDrive")
EXPECTED_DIR = DRIVE_ROOT / "Marine_Project" / "dataset_zips"

print("Expected folder:", EXPECTED_DIR)
print("Expected folder exists:", EXPECTED_DIR.exists())

if EXPECTED_DIR.exists():
    print("\nFiles in the expected folder:")
    for item in sorted(EXPECTED_DIR.iterdir()):
        if item.is_file():
            print(f"  {item.name}")

print("\nSearching Drive for training ZIP filenames...")
matches = []

# Search My Drive; skip common folders that can contain many unrelated files.
skip_dirs = {".Trash", ".shortcut-targets-by-id"}

for current_dir, dirs, files in __import__("os").walk(DRIVE_ROOT):
    dirs[:] = [d for d in dirs if d not in skip_dirs]

    for filename in files:
        lower = filename.lower()
        if lower.endswith(".zip") and ("training" in lower or "train" in lower):
            matches.append(Path(current_dir) / filename)

if matches:
    print("\nTraining ZIP files found:")
    for path in matches:
        print(" ", path)
else:
    print("No training ZIP filenames found in My Drive.")
    print("Check whether the upload finished or the files are in Shared with me.")


In [ ]:
from pathlib import Path

folder = Path("/content/drive/MyDrive/Marine_Project/dataset_zips")

for path in sorted(folder.glob("*.zip")):
    print(f"{path.name} — {path.stat().st_size / (1024**3):.2f} GB")

print("\nTotal ZIP files:", len(list(folder.glob("*.zip"))))

In [ ]:

from pathlib import Path
from zipfile import ZipFile

DRIVE_DATA_DIR = Path(
    "/content/drive/MyDrive/Marine_Project/dataset_zips"
)

classes = ["Amphibolis", "Background", "Halophila", "Posidonia"]

expected_counts = {
    "Amphibolis": 1044,
    "Background": 977,
    "Halophila": 1055,
    "Posidonia": 1085,
}

image_extensions = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff", ".webp"}
total_images = 0
all_counts_match = True

for class_name in classes:
    zip_path = DRIVE_DATA_DIR / f"ECU-MSS-2 {class_name}_training.zip"

    print("\n" + "=" * 60)
    print("Class:", class_name)

    if not zip_path.exists():
        print("ERROR: Archive not found:", zip_path.name)
        all_counts_match = False
        continue

    with ZipFile(zip_path, "r") as archive:
        members = [m for m in archive.infolist() if not m.is_dir()]
        images = [
            m for m in members
            if Path(m.filename).suffix.lower() in image_extensions
        ]

        print("Archive:", zip_path.name)
        print("Total files:", len(members))
        print("Image files:", len(images))
        print("Expected images:", expected_counts[class_name])
        print("Count matches:", len(images) == expected_counts[class_name])

        print("Example paths:")
        for item in images[:3]:
            print(" ", item.filename)

        total_images += len(images)
        if len(images) != expected_counts[class_name]:
            all_counts_match = False

print("\n" + "=" * 60)
print("Total training images found:", total_images)
print("Expected total:", sum(expected_counts.values()))
print("All class counts match:", all_counts_match)


In [ ]:

from pathlib import Path
from zipfile import ZipFile
from PIL import Image
import shutil

DRIVE_DATA_DIR = Path(
    "/content/drive/MyDrive/Marine_Project/dataset_zips"
)
TRAIN_DIR = Path("/content/marine_project/data/train")

classes = ["Amphibolis", "Background", "Halophila", "Posidonia"]
image_extensions = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff", ".webp"}

TRAIN_DIR.mkdir(parents=True, exist_ok=True)

for class_name in classes:
    zip_path = DRIVE_DATA_DIR / f"ECU-MSS-2 {class_name}_training.zip"
    output_class_dir = TRAIN_DIR / class_name
    output_class_dir.mkdir(parents=True, exist_ok=True)

    extracted = 0

    with ZipFile(zip_path, "r") as archive:
        for member in archive.infolist():
            if member.is_dir():
                continue

            source = Path(member.filename)

            # Accept only images inside the expected class folder.
            if (
                not source.parts
                or source.parts[0] != class_name
                or source.suffix.lower() not in image_extensions
            ):
                continue

            # Use the filename only, preventing unexpected nested paths.
            destination = output_class_dir / source.name

            with archive.open(member) as src, open(destination, "wb") as dst:
                shutil.copyfileobj(src, dst)

            extracted += 1

    print(f"{class_name}: extracted {extracted} images")

print("\nVerifying extracted training counts...")
total = 0

for class_name in classes:
    image_files = [
        p for p in (TRAIN_DIR / class_name).iterdir()
        if p.is_file() and p.suffix.lower() in image_extensions
    ]
    print(f"{class_name}: {len(image_files)}")
    total += len(image_files)

print("Total extracted training images:", total)
print("Expected total: 4161")
print("Counts match:", total == 4161)

# Test whether a sample image from each class can be opened.
print("\nChecking sample image readability...")
for class_name in classes:
    image_files = sorted(
        p for p in (TRAIN_DIR / class_name).iterdir()
        if p.is_file() and p.suffix.lower() in image_extensions
    )

    if not image_files:
        print(class_name, "ERROR: no images found")
        continue

    try:
        with Image.open(image_files[0]) as img:
            img.verify()
        print(class_name, "sample image readable:", image_files[0].name)
    except Exception as e:
        print(class_name, "sample image error:", repr(e))


In [ ]:

from pathlib import Path
from sklearn.model_selection import train_test_split
import pandas as pd

TRAIN_DIR = Path("/content/marine_project/data/train")
RESULTS_DIR = Path("/content/marine_project/results")
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

classes = ["Amphibolis", "Background", "Halophila", "Posidonia"]
extensions = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff", ".webp"}

records = []

for class_name in classes:
    class_dir = TRAIN_DIR / class_name

    for image_path in sorted(class_dir.iterdir()):
        if image_path.is_file() and image_path.suffix.lower() in extensions:
            records.append({
                "path": str(image_path),
                "label": class_name
            })

df = pd.DataFrame(records)

if df.empty:
    raise RuntimeError("No training images found. Complete Step 12 first.")

train_records = []
val_records = []

for class_name in classes:
    class_df = df[df["label"] == class_name]

    train_part, val_part = train_test_split(
        class_df,
        test_size=0.20,
        random_state=42,
        shuffle=True
    )

    train_records.append(train_part)
    val_records.append(val_part)

train_df = pd.concat(train_records).sample(frac=1, random_state=42).reset_index(drop=True)
val_df = pd.concat(val_records).sample(frac=1, random_state=42).reset_index(drop=True)

train_csv = RESULTS_DIR / "train_split.csv"
val_csv = RESULTS_DIR / "validation_split.csv"

train_df.to_csv(train_csv, index=False)
val_df.to_csv(val_csv, index=False)

print("Training images:", len(train_df))
print("Validation images:", len(val_df))
print("\nTraining counts by class:")
print(train_df["label"].value_counts().reindex(classes))
print("\nValidation counts by class:")
print(val_df["label"].value_counts().reindex(classes))

print("\nSaved split files:")
print(train_csv)
print(val_csv)


In [ ]:

from pathlib import Path
from PIL import Image
import pandas as pd
import matplotlib.pyplot as plt
import random

PROJECT_DIR = Path("/content/marine_project")
TRAIN_DIR = PROJECT_DIR / "data/train"
RESULTS_DIR = PROJECT_DIR / "results"
OUTPUT_DIR = PROJECT_DIR / "outputs"

classes = ["Amphibolis", "Background", "Halophila", "Posidonia"]
extensions = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff", ".webp"}

# Inspect image dimensions using a sample from each class.
print("Sample image dimensions:")
for class_name in classes:
    files = sorted(
        p for p in (TRAIN_DIR / class_name).iterdir()
        if p.is_file() and p.suffix.lower() in extensions
    )

    if not files:
        print(class_name, "- no images found")
        continue

    with Image.open(files[0]) as img:
        print(f"{class_name}: {img.width} x {img.height}, mode={img.mode}")

# Plot four randomly selected training examples.
random.seed(42)
fig, axes = plt.subplots(1, 4, figsize=(16, 5))

for ax, class_name in zip(axes, classes):
    files = sorted(
        p for p in (TRAIN_DIR / class_name).iterdir()
        if p.is_file() and p.suffix.lower() in extensions
    )

    image_path = random.choice(files)

    with Image.open(image_path) as img:
        ax.imshow(img.convert("RGB"))

    ax.set_title(class_name)
    ax.axis("off")

fig.suptitle("ECU-MSS-2 Training Dataset: Sample Images", fontsize=14)
plt.tight_layout()

sample_path = OUTPUT_DIR / "training_sample_images.png"
plt.savefig(sample_path, dpi=200, bbox_inches="tight")
plt.show()


In [ ]:
# Save a class-distribution chart.
train_df = pd.read_csv(RESULTS_DIR / "train_split.csv")
val_df = pd.read_csv(RESULTS_DIR / "validation_split.csv")

distribution = pd.DataFrame({
    "Training": train_df["label"].value_counts().reindex(classes),
    "Validation": val_df["label"].value_counts().reindex(classes)
})

ax = distribution.plot(kind="bar", figsize=(9, 5))
ax.set_title("Training and Validation Class Distribution")
ax.set_xlabel("Class")
ax.set_ylabel("Number of images")
ax.tick_params(axis="x", rotation=20)
plt.tight_layout()

distribution_path = OUTPUT_DIR / "class_distribution.png"
plt.savefig(distribution_path, dpi=200, bbox_inches="tight")
plt.show()

print("\nSaved visualization files:")
print(sample_path)
print(distribution_path)

In [ ]:

from pathlib import Path
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image

PROJECT_DIR = Path("/content/marine_project")
RESULTS_DIR = PROJECT_DIR / "results"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

if device.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))

class_names = ["Amphibolis", "Background", "Halophila", "Posidonia"]
class_to_idx = {name: idx for idx, name in enumerate(class_names)}
print("Class mapping:", class_to_idx)

# ImageNet normalization is standard for pretrained ResNet models.
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=10),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

eval_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

class MarineImageDataset(Dataset):
    def __init__(self, csv_path, transform=None):
        self.data = pd.read_csv(csv_path)
        self.transform = transform

        required_columns = {"path", "label"}
        if not required_columns.issubset(self.data.columns):
            raise ValueError(f"CSV must contain columns: {required_columns}")

        unknown_labels = set(self.data["label"]) - set(class_to_idx)
        if unknown_labels:
            raise ValueError(f"Unexpected labels: {unknown_labels}")

    def __len__(self):
        return len(self.data)

    def __getitem__(self, index):
        row = self.data.iloc[index]
        image_path = Path(row["path"])

        with Image.open(image_path) as image:
            image = image.convert("RGB")

        if self.transform:
            image = self.transform(image)

        label = class_to_idx[row["label"]]
        return image, label

train_dataset = MarineImageDataset(
    RESULTS_DIR / "train_split.csv",
    transform=train_transform
)

val_dataset = MarineImageDataset(
    RESULTS_DIR / "validation_split.csv",
    transform=eval_transform
)

# A moderate batch size is appropriate for a Colab T4.
BATCH_SIZE = 32
NUM_WORKERS = 2

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=(device.type == "cuda")
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=(device.type == "cuda")
)

# Verify a real batch.
images, labels = next(iter(train_loader))

print("\nData loader verification")
print("Training images:", len(train_dataset))
print("Validation images:", len(val_dataset))
print("Batch image shape:", tuple(images.shape))
print("Batch label shape:", tuple(labels.shape))
print("Labels in first batch:", sorted(labels.unique().tolist()))
print("Expected image shape: (batch_size, 3, 224, 224)")


In [ ]:

import torch
import torch.nn as nn
import pandas as pd
import matplotlib.pyplot as plt

from pathlib import Path
from torchvision.models import resnet18, ResNet18_Weights
from tqdm.auto import tqdm

PROJECT_DIR = Path("/content/marine_project")
MODEL_DIR = PROJECT_DIR / "models"
RESULTS_DIR = PROJECT_DIR / "results"
OUTPUT_DIR = PROJECT_DIR / "outputs"

for folder in [MODEL_DIR, RESULTS_DIR, OUTPUT_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
NUM_CLASSES = 4
LEARNING_RATE = 1e-4
EPOCHS = 1

# Reproducibility for this initial experiment
torch.manual_seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(42)

# Load pretrained ResNet-18 weights.
# The first run may download weights from the internet.
print("Loading pretrained ResNet-18...")
weights = ResNet18_Weights.DEFAULT
model = resnet18(weights=weights)

# Replace the ImageNet classifier with our four-class output layer.
in_features = model.fc.in_features
model.fc = nn.Linear(in_features, NUM_CLASSES)
model = model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)

# Mixed precision can reduce GPU memory use on the T4.
use_amp = device.type == "cuda"
scaler = torch.amp.GradScaler("cuda", enabled=use_amp)

def run_training_epoch(model, loader):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    progress = tqdm(loader, desc="Training", leave=True)

    for images, labels in progress:
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        with torch.autocast(
            device_type=device.type,
            dtype=torch.float16,
            enabled=use_amp
        ):
            outputs = model(images)
            loss = criterion(outputs, labels)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        batch_size = labels.size(0)
        running_loss += loss.item() * batch_size
        correct += (outputs.argmax(dim=1) == labels).sum().item()
        total += batch_size

        progress.set_postfix(loss=f"{loss.item():.4f}")

    return running_loss / total, correct / total

@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    running_loss = 0.0
    correct = 0
    total = 0

    for images, labels in tqdm(loader, desc="Validation", leave=True):
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        with torch.autocast(
            device_type=device.type,
            dtype=torch.float16,
            enabled=use_amp
        ):
            outputs = model(images)
            loss = criterion(outputs, labels)

        batch_size = labels.size(0)
        running_loss += loss.item() * batch_size
        correct += (outputs.argmax(dim=1) == labels).sum().item()
        total += batch_size

    return running_loss / total, correct / total

history = []

print(f"\nDevice: {device}")
print(f"Training images: {len(train_dataset)}")
print(f"Validation images: {len(val_dataset)}")
print(f"Learning rate: {LEARNING_RATE}")
print("\nStarting first epoch...")

for epoch in range(EPOCHS):
    train_loss, train_accuracy = run_training_epoch(model, train_loader)
    val_loss, val_accuracy = evaluate(model, val_loader)

    record = {
        "epoch": epoch + 1,
        "train_loss": train_loss,
        "train_accuracy": train_accuracy,
        "val_loss": val_loss,
        "val_accuracy": val_accuracy
    }
    history.append(record)

    print(f"\nEpoch {epoch + 1}/{EPOCHS}")
    print(f"Train loss:     {train_loss:.4f}")
    print(f"Train accuracy: {train_accuracy * 100:.2f}%")
    print(f"Val loss:       {val_loss:.4f}")
    print(f"Val accuracy:   {val_accuracy * 100:.2f}%")

# Save metrics and model checkpoint.
history_df = pd.DataFrame(history)
history_path = RESULTS_DIR / "resnet18_baseline_history.csv"
checkpoint_path = MODEL_DIR / "resnet18_baseline_epoch1.pth"

history_df.to_csv(history_path, index=False)

torch.save({
    "model_state_dict": model.state_dict(),
    "class_names": class_names,
    "class_to_idx": class_to_idx,
    "epoch": EPOCHS,
    "optimizer_state_dict": optimizer.state_dict(),
    "metrics": history[-1],
    "architecture": "ResNet-18",
    "pretrained_weights": "ImageNet ResNet18_Weights.DEFAULT"
}, checkpoint_path)

# Plot the actual first-epoch results.
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

axes[0].plot(history_df["epoch"], history_df["train_loss"], marker="o", label="Training")
axes[0].plot(history_df["epoch"], history_df["val_loss"], marker="o", label="Validation")
axes[0].set_title("Loss")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Cross-entropy loss")
axes[0].legend()

axes[1].plot(history_df["epoch"], history_df["train_accuracy"] * 100, marker="o", label="Training")
axes[1].plot(history_df["epoch"], history_df["val_accuracy"] * 100, marker="o", label="Validation")
axes[1].set_title("Accuracy")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Accuracy (%)")
axes[1].legend()

plt.tight_layout()
plot_path = OUTPUT_DIR / "resnet18_baseline_epoch1.png"
plt.savefig(plot_path, dpi=200, bbox_inches="tight")
plt.show()

print("\nSaved history:", history_path)
print("Saved checkpoint:", checkpoint_path)
print("Saved training plot:", plot_path)
print("\nBaseline first epoch completed.")


In [ ]:

from pathlib import Path
import pandas as pd
import torch
from torch.utils.data import DataLoader
from tqdm.auto import tqdm

PROJECT_DIR = Path("/content/marine_project")
MODEL_DIR = PROJECT_DIR / "models"
RESULTS_DIR = PROJECT_DIR / "results"
OUTPUT_DIR = PROJECT_DIR / "outputs"

MODEL_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Rebuild loaders with no multiprocessing workers.
BATCH_SIZE = 32

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0,
    pin_memory=(device.type == "cuda")
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=(device.type == "cuda")
)

# Restore the best available checkpoint from epoch 1.
checkpoint_path = MODEL_DIR / "resnet18_baseline_epoch1.pth"
checkpoint = torch.load(checkpoint_path, map_location=device, weights_only=False)
model.load_state_dict(checkpoint["model_state_dict"])

# Start with epoch 1's recorded metrics.
history_path = RESULTS_DIR / "resnet18_baseline_history.csv"
history_df = pd.read_csv(history_path)
history = history_df.to_dict("records")

best_val_loss = min(float(row["val_loss"]) for row in history)
best_checkpoint_path = MODEL_DIR / "resnet18_baseline_best.pth"

torch.save({
    "model_state_dict": model.state_dict(),
    "class_names": class_names,
    "class_to_idx": class_to_idx,
    "epoch": 1,
    "val_loss": best_val_loss,
    "architecture": "ResNet-18"
}, best_checkpoint_path)

MAX_ADDITIONAL_EPOCHS = 4
PATIENCE = 3
epochs_without_improvement = 0

def train_one_epoch():
    model.train()
    total_loss, correct, total = 0.0, 0, 0

    for images, labels in tqdm(train_loader, desc="Training", leave=False):
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        with torch.autocast(
            device_type=device.type,
            dtype=torch.float16,
            enabled=use_amp
        ):
            outputs = model(images)
            loss = criterion(outputs, labels)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        n = labels.size(0)
        total_loss += loss.item() * n
        correct += (outputs.argmax(1) == labels).sum().item()
        total += n

    return total_loss / total, correct / total

@torch.no_grad()
def validate():
    model.eval()
    total_loss, correct, total = 0.0, 0, 0

    for images, labels in tqdm(val_loader, desc="Validation", leave=False):
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        with torch.autocast(
            device_type=device.type,
            dtype=torch.float16,
            enabled=use_amp
        ):
            outputs = model(images)
            loss = criterion(outputs, labels)

        n = labels.size(0)
        total_loss += loss.item() * n
        correct += (outputs.argmax(1) == labels).sum().item()
        total += n

    return total_loss / total, correct / total

for epoch in range(2, MAX_ADDITIONAL_EPOCHS + 2):
    train_loss, train_acc = train_one_epoch()
    val_loss, val_acc = validate()

    row = {
        "epoch": epoch,
        "train_loss": train_loss,
        "train_accuracy": train_acc,
        "val_loss": val_loss,
        "val_accuracy": val_acc
    }
    history.append(row)

    print(
        f"Epoch {epoch}: "
        f"train_loss={train_loss:.4f}, "
        f"train_acc={train_acc * 100:.2f}%, "
        f"val_loss={val_loss:.4f}, "
        f"val_acc={val_acc * 100:.2f}%"
    )

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        epochs_without_improvement = 0

        torch.save({
            "model_state_dict": model.state_dict(),
            "class_names": class_names,
            "class_to_idx": class_to_idx,
            "epoch": epoch,
            "val_loss": val_loss,
            "val_accuracy": val_acc,
            "architecture": "ResNet-18"
        }, best_checkpoint_path)

        print("  Best checkpoint updated.")
    else:
        epochs_without_improvement += 1
        print("  Validation loss did not improve.")

    pd.DataFrame(history).to_csv(history_path, index=False)

    if epochs_without_improvement >= PATIENCE:
        print("Early stopping: validation loss did not improve.")
        break

print("\nTraining continuation complete.")
print("History:", history_path)
print("Best checkpoint:", best_checkpoint_path)


In [ ]:

import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

PROJECT_DIR = Path("/content/marine_project")
history_path = PROJECT_DIR / "results/resnet18_baseline_history.csv"
plot_path = PROJECT_DIR / "outputs/resnet18_baseline_learning_curves.png"

history_df = pd.read_csv(history_path)

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

axes[0].plot(history_df["epoch"], history_df["train_loss"], marker="o", label="Training")
axes[0].plot(history_df["epoch"], history_df["val_loss"], marker="o", label="Validation")
axes[0].set_title("Loss by Epoch")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Cross-entropy loss")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(
    history_df["epoch"], history_df["train_accuracy"] * 100,
    marker="o", label="Training"
)
axes[1].plot(
    history_df["epoch"], history_df["val_accuracy"] * 100,
    marker="o", label="Validation"
)
axes[1].set_title("Accuracy by Epoch")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Accuracy (%)")
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(plot_path, dpi=200, bbox_inches="tight")
plt.show()

print("Epochs recorded:", len(history_df))
print("Best validation loss:", history_df["val_loss"].min())
print("Highest validation accuracy:", history_df["val_accuracy"].max() * 100, "%")
print("Learning curves saved:", plot_path)


In [ ]:

import pandas as pd
import torch
import os

history_path = "/content/marine_project/results/resnet18_baseline_history.csv"
best_model_path = "/content/marine_project/models/resnet18_baseline_best.pth"

history_df = pd.read_csv(history_path)

print("TRAINING HISTORY")
display(history_df.round(5))

print("\nNumber of recorded epochs:", len(history_df))

# Show the row with the lowest validation loss
best_row = history_df.loc[history_df["val_loss"].idxmin()]
print("\nBest epoch based on validation loss:")
print(best_row)

print("\nBest checkpoint exists:", os.path.exists(best_model_path))

if os.path.exists(best_model_path):
    checkpoint = torch.load(
        best_model_path,
        map_location="cpu",
        weights_only=False
    )
    print("Checkpoint epoch:", checkpoint.get("epoch"))
    print("Checkpoint metrics:", checkpoint.get("metrics"))
    print("Architecture:", checkpoint.get("architecture"))
    print("Classes:", checkpoint.get("class_names"))


In [ ]:

import torch
import os

paths = [
    "/content/marine_project/models/resnet18_baseline_epoch1.pth",
    "/content/marine_project/models/resnet18_baseline_best.pth",
]

for path in paths:
    print("\n" + "=" * 60)
    print("File:", os.path.basename(path))
    print("Exists:", os.path.exists(path))

    if os.path.exists(path):
        ckpt = torch.load(path, map_location="cpu", weights_only=False)
        print("Keys:", list(ckpt.keys()))
        print("Epoch:", ckpt.get("epoch"))
        print("Metrics:", ckpt.get("metrics"))
        print("Architecture:", ckpt.get("architecture"))


In [ ]:

import os
import copy
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision.models import resnet18
from tqdm.auto import tqdm

# Paths
model_dir = "/content/marine_project/models"
results_dir = "/content/marine_project/results"

epoch1_path = f"{model_dir}/resnet18_baseline_epoch1.pth"
clean_history_path = f"{results_dir}/resnet18_baseline_clean_history.csv"
clean_best_path = f"{model_dir}/resnet18_baseline_best_clean.pth"

# Confirm that required variables from previous steps exist
required = [
    "train_dataset", "val_dataset", "class_names",
    "class_to_idx", "device"
]
missing = [name for name in required if name not in globals()]
if missing:
    raise RuntimeError(
        f"Missing variables: {missing}. Tell me this output; "
        "we will restore the model and data loaders first."
    )

# Rebuild loaders with num_workers=0 for Colab stability
train_loader = DataLoader(
    train_dataset, batch_size=32, shuffle=True,
    num_workers=0, pin_memory=(device.type == "cuda")
)
val_loader = DataLoader(
    val_dataset, batch_size=32, shuffle=False,
    num_workers=0, pin_memory=(device.type == "cuda")
)

# Recreate the same ResNet-18 architecture
model = resnet18(weights=None)
model.fc = nn.Linear(model.fc.in_features, len(class_names))

checkpoint = torch.load(
    epoch1_path, map_location=device, weights_only=False
)
model.load_state_dict(checkpoint["model_state_dict"])
model = model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

use_amp = device.type == "cuda"
scaler = torch.amp.GradScaler("cuda", enabled=use_amp)

# Preserve the genuine Epoch 1 metrics
first_metrics = checkpoint["metrics"]
history = [first_metrics.copy()]

best_val_loss = float(first_metrics["val_loss"])
best_epoch = 1
best_state = copy.deepcopy(model.state_dict())
epochs_without_improvement = 0

# Save a correct best checkpoint immediately
torch.save({
    "model_state_dict": best_state,
    "class_names": class_names,
    "class_to_idx": class_to_idx,
    "epoch": 1,
    "val_loss": best_val_loss,
    "metrics": first_metrics,
    "architecture": "ResNet-18"
}, clean_best_path)

# Train up to four additional epochs
for epoch in range(2, 6):
    model.train()
    train_loss_sum = 0.0
    train_correct = 0
    train_total = 0

    for images, labels in tqdm(train_loader, desc=f"Epoch {epoch} train"):
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)

        with torch.autocast(
            device_type="cuda",
            dtype=torch.float16,
            enabled=use_amp
        ):
            outputs = model(images)
            loss = criterion(outputs, labels)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        train_loss_sum += loss.item() * labels.size(0)
        train_correct += (outputs.argmax(1) == labels).sum().item()
        train_total += labels.size(0)

    train_loss = train_loss_sum / train_total
    train_acc = train_correct / train_total

    model.eval()
    val_loss_sum = 0.0
    val_correct = 0
    val_total = 0

    with torch.no_grad():
        for images, labels in tqdm(val_loader, desc=f"Epoch {epoch} val"):
            images = images.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)

            with torch.autocast(
                device_type="cuda",
                dtype=torch.float16,
                enabled=use_amp
            ):
                outputs = model(images)
                loss = criterion(outputs, labels)

            val_loss_sum += loss.item() * labels.size(0)
            val_correct += (outputs.argmax(1) == labels).sum().item()
            val_total += labels.size(0)

    val_loss = val_loss_sum / val_total
    val_acc = val_correct / val_total

    metrics = {
        "epoch": epoch,
        "train_loss": train_loss,
        "train_accuracy": train_acc,
        "val_loss": val_loss,
        "val_accuracy": val_acc
    }
    history.append(metrics)

    print(
        f"Epoch {epoch}: train_loss={train_loss:.4f}, "
        f"train_acc={train_acc:.2%}, "
        f"val_loss={val_loss:.4f}, val_acc={val_acc:.2%}"
    )

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_epoch = epoch
        epochs_without_improvement = 0
        best_state = copy.deepcopy(model.state_dict())

        torch.save({
            "model_state_dict": best_state,
            "class_names": class_names,
            "class_to_idx": class_to_idx,
            "epoch": epoch,
            "val_loss": val_loss,
            "metrics": metrics,
            "architecture": "ResNet-18"
        }, clean_best_path)

        print("Saved improved best checkpoint.")
    else:
        epochs_without_improvement += 1
        print("Validation loss did not improve.")

    pd.DataFrame(history).to_csv(clean_history_path, index=False)

    if epochs_without_improvement >= 3:
        print("Early stopping.")
        break

print("\nClean training run complete.")
print("Best epoch:", best_epoch)
print("Best validation loss:", best_val_loss)
print("Clean history:", clean_history_path)
print("Best checkpoint:", clean_best_path)


In [ ]:

import os
import copy
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

from PIL import Image
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from torchvision.models import resnet18
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay
)

# --------------------------------------------------
# 1. Paths and configuration
# --------------------------------------------------
data_dir = "/content/marine_project/data/test"
model_path = "/content/marine_project/models/resnet18_baseline_best_clean.pth"
results_dir = "/content/marine_project/results"
outputs_dir = "/content/marine_project/outputs"

os.makedirs(results_dir, exist_ok=True)
os.makedirs(outputs_dir, exist_ok=True)

checkpoint = torch.load(
    model_path,
    map_location=device,
    weights_only=False
)

class_names = checkpoint["class_names"]
class_to_idx = checkpoint["class_to_idx"]

print("Loaded model:", checkpoint["architecture"])
print("Selected epoch:", checkpoint["epoch"])
print("Best validation loss:", checkpoint["val_loss"])
print("Classes:", class_names)

# --------------------------------------------------
# 2. Collect test images and verify class counts
# --------------------------------------------------
valid_extensions = (".jpg", ".jpeg", ".png", ".bmp", ".webp")
test_records = []

for class_name in class_names:
    class_dir = os.path.join(data_dir, class_name)

    if not os.path.isdir(class_dir):
        raise FileNotFoundError(
            f"Test class folder not found: {class_dir}"
        )

    files = sorted([
        os.path.join(class_dir, f)
        for f in os.listdir(class_dir)
        if f.lower().endswith(valid_extensions)
    ])

    print(f"{class_name}: {len(files)} test images")

    for path in files:
        test_records.append({
            "path": path,
            "label": class_to_idx[class_name],
            "true_class": class_name
        })

test_df = pd.DataFrame(test_records)

if test_df.empty:
    raise RuntimeError("No test images were found.")

print("\nTotal test images:", len(test_df))

# Expected count based on the dataset extraction previously verified
expected_counts = {name: 260 for name in class_names}
actual_counts = test_df["true_class"].value_counts().to_dict()

if actual_counts != expected_counts:
    raise ValueError(
        f"Test counts differ from expected 260/class.\n"
        f"Actual counts: {actual_counts}\n"
        "Check the test folders before evaluating."
    )

# --------------------------------------------------
# 3. Test dataset and preprocessing
# --------------------------------------------------
eval_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

class MarineTestDataset(Dataset):
    def __init__(self, dataframe, transform):
        self.df = dataframe.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, index):
        row = self.df.iloc[index]

        with Image.open(row["path"]) as img:
            image = img.convert("RGB")

        image = self.transform(image)
        label = int(row["label"])

        return image, label, row["path"]

test_dataset = MarineTestDataset(test_df, eval_transform)
test_loader = DataLoader(
    test_dataset,
    batch_size=32,
    shuffle=False,
    num_workers=0,
    pin_memory=(device.type == "cuda")
)

# --------------------------------------------------
# 4. Rebuild model and load the selected weights
# --------------------------------------------------
model = resnet18(weights=None)
model.fc = nn.Linear(model.fc.in_features, len(class_names))
model.load_state_dict(checkpoint["model_state_dict"])
model = model.to(device)
model.eval()

# --------------------------------------------------
# 5. Run test predictions
# --------------------------------------------------
y_true = []
y_pred = []
probabilities_all = []
paths_all = []

with torch.no_grad():
    for images, labels, paths in test_loader:
        images = images.to(device)
        outputs = model(images)

        probabilities = torch.softmax(outputs, dim=1)
        predictions = outputs.argmax(dim=1)

        y_true.extend(labels.numpy().tolist())
        y_pred.extend(predictions.cpu().numpy().tolist())
        probabilities_all.extend(
            probabilities.cpu().numpy().tolist()
        )
        paths_all.extend(list(paths))

# --------------------------------------------------
# 6. Overall test accuracy and class-wise metrics
# --------------------------------------------------
test_accuracy = accuracy_score(y_true, y_pred)

print("\n" + "=" * 55)
print(f"TEST ACCURACY: {test_accuracy:.4%}")
print("=" * 55)

report_dict = classification_report(
    y_true,
    y_pred,
    labels=list(range(len(class_names))),
    target_names=class_names,
    output_dict=True,
    zero_division=0
)

print("\nCLASSIFICATION REPORT")
print(classification_report(
    y_true,
    y_pred,
    labels=list(range(len(class_names))),
    target_names=class_names,
    digits=4,
    zero_division=0
))

report_df = pd.DataFrame(report_dict).transpose()
report_df.to_csv(
    os.path.join(results_dir, "resnet18_test_classification_report.csv")
)

# --------------------------------------------------
# 7. Confusion matrix
# --------------------------------------------------
cm = confusion_matrix(
    y_true,
    y_pred,
    labels=list(range(len(class_names)))
)

print("CONFUSION MATRIX")
print(pd.DataFrame(
    cm,
    index=[f"Actual {name}" for name in class_names],
    columns=[f"Predicted {name}" for name in class_names]
))

fig, ax = plt.subplots(figsize=(8, 7))
disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=class_names
)
disp.plot(ax=ax, cmap="Blues", values_format="d", colorbar=False)
plt.title("ResNet-18 — Reserved Test Set Confusion Matrix")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()

cm_path = os.path.join(
    outputs_dir, "resnet18_test_confusion_matrix.png"
)
plt.savefig(cm_path, dpi=200, bbox_inches="tight")
plt.show()
plt.close()

# --------------------------------------------------
# 8. Save every prediction
# --------------------------------------------------
probabilities_all = np.asarray(probabilities_all)
prediction_df = pd.DataFrame({
    "image_path": paths_all,
    "actual_class": [class_names[i] for i in y_true],
    "predicted_class": [class_names[i] for i in y_pred],
    "correct": [
        actual == predicted
        for actual, predicted in zip(y_true, y_pred)
    ],
    "predicted_confidence": probabilities_all.max(axis=1)
})

for i, name in enumerate(class_names):
    prediction_df[f"prob_{name}"] = probabilities_all[:, i]

predictions_path = os.path.join(
    results_dir, "resnet18_test_predictions.csv"
)
prediction_df.to_csv(predictions_path, index=False)

print("\nSaved files:")
print("Classification report:", os.path.join(
    results_dir, "resnet18_test_classification_report.csv"
))
print("Predictions:", predictions_path)
print("Confusion matrix:", cm_path)

print("\nCorrect predictions:",
      int(prediction_df["correct"].sum()), "/", len(prediction_df))
print("Incorrect predictions:",
      int((~prediction_df["correct"]).sum()))


In [ ]:

import os
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

pred_path = "/content/marine_project/results/resnet18_test_predictions.csv"
output_dir = "/content/marine_project/outputs"
os.makedirs(output_dir, exist_ok=True)

df = pd.read_csv(pred_path)
errors = df[df["correct"] == False].copy()

print("Total test images:", len(df))
print("Incorrect predictions:", len(errors))
print("\nMost common mistakes:")
print(
    errors.groupby(["actual_class", "predicted_class"])
    .size()
    .sort_values(ascending=False)
    .head(15)
)

# Display up to 24 incorrect predictions
samples = errors.head(24)
fig, axes = plt.subplots(4, 6, figsize=(18, 13))
axes = axes.flatten()

for ax, (_, row) in zip(axes, samples.iterrows()):
    path = row["image_path"]

    try:
        img = Image.open(path).convert("RGB")
        ax.imshow(img)
        confidence = row.get("predicted_confidence", float("nan"))
        ax.set_title(
            f"Actual: {row['actual_class']}\n"
            f"Pred: {row['predicted_class']} ({confidence:.1%})",
            fontsize=8
        )
    except Exception as e:
        ax.text(0.5, 0.5, f"Image error\n{e}", ha="center", va="center")

    ax.axis("off")

for ax in axes[len(samples):]:
    ax.axis("off")

plt.tight_layout()
save_path = os.path.join(output_dir, "misclassified_test_samples.png")
plt.savefig(save_path, dpi=200, bbox_inches="tight")
plt.show()

print("\nSaved error analysis image:", save_path)


In [ ]:

import pandas as pd

path = "/content/marine_project/results/resnet18_test_predictions.csv"
df = pd.read_csv(path)

# Check the actual column names and confidence values
print("Columns:", df.columns.tolist())

# Errors where a seagrass class was predicted as Background
background_errors = df[
    (df["actual_class"] != "Background") &
    (df["predicted_class"] == "Background")
].copy()

print("\nSeagrass images incorrectly predicted as Background:",
      len(background_errors))

print("\nConfidence summary:")
print(background_errors["predicted_confidence"].describe())

print("\nConfidence by actual class:")
print(
    background_errors.groupby("actual_class")["predicted_confidence"]
    .agg(["count", "mean", "median", "min", "max"])
    .round(4)
)

print("\nLowest-confidence examples:")
print(
    background_errors[
        ["image_path", "actual_class", "predicted_class",
         "predicted_confidence"]
    ].sort_values("predicted_confidence").head(10).to_string(index=False)
)

print("\nHighest-confidence examples:")
print(
    background_errors[
        ["image_path", "actual_class", "predicted_class",
         "predicted_confidence"]
    ].sort_values("predicted_confidence", ascending=False).head(10)
    .to_string(index=False)
)


In [ ]:

import pandas as pd

path = "/content/marine_project/results/resnet18_test_predictions.csv"
df = pd.read_csv(path)

# Extract the source identifier from filenames such as:
# Halophila_CMER_test_153.jpg
# Posidonia_DBCA_test_133.jpg
df["source"] = df["image_path"].str.extract(
    r"_([A-Za-z]+)_test_\d+\."
)

print("Source counts:")
print(df["source"].value_counts(dropna=False))

print("\nAccuracy by source:")
print(
    df.groupby("source")
      .agg(
          images=("correct", "size"),
          accuracy=("correct", "mean")
      )
      .assign(accuracy=lambda x: (x["accuracy"] * 100).round(2))
)

print("\nAccuracy by actual class and source:")
print(
    df.groupby(["actual_class", "source"])
      .agg(
          images=("correct", "size"),
          accuracy=("correct", "mean")
      )
      .assign(accuracy=lambda x: (x["accuracy"] * 100).round(2))
      .to_string()
)

print("\nMost common errors by source:")
errors = df[df["correct"] == False]
print(
    errors.groupby(["source", "actual_class", "predicted_class"])
          .size()
          .sort_values(ascending=False)
          .head(20)
)


In [ ]:

import pandas as pd
import os

base = "/content/marine_project"

train_csv = os.path.join(base, "results", "train_split.csv")
val_csv = os.path.join(base, "results", "validation_split.csv")
test_csv = os.path.join(
    base, "results", "resnet18_test_predictions.csv"
)

def load_and_audit(path, split_name):
    df = pd.read_csv(path)

    # Find source identifiers anywhere in each filename/path
    df["source"] = (
        df["path" if "path" in df.columns else "image_path"]
        .str.extract(r"(CMER|DBCA)", expand=False)
        .fillna("Unknown")
        .str.upper()
    )

    label_col = "label" if "label" in df.columns else "actual_class"

    print(f"\n{'=' * 20} {split_name} {'=' * 20}")
    print("Images:", len(df))
    print("\nClass counts:")
    print(df[label_col].value_counts().sort_index())

    print("\nClass-by-source counts:")
    print(pd.crosstab(df[label_col], df["source"], margins=True))

    return df

train_df = load_and_audit(train_csv, "TRAINING")
val_df = load_and_audit(val_csv, "VALIDATION")
test_df = load_and_audit(test_csv, "TEST")


In [ ]:

from google.colab import drive
drive.mount('/content/drive')

import os
import shutil
from datetime import datetime

project_dir = "/content/marine_project"
drive_project_dir = "/content/drive/MyDrive/Marine_Project"

# Create a separate backup so existing files are not overwritten
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
backup_dir = os.path.join(drive_project_dir, f"backup_{timestamp}")

os.makedirs(backup_dir, exist_ok=True)

# Back up models, results and outputs
for folder in ["models", "results", "outputs"]:
    source = os.path.join(project_dir, folder)
    destination = os.path.join(backup_dir, folder)

    if os.path.isdir(source):
        shutil.copytree(source, destination, dirs_exist_ok=True)
        print(f"Backed up: {folder}")
    else:
        print(f"Not found: {source}")

print("\nBackup location:")
print(backup_dir)

# Verify the copied files
for root, dirs, files in os.walk(backup_dir):
    for filename in files:
        print(os.path.relpath(os.path.join(root, filename), backup_dir))


In [ ]:

import os

search_root = "/content/drive/MyDrive/Marine_Project"

extensions = (".pth", ".pt", ".csv", ".png", ".json")

print("Searching Google Drive for saved model files...\n")

found = []

for root, dirs, files in os.walk(search_root):
    # Skip the empty backup folders created in the previous step
    dirs[:] = [
        d for d in dirs
        if not d.startswith("backup_")
    ]

    for filename in files:
        if filename.lower().endswith(extensions):
            path = os.path.join(root, filename)
            found.append(path)
            print(path)

print("\nTotal matching files found:", len(found))


In [ ]:
import os

for path in [
    "/content/marine_project/models/resnet18_baseline_best_clean.pth",
    "/content/drive/MyDrive/Marine_Project"
]:
    print(path, "->", os.path.exists(path))

In [ ]:
import os

root = "/content/drive/MyDrive/Marine_Project"

for folder, dirs, files in os.walk(root):
    dirs[:] = [d for d in dirs if not d.startswith("backup_")]

    for file in files:
        if file.lower().endswith(
            (".pth", ".pt", ".ckpt", ".onnx",
             ".csv", ".png", ".json", ".ipynb")
        ):
            path = os.path.join(folder, file)
            print(path)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os

root = "/content/drive/MyDrive/Marine_Project"

print("Files and folders in Marine_Project:")
for item in sorted(os.listdir(root)):
    path = os.path.join(root, item)
    print("[FOLDER]" if os.path.isdir(path) else "[FILE]", item)

In [ ]:
import os

zip_dir = "/content/drive/MyDrive/Marine_Project/dataset_zips"

zip_files = sorted(
    f for f in os.listdir(zip_dir)
    if f.lower().endswith(".zip")
)

print("ZIP files found:", len(zip_files))

for f in zip_files:
    size_mb = os.path.getsize(os.path.join(zip_dir, f)) / (1024 * 1024)
    print(f"{f} — {size_mb:.2f} MB")

In [ ]:
import os
import shutil

project_dir = "/content/marine_project"
drive_project_dir = "/content/drive/MyDrive/Marine_Project"

folders = [
    "data/train",
    "data/test",
    "models",
    "results",
    "outputs"
]

for folder in folders:
    os.makedirs(os.path.join(project_dir, folder), exist_ok=True)
    os.makedirs(os.path.join(drive_project_dir, folder), exist_ok=True)

total, used, free = shutil.disk_usage("/content")

print("Project folders created.")
print(f"Colab storage free: {free / (1024**3):.2f} GB")
print(f"Colab storage total: {total / (1024**3):.2f} GB")
print(f"Google Drive project: {drive_project_dir}")

In [ ]:
import os
import shutil

drive_root = "/content/drive/MyDrive/Marine_Project"
backup_root = os.path.join(drive_root, "backup_20261028_074754")

for folder in ["models", "results", "outputs"]:
    source = os.path.join(backup_root, folder)
    destination = os.path.join(drive_root, folder)

    os.makedirs(destination, exist_ok=True)

    if os.path.isdir(source):
        for name in os.listdir(source):
            src = os.path.join(source, name)
            dst = os.path.join(destination, name)

            if os.path.isfile(src):
                shutil.copy2(src, dst)

        print(f"Restored: {folder}")
    else:
        print(f"Missing backup folder: {source}")

print("\nCheckpoint exists:",
      os.path.exists(
          os.path.join(
              drive_root, "models",
              "resnet18_baseline_best_clean.pth"
          )
      ))

In [ ]:
import os

root = "/content/drive/MyDrive/Marine_Project"

for folder in ["models", "results", "outputs"]:
    path = os.path.join(root, folder)
    print(f"\n--- {folder.upper()} ---")

    for name in sorted(os.listdir(path)):
        full_path = os.path.join(path, name)
        size = os.path.getsize(full_path) / (1024 * 1024)

        print(f"{name} — {size:.2f} MB")

In [ ]:

import os
import torch
import torchvision.models as models
import torch.nn as nn

checkpoint_path = (
    "/content/drive/MyDrive/Marine_Project/"
    "models/resnet18_baseline_best_clean.pth"
)

print("Checkpoint exists:", os.path.exists(checkpoint_path))
print("PyTorch version:", torch.__version__)

checkpoint = torch.load(
    checkpoint_path,
    map_location="cpu",
    weights_only=False
)

print("\nCheckpoint type:", type(checkpoint).__name__)

if isinstance(checkpoint, dict):
    print("Checkpoint keys:", list(checkpoint.keys()))

    if "model_state_dict" in checkpoint:
        state_dict = checkpoint["model_state_dict"]
    elif "state_dict" in checkpoint:
        state_dict = checkpoint["state_dict"]
    else:
        state_dict = checkpoint

    if all(isinstance(k, str) for k in state_dict.keys()):
        print("Number of parameter entries:", len(state_dict))
        print("First few parameter names:", list(state_dict.keys())[:5])


In [ ]:
import os

from google.colab import drive
drive.mount("/content/drive")

root = "/content/drive/MyDrive/Marine_Project"

for folder, dirs, files in os.walk(root):
    dirs[:] = [d for d in dirs if not d.startswith(".")]

    for file in files:
        if file.endswith((".pth", ".pt", ".ckpt")):
            print(os.path.join(folder, file))

In [ ]:

from google.colab import drive
drive.mount("/content/drive")

import os
import torch
import torch.nn as nn
import torchvision.models as models

checkpoint_path = (
    "/content/drive/MyDrive/Marine_Project/"
    "models/resnet18_baseline_best_clean.pth"
)

assert os.path.isfile(checkpoint_path), (
    f"Checkpoint not found: {checkpoint_path}"
)

checkpoint = torch.load(
    checkpoint_path,
    map_location="cpu",
    weights_only=False
)

# Support either a raw state dictionary or a saved checkpoint dictionary.
if isinstance(checkpoint, dict) and "model_state_dict" in checkpoint:
    state_dict = checkpoint["model_state_dict"]
elif isinstance(checkpoint, dict) and "state_dict" in checkpoint:
    state_dict = checkpoint["state_dict"]
else:
    state_dict = checkpoint

# Remove a possible DataParallel prefix.
state_dict = {
    k.removeprefix("module."): v
    for k, v in state_dict.items()
}

model = models.resnet18(weights=None)
model.fc = nn.Linear(model.fc.in_features, 4)

model.load_state_dict(state_dict)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)
model.eval()

print("Checkpoint loaded successfully!")
print("Device:", device)
print("Classes:", ["Amphibolis", "Background", "Halophila", "Posidonia"])
print("Training skipped.")


In [ ]:
import os
import pandas as pd

PROJECT_DIR = "/content/drive/MyDrive/Marine_Project"
RESULTS_DIR = os.path.join(PROJECT_DIR, "results")

print("Available result files:\n")
for file in os.listdir(RESULTS_DIR):
    print(file)

report_path = os.path.join(
    RESULTS_DIR, "resnet18_test_classification_report.csv"
)

predictions_path = os.path.join(
    RESULTS_DIR, "resnet18_test_predictions.csv"
)

if os.path.exists(report_path):
    report = pd.read_csv(report_path)
    print("\nCLASSIFICATION REPORT:")
    display(report)

if os.path.exists(predictions_path):
    predictions = pd.read_csv(predictions_path)
    print("\nPREDICTION FILE:")
    print("Shape:", predictions.shape)
    display(predictions.head())

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, classification_report

predictions_path = (
    "/content/drive/MyDrive/Marine_Project/results/"
    "resnet18_test_predictions.csv"
)

df = pd.read_csv(predictions_path)

classes = [
    "Amphibolis",
    "Background",
    "Halophila",
    "Posidonia"
]

# Confusion matrix
cm = confusion_matrix(
    df["actual_class"],
    df["predicted_class"],
    labels=classes
)

plt.figure(figsize=(9, 7))
sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=classes,
    yticklabels=classes
)

plt.xlabel("Predicted Class")
plt.ylabel("Actual Class")
plt.title("ResNet-18 Test Confusion Matrix")
plt.tight_layout()
plt.show()

# Incorrect predictions
errors = df[df["correct"] == False].copy()

print("Total incorrect predictions:", len(errors))

print("\nMisclassification breakdown:")
display(
    errors.groupby(
        ["actual_class", "predicted_class"]
    ).size().reset_index(name="count")
    .sort_values("count", ascending=False)
)

print("\nAverage confidence on incorrect predictions:")
print(errors["predicted_confidence"].mean())

In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

df = pd.read_csv(predictions_path)

errors = df[df["correct"] == False].copy()

# Check whether the original image paths still exist
errors["file_exists"] = errors["image_path"].apply(os.path.exists)

print("Total misclassified images:", len(errors))
print("Images currently accessible:", errors["file_exists"].sum())

# Display up to 12 accessible misclassified images
available = errors[errors["file_exists"]].head(12)

if len(available) > 0:
    fig, axes = plt.subplots(3, 4, figsize=(16, 11))
    axes = axes.flatten()

    for ax, (_, row) in zip(axes, available.iterrows()):
        img = Image.open(row["image_path"]).convert("RGB")
        ax.imshow(img)
        ax.set_title(
            f"Actual: {row['actual_class']}\n"
            f"Predicted: {row['predicted_class']}\n"
            f"Confidence: {row['predicted_confidence']:.2%}",
            fontsize=9
        )
        ax.axis("off")

    for ax in axes[len(available):]:
        ax.axis("off")

    plt.tight_layout()
    plt.show()
else:
    print("Original test images are not currently extracted.")
    print("We will locate the saved dataset before inspecting them.")

# Confidence analysis by error type
print("\nConfidence by misclassification:")
display(
    errors.groupby(
        ["actual_class", "predicted_class"]
    )["predicted_confidence"]
    .agg(["count", "mean", "median"])
    .sort_values("count", ascending=False)
)

In [ ]:
import os
import zipfile

ZIP_DIR = (
    "/content/drive/MyDrive/Marine_Project/"
    "dataset_zips"
)

test_zips = [
    "ECU-MSS-2 Amphibolis_testing.zip",
    "ECU-MSS-2 Background_testing.zip",
    "ECU-MSS-2 Halophila_testing.zip",
    "ECU-MSS-2 Posidonia_testing.zip"
]

for filename in test_zips:
    zip_path = os.path.join(ZIP_DIR, filename)

    print("\n" + "=" * 65)
    print("FILE:", filename)

    if not os.path.exists(zip_path):
        print("ZIP NOT FOUND")
        continue

    with zipfile.ZipFile(zip_path, "r") as z:
        members = [
            name for name in z.namelist()
            if not name.endswith("/")
        ]

        print("Total files:", len(members))
        print("Sample paths:")

        for name in members[:5]:
            print(" ", name)

In [ ]:
import os
import zipfile
import pandas as pd

ZIP_DIR = "/content/drive/MyDrive/Marine_Project/dataset_zips"

# Restore images to the original expected directory
TEST_DIR = "/content/marine_project/data/test"
os.makedirs(TEST_DIR, exist_ok=True)

# Load existing predictions
predictions_path = (
    "/content/drive/MyDrive/Marine_Project/results/"
    "resnet18_test_predictions.csv"
)

df = pd.read_csv(predictions_path)
errors = df[df["correct"] == False].copy()

# Map each class to its test ZIP
zip_mapping = {
    "Amphibolis": "ECU-MSS-2 Amphibolis_testing.zip",
    "Background": "ECU-MSS-2 Background_testing.zip",
    "Halophila": "ECU-MSS-2 Halophila_testing.zip",
    "Posidonia": "ECU-MSS-2 Posidonia_testing.zip"
}

extracted = 0
missing = []

for _, row in errors.iterrows():
    class_name = row["actual_class"]
    filename = os.path.basename(row["image_path"])

    zip_path = os.path.join(ZIP_DIR, zip_mapping[class_name])
    expected_member = f"{class_name}/{filename}"

    destination = os.path.join(TEST_DIR, class_name, filename)
    os.makedirs(os.path.dirname(destination), exist_ok=True)

    if os.path.exists(destination):
        extracted += 1
        continue

    with zipfile.ZipFile(zip_path, "r") as archive:
        if expected_member in archive.namelist():
            archive.extract(expected_member, TEST_DIR)
            extracted += 1
        else:
            missing.append(expected_member)

print("Misclassified images:", len(errors))
print("Recovered images:", extracted)
print("Missing images:", len(missing))

if missing:
    print("Missing files:", missing[:10])

# Verify original CSV paths
errors["file_exists"] = errors["image_path"].apply(os.path.exists)

print("\nAccessible original paths:", errors["file_exists"].sum())

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
from PIL import Image
import os

errors = df[df["correct"] == False].copy()

# Prioritize the most common mistakes
priority = [
    ("Posidonia", "Background"),
    ("Amphibolis", "Background"),
    ("Halophila", "Background")
]

selected = pd.concat([
    errors[
        (errors["actual_class"] == actual) &
        (errors["predicted_class"] == predicted)
    ].head(4)
    for actual, predicted in priority
])

fig, axes = plt.subplots(3, 4, figsize=(16, 12))
axes = axes.flatten()

for ax, (_, row) in zip(axes, selected.iterrows()):
    img = Image.open(row["image_path"]).convert("RGB")
    ax.imshow(img)
    ax.set_title(
        f"Actual: {row['actual_class']}\n"
        f"Predicted: {row['predicted_class']}\n"
        f"Confidence: {row['predicted_confidence']:.2%}",
        fontsize=9
    )
    ax.axis("off")

plt.suptitle("High-Priority ResNet-18 Misclassifications", fontsize=15)
plt.tight_layout()
plt.show()

In [ ]:
import torch
import torch.nn.functional as F
import torchvision.transforms as transforms
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
import os

# ImageNet preprocessing used by ResNet-18
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

# Select one high-confidence error from each class
samples = []

for class_name in ["Posidonia", "Amphibolis", "Halophila"]:
    sample = errors[
        (errors["actual_class"] == class_name) &
        (errors["predicted_class"] == "Background")
    ].sort_values(
        "predicted_confidence", ascending=False
    ).iloc[0]

    samples.append(sample)

# Grad-CAM hooks
activations = {}
gradients = {}

target_layer = model.layer4[-1].conv2

def forward_hook(module, input, output):
    activations["value"] = output

def backward_hook(module, grad_input, grad_output):
    gradients["value"] = grad_output[0]

forward_handle = target_layer.register_forward_hook(forward_hook)
backward_handle = target_layer.register_full_backward_hook(backward_hook)

fig, axes = plt.subplots(3, 3, figsize=(14, 13))

for i, row in enumerate(samples):

    image = Image.open(row["image_path"]).convert("RGB")
    input_tensor = transform(image).unsqueeze(0).to(device)

    model.zero_grad(set_to_none=True)

    output = model(input_tensor)
    predicted_idx = output.argmax(dim=1).item()

    output[0, predicted_idx].backward()

    feature_maps = activations["value"].detach()
    grads = gradients["value"].detach()

    weights = grads.mean(dim=(2, 3), keepdim=True)

    cam = (weights * feature_maps).sum(dim=1)
    cam = F.relu(cam)

    cam = F.interpolate(
        cam.unsqueeze(1),
        size=(224, 224),
        mode="bilinear",
        align_corners=False
    ).squeeze()

    cam = cam.cpu().numpy()
    cam = (cam - cam.min()) / (cam.max() - cam.min() + 1e-8)

    original = np.array(image.resize((224, 224)))
    heatmap = plt.get_cmap("jet")(cam)[:, :, :3]

    overlay = (
        0.55 * original / 255.0 +
        0.45 * heatmap
    )
    overlay = np.clip(overlay, 0, 1)

    axes[i, 0].imshow(original)
    axes[i, 0].set_title(
        f"Actual: {row['actual_class']}"
    )

    axes[i, 1].imshow(cam, cmap="jet")
    axes[i, 1].set_title("Grad-CAM")

    axes[i, 2].imshow(overlay)
    axes[i, 2].set_title(
        f"Predicted: {row['predicted_class']}\n"
        f"Confidence: {row['predicted_confidence']:.2%}"
    )

    for ax in axes[i]:
        ax.axis("off")

plt.tight_layout()
plt.show()

forward_handle.remove()
backward_handle.remove()

In [ ]:
import torch
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

class_names = [
    "Amphibolis",
    "Background",
    "Halophila",
    "Posidonia"
]

class_to_idx = {
    name: i for i, name in enumerate(class_names)
}

target_layer = model.layer4[-1].conv2

def generate_gradcam(input_tensor, target_class):

    captured = {}

    def hook(module, inputs, output):
        captured["features"] = output

    handle = target_layer.register_forward_hook(hook)

    model.zero_grad(set_to_none=True)

    output = model(input_tensor)
    features = captured["features"]

    score = output[0, target_class]

    gradients = torch.autograd.grad(
        score, features
    )[0]

    weights = gradients.mean(
        dim=(2, 3), keepdim=True
    )

    cam = (weights * features).sum(dim=1)
    cam = F.relu(cam)

    cam = F.interpolate(
        cam.unsqueeze(1),
        size=(224, 224),
        mode="bilinear",
        align_corners=False
    )[0, 0]

    cam = cam.detach().cpu().numpy()
    cam = (cam - cam.min()) / (
        cam.max() - cam.min() + 1e-8
    )

    handle.remove()

    return cam, output.detach()

fig, axes = plt.subplots(3, 3, figsize=(15, 13))

for i, row in enumerate(samples):

    image = Image.open(
        row["image_path"]
    ).convert("RGB")

    input_tensor = transform(image).unsqueeze(0).to(device)

    actual_idx = class_to_idx[row["actual_class"]]
    predicted_idx = class_to_idx[row["predicted_class"]]

    # Actual-class Grad-CAM
    actual_cam, output = generate_gradcam(
        input_tensor, actual_idx
    )

    # Predicted-class Grad-CAM
    predicted_cam, _ = generate_gradcam(
        input_tensor, predicted_idx
    )

    probabilities = torch.softmax(
        output, dim=1
    )[0]

    original = np.array(
        image.resize((224, 224))
    )

    axes[i, 0].imshow(original)
    axes[i, 0].set_title(
        f"Original: {row['actual_class']}"
    )

    axes[i, 1].imshow(original)
    axes[i, 1].imshow(
        actual_cam, cmap="jet", alpha=0.45
    )
    axes[i, 1].set_title(
        f"Actual-class CAM\n"
        f"Score: {probabilities[actual_idx]:.4f}"
    )

    axes[i, 2].imshow(original)
    axes[i, 2].imshow(
        predicted_cam, cmap="jet", alpha=0.45
    )
    axes[i, 2].set_title(
        f"Predicted-class CAM\n"
        f"Score: {probabilities[predicted_idx]:.4f}"
    )

    for ax in axes[i]:
        ax.axis("off")

plt.suptitle(
    "Actual Class vs Predicted Class Grad-CAM",
    fontsize=15
)
plt.tight_layout()
plt.show()

In [ ]:
import torch
import pandas as pd

results = []

for _, row in pd.DataFrame(samples).iterrows():

    image = Image.open(row["image_path"]).convert("RGB")
    input_tensor = transform(image).unsqueeze(0).to(device)

    with torch.no_grad():
        logits = model(input_tensor)
        probabilities = torch.softmax(logits, dim=1)[0]

    result = {
        "Actual Class": row["actual_class"],
        "Predicted Class": row["predicted_class"],
        "Confidence": float(probabilities.max().item())
    }

    for i, class_name in enumerate(class_names):
        result[class_name] = float(probabilities[i].item())

    results.append(result)

probability_df = pd.DataFrame(results)

pd.set_option("display.precision", 8)
display(probability_df)

In [ ]:
import os
import zipfile
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from io import BytesIO

ZIP_DIR = "/content/drive/MyDrive/Marine_Project/dataset_zips"

RESULTS_DIR = "/content/drive/MyDrive/Marine_Project/results"

df = pd.read_csv(
    os.path.join(RESULTS_DIR, "resnet18_test_predictions.csv")
)

classes = ["Amphibolis", "Halophila", "Posidonia"]

zip_mapping = {
    "Amphibolis": "ECU-MSS-2 Amphibolis_testing.zip",
    "Halophila": "ECU-MSS-2 Halophila_testing.zip",
    "Posidonia": "ECU-MSS-2 Posidonia_testing.zip"
}

fig, axes = plt.subplots(3, 8, figsize=(20, 11))

for i, class_name in enumerate(classes):

    class_df = df[df["actual_class"] == class_name]

    correct = class_df[class_df["correct"] == True].head(4)
    incorrect = class_df[class_df["correct"] == False].head(4)

    selected = list(correct.iterrows()) + list(incorrect.iterrows())

    zip_path = os.path.join(ZIP_DIR, zip_mapping[class_name])

    with zipfile.ZipFile(zip_path, "r") as archive:

        for j, (_, row) in enumerate(selected):

            filename = os.path.basename(row["image_path"])
            member = f"{class_name}/{filename}"

            with archive.open(member) as file:
                img = Image.open(BytesIO(file.read())).convert("RGB")

            axes[i, j].imshow(img)

            status = "CORRECT" if row["correct"] else "WRONG"

            axes[i, j].set_title(
                f"{status}\nPredicted: {row['predicted_class']}",
                fontsize=8
            )

            axes[i, j].axis("off")

    axes[i, 0].set_ylabel(class_name, fontsize=12)

plt.suptitle(
    "Correct vs Incorrect Test Predictions",
    fontsize=16
)

plt.tight_layout()
plt.show()

In [ ]:
import os
import zipfile
import cv2
import numpy as np
import pandas as pd
from PIL import Image
from io import BytesIO

ZIP_DIR = "/content/drive/MyDrive/Marine_Project/dataset_zips"
RESULTS_DIR = "/content/drive/MyDrive/Marine_Project/results"

df = pd.read_csv(
    os.path.join(RESULTS_DIR, "resnet18_test_predictions.csv")
)

zip_mapping = {
    "Amphibolis": "ECU-MSS-2 Amphibolis_testing.zip",
    "Background": "ECU-MSS-2 Background_testing.zip",
    "Halophila": "ECU-MSS-2 Halophila_testing.zip",
    "Posidonia": "ECU-MSS-2 Posidonia_testing.zip"
}

quality_records = []

for class_name, zip_name in zip_mapping.items():

    zip_path = os.path.join(ZIP_DIR, zip_name)

    print("Analysing:", class_name)

    with zipfile.ZipFile(zip_path, "r") as archive:

        for member in archive.namelist():

            if not member.lower().endswith((".jpg", ".jpeg", ".png")):
                continue

            filename = os.path.basename(member)

            with archive.open(member) as file:
                image = Image.open(
                    BytesIO(file.read())
                ).convert("RGB")

            image = image.resize((224, 224))

            rgb = np.array(image)
            gray = cv2.cvtColor(
                rgb, cv2.COLOR_RGB2GRAY
            )

            hsv = cv2.cvtColor(
                rgb, cv2.COLOR_RGB2HSV
            )

            sharpness = cv2.Laplacian(
                gray, cv2.CV_64F
            ).var()

            contrast = gray.std()
            brightness = gray.mean()
            saturation = hsv[:, :, 1].mean()

            quality_records.append({
                "filename": filename,
                "actual_class": class_name,
                "sharpness": sharpness,
                "contrast": contrast,
                "brightness": brightness,
                "saturation": saturation
            })

quality_df = pd.DataFrame(quality_records)

# Match image measurements with existing predictions
df["filename"] = df["image_path"].apply(
    lambda x: os.path.basename(x)
)

analysis_df = df.merge(
    quality_df,
    on=["filename", "actual_class"],
    how="inner"
)

print("\nImages analysed:", len(analysis_df))

print("\nImage quality: Correct vs Incorrect predictions")

display(
    analysis_df.groupby("correct")[
        ["sharpness", "contrast", "brightness", "saturation"]
    ].agg(["mean", "median"]).round(2)
)

print("\nImage quality by actual class and prediction status")

display(
    analysis_df.groupby(
        ["actual_class", "correct"]
    )[
        ["sharpness", "contrast", "brightness", "saturation"]
    ].mean().round(2)
)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

classes = [
    "Amphibolis",
    "Halophila",
    "Posidonia"
]

fig, axes = plt.subplots(2, 3, figsize=(16, 9))

for i, class_name in enumerate(classes):

    subset = analysis_df[
        analysis_df["actual_class"] == class_name
    ].copy()

    subset["Prediction Status"] = subset["correct"].map({
        True: "Correct",
        False: "Incorrect"
    })

    sns.boxplot(
        data=subset,
        x="Prediction Status",
        y="sharpness",
        ax=axes[0, i]
    )

    axes[0, i].set_title(
        f"{class_name} - Sharpness"
    )

    sns.boxplot(
        data=subset,
        x="Prediction Status",
        y="contrast",
        ax=axes[1, i]
    )

    axes[1, i].set_title(
        f"{class_name} - Contrast"
    )

plt.suptitle(
    "Image Quality Distribution: Correct vs Incorrect Predictions",
    fontsize=15
)

plt.tight_layout()
plt.show()

In [ ]:
from scipy.stats import mannwhitneyu
import pandas as pd

metrics = [
    "sharpness",
    "contrast",
    "brightness",
    "saturation"
]

classes = [
    "Amphibolis",
    "Halophila",
    "Posidonia"
]

statistical_results = []

for class_name in classes:

    subset = analysis_df[
        analysis_df["actual_class"] == class_name
    ]

    correct = subset[subset["correct"] == True]
    incorrect = subset[subset["correct"] == False]

    for metric in metrics:

        x = correct[metric].dropna()
        y = incorrect[metric].dropna()

        if len(y) == 0:
            continue

        statistic, p_value = mannwhitneyu(
            x, y, alternative="two-sided"
        )

        statistical_results.append({
            "Class": class_name,
            "Metric": metric,
            "Correct_count": len(x),
            "Incorrect_count": len(y),
            "Correct_median": x.median(),
            "Incorrect_median": y.median(),
            "P_value": p_value
        })

stats_df = pd.DataFrame(statistical_results)

stats_df["Significant (p < 0.05)"] = (
    stats_df["P_value"] < 0.05
)

display(stats_df.round(5))

In [ ]:
from statsmodels.stats.multitest import multipletests

stats_df["Adjusted_P_value"] = multipletests(
    stats_df["P_value"],
    method="fdr_bh"
)[1]

stats_df["Significant_after_FDR"] = (
    stats_df["Adjusted_P_value"] < 0.05
)

display(
    stats_df[
        [
            "Class",
            "Metric",
            "P_value",
            "Adjusted_P_value",
            "Significant_after_FDR"
        ]
    ].round(6)
)

In [ ]:
print("DEVICE:", device)

print("\nMODEL:")
print(type(model).__name__)

print("\nTRAIN LOADER:")
if "train_loader" in globals():
    print("Available")
    print("Images:", len(train_loader.dataset))
else:
    print("Not currently loaded")

print("\nVALIDATION LOADER:")
if "val_loader" in globals():
    print("Available")
    print("Images:", len(val_loader.dataset))
else:
    print("Not currently loaded")

print("\nDATASET PATH:")
import os

for path in [
    "/content/marine_project/data/train",
    "/content/marine_project/data/val",
    "/content/drive/MyDrive/Marine_Project/dataset_zips"
]:
    print(path, "->", os.path.exists(path))

In [ ]:
import os
import pandas as pd
import zipfile

results_dir = "/content/drive/MyDrive/Marine_Project/results"
zip_dir = "/content/drive/MyDrive/Marine_Project/dataset_zips"

# Inspect saved training split
train_csv = pd.read_csv(
    os.path.join(results_dir, "train_split.csv")
)

val_csv = pd.read_csv(
    os.path.join(results_dir, "validation_split.csv")
)

print("TRAIN SPLIT")
print("Shape:", train_csv.shape)
print("Columns:", train_csv.columns.tolist())
display(train_csv.head(3))

print("\nVALIDATION SPLIT")
print("Shape:", val_csv.shape)
print("Columns:", val_csv.columns.tolist())
display(val_csv.head(3))

# Inspect archive contents
print("\nZIP ARCHIVES")

for filename in sorted(os.listdir(zip_dir)):
    if filename.endswith(".zip"):
        print(filename)

In [ ]:
import os
import zipfile
import pandas as pd
import torch
import numpy as np

from PIL import Image
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from torchvision.transforms import InterpolationMode

# --------------------------------------------------
# 1. PATHS AND CLASS ORDER
# --------------------------------------------------

ZIP_DIR = "/content/drive/MyDrive/Marine_Project/dataset_zips"
RESULTS_DIR = "/content/drive/MyDrive/Marine_Project/results"
MODEL_DIR = "/content/drive/MyDrive/Marine_Project/models"

os.makedirs(MODEL_DIR, exist_ok=True)

class_names = [
    "Amphibolis",
    "Background",
    "Halophila",
    "Posidonia"
]

class_to_idx = {
    name: i for i, name in enumerate(class_names)
}

zip_mapping = {
    "Amphibolis": "ECU-MSS-2 Amphibolis_training.zip",
    "Background": "ECU-MSS-2 Background_training.zip",
    "Halophila": "ECU-MSS-2 Halophila_training.zip",
    "Posidonia": "ECU-MSS-2 Posidonia_training.zip"
}

# --------------------------------------------------
# 2. LOAD SAVED SPLITS
# --------------------------------------------------

train_df = pd.read_csv(
    os.path.join(RESULTS_DIR, "train_split.csv")
)

val_df = pd.read_csv(
    os.path.join(RESULTS_DIR, "validation_split.csv")
)

# --------------------------------------------------
# 3. TRANSFORMS
# --------------------------------------------------

# E1: Original augmentation + underwater variations
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),

    transforms.RandomHorizontalFlip(p=0.5),

    transforms.RandomRotation(
        10,
        interpolation=InterpolationMode.BILINEAR
    ),

    transforms.RandomApply([
        transforms.GaussianBlur(
            kernel_size=3,
            sigma=(0.1, 1.5)
        )
    ], p=0.25),

    transforms.ColorJitter(
        brightness=0.15,
        contrast=0.25
    ),

    transforms.ToTensor(),

    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

# --------------------------------------------------
# 4. CUSTOM ZIP DATASET
# --------------------------------------------------

class MarineZipDataset(Dataset):

    def __init__(self, dataframe, zip_dir, zip_mapping, transform=None):

        self.dataframe = dataframe.reset_index(drop=True)
        self.zip_dir = zip_dir
        self.zip_mapping = zip_mapping
        self.transform = transform
        self.archives = {}

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, index):

        row = self.dataframe.iloc[index]

        label = row["label"]
        filename = os.path.basename(row["path"])

        zip_name = self.zip_mapping[label]
        zip_path = os.path.join(self.zip_dir, zip_name)

        if zip_name not in self.archives:
            self.archives[zip_name] = zipfile.ZipFile(
                zip_path, "r"
            )

        member = f"{label}/{filename}"

        with self.archives[zip_name].open(member) as file:
            image = Image.open(file).convert("RGB")

        if self.transform:
            image = self.transform(image)

        target = class_to_idx[label]

        return image, target

# --------------------------------------------------
# 5. CREATE DATASETS AND LOADERS
# --------------------------------------------------

train_dataset = MarineZipDataset(
    train_df,
    ZIP_DIR,
    zip_mapping,
    transform=train_transform
)

val_dataset = MarineZipDataset(
    val_df,
    ZIP_DIR,
    zip_mapping,
    transform=val_transform
)

train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True,
    num_workers=0,
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=32,
    shuffle=False,
    num_workers=0,
    pin_memory=True
)

# --------------------------------------------------
# 6. VERIFY
# --------------------------------------------------

print("TRAINING SAMPLES:", len(train_dataset))
print("VALIDATION SAMPLES:", len(val_dataset))

print("\nCLASS DISTRIBUTION:")
print(train_df["label"].value_counts())

images, labels = next(iter(train_loader))

print("\nBATCH VERIFICATION")
print("Image shape:", images.shape)
print("Label shape:", labels.shape)
print("Device:", device)

print("\nE1 DATA LOADERS READY!")

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import copy
import os

# 1. Load original best baseline checkpoint
checkpoint_path = (
    "/content/drive/MyDrive/Marine_Project/models/"
    "resnet18_baseline_best_clean.pth"
)

checkpoint = torch.load(
    checkpoint_path,
    map_location=device,
    weights_only=False
)

if "model_state_dict" in checkpoint:
    state_dict = checkpoint["model_state_dict"]
elif "state_dict" in checkpoint:
    state_dict = checkpoint["state_dict"]
else:
    state_dict = checkpoint

state_dict = {
    k.removeprefix("module."): v
    for k, v in state_dict.items()
}

# 2. Create independent E1 model
import torchvision.models as models

E1_model = models.resnet18(weights=None)
E1_model.fc = nn.Linear(
    E1_model.fc.in_features, 4
)

E1_model.load_state_dict(state_dict)
E1_model = E1_model.to(device)

# 3. Optimizer and loss
criterion = nn.CrossEntropyLoss()

optimizer = optim.Adam(
    E1_model.parameters(),
    lr=0.0001
)

# 4. Training configuration
E1_epochs = 5
best_val_loss = float("inf")
patience = 3
epochs_without_improvement = 0

E1_checkpoint_path = os.path.join(
    MODEL_DIR,
    "resnet18_E1_best.pth"
)

E1_history = []

print("E1 MODEL READY")
print("Architecture:", E1_model.__class__.__name__)
print("Device:", device)
print("Epochs:", E1_epochs)
print("Learning rate:", optimizer.param_groups[0]["lr"])
print("Checkpoint:", E1_checkpoint_path)

In [ ]:
import time
import pandas as pd
from tqdm.auto import tqdm

scaler = torch.amp.GradScaler(
    "cuda",
    enabled=(device.type == "cuda")
)

for epoch in range(E1_epochs):
    start_time = time.time()

    # ---------------- TRAINING ----------------
    E1_model.train()
    train_loss = 0.0
    train_correct = 0
    train_total = 0

    train_bar = tqdm(train_loader, desc=f"E1 Epoch {epoch+1}/{E1_epochs}")

    for images, labels in train_bar:
        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad(set_to_none=True)

        with torch.amp.autocast(
            device_type=device.type,
            enabled=(device.type == "cuda")
        ):
            outputs = E1_model(images)
            loss = criterion(outputs, labels)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        train_loss += loss.item() * images.size(0)
        train_correct += (outputs.argmax(1) == labels).sum().item()
        train_total += labels.size(0)

        train_bar.set_postfix(loss=f"{loss.item():.4f}")

    epoch_train_loss = train_loss / train_total
    epoch_train_acc = train_correct / train_total

    # ---------------- VALIDATION ----------------
    E1_model.eval()
    val_loss = 0.0
    val_correct = 0
    val_total = 0

    with torch.no_grad():
        for images, labels in tqdm(val_loader, desc="Validation"):
            images = images.to(device)
            labels = labels.to(device)

            with torch.amp.autocast(
                device_type=device.type,
                enabled=(device.type == "cuda")
            ):
                outputs = E1_model(images)
                loss = criterion(outputs, labels)

            val_loss += loss.item() * images.size(0)
            val_correct += (outputs.argmax(1) == labels).sum().item()
            val_total += labels.size(0)

    epoch_val_loss = val_loss / val_total
    epoch_val_acc = val_correct / val_total

    elapsed = time.time() - start_time

    E1_history.append({
        "epoch": epoch + 1,
        "train_loss": epoch_train_loss,
        "train_accuracy": epoch_train_acc,
        "val_loss": epoch_val_loss,
        "val_accuracy": epoch_val_acc
    })

    print(f"\nEpoch {epoch+1}/{E1_epochs}")
    print(f"Train Loss: {epoch_train_loss:.4f}")
    print(f"Train Accuracy: {epoch_train_acc*100:.2f}%")
    print(f"Validation Loss: {epoch_val_loss:.4f}")
    print(f"Validation Accuracy: {epoch_val_acc*100:.2f}%")
    print(f"Time: {elapsed/60:.2f} minutes")

    # ---------------- SAVE BEST CHECKPOINT ----------------
    if epoch_val_loss < best_val_loss:
        best_val_loss = epoch_val_loss
        epochs_without_improvement = 0

        torch.save({
            "epoch": epoch + 1,
            "model_state_dict": E1_model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "best_val_loss": best_val_loss,
            "class_names": class_names,
            "history": E1_history
        }, E1_checkpoint_path)

        print("Best E1 checkpoint saved!")

    else:
        epochs_without_improvement += 1
        print(f"No improvement: {epochs_without_improvement}/{patience}")

    # Save history after every epoch
    pd.DataFrame(E1_history).to_csv(
        os.path.join(RESULTS_DIR, "resnet18_E1_history.csv"),
        index=False
    )

    if epochs_without_improvement >= patience:
        print("\nEarly stopping triggered.")
        break

print("\nE1 TRAINING COMPLETED")
print("Best validation loss:", best_val_loss)
print("Checkpoint:", E1_checkpoint_path)

In [ ]:
import torch
import torch.nn as nn
import torchvision.models as models
import pandas as pd
import numpy as np
import os

from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix
)

# Load both saved models
baseline_path = os.path.join(
    MODEL_DIR, "resnet18_baseline_best_clean.pth"
)

E1_path = os.path.join(
    MODEL_DIR, "resnet18_E1_best.pth"
)

def load_resnet_checkpoint(path):
    checkpoint = torch.load(
        path, map_location=device, weights_only=False
    )

    if "model_state_dict" in checkpoint:
        state = checkpoint["model_state_dict"]
    elif "state_dict" in checkpoint:
        state = checkpoint["state_dict"]
    else:
        state = checkpoint

    state = {
        k.removeprefix("module."): v
        for k, v in state.items()
    }

    model = models.resnet18(weights=None)
    model.fc = nn.Linear(model.fc.in_features, 4)
    model.load_state_dict(state)
    model = model.to(device)
    model.eval()

    return model

baseline_model = load_resnet_checkpoint(baseline_path)
E1_best_model = load_resnet_checkpoint(E1_path)

# Evaluate on the same validation set
def evaluate_model(model, model_name):
    y_true = []
    y_pred = []

    model.eval()

    with torch.no_grad():
        for images, labels in val_loader:
            images = images.to(device)

            with torch.amp.autocast(
                device_type=device.type,
                enabled=(device.type == "cuda")
            ):
                outputs = model(images)

            predictions = outputs.argmax(dim=1)

            y_true.extend(labels.numpy())
            y_pred.extend(predictions.cpu().numpy())

    accuracy = accuracy_score(y_true, y_pred)

    print(f"\n{'='*55}")
    print(f"{model_name} VALIDATION RESULTS")
    print(f"{'='*55}")
    print(f"Accuracy: {accuracy*100:.2f}%")

    print("\nClassification Report:")
    print(classification_report(
        y_true,
        y_pred,
        labels=[0, 1, 2, 3],
        target_names=class_names,
        digits=4,
        zero_division=0
    ))

    print("Confusion Matrix:")
    cm = confusion_matrix(y_true, y_pred, labels=[0, 1, 2, 3])
    print(cm)

    return {
        "model": model_name,
        "accuracy": accuracy,
        "y_true": y_true,
        "y_pred": y_pred,
        "confusion_matrix": cm
    }

baseline_results = evaluate_model(
    baseline_model, "BASELINE"
)

E1_results = evaluate_model(
    E1_best_model, "E1"
)

# Summary comparison
comparison = pd.DataFrame([
    {
        "Model": "Baseline",
        "Validation Accuracy (%)":
            baseline_results["accuracy"] * 100
    },
    {
        "Model": "E1",
        "Validation Accuracy (%)":
            E1_results["accuracy"] * 100
    }
])

print("\nFINAL COMPARISON")
display(comparison)

comparison.to_csv(
    os.path.join(RESULTS_DIR, "baseline_vs_E1_validation.csv"),
    index=False
)

print("Comparison saved successfully.")

In [ ]:
import pandas as pd
import os

RESULTS_DIR = "/content/drive/MyDrive/Marine_Project/results"

train_df = pd.read_csv(os.path.join(RESULTS_DIR, "train_split.csv"))
val_df = pd.read_csv(os.path.join(RESULTS_DIR, "validation_split.csv"))

print("TRAINING DATASET")
print("Total images:", len(train_df))
print(train_df["label"].value_counts().sort_index())

print("\nVALIDATION DATASET")
print("Total images:", len(val_df))
print(val_df["label"].value_counts().sort_index())

print("\nSPLIT SUMMARY")
print("Training:", len(train_df))
print("Validation:", len(val_df))
print("Combined:", len(train_df) + len(val_df))

print("\nDUPLICATE PATH CHECK")
print("Train duplicate paths:", train_df["path"].duplicated().sum())
print("Validation duplicate paths:", val_df["path"].duplicated().sum())

print("\nLABEL DISTRIBUTION (%)")
distribution = pd.DataFrame({
    "Train (%)": train_df["label"].value_counts(normalize=True) * 100,
    "Validation (%)": val_df["label"].value_counts(normalize=True) * 100
}).fillna(0)

display(distribution.round(2))

In [ ]:
import zipfile
import os
import pandas as pd

ZIP_DIR = "/content/drive/MyDrive/Marine_Project/dataset_zips"

class_names = [
    "Amphibolis",
    "Background",
    "Halophila",
    "Posidonia"
]

test_counts = {}

for class_name in class_names:

    zip_path = os.path.join(
        ZIP_DIR,
        f"ECU-MSS-2 {class_name}_testing.zip"
    )

    with zipfile.ZipFile(zip_path, "r") as z:
        image_files = [
            name for name in z.namelist()
            if name.lower().endswith(
                (".jpg", ".jpeg", ".png", ".bmp")
            )
        ]

    test_counts[class_name] = len(image_files)

test_df = pd.DataFrame({
    "Class": class_names,
    "Training": [
        (train_df["label"] == c).sum()
        for c in class_names
    ],
    "Validation": [
        (val_df["label"] == c).sum()
        for c in class_names
    ],
    "Test": [
        test_counts[c]
        for c in class_names
    ]
})

test_df["Test (%)"] = (
    test_df["Test"] / test_df["Test"].sum() * 100
)

display(test_df)

print("\nTOTAL TEST IMAGES:", sum(test_counts.values()))
print("EXPECTED TEST IMAGES: 1040")

In [ ]:
import os
import zipfile
import cv2
import numpy as np
import pandas as pd
from tqdm.auto import tqdm

ZIP_DIR = "/content/drive/MyDrive/Marine_Project/dataset_zips"

# Sample 100 images per class per split
SAMPLES_PER_CLASS = 100

def get_image_quality(img_bytes):
    import io
    from PIL import Image

    img = Image.open(io.BytesIO(img_bytes)).convert("RGB")
    img = np.array(img)

    gray = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)
    hsv = cv2.cvtColor(img, cv2.COLOR_RGB2HSV)

    return {
        "sharpness": cv2.Laplacian(
            gray, cv2.CV_64F
        ).var(),
        "contrast": gray.std(),
        "brightness": gray.mean(),
        "saturation": hsv[:, :, 1].mean()
    }

quality_records = []

# TRAINING and VALIDATION images from saved CSVs
for split_name, df in [
    ("Training", train_df),
    ("Validation", val_df)
]:
    for class_name in class_names:
        subset = df[df["label"] == class_name]

        subset = subset.sample(
            n=min(SAMPLES_PER_CLASS, len(subset)),
            random_state=42
        )

        zip_path = os.path.join(
            ZIP_DIR,
            f"ECU-MSS-2 {class_name}_training.zip"
        )

        with zipfile.ZipFile(zip_path, "r") as z:
            for _, row in tqdm(
                subset.iterrows(),
                total=len(subset),
                desc=f"{split_name}-{class_name}"
            ):
                filename = os.path.basename(row["path"])
                member = f"{class_name}/{filename}"

                try:
                    img_bytes = z.read(member)
                    metrics = get_image_quality(img_bytes)

                    quality_records.append({
                        "split": split_name,
                        "class": class_name,
                        **metrics
                    })
                except Exception as e:
                    print("Skipped:", member, str(e))

# TEST images directly from ZIP archives
for class_name in class_names:
    zip_path = os.path.join(
        ZIP_DIR,
        f"ECU-MSS-2 {class_name}_testing.zip"
    )

    with zipfile.ZipFile(zip_path, "r") as z:
        image_files = [
            n for n in z.namelist()
            if n.lower().endswith(
                (".jpg", ".jpeg", ".png", ".bmp")
            )
        ]

        rng = np.random.default_rng(42)
        selected = rng.choice(
            image_files,
            size=min(SAMPLES_PER_CLASS, len(image_files)),
            replace=False
        )

        for member in tqdm(
            selected,
            desc=f"Test-{class_name}"
        ):
            try:
                img_bytes = z.read(member)
                metrics = get_image_quality(img_bytes)

                quality_records.append({
                    "split": "Test",
                    "class": class_name,
                    **metrics
                })
            except Exception as e:
                print("Skipped:", member, str(e))

quality_df = pd.DataFrame(quality_records)

# Save the investigation data
quality_path = os.path.join(
    RESULTS_DIR,
    "split_image_quality_analysis.csv"
)

quality_df.to_csv(quality_path, index=False)

print("\nIMAGE QUALITY ANALYSIS COMPLETED")
print("Images analyzed:", len(quality_df))
print("Saved:", quality_path)

summary = quality_df.groupby("split")[
    ["sharpness", "contrast", "brightness", "saturation"]
].agg(["mean", "median"]).round(2)

display(summary)

In [ ]:
# Compare image quality separately for each class and split

metrics = [
    "sharpness",
    "contrast",
    "brightness",
    "saturation"
]

class_quality = (
    quality_df
    .groupby(["class", "split"])[metrics]
    .agg(["mean", "median"])
    .round(2)
)

display(class_quality)

# Easier comparison: mean values only
class_mean = (
    quality_df
    .groupby(["class", "split"])[metrics]
    .mean()
    .round(2)
    .reset_index()
)

print("\nCLASS-WISE MEAN IMAGE QUALITY")
display(
    class_mean.pivot(
        index="class",
        columns="split",
        values=metrics
    )
)

# Save results
class_mean.to_csv(
    os.path.join(
        RESULTS_DIR,
        "classwise_image_quality_comparison.csv"
    ),
    index=False
)

print("Saved class-wise comparison.")

In [ ]:
import pandas as pd
import os

RESULTS_DIR = "/content/drive/MyDrive/Marine_Project/results"

quality_path = os.path.join(
    RESULTS_DIR,
    "split_image_quality_analysis.csv"
)

quality_df = pd.read_csv(quality_path)

print("QUALITY DATA RESTORED")
print("Total records:", len(quality_df))
print("\nColumns:", quality_df.columns.tolist())
print("\nSplit counts:")
print(quality_df["split"].value_counts())

print("\nClass counts:")
print(quality_df["class"].value_counts())

In [ ]:
import os
import numpy as np
import pandas as pd
from scipy.stats import mannwhitneyu
from statsmodels.stats.multitest import multipletests

class_names = ["Amphibolis", "Background", "Halophila", "Posidonia"]
metrics = ["sharpness", "contrast", "brightness", "saturation"]

test_results = []

for class_name in class_names:
    for metric in metrics:

        train_values = quality_df[
            (quality_df["class"] == class_name) &
            (quality_df["split"] == "Training")
        ][metric].dropna()

        test_values = quality_df[
            (quality_df["class"] == class_name) &
            (quality_df["split"] == "Test")
        ][metric].dropna()

        statistic, p_value = mannwhitneyu(
            train_values,
            test_values,
            alternative="two-sided"
        )

        test_results.append({
            "class": class_name,
            "metric": metric,
            "training_median": train_values.median(),
            "test_median": test_values.median(),
            "training_mean": train_values.mean(),
            "test_mean": test_values.mean(),
            "p_value": p_value
        })

stats_df = pd.DataFrame(test_results)

# Correct for 16 simultaneous statistical comparisons
stats_df["p_adjusted"] = multipletests(
    stats_df["p_value"],
    method="fdr_bh"
)[1]

stats_df["significant"] = stats_df["p_adjusted"] < 0.05

display(stats_df.round(5))

save_path = os.path.join(
    RESULTS_DIR,
    "train_test_quality_statistical_tests.csv"
)

stats_df.to_csv(save_path, index=False)

print("\nSaved to:", save_path)
print("Significant comparisons:", stats_df["significant"].sum(), "/ 16")

In [ ]:
import os
import zipfile

ZIP_DIR = "/content/drive/MyDrive/Marine_Project/dataset_zips"

zip_files = sorted([
    f for f in os.listdir(ZIP_DIR)
    if f.lower().endswith(".zip")
])

for zip_name in zip_files:
    zip_path = os.path.join(ZIP_DIR, zip_name)

    with zipfile.ZipFile(zip_path, "r") as z:
        image_files = [
            name for name in z.namelist()
            if name.lower().endswith((".jpg", ".jpeg", ".png"))
        ]

        print(f"\n{zip_name}")
        print("Image count:", len(image_files))
        print("Sample paths:")
        for path in image_files[:3]:
            print(" ", path)

In [ ]:
import os
import zipfile
import hashlib
import pandas as pd
from collections import defaultdict

ZIP_DIR = "/content/drive/MyDrive/Marine_Project/dataset_zips"

zip_files = {
    "Training": [],
    "Test": []
}

for filename in os.listdir(ZIP_DIR):
    if filename.endswith(".zip"):
        if "training" in filename.lower():
            zip_files["Training"].append(filename)
        elif "testing" in filename.lower():
            zip_files["Test"].append(filename)

hash_records = []

for split, files in zip_files.items():
    print(f"\nProcessing {split}...")

    for filename in sorted(files):
        zip_path = os.path.join(ZIP_DIR, filename)

        with zipfile.ZipFile(zip_path, "r") as archive:
            image_names = [
                n for n in archive.namelist()
                if n.lower().endswith((".jpg", ".jpeg", ".png"))
            ]

            for image_name in image_names:
                image_bytes = archive.read(image_name)
                image_hash = hashlib.sha256(image_bytes).hexdigest()

                hash_records.append({
                    "split": split,
                    "zip_file": filename,
                    "image_name": image_name,
                    "hash": image_hash
                })

        print("Completed:", filename)

hash_df = pd.DataFrame(hash_records)

print("\nTotal images hashed:", len(hash_df))
print("\nImages per split:")
print(hash_df["split"].value_counts())

In [ ]:
import os
import pandas as pd

RESULTS_DIR = "/content/drive/MyDrive/Marine_Project/results"

# Load the saved split definitions
train_df = pd.read_csv(os.path.join(RESULTS_DIR, "train_split.csv"))
val_df = pd.read_csv(os.path.join(RESULTS_DIR, "validation_split.csv"))

# Match images using class + filename
validation_keys = set(
    zip(
        val_df["label"].astype(str),
        val_df["path"].apply(os.path.basename)
    )
)

training_keys = set(
    zip(
        train_df["label"].astype(str),
        train_df["path"].apply(os.path.basename)
    )
)

# Assign the correct split to every hashed image
hash_df["class"] = hash_df["image_name"].apply(
    lambda x: x.split("/")[0]
)

hash_df["basename"] = hash_df["image_name"].apply(
    os.path.basename
)

def assign_split(row):
    if row["split"] == "Test":
        return "Test"

    key = (row["class"], row["basename"])

    if key in validation_keys:
        return "Validation"

    if key in training_keys:
        return "Training"

    return "Unassigned"

hash_df["actual_split"] = hash_df.apply(assign_split, axis=1)

print("Actual split counts:")
print(hash_df["actual_split"].value_counts())

# Find exact duplicate hashes
duplicate_groups = (
    hash_df.groupby("hash")
    .filter(lambda group: len(group) > 1)
    .sort_values(["hash", "actual_split"])
)

print("\nTotal duplicate image records:", len(duplicate_groups))
print("Unique duplicated image contents:",
      duplicate_groups["hash"].nunique())

if len(duplicate_groups):
    print("\nDuplicate details:")
    display(
        duplicate_groups[
            ["actual_split", "class", "image_name", "hash"]
        ]
    )

# Check duplicates crossing different splits
cross_split = (
    duplicate_groups.groupby("hash")["actual_split"]
    .nunique()
)

cross_split_hashes = cross_split[cross_split > 1].index

print("\nExact duplicates crossing splits:",
      len(cross_split_hashes))

# Save results
duplicate_groups.to_csv(
    os.path.join(RESULTS_DIR, "exact_duplicate_images.csv"),
    index=False
)

print("\nDuplicate report saved.")

In [ ]:
# Summarise each duplicated image by its split membership

cross_split_details = (
    duplicate_groups
    .groupby("hash")
    .agg(
        image_count=("hash", "size"),
        splits=("actual_split", lambda x: ", ".join(sorted(set(x)))),
        classes=("class", lambda x: ", ".join(sorted(set(x)))),
        filenames=("image_name", lambda x: " | ".join(x))
    )
    .reset_index()
)

# Keep only duplicates appearing in multiple splits
cross_split_details = cross_split_details[
    cross_split_details["splits"].str.contains(",")
].copy()

print("Cross-split duplicate groups:", len(cross_split_details))

print("\nSplit combination counts:")
print(cross_split_details["splits"].value_counts())

print("\nDuplicate groups by class:")
print(
    cross_split_details["classes"].value_counts()
)

print("\nFull cross-split duplicate details:")
display(cross_split_details)

# Save a focused report
cross_split_details.to_csv(
    os.path.join(RESULTS_DIR, "cross_split_duplicate_details.csv"),
    index=False
)

In [ ]:
# Count affected images in each split

affected = (
    duplicate_groups.groupby("actual_split")
    .agg(
        affected_images=("hash", "count"),
        unique_duplicate_groups=("hash", "nunique")
    )
)

print("Affected images by split:")
display(affected)

# Count affected images by class and split
class_affected = (
    duplicate_groups.groupby(["class", "actual_split"])
    .size()
    .unstack(fill_value=0)
)

print("\nAffected images by class:")
display(class_affected)

# Count unique images that belong to cross-split groups
cross_hashes = set(cross_split_hashes)

cross_records = hash_df[
    hash_df["hash"].isin(cross_hashes)
]

print("\nCross-split duplicate records:", len(cross_records))
print("\nCross-split records by split:")
print(cross_records["actual_split"].value_counts())

print("\nCross-split records by class:")
print(cross_records["class"].value_counts())

In [ ]:
import pandas as pd
import os

# Build one row per image, preserving all original information
all_images = hash_df[
    ["hash", "class", "image_name", "actual_split"]
].copy()

# Give identical images the same group ID
all_images["duplicate_group"] = (
    pd.factorize(all_images["hash"])[0]
)

# Verify group consistency
group_summary = (
    all_images.groupby("duplicate_group")
    .agg(
        image_count=("hash", "size"),
        classes=("class", "nunique"),
        splits=("actual_split", "nunique"),
        filenames=("image_name", lambda x: " | ".join(x))
    )
    .reset_index()
)

print("Total unique image groups:", len(group_summary))
print("Groups containing duplicates:",
      (group_summary["image_count"] > 1).sum())

print("\nGroups crossing splits:",
      (group_summary["splits"] > 1).sum())

print("\nGroups crossing classes:",
      (group_summary["classes"] > 1).sum())

# Save full grouping information
group_path = os.path.join(
    RESULTS_DIR,
    "image_duplicate_groups.csv"
)

all_images.to_csv(group_path, index=False)

print("\nSaved:", group_path)

In [ ]:
# Compare official training ZIP hashes against official test ZIP hashes

official_train = hash_df[
    hash_df["split"] == "Training"
].copy()

official_test = hash_df[
    hash_df["split"] == "Test"
].copy()

train_hashes = set(official_train["hash"])

official_test["duplicate_with_training"] = (
    official_test["hash"].isin(train_hashes)
)

clean_test = official_test[
    ~official_test["duplicate_with_training"]
].copy()

removed_test = official_test[
    official_test["duplicate_with_training"]
].copy()

print("OFFICIAL TEST SET")
print("Original images:", len(official_test))
print("Exact duplicates with training:", len(removed_test))
print("Clean test images:", len(clean_test))

print("\nRemoved duplicates by class:")
print(removed_test["class"].value_counts())

print("\nRemaining clean test images by class:")
print(clean_test["class"].value_counts())

print("\nUnique hashes in clean test:",
      clean_test["hash"].nunique())

In [ ]:
# Remove exact duplicates within the remaining clean test set.
# Keep one representative per unique SHA-256 hash.

clean_test_unique = (
    clean_test
    .sort_values(["class", "image_name"])
    .drop_duplicates(subset="hash", keep="first")
    .copy()
)

print("CLEAN TEST SET SUMMARY")
print("Before within-test deduplication:", len(clean_test))
print("After within-test deduplication:", len(clean_test_unique))
print("Removed internal duplicate records:",
      len(clean_test) - len(clean_test_unique))

print("\nFinal class distribution:")
print(clean_test_unique["class"].value_counts())

print("\nTotal unique hashes:",
      clean_test_unique["hash"].nunique())

# Save the clean test manifest
clean_test_path = os.path.join(
    RESULTS_DIR,
    "clean_test_manifest.csv"
)

clean_test_unique.to_csv(clean_test_path, index=False)

print("\nSaved:", clean_test_path)

In [ ]:
import os
import pandas as pd
from sklearn.model_selection import StratifiedKFold

RESULTS_DIR = "/content/drive/MyDrive/Marine_Project/results"

# Original training ZIP images only
official_train = hash_df[
    hash_df["split"] == "Training"
].copy()

print("Original training records:", len(official_train))

# Remove exact duplicate image contents
unique_train = (
    official_train
    .sort_values(["class", "image_name"])
    .drop_duplicates(subset="hash", keep="first")
    .reset_index(drop=True)
)

print("Unique training images:", len(unique_train))
print("Removed internal duplicate records:",
      len(official_train) - len(unique_train))

# Stratified 80:20 split
skf = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

train_indices, val_indices = next(
    skf.split(
        unique_train,
        unique_train["class"]
    )
)

new_train = unique_train.iloc[train_indices].copy()
new_val = unique_train.iloc[val_indices].copy()

new_train["split"] = "Training"
new_val["split"] = "Validation"

# Verify no hash overlaps
train_hash_set = set(new_train["hash"])
val_hash_set = set(new_val["hash"])

assert train_hash_set.isdisjoint(val_hash_set)

# Verify class distributions
print("\nNEW TRAINING DISTRIBUTION:")
print(new_train["class"].value_counts())

print("\nNEW VALIDATION DISTRIBUTION:")
print(new_val["class"].value_counts())

print("\nTotal training:", len(new_train))
print("Total validation:", len(new_val))
print("Training-validation hash overlap:",
      len(train_hash_set & val_hash_set))

# Save manifests
new_train.to_csv(
    os.path.join(RESULTS_DIR, "clean_train_manifest.csv"),
    index=False
)

new_val.to_csv(
    os.path.join(RESULTS_DIR, "clean_validation_manifest.csv"),
    index=False
)

print("\nBoth clean manifests saved successfully.")

In [ ]:
import os
import pandas as pd

RESULTS_DIR = "/content/drive/MyDrive/Marine_Project/results"

train_df = pd.read_csv(
    os.path.join(RESULTS_DIR, "clean_train_manifest.csv")
)

val_df = pd.read_csv(
    os.path.join(RESULTS_DIR, "clean_validation_manifest.csv")
)

test_df = pd.read_csv(
    os.path.join(RESULTS_DIR, "clean_test_manifest.csv")
)

datasets = {
    "Training": train_df,
    "Validation": val_df,
    "Test": test_df
}

print("DATASET VERIFICATION")
print("-" * 45)

for name, df in datasets.items():
    print(f"\n{name}: {len(df)} images")
    print("Duplicate hashes:", df["hash"].duplicated().sum())
    print("Missing hashes:", df["hash"].isna().sum())
    print("Missing filenames:", df["image_name"].isna().sum())
    print("Missing ZIP names:", df["zip_file"].isna().sum())
    print("Classes:", sorted(df["class"].unique()))

# Check overlap between every pair
print("\nCROSS-SPLIT HASH OVERLAP")
print("-" * 45)

for first, second in [
    ("Training", "Validation"),
    ("Training", "Test"),
    ("Validation", "Test")
]:
    overlap = (
        set(datasets[first]["hash"])
        & set(datasets[second]["hash"])
    )
    print(f"{first} vs {second}: {len(overlap)}")

# Confirm required metadata
required_columns = {"hash", "image_name", "zip_file", "class"}

for name, df in datasets.items():
    assert required_columns.issubset(df.columns), (
        f"{name} is missing required columns"
    )
    assert df["hash"].is_unique, (
        f"{name} contains duplicate hashes"
    )
    assert df[list(required_columns)].notna().all().all(), (
        f"{name} contains missing required values"
    )

print("\nVerification completed.")

In [ ]:
import os
import zipfile
import pandas as pd

ZIP_DIR = "/content/drive/MyDrive/Marine_Project/dataset_zips"
RESULTS_DIR = "/content/drive/MyDrive/Marine_Project/results"

train_df = pd.read_csv(
    os.path.join(RESULTS_DIR, "clean_train_manifest.csv")
)
val_df = pd.read_csv(
    os.path.join(RESULTS_DIR, "clean_validation_manifest.csv")
)
test_df = pd.read_csv(
    os.path.join(RESULTS_DIR, "clean_test_manifest.csv")
)

all_df = pd.concat(
    [train_df, val_df, test_df],
    ignore_index=True
)

print("Manifest ZIP names:")
print(sorted(all_df["zip_file"].unique()))

print("\nChecking ZIP archives and image filenames...")

zip_cache = {}
missing_files = []

for zip_name in all_df["zip_file"].unique():

    zip_path = os.path.join(ZIP_DIR, zip_name)

    if not os.path.exists(zip_path):
        print("MISSING ZIP:", zip_name)
        continue

    with zipfile.ZipFile(zip_path, "r") as z:
        zip_cache[zip_name] = {
            os.path.basename(member).lower()
            for member in z.namelist()
            if not member.endswith("/")
        }

for row in all_df.itertuples(index=False):

    available = zip_cache.get(row.zip_file, set())

    if str(row.image_name).lower() not in available:
        missing_files.append(
            (row.zip_file, row.image_name)
        )

print("\nTOTAL MANIFEST IMAGES:", len(all_df))
print("ZIP ARCHIVES FOUND:", len(zip_cache))
print("MISSING IMAGE FILES:", len(missing_files))

if missing_files:
    print("\nFirst 10 missing entries:")
    print(missing_files[:10])
else:
    print("\nSUCCESS: All manifest images were found in their ZIP archives.")

In [ ]:
import os
import zipfile
import pandas as pd

ZIP_DIR = "/content/drive/MyDrive/Marine_Project/dataset_zips"
RESULTS_DIR = "/content/drive/MyDrive/Marine_Project/results"

train_df = pd.read_csv(
    os.path.join(RESULTS_DIR, "clean_train_manifest.csv")
)
val_df = pd.read_csv(
    os.path.join(RESULTS_DIR, "clean_validation_manifest.csv")
)
test_df = pd.read_csv(
    os.path.join(RESULTS_DIR, "clean_test_manifest.csv")
)

all_df = pd.concat(
    [train_df, val_df, test_df],
    ignore_index=True
)

zip_cache = {}
missing_files = []
missing_zips = []

print("Checking ZIP archive paths...\n")

for zip_name in all_df["zip_file"].unique():

    zip_path = os.path.join(ZIP_DIR, zip_name)

    if not os.path.exists(zip_path):
        missing_zips.append(zip_name)
        continue

    with zipfile.ZipFile(zip_path, "r") as z:
        zip_cache[zip_name] = {
            member.replace("\\", "/").lstrip("./").lower()
            for member in z.namelist()
            if not member.endswith("/")
        }

for row in all_df.itertuples(index=False):

    expected_path = str(row.image_name).replace("\\", "/").lstrip("./").lower()

    available = zip_cache.get(row.zip_file, set())

    if expected_path not in available:
        missing_files.append(
            (row.zip_file, row.image_name)
        )

print("TOTAL IMAGES:", len(all_df))
print("ZIP ARCHIVES FOUND:", len(zip_cache))
print("MISSING ZIP ARCHIVES:", len(missing_zips))
print("MISSING IMAGE FILES:", len(missing_files))

if missing_zips:
    print("\nMissing ZIPs:", missing_zips)

if missing_files:
    print("\nFirst 10 missing images:")
    print(missing_files[:10])
else:
    print("\nSUCCESS: All manifest images exist in their corresponding ZIP archives.")

In [ ]:

from google.colab import drive
drive.mount("/content/drive")

import os
import torch
import torch.nn as nn
import torchvision.models as models

checkpoint_path = (
    "/content/drive/MyDrive/Marine_Project/"
    "models/resnet18_baseline_best_clean.pth"
)

assert os.path.isfile(checkpoint_path), (
    f"Checkpoint not found: {checkpoint_path}"
)

checkpoint = torch.load(
    checkpoint_path,
    map_location="cpu",
    weights_only=False
)

# Support either a raw state dictionary or a saved checkpoint dictionary.
if isinstance(checkpoint, dict) and "model_state_dict" in checkpoint:
    state_dict = checkpoint["model_state_dict"]
elif isinstance(checkpoint, dict) and "state_dict" in checkpoint:
    state_dict = checkpoint["state_dict"]
else:
    state_dict = checkpoint

# Remove a possible DataParallel prefix.
state_dict = {
    k.removeprefix("module."): v
    for k, v in state_dict.items()
}

model = models.resnet18(weights=None)
model.fc = nn.Linear(model.fc.in_features, 4)

model.load_state_dict(state_dict)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)
model.eval()

print("Checkpoint loaded successfully!")
print("Device:", device)
print("Classes:", ["Amphibolis", "Background", "Halophila", "Posidonia"])
print("Training skipped.")


In [ ]:
import os
import zipfile
import io
import pandas as pd
import torch

from PIL import Image
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

# --------------------------------------------------
# 1. PATHS AND DEVICE
# --------------------------------------------------

ZIP_DIR = "/content/drive/MyDrive/Marine_Project/dataset_zips"
RESULTS_DIR = "/content/drive/MyDrive/Marine_Project/results"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Device:", device)

# --------------------------------------------------
# 2. LOAD CLEAN MANIFESTS
# --------------------------------------------------

train_df = pd.read_csv(
    os.path.join(RESULTS_DIR, "clean_train_manifest.csv")
)

val_df = pd.read_csv(
    os.path.join(RESULTS_DIR, "clean_validation_manifest.csv")
)

test_df = pd.read_csv(
    os.path.join(RESULTS_DIR, "clean_test_manifest.csv")
)

classes = sorted(train_df["class"].unique())

class_to_idx = {
    class_name: index
    for index, class_name in enumerate(classes)
}

print("Class mapping:", class_to_idx)

# --------------------------------------------------
# 3. TRANSFORMS
# --------------------------------------------------

train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(
        10,
        interpolation=transforms.InterpolationMode.BILINEAR
    ),
    transforms.RandomApply([
        transforms.GaussianBlur(kernel_size=3, sigma=(0.1, 1.5))
    ], p=0.25),
    transforms.ColorJitter(
        brightness=0.15,
        contrast=0.25
    ),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

eval_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

# --------------------------------------------------
# 4. CUSTOM ZIP DATASET
# --------------------------------------------------

class MarineZipDataset(Dataset):

    def __init__(self, dataframe, zip_dir, class_mapping, transform=None):

        self.dataframe = dataframe.reset_index(drop=True)
        self.zip_dir = zip_dir
        self.class_mapping = class_mapping
        self.transform = transform
        self.zip_handles = {}

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, index):

        row = self.dataframe.iloc[index]

        zip_name = row["zip_file"]
        image_name = row["image_name"]

        if zip_name not in self.zip_handles:
            zip_path = os.path.join(self.zip_dir, zip_name)
            self.zip_handles[zip_name] = zipfile.ZipFile(
                zip_path, "r"
            )

        with self.zip_handles[zip_name].open(image_name) as image_file:
            image = Image.open(io.BytesIO(image_file.read())).convert("RGB")

        if self.transform:
            image = self.transform(image)

        label = self.class_mapping[row["class"]]

        return image, label

    def close(self):
        for archive in self.zip_handles.values():
            archive.close()
        self.zip_handles.clear()

# --------------------------------------------------
# 5. CREATE DATASETS
# --------------------------------------------------

train_dataset = MarineZipDataset(
    train_df, ZIP_DIR, class_to_idx, train_transform
)

val_dataset = MarineZipDataset(
    val_df, ZIP_DIR, class_to_idx, eval_transform
)

test_dataset = MarineZipDataset(
    test_df, ZIP_DIR, class_to_idx, eval_transform
)

# --------------------------------------------------
# 6. CREATE DATALOADERS
# --------------------------------------------------

BATCH_SIZE = 32

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0,
    pin_memory=(device.type == "cuda")
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=(device.type == "cuda")
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=(device.type == "cuda")
)

# --------------------------------------------------
# 7. VERIFY ACTUAL IMAGE BATCH
# --------------------------------------------------

images, labels = next(iter(train_loader))

print("\nDATASET SIZES")
print("Training:", len(train_dataset))
print("Validation:", len(val_dataset))
print("Test:", len(test_dataset))

print("\nBATCH VERIFICATION")
print("Image batch shape:", images.shape)
print("Label batch shape:", labels.shape)
print("Image data type:", images.dtype)
print("Label data type:", labels.dtype)
print("Image values:", images.min().item(), "to", images.max().item())

print("\nLABEL VERIFICATION")
for index, class_name in enumerate(classes):
    print(class_name, "=", index)

print("\nSUCCESS: ZIP-based PyTorch DataLoaders are working.")

In [ ]:
import torch
import subprocess

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("CUDA version:", torch.version.cuda)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("GPU count:", torch.cuda.device_count())
else:
    print("\nNo CUDA GPU detected.")

    try:
        result = subprocess.run(
            ["nvidia-smi"],
            capture_output=True,
            text=True
        )
        print(result.stdout if result.returncode == 0 else result.stderr)
    except Exception as e:
        print("GPU diagnostic:", e)

In [ ]:
import torch

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("CUDA version:", torch.version.cuda)
else:
    print("GPU is still unavailable.")

In [ ]:
import sys
import torch
import subprocess

print("PYTHON:", sys.executable)
print("PYTORCH:", torch.__version__)
print("TORCH FILE:", torch.__file__)

print("\n--- NVIDIA GPU CHECK ---")

result = subprocess.run(
    ["bash", "-c", "nvidia-smi"],
    capture_output=True,
    text=True
)

print("Return code:", result.returncode)
print(result.stdout if result.stdout else result.stderr)

print("\n--- PIP TORCH DETAILS ---")

result = subprocess.run(
    [sys.executable, "-m", "pip", "show", "torch"],
    capture_output=True,
    text=True
)

print(result.stdout)

In [ ]:
import torch

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("CUDA version:", torch.version.cuda)
else:
    print("GPU not attached yet.")

In [ ]:
import subprocess
import torch

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

result = subprocess.run(
    ["bash", "-c", "nvidia-smi"],
    capture_output=True,
    text=True
)

print("\nGPU STATUS:")
print(result.stdout if result.returncode == 0 else result.stderr)

if torch.cuda.is_available():
    print("GPU READY:", torch.cuda.get_device_name(0))
else:
    print("CUDA is not available in PyTorch yet.")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
import torch
import pandas as pd

ZIP_DIR = "/content/drive/MyDrive/Marine_Project/dataset_zips"
RESULTS_DIR = "/content/drive/MyDrive/Marine_Project/results"
MODELS_DIR = "/content/drive/MyDrive/Marine_Project/models"
OUTPUTS_DIR = "/content/drive/MyDrive/Marine_Project/outputs"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Device:", device)
print("GPU:", torch.cuda.get_device_name(0))

print("\nMANIFEST FILES:")
for filename in [
    "clean_train_manifest.csv",
    "clean_validation_manifest.csv",
    "clean_test_manifest.csv"
]:
    path = os.path.join(RESULTS_DIR, filename)
    print(filename, ":", os.path.exists(path))

print("\nEXISTING CHECKPOINTS:")
for filename in os.listdir(MODELS_DIR):
    if filename.endswith(".pth"):
        print(filename)

In [ ]:

from google.colab import drive
drive.mount("/content/drive")

import os
import torch
import torch.nn as nn
import torchvision.models as models

checkpoint_path = (
    "/content/drive/MyDrive/Marine_Project/"
    "models/resnet18_baseline_best_clean.pth"
)

assert os.path.isfile(checkpoint_path), (
    f"Checkpoint not found: {checkpoint_path}"
)

checkpoint = torch.load(
    checkpoint_path,
    map_location="cpu",
    weights_only=False
)

# Support either a raw state dictionary or a saved checkpoint dictionary.
if isinstance(checkpoint, dict) and "model_state_dict" in checkpoint:
    state_dict = checkpoint["model_state_dict"]
elif isinstance(checkpoint, dict) and "state_dict" in checkpoint:
    state_dict = checkpoint["state_dict"]
else:
    state_dict = checkpoint

# Remove a possible DataParallel prefix.
state_dict = {
    k.removeprefix("module."): v
    for k, v in state_dict.items()
}

model = models.resnet18(weights=None)
model.fc = nn.Linear(model.fc.in_features, 4)

model.load_state_dict(state_dict)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)
model.eval()

print("Checkpoint loaded successfully!")
print("Device:", device)
print("Classes:", ["Amphibolis", "Background", "Halophila", "Posidonia"])
print("Training skipped.")


In [ ]:
import sys
import torch
import subprocess

print("PYTHON:", sys.executable)
print("PYTORCH:", torch.__version__)
print("PYTORCH LOCATION:", torch.__file__)

print("\nCUDA AVAILABLE:", torch.cuda.is_available())
print("CUDA BUILD:", torch.version.cuda)

print("\nNVIDIA GPU CHECK:")

result = subprocess.run(
    ["bash", "-c", "nvidia-smi"],
    capture_output=True,
    text=True
)

print(result.stdout if result.returncode == 0 else result.stderr)

In [ ]:
import torch
import os

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("GPU unavailable")

print("\nGoogle Drive mounted:", os.path.exists(
    "/content/drive/MyDrive/Marine_Project"
))

In [ ]:
import os
import torch

print("DEVICE:", device)

# Verify all three DataLoaders
for name, loader in [
    ("Training", train_loader),
    ("Validation", val_loader),
    ("Test", test_loader)
]:
    images, labels = next(iter(loader))
    print(f"\n{name}:")
    print("Dataset size:", len(loader.dataset))
    print("Batch shape:", tuple(images.shape))
    print("Labels:", sorted(labels.unique().tolist()))

# Define a completely new checkpoint name
NEW_CHECKPOINT = os.path.join(
    "/content/drive/MyDrive/Marine_Project/models",
    "resnet18_clean_split_baseline_best.pth"
)

NEW_HISTORY = os.path.join(
    "/content/drive/MyDrive/Marine_Project/results",
    "resnet18_clean_split_baseline_history.csv"
)

print("\nNEW CHECKPOINT ALREADY EXISTS:", os.path.exists(NEW_CHECKPOINT))
print("NEW HISTORY ALREADY EXISTS:", os.path.exists(NEW_HISTORY))

if device.type == "cuda":
    print("\nGPU MEMORY:")
    print("Allocated:", round(torch.cuda.memory_allocated() / 1024**2), "MB")
    print("Reserved:", round(torch.cuda.memory_reserved() / 1024**2), "MB")

print("\nPRE-TRAINING VERIFICATION COMPLETE")

In [ ]:
import torch

class_names = {
    0: "Amphibolis",
    1: "Background",
    2: "Halophila",
    3: "Posidonia"
}

for name, loader in [
    ("Training", train_loader),
    ("Validation", val_loader),
    ("Test", test_loader)
]:
    counts = torch.zeros(4, dtype=torch.long)

    for _, labels in loader:
        counts += torch.bincount(labels.cpu(), minlength=4)

    print(f"\n{name} class distribution:")
    for i in range(4):
        print(f"{class_names[i]}: {counts[i].item()}")

    print("Total:", counts.sum().item())

In [ ]:
import torch
import torch.nn as nn
import torchvision.models as models
from torch.optim import Adam

# Reproducibility
torch.manual_seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(42)

# Fresh ResNet-18
weights = models.ResNet18_Weights.DEFAULT
model = models.resnet18(weights=weights)

# Four-class output
model.fc = nn.Linear(model.fc.in_features, 4)
model = model.to(device)

# Loss and optimizer
criterion = nn.CrossEntropyLoss()
optimizer = Adam(model.parameters(), lr=1e-4)

# Training configuration
EPOCHS = 10
PATIENCE = 3
BATCH_SIZE = 32

# Mixed precision
scaler = torch.amp.GradScaler("cuda", enabled=(device.type == "cuda"))

print("MODEL:", model.__class__.__name__)
print("OUTPUT CLASSES:", model.fc.out_features)
print("DEVICE:", device)
print("LEARNING RATE:", optimizer.param_groups[0]["lr"])
print("EPOCHS:", EPOCHS)
print("PATIENCE:", PATIENCE)
print("CHECKPOINT:", NEW_CHECKPOINT)
print("HISTORY:", NEW_HISTORY)

print("\nMODEL CONFIGURATION READY")

In [ ]:
import time
import pandas as pd
from tqdm.auto import tqdm

history = []
best_val_loss = float("inf")
patience_counter = 0

for epoch in range(EPOCHS):
    start_time = time.time()

    # -------------------------
    # TRAINING
    # -------------------------
    model.train()
    train_loss = 0.0
    train_correct = 0
    train_total = 0

    train_bar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS} [Train]")

    for images, labels in train_bar:
        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad(set_to_none=True)

        with torch.autocast(
            device_type="cuda",
            dtype=torch.float16,
            enabled=(device.type == "cuda")
        ):
            outputs = model(images)
            loss = criterion(outputs, labels)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        train_loss += loss.item() * images.size(0)
        train_correct += (outputs.argmax(1) == labels).sum().item()
        train_total += labels.size(0)

    train_loss /= train_total
    train_acc = train_correct / train_total

    # -------------------------
    # VALIDATION
    # -------------------------
    model.eval()
    val_loss = 0.0
    val_correct = 0
    val_total = 0

    with torch.no_grad():
        for images, labels in tqdm(
            val_loader, desc=f"Epoch {epoch+1}/{EPOCHS} [Val]"
        ):
            images = images.to(device)
            labels = labels.to(device)

            with torch.autocast(
                device_type="cuda",
                dtype=torch.float16,
                enabled=(device.type == "cuda")
            ):
                outputs = model(images)
                loss = criterion(outputs, labels)

            val_loss += loss.item() * images.size(0)
            val_correct += (outputs.argmax(1) == labels).sum().item()
            val_total += labels.size(0)

    val_loss /= val_total
    val_acc = val_correct / val_total

    epoch_time = time.time() - start_time

    # -------------------------
    # RECORD RESULTS
    # -------------------------
    row = {
        "epoch": epoch + 1,
        "train_loss": train_loss,
        "train_accuracy": train_acc,
        "val_loss": val_loss,
        "val_accuracy": val_acc,
        "epoch_time_seconds": epoch_time
    }

    history.append(row)

    print(f"\nEpoch {epoch+1}/{EPOCHS}")
    print(f"Train Loss: {train_loss:.5f} | Train Accuracy: {train_acc*100:.2f}%")
    print(f"Val Loss:   {val_loss:.5f} | Val Accuracy:   {val_acc*100:.2f}%")
    print(f"Time: {epoch_time:.1f} seconds")

    # -------------------------
    # SAVE BEST CHECKPOINT
    # -------------------------
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        patience_counter = 0

        torch.save({
            "epoch": epoch + 1,
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "best_val_loss": best_val_loss,
            "class_mapping": class_to_idx
        }, NEW_CHECKPOINT)

        print("BEST CHECKPOINT SAVED")
    else:
        patience_counter += 1
        print(f"No improvement: {patience_counter}/{PATIENCE}")

    # Save history after every epoch
    pd.DataFrame(history).to_csv(NEW_HISTORY, index=False)

    if patience_counter >= PATIENCE:
        print("\nEarly stopping triggered.")
        break

print("\nTRAINING FINISHED")
print("Best validation loss:", best_val_loss)
print("Checkpoint:", NEW_CHECKPOINT)
print("History:", NEW_HISTORY)

In [ ]:
import torch
import numpy as np
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    accuracy_score
)

# Load best checkpoint
checkpoint = torch.load(
    NEW_CHECKPOINT,
    map_location=device,
    weights_only=False
)

model.load_state_dict(checkpoint["model_state_dict"])
model.eval()

print("BEST CHECKPOINT EPOCH:", checkpoint["epoch"])
print("BEST VALIDATION LOSS:", checkpoint["best_val_loss"])

# Validation predictions
all_labels = []
all_predictions = []

with torch.no_grad():
    for images, labels in val_loader:
        images = images.to(device)
        labels = labels.to(device)

        with torch.autocast(
            device_type="cuda",
            dtype=torch.float16,
            enabled=(device.type == "cuda")
        ):
            outputs = model(images)

        predictions = outputs.argmax(dim=1)

        all_labels.extend(labels.cpu().numpy())
        all_predictions.extend(predictions.cpu().numpy())

# Metrics
class_names = [
    "Amphibolis",
    "Background",
    "Halophila",
    "Posidonia"
]

print("\nVALIDATION ACCURACY:",
      f"{accuracy_score(all_labels, all_predictions)*100:.2f}%")

print("\nCLASSIFICATION REPORT:")
print(classification_report(
    all_labels,
    all_predictions,
    target_names=class_names,
    digits=4
))

print("\nCONFUSION MATRIX:")
print(confusion_matrix(
    all_labels,
    all_predictions
))

In [ ]:
import os
import pandas as pd
import torch
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix
)

# Load the selected best checkpoint
checkpoint = torch.load(
    NEW_CHECKPOINT,
    map_location=device,
    weights_only=False
)

model.load_state_dict(checkpoint["model_state_dict"])
model.eval()

class_names = [
    "Amphibolis",
    "Background",
    "Halophila",
    "Posidonia"
]

all_labels = []
all_predictions = []
all_confidences = []

# Test evaluation
with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device)
        labels = labels.to(device)

        with torch.autocast(
            device_type="cuda",
            dtype=torch.float16,
            enabled=(device.type == "cuda")
        ):
            outputs = model(images)

        probabilities = torch.softmax(outputs.float(), dim=1)
        confidences, predictions = probabilities.max(dim=1)

        all_labels.extend(labels.cpu().numpy())
        all_predictions.extend(predictions.cpu().numpy())
        all_confidences.extend(confidences.cpu().numpy())

# Metrics
test_accuracy = accuracy_score(
    all_labels, all_predictions
)

report = classification_report(
    all_labels,
    all_predictions,
    target_names=class_names,
    output_dict=True,
    digits=4
)

cm = confusion_matrix(
    all_labels,
    all_predictions
)

print("TEST CHECKPOINT EPOCH:", checkpoint["epoch"])
print("TEST IMAGES:", len(all_labels))
print(f"TEST ACCURACY: {test_accuracy * 100:.2f}%")

print("\nCLASSIFICATION REPORT:")
print(classification_report(
    all_labels,
    all_predictions,
    target_names=class_names,
    digits=4
))

print("\nCONFUSION MATRIX:")
print(cm)

# Save results with new filenames
results_dir = "/content/drive/MyDrive/Marine_Project/results"

pd.DataFrame(report).transpose().to_csv(
    os.path.join(results_dir, "clean_split_baseline_test_report.csv")
)

pd.DataFrame(
    cm,
    index=class_names,
    columns=class_names
).to_csv(
    os.path.join(results_dir, "clean_split_baseline_test_confusion_matrix.csv")
)

pd.DataFrame({
    "actual": [class_names[x] for x in all_labels],
    "predicted": [class_names[x] for x in all_predictions],
    "confidence": all_confidences
}).to_csv(
    os.path.join(results_dir, "clean_split_baseline_test_predictions.csv"),
    index=False
)

print("\nTEST RESULTS SAVED SUCCESSFULLY")

In [ ]:
import pandas as pd
import numpy as np

pred_path = (
    "/content/drive/MyDrive/Marine_Project/results/"
    "clean_split_baseline_test_predictions.csv"
)

df = pd.read_csv(pred_path)

print("TOTAL PREDICTIONS:", len(df))
print("CORRECT:", (df["actual"] == df["predicted"]).sum())
print("INCORRECT:", (df["actual"] != df["predicted"]).sum())

df["correct"] = df["actual"] == df["predicted"]

print("\nMEAN CONFIDENCE:")
print(df.groupby("correct")["confidence"].mean())

print("\nERROR DISTRIBUTION:")
errors = df[~df["correct"]]

print(
    errors.groupby(["actual", "predicted"])
    .size()
    .sort_values(ascending=False)
)

print("\nCONFIDENCE OF INCORRECT PREDICTIONS:")
print(errors["confidence"].describe())

In [ ]:
import pandas as pd

print("TEST DATASET TYPE:")
print(type(test_loader.dataset))

print("\nTEST DATASET ATTRIBUTES:")
print([
    attr for attr in dir(test_loader.dataset)
    if not attr.startswith("__")
    and not callable(getattr(test_loader.dataset, attr, None))
])

print("\nDATASET INTERNAL DATAFRAME:")
for attr in ["df", "data", "manifest", "samples", "image_paths"]:
    if hasattr(test_loader.dataset, attr):
        value = getattr(test_loader.dataset, attr)
        print(f"\n{attr}:")
        if isinstance(value, pd.DataFrame):
            print("Columns:", value.columns.tolist())
            print(value.head(3))
        elif isinstance(value, (list, tuple)):
            print("Length:", len(value))
            print("First 3:", value[:3])
        else:
            print(type(value))

print("\nTEST LOADER SHUFFLE CHECK:")
print("Sampler:", type(test_loader.sampler).__name__)
print("Dataset length:", len(test_loader.dataset))

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import torch

# Load saved predictions
pred_path = (
    "/content/drive/MyDrive/Marine_Project/results/"
    "clean_split_baseline_test_predictions.csv"
)

pred_df = pd.read_csv(pred_path)

# Retrieve test manifest in exact loader order
test_manifest = test_loader.dataset.dataframe.reset_index(drop=True)

# Verify alignment
assert len(pred_df) == len(test_manifest)

# Combine prediction and manifest information
error_df = pd.concat(
    [pred_df.reset_index(drop=True), test_manifest],
    axis=1
)

# Select incorrect predictions
errors = error_df[
    error_df["actual"] != error_df["predicted"]
].copy()

# Sort by confidence, highest first
errors = errors.sort_values(
    "confidence",
    ascending=False
)

print("TOTAL ERRORS:", len(errors))
print("\nTOP 12 CONFIDENTLY INCORRECT PREDICTIONS:")

display(
    errors.head(12)[
        ["actual", "predicted", "confidence", "image_name", "zip_file"]
    ]
)

In [ ]:
import os
import zipfile
import matplotlib.pyplot as plt
from PIL import Image
from io import BytesIO

# Use the 12 highest-confidence errors
top_errors = errors.head(12).reset_index(drop=True)

fig, axes = plt.subplots(3, 4, figsize=(18, 13))
axes = axes.flatten()

zip_dir = test_loader.dataset.zip_dir

for i, row in top_errors.iterrows():

    zip_path = os.path.join(zip_dir, row["zip_file"])

    with zipfile.ZipFile(zip_path, "r") as z:
        image_bytes = z.read(row["image_name"])

    img = Image.open(BytesIO(image_bytes)).convert("RGB")

    axes[i].imshow(img)
    axes[i].set_title(
        f"Actual: {row['actual']}\n"
        f"Predicted: {row['predicted']}\n"
        f"Confidence: {row['confidence']:.4%}",
        fontsize=10
    )
    axes[i].axis("off")

plt.suptitle(
    "Top 12 High-Confidence Misclassified Test Images",
    fontsize=16
)
plt.tight_layout()
plt.show()

In [ ]:
import cv2
import numpy as np
import zipfile
import os
import pandas as pd
from PIL import Image
from io import BytesIO

# Select 12 highest-confidence errors
error_samples = errors.head(12).copy()

# Select correctly classified comparison images
correct_samples = error_df[
    error_df["actual"] == error_df["predicted"]
].groupby("actual", group_keys=False).head(3)

samples = pd.concat(
    [error_samples, correct_samples],
    ignore_index=True
)

results = []

for _, row in samples.iterrows():

    zip_path = os.path.join(
        test_loader.dataset.zip_dir,
        row["zip_file"]
    )

    with zipfile.ZipFile(zip_path, "r") as z:
        image_bytes = z.read(row["image_name"])

    img = Image.open(BytesIO(image_bytes)).convert("RGB")

    image_array = np.array(img)
    gray = cv2.cvtColor(image_array, cv2.COLOR_RGB2GRAY)

    sharpness = cv2.Laplacian(
        gray, cv2.CV_64F
    ).var()

    results.append({
        "actual": row["actual"],
        "predicted": row["predicted"],
        "correct": row["actual"] == row["predicted"],
        "width": img.width,
        "height": img.height,
        "sharpness": sharpness,
        "confidence": row["confidence"]
    })

sharpness_df = pd.DataFrame(results)

print("IMAGE QUALITY COMPARISON")
print(
    sharpness_df.groupby("correct")[
        ["sharpness", "width", "height"]
    ].agg(["mean", "median"])
)

print("\nINDIVIDUAL RESULTS")
display(sharpness_df)

In [ ]:
import cv2
import numpy as np
import pandas as pd
import os
import zipfile
from PIL import Image
from io import BytesIO

# Create the correct/incorrect indicator
error_df = error_df.copy()
error_df["correct"] = (
    error_df["actual"] == error_df["predicted"]
)

# All incorrect predictions
error_samples = error_df[
    ~error_df["correct"]
].copy()

# Select matched correct images by actual class
correct_parts = []

for class_name, count in error_samples["actual"].value_counts().items():

    candidates = error_df[
        (error_df["actual"] == class_name) &
        (error_df["correct"])
    ]

    selected = candidates.sample(
        n=count,
        random_state=42
    )

    correct_parts.append(selected)

correct_samples = pd.concat(
    correct_parts,
    ignore_index=True
)

samples = pd.concat(
    [error_samples, correct_samples],
    ignore_index=True
)

print("ERROR IMAGES:", len(error_samples))
print("MATCHED CORRECT IMAGES:", len(correct_samples))

print("\nCLASS DISTRIBUTION:")
print(pd.crosstab(samples["actual"], samples["correct"]))

# Image quality measurements
quality_results = []

for _, row in samples.iterrows():

    zip_path = os.path.join(
        test_loader.dataset.zip_dir,
        row["zip_file"]
    )

    with zipfile.ZipFile(zip_path, "r") as z:
        image_bytes = z.read(row["image_name"])

    img = Image.open(BytesIO(image_bytes)).convert("RGB")
    rgb = np.array(img)

    gray = cv2.cvtColor(rgb, cv2.COLOR_RGB2GRAY)
    hsv = cv2.cvtColor(rgb, cv2.COLOR_RGB2HSV)

    quality_results.append({
        "actual": row["actual"],
        "correct": bool(row["correct"]),
        "sharpness": cv2.Laplacian(
            gray, cv2.CV_64F
        ).var(),
        "contrast": gray.std(),
        "brightness": gray.mean(),
        "saturation": hsv[:, :, 1].mean()
    })

quality_df = pd.DataFrame(quality_results)

print("\nMATCHED IMAGE QUALITY COMPARISON:")
print(
    quality_df.groupby("correct")[
        ["sharpness", "contrast", "brightness", "saturation"]
    ].agg(["mean", "median"]).round(3)
)

print("\nCLASS-WISE COMPARISON:")
print(
    quality_df.groupby(["actual", "correct"])[
        ["sharpness", "contrast", "brightness", "saturation"]
    ].mean().round(3)
)

# Save without overwriting an existing file
quality_path = (
    "/content/drive/MyDrive/Marine_Project/results/"
    "matched_test_image_quality_analysis.csv"
)

if os.path.exists(quality_path):
    quality_path = quality_path.replace(
        ".csv", "_v2.csv"
    )

quality_df.to_csv(quality_path, index=False)

print("\nSAVED:", quality_path)

In [ ]:
import numpy as np
import pandas as pd
from scipy.stats import mannwhitneyu

metrics = [
    "sharpness",
    "contrast",
    "brightness",
    "saturation"
]

incorrect = quality_df[quality_df["correct"] == False]
correct = quality_df[quality_df["correct"] == True]

results = []

for metric in metrics:

    x = incorrect[metric].dropna().values
    y = correct[metric].dropna().values

    test = mannwhitneyu(
        x, y,
        alternative="two-sided",
        method="auto"
    )

    # Rank-biserial correlation:
    # Negative value means incorrect group tends to have lower values.
    effect_size = (2 * test.statistic) / (len(x) * len(y)) - 1

    results.append({
        "metric": metric,
        "incorrect_median": np.median(x),
        "correct_median": np.median(y),
        "U_statistic": test.statistic,
        "p_value": test.pvalue,
        "rank_biserial": effect_size
    })

stats_df = pd.DataFrame(results)

# Benjamini-Hochberg FDR correction
p_values = stats_df["p_value"].values
m = len(p_values)
order = np.argsort(p_values)
adjusted = np.empty(m)

sorted_p = p_values[order]
sorted_adj = sorted_p * m / np.arange(1, m + 1)
sorted_adj = np.minimum.accumulate(sorted_adj[::-1])[::-1]
adjusted[order] = np.minimum(sorted_adj, 1.0)

stats_df["FDR_adjusted_p"] = adjusted
stats_df["significant_FDR_0.05"] = adjusted < 0.05

print("STATISTICAL COMPARISON")
display(stats_df.round(6))

print("\nSIGNIFICANT METRICS AFTER FDR:")
print(
    stats_df.loc[
        stats_df["significant_FDR_0.05"], "metric"
    ].tolist()
)

In [ ]:
import numpy as np
import pandas as pd
from scipy.stats import mannwhitneyu

classes = ["Amphibolis", "Halophila", "Posidonia"]

metrics = [
    "sharpness",
    "contrast",
    "brightness",
    "saturation"
]

class_results = []

for cls in classes:

    subset = quality_df[quality_df["actual"] == cls]

    incorrect = subset[subset["correct"] == False]
    correct = subset[subset["correct"] == True]

    print(f"\n{cls}: Errors={len(incorrect)}, Correct={len(correct)}")

    for metric in metrics:

        x = incorrect[metric].dropna().values
        y = correct[metric].dropna().values

        test = mannwhitneyu(
            x, y,
            alternative="two-sided",
            method="auto"
        )

        effect = (
            2 * test.statistic / (len(x) * len(y))
        ) - 1

        class_results.append({
            "class": cls,
            "metric": metric,
            "incorrect_median": np.median(x),
            "correct_median": np.median(y),
            "p_value": test.pvalue,
            "rank_biserial": effect
        })

class_stats = pd.DataFrame(class_results)

# FDR correction across all 12 class-wise comparisons
p = class_stats["p_value"].values
m = len(p)
order = np.argsort(p)

adjusted = np.empty(m)
sorted_p = p[order]
sorted_adj = sorted_p * m / np.arange(1, m + 1)
sorted_adj = np.minimum.accumulate(sorted_adj[::-1])[::-1]

adjusted[order] = np.minimum(sorted_adj, 1.0)

class_stats["FDR_adjusted_p"] = adjusted
class_stats["significant"] = adjusted < 0.05

display(class_stats.round(6))

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

metrics = [
    "sharpness",
    "contrast",
    "brightness",
    "saturation"
]

class_order = [
    "Amphibolis",
    "Halophila",
    "Posidonia"
]

for metric in metrics:

    plt.figure(figsize=(10, 5))

    sns.boxplot(
        data=quality_df,
        x="actual",
        y=metric,
        hue="correct",
        order=class_order
    )

    plt.title(
        f"{metric.capitalize()}: Correct vs Incorrect Predictions"
    )
    plt.xlabel("Actual Class")
    plt.ylabel(metric.capitalize())
    plt.legend(
        title="Prediction Correct",
        labels=["Incorrect", "Correct"]
    )

    plt.tight_layout()
    plt.show()

In [ ]:
import inspect

print("Dataset class:", type(train_loader.dataset).__name__)

print("\nDataset attributes:")
print(vars(train_loader.dataset).keys())

print("\nDataset __getitem__ implementation:")
print(inspect.getsource(type(train_loader.dataset).__getitem__))

print("\nCurrent image transform:")
print(inspect.getsource(type(train_loader.dataset).__init__))

In [ ]:
print("Training transform:")
print(train_loader.dataset.transform)

print("\nValidation transform:")
print(val_loader.dataset.transform)

print("\nOpenCV availability:")

try:
    import cv2
    print("OpenCV version:", cv2.__version__)
except ImportError:
    print("OpenCV is not installed.")

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
import zipfile
import os
import io
from PIL import Image

# Select one validation image from each class
class_order = ["Amphibolis", "Background", "Halophila", "Posidonia"]

def apply_clahe(rgb):
    lab = cv2.cvtColor(rgb, cv2.COLOR_RGB2LAB)
    l, a, b = cv2.split(lab)

    clahe = cv2.createCLAHE(
        clipLimit=2.0,
        tileGridSize=(8, 8)
    )

    enhanced_l = clahe.apply(l)
    enhanced = cv2.merge((enhanced_l, a, b))

    return cv2.cvtColor(enhanced, cv2.COLOR_LAB2RGB)


def gray_world_correction(rgb):
    image = rgb.astype(np.float32)

    channel_means = image.mean(axis=(0, 1))
    gray_mean = channel_means.mean()

    scale = gray_mean / np.maximum(channel_means, 1e-6)
    corrected = image * scale

    return np.clip(corrected, 0, 255).astype(np.uint8)


fig, axes = plt.subplots(4, 4, figsize=(16, 14))

for row_idx, class_name in enumerate(class_order):

    row = val_loader.dataset.dataframe[
        val_loader.dataset.dataframe["class"] == class_name
    ].iloc[0]

    zip_path = os.path.join(
        val_loader.dataset.zip_dir,
        row["zip_file"]
    )

    with zipfile.ZipFile(zip_path, "r") as zf:
        with zf.open(row["image_name"]) as image_file:
            original = Image.open(
                io.BytesIO(image_file.read())
            ).convert("RGB")

    original = np.array(
        original.resize((224, 224), Image.Resampling.BILINEAR)
    )

    clahe_img = apply_clahe(original)
    color_img = gray_world_correction(original)
    combined_img = apply_clahe(color_img)

    images = [
        original,
        clahe_img,
        color_img,
        combined_img
    ]

    titles = [
        "Original",
        "CLAHE",
        "Gray-world",
        "Combined"
    ]

    for col_idx, (img, title) in enumerate(zip(images, titles)):
        axes[row_idx, col_idx].imshow(img)
        axes[row_idx, col_idx].set_title(title)
        axes[row_idx, col_idx].axis("off")

    axes[row_idx, 0].set_ylabel(
        class_name,
        fontsize=11,
        rotation=90,
        labelpad=15
    )

plt.tight_layout()
plt.show()

In [ ]:
import torch

checkpoint_path = (
    "/content/drive/MyDrive/Marine_Project/models/"
    "resnet18_clean_split_baseline_best.pth"
)

checkpoint = torch.load(
    checkpoint_path,
    map_location="cpu",
    weights_only=False
)

print("Checkpoint type:", type(checkpoint))

if isinstance(checkpoint, dict):
    print("Checkpoint keys:", checkpoint.keys())

    for key, value in checkpoint.items():
        if isinstance(value, dict):
            print(f"{key}: dictionary with {len(value)} entries")
        else:
            print(f"{key}: {type(value).__name__}")

In [ ]:
import torch
import torch.nn as nn
import numpy as np
import cv2
from PIL import Image
from torchvision import models, transforms as T
from torch.utils.data import DataLoader
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix
)

# --------------------------------------------------
# 1. Load the existing trained model
# --------------------------------------------------

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = models.resnet18(weights=None)
model.fc = nn.Linear(model.fc.in_features, 4)

model.load_state_dict(checkpoint["model_state_dict"])
model = model.to(device)
model.eval()

class_mapping = checkpoint["class_mapping"]
class_names = [
    name for name, index in
    sorted(class_mapping.items(), key=lambda x: x[1])
]

print("Model loaded successfully")
print("Device:", device)
print("Classes:", class_names)


# --------------------------------------------------
# 2. Define CLAHE preprocessing
# --------------------------------------------------

def clahe_pil(image):

    rgb = np.array(image.convert("RGB"))

    lab = cv2.cvtColor(rgb, cv2.COLOR_RGB2LAB)
    l, a, b = cv2.split(lab)

    clahe = cv2.createCLAHE(
        clipLimit=2.0,
        tileGridSize=(8, 8)
    )

    enhanced_l = clahe.apply(l)

    enhanced = cv2.merge((enhanced_l, a, b))

    result = cv2.cvtColor(
        enhanced,
        cv2.COLOR_LAB2RGB
    )

    return Image.fromarray(result)


# --------------------------------------------------
# 3. Create a separate CLAHE validation dataset
# --------------------------------------------------

clahe_transform = T.Compose([
    T.Resize((224, 224)),
    T.Lambda(clahe_pil),
    T.ToTensor(),
    T.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

clahe_val_dataset = MarineZipDataset(
    dataframe=val_loader.dataset.dataframe,
    zip_dir=val_loader.dataset.zip_dir,
    class_mapping=class_mapping,
    transform=clahe_transform
)

clahe_val_loader = DataLoader(
    clahe_val_dataset,
    batch_size=32,
    shuffle=False,
    num_workers=0,
    pin_memory=(device.type == "cuda")
)


# --------------------------------------------------
# 4. Evaluation function
# --------------------------------------------------

def evaluate_model(model, loader, device):

    all_labels = []
    all_predictions = []

    model.eval()

    with torch.no_grad():

        for images, labels in loader:

            images = images.to(device)

            outputs = model(images)

            predictions = torch.argmax(outputs, dim=1)

            all_labels.extend(labels.numpy())
            all_predictions.extend(
                predictions.cpu().numpy()
            )

    return np.array(all_labels), np.array(all_predictions)


# --------------------------------------------------
# 5. Evaluate both versions
# --------------------------------------------------

original_labels, original_preds = evaluate_model(
    model, val_loader, device
)

clahe_labels, clahe_preds = evaluate_model(
    model, clahe_val_loader, device
)

# Confirm identical evaluation order and labels
assert np.array_equal(original_labels, clahe_labels)

# --------------------------------------------------
# 6. Display comparison
# --------------------------------------------------

for name, labels, predictions in [
    ("ORIGINAL", original_labels, original_preds),
    ("CLAHE", clahe_labels, clahe_preds)
]:

    print("\n" + "=" * 55)
    print(name)
    print("=" * 55)

    print(
        "Accuracy:",
        round(accuracy_score(labels, predictions) * 100, 2),
        "%"
    )

    print(
        "Macro F1:",
        round(f1_score(labels, predictions, average="macro"), 4)
    )

    print("\nClassification Report:")

    print(classification_report(
        labels,
        predictions,
        target_names=class_names,
        digits=4
    ))

    print("Confusion Matrix:")

    print(confusion_matrix(labels, predictions))

In [ ]:
import pandas as pd
import numpy as np

comparison_df = pd.DataFrame({
    "actual": original_labels,
    "original_prediction": original_preds,
    "clahe_prediction": clahe_preds
})

comparison_df["original_correct"] = (
    comparison_df["actual"] == comparison_df["original_prediction"]
)

comparison_df["clahe_correct"] = (
    comparison_df["actual"] == comparison_df["clahe_prediction"]
)

original_fixed = comparison_df[
    (~comparison_df["original_correct"]) &
    (comparison_df["clahe_correct"])
]

new_errors = comparison_df[
    (comparison_df["original_correct"]) &
    (~comparison_df["clahe_correct"])
]

print("Original errors corrected by CLAHE:", len(original_fixed))
print("New errors introduced by CLAHE:", len(new_errors))

print("\nOriginal errors corrected:")
print(original_fixed.to_string(index=False))

print("\nNew errors introduced:")
print(new_errors.to_string(index=False))

print("\nPrediction changes:")
print(
    (comparison_df["original_prediction"] !=
     comparison_df["clahe_prediction"]).sum()
)

In [ ]:
# Mild CLAHE: clipLimit = 1.0

def mild_clahe_pil(image):

    rgb = np.array(image.convert("RGB"))

    lab = cv2.cvtColor(rgb, cv2.COLOR_RGB2LAB)
    l, a, b = cv2.split(lab)

    clahe = cv2.createCLAHE(
        clipLimit=1.0,
        tileGridSize=(8, 8)
    )

    enhanced_l = clahe.apply(l)

    result = cv2.cvtColor(
        cv2.merge((enhanced_l, a, b)),
        cv2.COLOR_LAB2RGB
    )

    return Image.fromarray(result)


mild_clahe_transform = T.Compose([
    T.Resize((224, 224)),
    T.Lambda(mild_clahe_pil),
    T.ToTensor(),
    T.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

mild_clahe_dataset = MarineZipDataset(
    dataframe=val_loader.dataset.dataframe,
    zip_dir=val_loader.dataset.zip_dir,
    class_mapping=class_mapping,
    transform=mild_clahe_transform
)

mild_clahe_loader = DataLoader(
    mild_clahe_dataset,
    batch_size=32,
    shuffle=False,
    num_workers=0
)

mild_labels, mild_preds = evaluate_model(
    model,
    mild_clahe_loader,
    device
)

assert np.array_equal(original_labels, mild_labels)

print("MILD CLAHE (clipLimit=1.0)")
print("Accuracy:", round(
    accuracy_score(mild_labels, mild_preds) * 100, 2
), "%")

print("Macro F1:", round(
    f1_score(mild_labels, mild_preds, average="macro"), 4
))

print("\nClassification Report:")
print(classification_report(
    mild_labels,
    mild_preds,
    target_names=class_names,
    digits=4
))

print("Confusion Matrix:")
print(confusion_matrix(mild_labels, mild_preds))

In [ ]:
import pandas as pd

splits = {
    "Training": train_loader.dataset.dataframe,
    "Validation": val_loader.dataset.dataframe,
    "Test": test_loader.dataset.dataframe
}

source_tables = []

for split_name, df in splits.items():

    temp = df.copy()

    temp["source"] = temp["zip_file"].str.extract(
        r"(DBCA|CMER)",
        expand=False
    )

    counts = pd.crosstab(
        temp["class"],
        temp["source"]
    )

    counts["Split"] = split_name

    source_tables.append(counts.reset_index())

source_distribution = pd.concat(
    source_tables,
    ignore_index=True
)

print(source_distribution.to_string(index=False))

In [ ]:
for split_name, df in splits.items():

    print(f"\n{'='*45}")
    print(split_name)
    print(f"{'='*45}")

    print("\nZIP filenames:")
    print(df["zip_file"].drop_duplicates().tolist())

    print("\nSample image filenames:")
    print(df["image_name"].head(10).tolist())

    print("\nFilename source keywords:")
    print(
        df["image_name"].str.extract(
            r"(DBCA|CMER)",
            expand=False
        ).value_counts(dropna=False)
    )

In [ ]:
import pandas as pd

source_tables = []

for split_name, df in splits.items():

    temp = df.copy()

    temp["source"] = temp["image_name"].str.extract(
        r"(DBCA|CMER)",
        expand=False
    )

    counts = pd.crosstab(
        [temp["class"], temp["source"]],
        columns=pd.Index([split_name], name="Split")
    )

    source_tables.append(counts)

source_distribution = pd.concat(
    source_tables,
    axis=1
).fillna(0).astype(int)

print("SOURCE DISTRIBUTION BY CLASS AND SPLIT")
print(source_distribution)

print("\nSOURCE PROPORTIONS WITHIN EACH CLASS AND SPLIT")

for split_name, df in splits.items():

    temp = df.copy()
    temp["source"] = temp["image_name"].str.extract(
        r"(DBCA|CMER)",
        expand=False
    )

    print(f"\n{split_name}")

    print(
        pd.crosstab(
            temp["class"],
            temp["source"],
            normalize="index"
        ).round(3)
    )

In [ ]:
try:
    import imagehash
    print("ImageHash version:", imagehash.__version__)
    print("pHash is available.")
except ImportError:
    print("ImageHash is not installed.")

In [ ]:
%pip install ImageHash

In [ ]:
import imagehash
print("ImageHash installed successfully:", imagehash.__version__)

In [ ]:
import imagehash
import zipfile
import os
import io
import pandas as pd
from PIL import Image
from tqdm.auto import tqdm

# --------------------------------------------------
# 1. Calculate perceptual hashes
# --------------------------------------------------

def calculate_phashes(df, split_name):

    hashes = []

    zip_handles = {}

    for _, row in tqdm(
        df.iterrows(),
        total=len(df),
        desc=f"Hashing {split_name}"
    ):

        zip_name = row["zip_file"]
        image_name = row["image_name"]

        if zip_name not in zip_handles:
            zip_path = os.path.join(
                train_loader.dataset.zip_dir,
                zip_name
            )
            zip_handles[zip_name] = zipfile.ZipFile(
                zip_path, "r"
            )

        with zip_handles[zip_name].open(image_name) as f:
            image = Image.open(
                io.BytesIO(f.read())
            ).convert("RGB")

        phash = imagehash.phash(image)

        hashes.append({
            "class": row["class"],
            "image_name": image_name,
            "zip_file": zip_name,
            "phash_int": int(str(phash), 16),
            "split": split_name
        })

    for zf in zip_handles.values():
        zf.close()

    return pd.DataFrame(hashes)


train_phashes = calculate_phashes(
    train_loader.dataset.dataframe,
    "Training"
)

val_phashes = calculate_phashes(
    val_loader.dataset.dataframe,
    "Validation"
)

print("\nTraining hashes:", len(train_phashes))
print("Validation hashes:", len(val_phashes))


# --------------------------------------------------
# 2. Compare hashes across splits
# --------------------------------------------------

near_duplicate_pairs = []

for train_idx, train_row in tqdm(
    train_phashes.iterrows(),
    total=len(train_phashes),
    desc="Comparing cross-split hashes"
):

    train_hash = train_row["phash_int"]

    for val_idx, val_row in val_phashes.iterrows():

        distance = (
            train_hash ^ val_row["phash_int"]
        ).bit_count()

        if distance <= 10:

            near_duplicate_pairs.append({
                "train_class": train_row["class"],
                "train_image": train_row["image_name"],
                "val_class": val_row["class"],
                "val_image": val_row["image_name"],
                "hamming_distance": distance
            })


near_duplicate_df = pd.DataFrame(
    near_duplicate_pairs
)

print("\nNear-duplicate candidate pairs:", len(near_duplicate_df))

if len(near_duplicate_df) > 0:

    print("\nDistance distribution:")
    print(
        near_duplicate_df["hamming_distance"]
        .value_counts()
        .sort_index()
    )

    print("\nClosest candidate pairs:")
    display(
        near_duplicate_df.sort_values(
            "hamming_distance"
        ).head(20)
    )

else:
    print("No pairs found within Hamming distance 10.")

In [ ]:
import os
import pandas as pd

results_dir = "/content/drive/MyDrive/Marine_Project/results"

# Preserve existing files: choose a new filename if necessary
base_name = "phash_train_validation_candidates"
csv_path = os.path.join(results_dir, base_name + ".csv")

version = 1
while os.path.exists(csv_path):
    csv_path = os.path.join(
        results_dir, f"{base_name}_v{version}.csv"
    )
    version += 1

# Save all candidate pairs
near_duplicate_df.to_csv(csv_path, index=False)

print("Saved candidate report:", csv_path)
print("Total candidate pairs:", len(near_duplicate_df))

print("\nClass-pair distribution:")
print(
    near_duplicate_df.groupby(
        ["train_class", "val_class"]
    ).size().sort_values(ascending=False)
)

print("\nSame-class vs cross-class candidates:")
print(
    "Same class:",
    (near_duplicate_df["train_class"] ==
     near_duplicate_df["val_class"]).sum()
)
print(
    "Cross class:",
    (near_duplicate_df["train_class"] !=
     near_duplicate_df["val_class"]).sum()
)

print("\nUnique training images involved:",
      near_duplicate_df["train_image"].nunique())

print("Unique validation images involved:",
      near_duplicate_df["val_image"].nunique())

In [ ]:
print("Training hash columns:")
print(train_phashes.columns.tolist())

print("\nValidation hash columns:")
print(val_phashes.columns.tolist())

print("\nCandidate pair columns:")
print(near_duplicate_df.columns.tolist())

print("\nTraining hash sample:")
display(train_phashes.head(3))

print("\nValidation hash sample:")
display(val_phashes.head(3))

print("\nCandidate pair sample:")
display(near_duplicate_df.head(3))

In [ ]:
import os
import zipfile
import io
import matplotlib.pyplot as plt
from PIL import Image

zip_dir = "/content/drive/MyDrive/Marine_Project/dataset_zips"

# Select 8 candidates with identical perceptual hashes
inspection_df = (
    near_duplicate_df[
        near_duplicate_df["hamming_distance"] == 0
    ]
    .head(8)
    .reset_index(drop=True)
)

# Open each required ZIP archive only once
zip_cache = {}

def get_image(row, split):
    if split == "Training":
        source_df = train_phashes
    else:
        source_df = val_phashes

    match = source_df[
        (source_df["class"] == row["train_class" if split == "Training" else "val_class"]) &
        (source_df["image_name"] == row["train_image" if split == "Training" else "val_image"])
    ]

    if len(match) != 1:
        raise ValueError(f"Expected one matching image, found {len(match)}")

    zip_name = match.iloc[0]["zip_file"]

    if zip_name not in zip_cache:
        zip_cache[zip_name] = zipfile.ZipFile(
            os.path.join(zip_dir, zip_name), "r"
        )

    image_path = row["train_image" if split == "Training" else "val_image"]

    image_bytes = zip_cache[zip_name].read(image_path)
    return Image.open(io.BytesIO(image_bytes)).convert("RGB")

fig, axes = plt.subplots(8, 2, figsize=(12, 32))

for i, row in inspection_df.iterrows():
    train_img = get_image(row, "Training")
    val_img = get_image(row, "Validation")

    axes[i, 0].imshow(train_img)
    axes[i, 0].set_title(
        f"TRAIN | {row['train_class']}\n{row['train_image']}",
        fontsize=8
    )
    axes[i, 0].axis("off")

    axes[i, 1].imshow(val_img)
    axes[i, 1].set_title(
        f"VALIDATION | Distance: {row['hamming_distance']}\n{row['val_image']}",
        fontsize=8
    )
    axes[i, 1].axis("off")

plt.tight_layout()
plt.show()

for z in zip_cache.values():
    z.close()

In [ ]:
import os
import zipfile
import io
import numpy as np
import pandas as pd
from PIL import Image
from skimage.metrics import structural_similarity

zip_dir = "/content/drive/MyDrive/Marine_Project/dataset_zips"

pairs_to_check = (
    near_duplicate_df[
        near_duplicate_df["hamming_distance"] == 0
    ].head(8).reset_index(drop=True)
)

zip_cache = {}

def load_original_image(image_name, class_name, hash_df):
    match = hash_df[
        (hash_df["class"] == class_name) &
        (hash_df["image_name"] == image_name)
    ]

    if len(match) != 1:
        raise ValueError(f"Image lookup issue: {image_name}")

    zip_name = match.iloc[0]["zip_file"]

    if zip_name not in zip_cache:
        zip_cache[zip_name] = zipfile.ZipFile(
            os.path.join(zip_dir, zip_name), "r"
        )

    image_bytes = zip_cache[zip_name].read(image_name)
    return Image.open(io.BytesIO(image_bytes)).convert("RGB")

verification_results = []

for _, pair in pairs_to_check.iterrows():

    train_img = load_original_image(
        pair["train_image"], pair["train_class"], train_phashes
    )

    val_img = load_original_image(
        pair["val_image"], pair["val_class"], val_phashes
    )

    train_arr = np.array(train_img)
    val_arr = np.array(val_img)

    same_dimensions = train_arr.shape == val_arr.shape

    exact_pixel_match = (
        same_dimensions and np.array_equal(train_arr, val_arr)
    )

    # Resize both images to a common size for SSIM comparison
    common_size = (min(train_img.width, val_img.width),
                   min(train_img.height, val_img.height))

    train_resized = np.array(train_img.resize(common_size))
    val_resized = np.array(val_img.resize(common_size))

    ssim_score = structural_similarity(
        train_resized,
        val_resized,
        channel_axis=2,
        data_range=255
    )

    verification_results.append({
        "train_image": pair["train_image"],
        "val_image": pair["val_image"],
        "train_dimensions": str(train_arr.shape),
        "val_dimensions": str(val_arr.shape),
        "same_dimensions": same_dimensions,
        "exact_pixel_match": exact_pixel_match,
        "SSIM": round(float(ssim_score), 5)
    })

verification_df = pd.DataFrame(verification_results)

display(verification_df)

for z in zip_cache.values():
    z.close()

In [ ]:
import cv2
import numpy as np
import pandas as pd
import os
import zipfile
import io
from PIL import Image

zip_dir = "/content/drive/MyDrive/Marine_Project/dataset_zips"
zip_cache_sift = {}

def load_image_for_sift(image_name, class_name, hash_df):

    match = hash_df[
        (hash_df["class"] == class_name) &
        (hash_df["image_name"] == image_name)
    ]

    zip_name = match.iloc[0]["zip_file"]

    if zip_name not in zip_cache_sift:
        zip_cache_sift[zip_name] = zipfile.ZipFile(
            os.path.join(zip_dir, zip_name), "r"
        )

    image_bytes = zip_cache_sift[zip_name].read(image_name)

    return np.array(
        Image.open(io.BytesIO(image_bytes)).convert("RGB")
    )

sift = cv2.SIFT_create(nfeatures=3000)
matcher = cv2.BFMatcher(cv2.NORM_L2)

sift_results = []

for _, pair in pairs_to_check.iterrows():

    img1 = load_image_for_sift(
        pair["train_image"], pair["train_class"], train_phashes
    )

    img2 = load_image_for_sift(
        pair["val_image"], pair["val_class"], val_phashes
    )

    # Resize while preserving aspect ratio
    def resize_image(img, max_dim=1200):
        h, w = img.shape[:2]
        scale = max_dim / max(h, w)
        return cv2.resize(
            img, (int(w * scale), int(h * scale))
        )

    gray1 = cv2.cvtColor(
        resize_image(img1), cv2.COLOR_RGB2GRAY
    )
    gray2 = cv2.cvtColor(
        resize_image(img2), cv2.COLOR_RGB2GRAY
    )

    kp1, des1 = sift.detectAndCompute(gray1, None)
    kp2, des2 = sift.detectAndCompute(gray2, None)

    good_matches = []

    if des1 is not None and des2 is not None:
        matches = matcher.knnMatch(des1, des2, k=2)

        good_matches = [
            m for m, n in matches
            if m.distance < 0.70 * n.distance
        ]

    inliers = 0
    inlier_ratio = 0.0

    if len(good_matches) >= 4:

        src_pts = np.float32(
            [kp1[m.queryIdx].pt for m in good_matches]
        ).reshape(-1, 1, 2)

        dst_pts = np.float32(
            [kp2[m.trainIdx].pt for m in good_matches]
        ).reshape(-1, 1, 2)

        H, mask = cv2.findHomography(
            src_pts, dst_pts, cv2.RANSAC, 5.0
        )

        if mask is not None:
            inliers = int(mask.sum())
            inlier_ratio = inliers / len(good_matches)

    sift_results.append({
        "train_image": pair["train_image"],
        "val_image": pair["val_image"],
        "train_keypoints": len(kp1),
        "val_keypoints": len(kp2),
        "good_matches": len(good_matches),
        "homography_inliers": inliers,
        "inlier_ratio": round(inlier_ratio, 4)
    })

sift_df = pd.DataFrame(sift_results)

display(sift_df)

for z in zip_cache_sift.values():
    z.close()

In [ ]:
# ============================================================
# COMPLETE SIFT VERIFICATION — ALL 137 pHash CANDIDATE PAIRS
# ============================================================

import os
import zipfile
import cv2
import numpy as np
import pandas as pd
import io

from PIL import Image
from tqdm.auto import tqdm

# ------------------------------------------------------------
# 1. PATHS AND INPUT VALIDATION
# ------------------------------------------------------------

zip_dir = "/content/drive/MyDrive/Marine_Project/dataset_zips"

results_dir = "/content/drive/MyDrive/Marine_Project/results"

assert len(near_duplicate_df) == 137, (
    f"Expected 137 candidates, found {len(near_duplicate_df)}"
)

assert len(train_phashes) > 0
assert len(val_phashes) > 0

# ------------------------------------------------------------
# 2. INITIALIZE SIFT AND MATCHER
# ------------------------------------------------------------

sift = cv2.SIFT_create(nfeatures=3000)

matcher = cv2.BFMatcher(cv2.NORM_L2)

# ------------------------------------------------------------
# 3. ZIP FILE CACHE
# ------------------------------------------------------------

zip_cache = {}

def load_original_image(image_name, class_name, hash_df):

    match = hash_df[
        (hash_df["class"] == class_name) &
        (hash_df["image_name"] == image_name)
    ]

    if len(match) != 1:
        raise ValueError(
            f"Image lookup failed: {image_name}, matches={len(match)}"
        )

    zip_name = match.iloc[0]["zip_file"]

    if zip_name not in zip_cache:

        archive_path = os.path.join(zip_dir, zip_name)

        if not os.path.exists(archive_path):
            raise FileNotFoundError(archive_path)

        zip_cache[zip_name] = zipfile.ZipFile(
            archive_path, "r"
        )

    image_bytes = zip_cache[zip_name].read(image_name)

    return np.array(
        Image.open(io.BytesIO(image_bytes)).convert("RGB")
    )

# ------------------------------------------------------------
# 4. RESIZE FUNCTION — PRESERVE ASPECT RATIO
# ------------------------------------------------------------

def resize_for_sift(image, max_dim=1200):

    height, width = image.shape[:2]

    scale = max_dim / max(height, width)

    new_width = max(1, int(width * scale))
    new_height = max(1, int(height * scale))

    return cv2.resize(
        image,
        (new_width, new_height),
        interpolation=cv2.INTER_AREA
    )

# ------------------------------------------------------------
# 5. SIFT VERIFICATION FOR ALL 137 PAIRS
# ------------------------------------------------------------

all_sift_results = []

try:

    for idx, pair in tqdm(
        near_duplicate_df.reset_index(drop=True).iterrows(),
        total=len(near_duplicate_df),
        desc="SIFT Verification"
    ):

        # Load original images
        img1 = load_original_image(
            pair["train_image"],
            pair["train_class"],
            train_phashes
        )

        img2 = load_original_image(
            pair["val_image"],
            pair["val_class"],
            val_phashes
        )

        # Resize and convert to grayscale
        gray1 = cv2.cvtColor(
            resize_for_sift(img1),
            cv2.COLOR_RGB2GRAY
        )

        gray2 = cv2.cvtColor(
            resize_for_sift(img2),
            cv2.COLOR_RGB2GRAY
        )

        # Extract SIFT features
        kp1, des1 = sift.detectAndCompute(gray1, None)
        kp2, des2 = sift.detectAndCompute(gray2, None)

        good_matches = []

        # Feature matching
        if (
            des1 is not None and
            des2 is not None and
            len(des1) >= 2 and
            len(des2) >= 2
        ):

            matches = matcher.knnMatch(
                des1,
                des2,
                k=2
            )

            # Safely handle entries containing fewer than 2 matches
            for neighbors in matches:

                if len(neighbors) == 2:

                    m, n = neighbors

                    if m.distance < 0.70 * n.distance:
                        good_matches.append(m)

        # Homography estimation
        inliers = 0
        inlier_ratio = 0.0
        homography_found = False

        if len(good_matches) >= 4:

            src_pts = np.float32([
                kp1[m.queryIdx].pt
                for m in good_matches
            ]).reshape(-1, 1, 2)

            dst_pts = np.float32([
                kp2[m.trainIdx].pt
                for m in good_matches
            ]).reshape(-1, 1, 2)

            H, mask = cv2.findHomography(
                src_pts,
                dst_pts,
                cv2.RANSAC,
                5.0
            )

            if H is not None and mask is not None:

                homography_found = True

                inliers = int(mask.sum())

                inlier_ratio = (
                    inliers / len(good_matches)
                )

        # Store results
        all_sift_results.append({

            "train_image": pair["train_image"],
            "val_image": pair["val_image"],

            "train_class": pair["train_class"],
            "val_class": pair["val_class"],

            "phash_distance": int(
                pair["hamming_distance"]
            ),

            "train_keypoints": len(kp1),
            "val_keypoints": len(kp2),

            "good_matches": len(good_matches),

            "homography_found": homography_found,

            "homography_inliers": inliers,

            "inlier_ratio": round(
                inlier_ratio, 5
            )
        })

finally:

    # Close all ZIP archives safely
    for archive in zip_cache.values():
        archive.close()

    zip_cache.clear()

# ------------------------------------------------------------
# 6. CREATE FINAL RESULTS DATAFRAME
# ------------------------------------------------------------

all_sift_df = pd.DataFrame(all_sift_results)

# ------------------------------------------------------------
# 7. SAVE WITHOUT OVERWRITING PREVIOUS RESULTS
# ------------------------------------------------------------

base_name = "phash_all_137_sift_verification"

save_path = os.path.join(
    results_dir,
    base_name + ".csv"
)

version = 1

while os.path.exists(save_path):

    save_path = os.path.join(
        results_dir,
        f"{base_name}_v{version}.csv"
    )

    version += 1

all_sift_df.to_csv(
    save_path,
    index=False
)

# ------------------------------------------------------------
# 8. SUMMARY
# ------------------------------------------------------------

print("\n========== SIFT VERIFICATION SUMMARY ==========")

print("Total candidate pairs:", len(all_sift_df))

print(
    "Pairs with homography:",
    all_sift_df["homography_found"].sum()
)

print(
    "Pairs with 100+ inliers:",
    (all_sift_df["homography_inliers"] >= 100).sum()
)

print(
    "Pairs with 1000+ inliers:",
    (all_sift_df["homography_inliers"] >= 1000).sum()
)

print(
    "Average good matches:",
    round(all_sift_df["good_matches"].mean(), 2)
)

print(
    "Average homography inliers:",
    round(all_sift_df["homography_inliers"].mean(), 2)
)

print("\nResults saved at:")
print(save_path)

print("\n========== TOP 20 MATCHES ==========")

display(
    all_sift_df.sort_values(
        ["homography_inliers", "inlier_ratio"],
        ascending=False
    ).head(20)
)

print("\n========== LOWEST 20 MATCHES ==========")

display(
    all_sift_df.sort_values(
        ["homography_inliers", "inlier_ratio"],
        ascending=True
    ).head(20)
)

In [ ]:
import pandas as pd
import numpy as np

sift_path = (
    "/content/drive/MyDrive/Marine_Project/results/"
    "phash_all_137_sift_verification.csv"
)

sift_df = pd.read_csv(sift_path)

# Review categories: deliberately conservative
conditions = [
    (sift_df["homography_inliers"] >= 1000) &
    (sift_df["inlier_ratio"] >= 0.90),

    (sift_df["homography_inliers"] >= 100) &
    (sift_df["inlier_ratio"] >= 0.80),

    (sift_df["homography_found"] == True),

    (sift_df["good_matches"] > 0)
]

labels = [
    "Very strong correspondence",
    "Strong correspondence",
    "Homography found - inspect",
    "Weak feature evidence"
]

sift_df["review_category"] = np.select(
    conditions,
    labels,
    default="No useful SIFT evidence"
)

print("========== REVIEW CATEGORY COUNTS ==========")
print(sift_df["review_category"].value_counts().to_string())

print("\n========== CATEGORY BY CLASS ==========")
print(pd.crosstab(
    sift_df["train_class"],
    sift_df["review_category"]
).to_string())

print("\n========== STRONGEST CANDIDATES ==========")

strong = sift_df[
    (sift_df["homography_inliers"] >= 100) &
    (sift_df["inlier_ratio"] >= 0.80)
].sort_values(
    ["homography_inliers", "inlier_ratio"],
    ascending=False
)

display(strong[[
    "train_image",
    "val_image",
    "train_class",
    "phash_distance",
    "good_matches",
    "homography_inliers",
    "inlier_ratio"
]].head(25))

print("\nTotal candidate pairs:", len(sift_df))
print("Strong correspondence candidates:", len(strong))

In [ ]:
import pandas as pd

sift_path = (
    "/content/drive/MyDrive/Marine_Project/results/"
    "phash_all_137_sift_verification.csv"
)

sift_df = pd.read_csv(sift_path)

print("Shape:", sift_df.shape)
print("\nActual column names:")
print(sift_df.columns.tolist())

print("\nFirst 5 rows:")
display(sift_df.head())

print("\nData types:")
print(sift_df.dtypes)

In [ ]:
import pandas as pd
from collections import defaultdict

sift_df = pd.read_csv(
    "/content/drive/MyDrive/Marine_Project/results/"
    "phash_all_137_sift_verification.csv"
)

# Recreate review categories
def assign_category(row):
    if row["homography_found"]:
        if row["homography_inliers"] >= 1000:
            return "Very strong correspondence"
        elif row["homography_inliers"] >= 100:
            return "Strong correspondence"
        else:
            return "Homography found - inspect"

    elif row["good_matches"] > 0:
        return "Weak feature evidence"

    return "No useful SIFT evidence"


sift_df["review_category"] = sift_df.apply(
    assign_category, axis=1
)

# Select strong candidates
strong_df = sift_df[
    sift_df["review_category"].isin([
        "Very strong correspondence",
        "Strong correspondence"
    ])
].copy()

# Build graph identifiers
strong_df["train_node"] = "TRAIN::" + strong_df["train_image"]
strong_df["val_node"] = "VAL::" + strong_df["val_image"]

# Union-Find
parent = {}

def find(x):
    parent.setdefault(x, x)

    if parent[x] != x:
        parent[x] = find(parent[x])

    return parent[x]


def union(a, b):
    root_a = find(a)
    root_b = find(b)

    if root_a != root_b:
        parent[root_b] = root_a


for _, row in strong_df.iterrows():
    union(row["train_node"], row["val_node"])


# Build connected components
components = defaultdict(list)

for node in parent:
    components[find(node)].append(node)


# Display summary
print("========== STRONG MATCH GROUP ANALYSIS ==========")

print("Strong candidate pairs:", len(strong_df))
print("Unique training images:", strong_df["train_image"].nunique())
print("Unique validation images:", strong_df["val_image"].nunique())
print("Connected groups:", len(components))

# Group details
group_rows = []

for group_id, nodes in enumerate(
    sorted(components.values(), key=len, reverse=True),
    start=1
):
    train_images = [
        n.replace("TRAIN::", "")
        for n in nodes if n.startswith("TRAIN::")
    ]

    val_images = [
        n.replace("VAL::", "")
        for n in nodes if n.startswith("VAL::")
    ]

    group_rows.append({
        "group_id": group_id,
        "train_images": len(train_images),
        "validation_images": len(val_images),
        "total_images": len(nodes),
        "train_files": train_images,
        "validation_files": val_images
    })


group_df = pd.DataFrame(group_rows)

print("\n========== GROUP DETAILS ==========")

display(
    group_df[
        ["group_id", "train_images", "validation_images", "total_images"]
    ]
)

print("\n========== LARGEST GROUPS ==========")

display(group_df.head(10))

In [ ]:
import os
import glob
import zipfile
import matplotlib.pyplot as plt
from PIL import Image
from io import BytesIO

zip_dir = "/content/drive/MyDrive/Marine_Project/dataset_zips/"

review_pairs = strong_df.head(20).reset_index(drop=True)

zip_cache = {}

def get_image_from_zip(image_path, split):
    class_name, image_name = image_path.split("/")

    # Find the corresponding class and split ZIP
    pattern = os.path.join(
        zip_dir,
        f"ECU-MSS-2 {class_name}_{split}*.zip"
    )

    matches = glob.glob(pattern)

    if len(matches) != 1:
        raise FileNotFoundError(
            f"Expected one ZIP for {class_name}/{split}, found: {matches}"
        )

    zip_path = matches[0]

    if zip_path not in zip_cache:
        zip_cache[zip_path] = zipfile.ZipFile(zip_path)

    zf = zip_cache[zip_path]

    members = [
        name for name in zf.namelist()
        if name.endswith("/" + image_name) or name == image_name
    ]

    if len(members) != 1:
        raise FileNotFoundError(
            f"Expected one image {image_name}, found {len(members)}"
        )

    with zf.open(members[0]) as f:
        return Image.open(BytesIO(f.read())).convert("RGB")


try:
    for start in range(0, len(review_pairs), 5):
        subset = review_pairs.iloc[start:start+5]

        fig, axes = plt.subplots(
            len(subset), 2,
            figsize=(12, 4 * len(subset))
        )

        if len(subset) == 1:
            axes = [axes]

        for ax_row, (_, row) in zip(axes, subset.iterrows()):

            train_img = get_image_from_zip(
                row["train_image"], "training"
            )

            val_img = get_image_from_zip(
                row["val_image"], "training"
            )

            ax_row[0].imshow(train_img)
            ax_row[0].set_title(
                "TRAIN: " + row["train_image"].split("/")[-1]
            )
            ax_row[0].axis("off")

            ax_row[1].imshow(val_img)
            ax_row[1].set_title(
                "VALIDATION: " + row["val_image"].split("/")[-1]
            )
            ax_row[1].axis("off")

        plt.tight_layout()
        plt.show()

finally:
    for zf in zip_cache.values():
        zf.close()

    zip_cache.clear()

In [ ]:
import matplotlib.pyplot as plt

# Review pairs 21–40 (zero-based rows 20–39)
review_pairs_next = strong_df.iloc[20:40].reset_index(drop=True)

print("Reviewing pairs:", len(review_pairs_next))

try:
    for start in range(0, len(review_pairs_next), 5):
        subset = review_pairs_next.iloc[start:start+5]

        fig, axes = plt.subplots(
            len(subset), 2,
            figsize=(12, 4 * len(subset))
        )

        if len(subset) == 1:
            axes = [axes]

        for ax_row, (_, row) in zip(axes, subset.iterrows()):

            train_img = get_image_from_zip(
                row["train_image"], "training"
            )

            val_img = get_image_from_zip(
                row["val_image"], "training"
            )

            ax_row[0].imshow(train_img)
            ax_row[0].set_title(
                "TRAIN: " + row["train_image"].split("/")[-1]
            )
            ax_row[0].axis("off")

            ax_row[1].imshow(val_img)
            ax_row[1].set_title(
                "VALIDATION: " + row["val_image"].split("/")[-1]
            )
            ax_row[1].axis("off")

        plt.tight_layout()
        plt.show()

finally:
    for zf in zip_cache.values():
        zf.close()

    zip_cache.clear()

In [ ]:
review_pairs_next = strong_df.iloc[40:60].reset_index(drop=True)

print("Reviewing pairs:", len(review_pairs_next))

try:
    for start in range(0, len(review_pairs_next), 5):
        subset = review_pairs_next.iloc[start:start+5]

        fig, axes = plt.subplots(
            len(subset), 2,
            figsize=(12, 4 * len(subset))
        )

        if len(subset) == 1:
            axes = [axes]

        for ax_row, (_, row) in zip(axes, subset.iterrows()):

            train_img = get_image_from_zip(
                row["train_image"], "training"
            )

            val_img = get_image_from_zip(
                row["val_image"], "training"
            )

            ax_row[0].imshow(train_img)
            ax_row[0].set_title(
                "TRAIN: " + row["train_image"].split("/")[-1]
            )
            ax_row[0].axis("off")

            ax_row[1].imshow(val_img)
            ax_row[1].set_title(
                "VALIDATION: " + row["val_image"].split("/")[-1]
            )
            ax_row[1].axis("off")

        plt.tight_layout()
        plt.show()

finally:
    for zf in zip_cache.values():
        zf.close()

    zip_cache.clear()

In [ ]:
review_pairs_final = strong_df.iloc[60:70].reset_index(drop=True)

print("Final pairs:", len(review_pairs_final))

fig, axes = plt.subplots(
    len(review_pairs_final), 2,
    figsize=(12, 4 * len(review_pairs_final))
)

try:
    for i, (_, row) in enumerate(review_pairs_final.iterrows()):

        train_img = get_image_from_zip(
            row["train_image"], "training"
        )

        val_img = get_image_from_zip(
            row["val_image"], "training"
        )

        axes[i, 0].imshow(train_img)
        axes[i, 0].set_title(
            "TRAIN: " + row["train_image"].split("/")[-1]
        )
        axes[i, 0].axis("off")

        axes[i, 1].imshow(val_img)
        axes[i, 1].set_title(
            "VALIDATION: " + row["val_image"].split("/")[-1]
        )
        axes[i, 1].axis("off")

    plt.tight_layout()
    plt.show()

finally:
    for zf in zip_cache.values():
        zf.close()
    zip_cache.clear()

In [ ]:
import pandas as pd
import os

results_dir = "/content/drive/MyDrive/Marine_Project/results/"

train_path = os.path.join(results_dir, "clean_train_manifest.csv")
val_path = os.path.join(results_dir, "clean_validation_manifest.csv")

train_manifest = pd.read_csv(train_path)
val_manifest = pd.read_csv(val_path)

print("========== MANIFEST STRUCTURE ==========")

print("\nTraining shape:", train_manifest.shape)
print("Training columns:", train_manifest.columns.tolist())

print("\nValidation shape:", val_manifest.shape)
print("Validation columns:", val_manifest.columns.tolist())

print("\nTraining sample:")
display(train_manifest.head(3))

print("\nValidation sample:")
display(val_manifest.head(3))

print("\nClass distributions:")
print("\nTraining:")
print(train_manifest["class"].value_counts())

print("\nValidation:")
print(val_manifest["class"].value_counts())

In [ ]:
# Select the 28 candidates requiring further inspection
inspect_df = sift_df[
    (sift_df["homography_found"] == True) &
    (sift_df["homography_inliers"] < 100)
].copy()

print("Candidates requiring inspection:", len(inspect_df))

display(
    inspect_df[
        [
            "train_image",
            "val_image",
            "train_class",
            "phash_distance",
            "good_matches",
            "homography_inliers",
            "inlier_ratio"
        ]
    ].reset_index(drop=True)
)

In [ ]:
import matplotlib.pyplot as plt

review_batch = inspect_df.iloc[0:7].reset_index(drop=True)

fig, axes = plt.subplots(len(review_batch), 2, figsize=(12, 4 * len(review_batch)))

for i, row in review_batch.iterrows():
    train_img = get_image_from_zip(row["train_image"], "training")
    val_img = get_image_from_zip(row["val_image"], "training")

    axes[i, 0].imshow(train_img)
    axes[i, 0].set_title(f"Pair {i} — TRAIN\n{row['train_image'].split('/')[-1]}")
    axes[i, 0].axis("off")

    axes[i, 1].imshow(val_img)
    axes[i, 1].set_title(f"Pair {i} — VALIDATION\n{row['val_image'].split('/')[-1]}")
    axes[i, 1].axis("off")

plt.tight_layout()
plt.show()

In [ ]:
review_batch = inspect_df.iloc[7:14].reset_index(drop=True)

fig, axes = plt.subplots(len(review_batch), 2, figsize=(12, 4 * len(review_batch)))

for i, row in review_batch.iterrows():
    train_img = get_image_from_zip(row["train_image"], "training")
    val_img = get_image_from_zip(row["val_image"], "training")

    axes[i, 0].imshow(train_img)
    axes[i, 0].set_title(f"Pair {i+7} — TRAIN\n{row['train_image'].split('/')[-1]}")
    axes[i, 0].axis("off")

    axes[i, 1].imshow(val_img)
    axes[i, 1].set_title(f"Pair {i+7} — VALIDATION\n{row['val_image'].split('/')[-1]}")
    axes[i, 1].axis("off")

plt.tight_layout()
plt.show()

In [ ]:
review_batch = inspect_df.iloc[14:21].reset_index(drop=True)

fig, axes = plt.subplots(len(review_batch), 2, figsize=(12, 4 * len(review_batch)))

for i, row in review_batch.iterrows():
    train_img = get_image_from_zip(row["train_image"], "training")
    val_img = get_image_from_zip(row["val_image"], "training")

    axes[i, 0].imshow(train_img)
    axes[i, 0].set_title(f"Pair {i+15} — TRAIN\n{row['train_image'].split('/')[-1]}")
    axes[i, 0].axis("off")

    axes[i, 1].imshow(val_img)
    axes[i, 1].set_title(f"Pair {i+15} — VALIDATION\n{row['val_image'].split('/')[-1]}")
    axes[i, 1].axis("off")

plt.tight_layout()
plt.show()

In [ ]:
review_batch = inspect_df.iloc[21:28].reset_index(drop=True)

fig, axes = plt.subplots(len(review_batch), 2, figsize=(12, 4 * len(review_batch)))

for i, row in review_batch.iterrows():
    train_img = get_image_from_zip(row["train_image"], "training")
    val_img = get_image_from_zip(row["val_image"], "training")

    axes[i, 0].imshow(train_img)
    axes[i, 0].set_title(f"Pair {i+22} — TRAIN\n{row['train_image'].split('/')[-1]}")
    axes[i, 0].axis("off")

    axes[i, 1].imshow(val_img)
    axes[i, 1].set_title(f"Pair {i+22} — VALIDATION\n{row['val_image'].split('/')[-1]}")
    axes[i, 1].axis("off")

plt.tight_layout()
plt.show()

In [ ]:
no_sift_df = sift_df[
    sift_df["homography_found"] == False
].copy()

print("No-homography candidates:", len(no_sift_df))

display(
    no_sift_df[
        [
            "train_image",
            "val_image",
            "train_class",
            "phash_distance",
            "good_matches",
            "homography_inliers"
        ]
    ].reset_index(drop=True)
)

In [ ]:
# Prioritize candidates with pHash distance <= 2
high_risk_df = (
    no_sift_df[no_sift_df["phash_distance"] <= 2]
    .sort_values(["phash_distance", "train_image"])
    .reset_index(drop=True)
)

print("High-risk candidates:", len(high_risk_df))

review_batch = high_risk_df.iloc[0:7].reset_index(drop=True)

fig, axes = plt.subplots(len(review_batch), 2, figsize=(12, 4 * len(review_batch)))

for i, row in review_batch.iterrows():
    train_img = get_image_from_zip(row["train_image"], "training")
    val_img = get_image_from_zip(row["val_image"], "training")

    axes[i, 0].imshow(train_img)
    axes[i, 0].set_title(f"TRAIN | pHash: {row['phash_distance']}")
    axes[i, 0].axis("off")

    axes[i, 1].imshow(val_img)
    axes[i, 1].set_title("VALIDATION")
    axes[i, 1].axis("off")

plt.tight_layout()
plt.show()

In [ ]:
review_batch = high_risk_df.iloc[7:14].reset_index(drop=True)

fig, axes = plt.subplots(len(review_batch), 2, figsize=(12, 4 * len(review_batch)))

for i, row in review_batch.iterrows():
    train_img = get_image_from_zip(row["train_image"], "training")
    val_img = get_image_from_zip(row["val_image"], "training")

    axes[i, 0].imshow(train_img)
    axes[i, 0].set_title(f"TRAIN | pHash: {row['phash_distance']}")
    axes[i, 0].axis("off")

    axes[i, 1].imshow(val_img)
    axes[i, 1].set_title("VALIDATION")
    axes[i, 1].axis("off")

plt.tight_layout()
plt.show()

In [ ]:
review_batch = high_risk_df.iloc[14:21].reset_index(drop=True)

fig, axes = plt.subplots(len(review_batch), 2, figsize=(12, 4 * len(review_batch)))

for i, row in review_batch.iterrows():
    train_img = get_image_from_zip(row["train_image"], "training")
    val_img = get_image_from_zip(row["val_image"], "training")

    axes[i, 0].imshow(train_img)
    axes[i, 0].set_title(f"TRAIN | pHash: {row['phash_distance']}")
    axes[i, 0].axis("off")

    axes[i, 1].imshow(val_img)
    axes[i, 1].set_title("VALIDATION")
    axes[i, 1].axis("off")

plt.tight_layout()
plt.show()

In [ ]:
review_batch = high_risk_df.iloc[21:28].reset_index(drop=True)

fig, axes = plt.subplots(len(review_batch), 2, figsize=(12, 4 * len(review_batch)))

for i, row in review_batch.iterrows():
    train_img = get_image_from_zip(row["train_image"], "training")
    val_img = get_image_from_zip(row["val_image"], "training")

    axes[i, 0].imshow(train_img)
    axes[i, 0].set_title(f"TRAIN | pHash: {row['phash_distance']}")
    axes[i, 0].axis("off")

    axes[i, 1].imshow(val_img)
    axes[i, 1].set_title("VALIDATION")
    axes[i, 1].axis("off")

plt.tight_layout()
plt.show()

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os

DRIVE_ROOT = "/content/drive/MyDrive/Marine_Project"

print("Drive mounted:", os.path.exists("/content/drive/MyDrive"))
print("Project folder exists:", os.path.exists(DRIVE_ROOT))

if os.path.exists(DRIVE_ROOT):
    print("\nProject folders:")
    print(os.listdir(DRIVE_ROOT))

In [ ]:
# ============================================================
# MARINE PROJECT — SESSION BACKUP BEFORE COLAB DISCONNECT
# ============================================================

import os
import json
import pandas as pd
from datetime import datetime

# Confirm Drive is mounted
DRIVE_ROOT = "/content/drive/MyDrive/Marine_Project"

if not os.path.exists(DRIVE_ROOT):
    raise FileNotFoundError(
        "Google Drive is not mounted. Mount Drive before running this cell."
    )

# Create a unique backup folder
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

BACKUP_DIR = os.path.join(
    DRIVE_ROOT,
    "session_checkpoints",
    f"audit_pause_{timestamp}"
)

os.makedirs(BACKUP_DIR, exist_ok=False)

print("Backup folder:", BACKUP_DIR)

# Save important DataFrames if they exist in this runtime
important_dfs = [
    "hash_df",
    "train_manifest",
    "val_manifest",
    "test_manifest",
    "sift_df",
    "inspect_df",
    "no_sift_df",
    "high_risk_df",
    "remaining_df",
    "quality_df",
]

saved_dataframes = []

for name in important_dfs:
    if name in globals() and isinstance(globals()[name], pd.DataFrame):
        df = globals()[name]

        # CSV for easy inspection and reuse
        csv_path = os.path.join(BACKUP_DIR, f"{name}.csv")
        df.to_csv(csv_path, index=False)

        # Pickle preserves DataFrame structure and data types
        pkl_path = os.path.join(BACKUP_DIR, f"{name}.pkl")
        df.to_pickle(pkl_path)

        saved_dataframes.append({
            "name": name,
            "rows": len(df),
            "columns": list(df.columns),
            "csv": csv_path,
            "pickle": pkl_path
        })

        print(f"Saved {name}: {df.shape}")

# Record current project progress
progress = {
    "backup_time": timestamp,
    "project_title": (
        "Fine-Grained Discrimination of Invasive and Native "
        "Marine Vegetation Using a Hybrid Deep Learning Model "
        "for Selective Underwater Plant Removal"
    ),
    "current_stage": "M1 - Data Audit",
    "completed": [
        "Original exact duplicate audit",
        "Clean test manifest created",
        "Group-aware clean train and validation manifests created",
        "Clean split ResNet18 baseline trained and evaluated",
        "Image quality analysis",
        "CLAHE inference comparison",
        "Source-class confounding audit",
        "pHash and SIFT near-duplicate screening",
        "98 SIFT-supported train-validation pairs visually inspected",
        "25 additional no-homography, pHash-distance <= 2 pairs visually inspected"
    ],
    "visual_inspection": {
        "very_strong_sift_pairs": 45,
        "strong_sift_pairs": 25,
        "homography_under_100_pairs": 28,
        "no_homography_phash_le_2_pairs": 25,
        "total_visually_confirmed_pairs": 123,
        "finding": (
            "All 123 inspected pairs were reported by the user "
            "to show the same underlying image content with "
            "different crops, orientations or layouts."
        )
    },
    "pending": [
        "Inspect remaining no-homography candidates with pHash distance > 2",
        "Complete perceptual duplicate grouping",
        "Create new group-aware train-validation manifests",
        "Re-evaluate model using new grouped split",
        "Begin M3 segmentation/localization",
        "Begin M4 invasive/native data acquisition and label preparation",
        "M5 invasive/native discrimination",
        "M6 selective removal and human-review pipeline"
    ],
    "saved_dataframes": saved_dataframes
}

progress_path = os.path.join(BACKUP_DIR, "session_progress.json")

with open(progress_path, "w") as f:
    json.dump(progress, f, indent=4, default=str)

# Human-readable continuation notes
notes = """
MARINE PROJECT — COLAB SESSION PAUSE
====================================

Current stage: M1 Data Audit.

IMPORTANT:
The existing M2 four-class ResNet18 baseline is COMPLETE.
Do not describe it as incomplete.

Dataset:
ECU-MSS-2
Classes: Amphibolis, Background, Halophila, Posidonia.

Existing clean manifests:
results/clean_train_manifest.csv
results/clean_validation_manifest.csv
results/clean_test_manifest.csv

Existing clean-split baseline checkpoint:
models/resnet18_clean_split_baseline_best.pth

Existing model results and audit files remain in the original
Marine_Project/models and Marine_Project/results folders.

Near-duplicate investigation:
123 train-validation candidate pairs have been visually
confirmed as the same underlying image content, despite
cropping, orientation or layout differences.

The remaining no-homography candidates with pHash distance > 2
have NOT yet been inspected.

Do not delete images or modify existing manifests.
Do not overwrite existing model checkpoints or results.

NEXT SESSION:
1. Mount Google Drive.
2. Restore DataFrames from this backup folder.
3. Continue the remaining candidate audit.
4. Build new group-aware manifests using new filenames.
5. Move toward segmentation and invasive/native data preparation.

Scientific limitation:
ECU-MSS-2 does not contain invasive/native labels.
Do not claim that its four classes represent invasive/native status.
External ecological labels/data are required for that research stage.
"""

notes_path = os.path.join(BACKUP_DIR, "CONTINUE_TOMORROW.txt")

with open(notes_path, "w") as f:
    f.write(notes)

# List existing model and result files without changing them
for folder_name in ["models", "results"]:
    folder_path = os.path.join(DRIVE_ROOT, folder_name)
    if os.path.exists(folder_path):
        files = sorted(os.listdir(folder_path))
        print(f"\nExisting {folder_name} files: {len(files)}")
        print("\n".join(files[:15]))

print("\n" + "=" * 55)
print("BACKUP COMPLETED")
print("Location:", BACKUP_DIR)
print("DataFrames saved:", len(saved_dataframes))
print("Progress note:", notes_path)
print("=" * 55)

In [ ]:
import os
import pandas as pd

BACKUP_DIR = (
    "/content/drive/MyDrive/Marine_Project/"
    "session_checkpoints/audit_pause_REPLACE_WITH_FOLDER_NAME"
)

for name in [
    "hash_df",
    "train_manifest",
    "val_manifest",
    "test_manifest",
    "sift_df",
    "inspect_df",
    "no_sift_df",
    "high_risk_df",
    "remaining_df",
    "quality_df"
]:
    path = os.path.join(BACKUP_DIR, f"{name}.pkl")

    if os.path.exists(path):
        globals()[name] = pd.read_pickle(path)
        print(f"Restored {name}: {globals()[name].shape}")

In [ ]:
audit_variables = [
    "train_manifest",
    "val_manifest",
    "sift_df",
    "no_sift_df",
    "high_risk_df",
    "remaining_df"
]

for name in audit_variables:
    if name in globals():
        obj = globals()[name]
        print(f"✅ {name}: {obj.shape}")
    else:
        print(f"❌ {name}: Not loaded")

In [ ]:
import os

checkpoint_root = (
    "/content/drive/MyDrive/Marine_Project/session_checkpoints"
)

print("Session checkpoint folders:")

for folder in sorted(os.listdir(checkpoint_root)):
    path = os.path.join(checkpoint_root, folder)

    if os.path.isdir(path):
        print("\n📁", folder)
        print("Files:", os.listdir(path))

In [ ]:
import os
import json

ROOT = "/content/drive/MyDrive/Marine_Project"

# Check what the latest backup recorded
progress_path = os.path.join(
    ROOT,
    "session_checkpoints",
    "audit_pause_20261114_055620",
    "session_progress.json"
)

with open(progress_path, "r") as f:
    progress = json.load(f)

print("Previously recorded DataFrames:")
print(progress.get("saved_dataframes", []))

# Locate original audit files
results_dir = os.path.join(ROOT, "results")

print("\nRelevant files in results folder:")

for filename in sorted(os.listdir(results_dir)):
    if any(word in filename.lower() for word in [
        "manifest", "phash", "sift", "duplicate", "split"
    ]):
        print(filename)

In [ ]:
import os
import pandas as pd

ROOT = "/content/drive/MyDrive/Marine_Project"
RESULTS = os.path.join(ROOT, "results")

# Restore original clean manifests
train_manifest = pd.read_csv(
    os.path.join(RESULTS, "clean_train_manifest.csv")
)

val_manifest = pd.read_csv(
    os.path.join(RESULTS, "clean_validation_manifest.csv")
)

test_manifest = pd.read_csv(
    os.path.join(RESULTS, "clean_test_manifest.csv")
)

# Restore perceptual hash and SIFT investigation
phash_df = pd.read_csv(
    os.path.join(RESULTS, "phash_train_validation_candidates.csv")
)

sift_df = pd.read_csv(
    os.path.join(RESULTS, "phash_all_137_sift_verification.csv")
)

print("=== RESTORED DATAFRAMES ===")
for name, df in [
    ("train_manifest", train_manifest),
    ("val_manifest", val_manifest),
    ("test_manifest", test_manifest),
    ("phash_df", phash_df),
    ("sift_df", sift_df)
]:
    print(f"{name}: {df.shape}")
    print("Columns:", list(df.columns))
    print()

print("=== SIFT HOMOGRAPHY DISTRIBUTION ===")
print(sift_df["homography_found"].value_counts(dropna=False))

print("\n=== PHASH DISTANCE DISTRIBUTION ===")
print(sift_df["phash_distance"].value_counts().sort_index())

In [ ]:
# Reconstruct candidate groups from the saved SIFT results

inspect_df = sift_df[
    (sift_df["homography_found"] == True) &
    (sift_df["homography_inliers"] < 100)
].copy().reset_index(drop=True)

no_sift_df = sift_df[
    sift_df["homography_found"] == False
].copy().reset_index(drop=True)

high_risk_df = (
    no_sift_df[no_sift_df["phash_distance"] <= 2]
    .sort_values(["phash_distance", "train_image"])
    .reset_index(drop=True)
)

remaining_df = (
    no_sift_df[no_sift_df["phash_distance"] > 2]
    .sort_values(["phash_distance", "train_class"])
    .reset_index(drop=True)
)

print("=== RECONSTRUCTED AUDIT ===")
print("Homography under 100 inliers:", len(inspect_df))
print("No-homography candidates:", len(no_sift_df))
print("Visually inspected high-risk candidates:", len(high_risk_df))
print("Remaining candidates:", len(remaining_df))

print("\nRemaining pHash distribution:")
print(remaining_df["phash_distance"].value_counts().sort_index())

print("\nRemaining class distribution:")
print(remaining_df["train_class"].value_counts())

display(
    remaining_df[
        [
            "train_image",
            "val_image",
            "train_class",
            "phash_distance",
            "good_matches"
        ]
    ].reset_index(drop=True)
)

In [ ]:
import zipfile
from PIL import Image
from io import BytesIO
import os

ZIP_DIR = "/content/drive/MyDrive/Marine_Project/dataset_zips"

# Open ZIP archives lazily and reuse their handles
zip_handles = {}

def get_image_from_zip(image_path, split="training"):
    zip_name = os.path.basename(image_path)

    # Find the archive containing this image
    matching_zips = [
        f for f in os.listdir(ZIP_DIR)
        if f.endswith(".zip")
        and split.lower() in f.lower()
        and zip_name.split("/")[0].lower() in f.lower()
    ]

    if not matching_zips:
        raise FileNotFoundError(f"No matching ZIP found for {image_path}")

    zip_path = os.path.join(ZIP_DIR, matching_zips[0])

    if zip_path not in zip_handles:
        zip_handles[zip_path] = zipfile.ZipFile(zip_path, "r")

    zf = zip_handles[zip_path]

    matches = [
        name for name in zf.namelist()
        if name.endswith("/" + zip_name)
        or name == zip_name
    ]

    if not matches:
        raise FileNotFoundError(f"{zip_name} not found in {matching_zips[0]}")

    with zf.open(matches[0]) as file:
        return Image.open(BytesIO(file.read())).convert("RGB")

print("Image-reading function restored.")

In [ ]:
import os

print("ZIP directory:", ZIP_DIR)
print("\nAvailable training ZIP files:")

for filename in sorted(os.listdir(ZIP_DIR)):
    if filename.lower().endswith(".zip") and "training" in filename.lower():
        print(filename)

In [ ]:
import zipfile
from PIL import Image
from io import BytesIO
import os

zip_handles = {}

def get_image_from_zip(image_path, split="training"):
    class_name = image_path.split("/")[0]

    zip_filename = f"ECU-MSS-2 {class_name}_{split}.zip"
    zip_path = os.path.join(ZIP_DIR, zip_filename)

    if not os.path.exists(zip_path):
        raise FileNotFoundError(f"ZIP not found: {zip_path}")

    if zip_path not in zip_handles:
        zip_handles[zip_path] = zipfile.ZipFile(zip_path, "r")

    zf = zip_handles[zip_path]

    image_name = image_path.split("/")[-1]

    matches = [
        name for name in zf.namelist()
        if name.endswith("/" + image_name) or name == image_name
    ]

    if not matches:
        raise FileNotFoundError(f"{image_name} not found in {zip_filename}")

    with zf.open(matches[0]) as file:
        return Image.open(BytesIO(file.read())).convert("RGB")

print("Corrected ZIP image loader successfully.")

In [ ]:
review_batch = remaining_df.iloc[0:7].reset_index(drop=True)

fig, axes = plt.subplots(
    len(review_batch), 2,
    figsize=(12, 4 * len(review_batch))
)

for i, row in review_batch.iterrows():
    train_img = get_image_from_zip(row["train_image"], "training")
    val_img = get_image_from_zip(row["val_image"], "training")

    axes[i, 0].imshow(train_img)
    axes[i, 0].set_title(f"Pair {i+1} TRAIN | pHash {row['phash_distance']}")
    axes[i, 0].axis("off")

    axes[i, 1].imshow(val_img)
    axes[i, 1].set_title(f"Pair {i+1} VALIDATION")
    axes[i, 1].axis("off")

plt.tight_layout()
plt.show()

In [ ]:
review_batch = remaining_df.iloc[7:14].reset_index(drop=True)

fig, axes = plt.subplots(
    len(review_batch), 2,
    figsize=(12, 4 * len(review_batch))
)

for i, row in review_batch.iterrows():
    train_img = get_image_from_zip(row["train_image"], "training")
    val_img = get_image_from_zip(row["val_image"], "training")

    axes[i, 0].imshow(train_img)
    axes[i, 0].set_title(f"Pair {i+8} TRAIN | pHash {row['phash_distance']}")
    axes[i, 0].axis("off")

    axes[i, 1].imshow(val_img)
    axes[i, 1].set_title(f"Pair {i+8} VALIDATION")
    axes[i, 1].axis("off")

plt.tight_layout()
plt.show()

In [ ]:
confirmed_pairs = sift_df.copy()

confirmed_pairs["visual_confirmed"] = True
confirmed_pairs["verification_method"] = "Manual visual inspection"
confirmed_pairs["verification_note"] = (
    "Same underlying image content despite possible "
    "cropping, orientation, resolution, or layout differences"
)

audit_path = os.path.join(
    RESULTS,
    "phash_137_visual_confirmation_audit.csv"
)

confirmed_pairs.to_csv(audit_path, index=False)

print("Visual confirmation audit saved:", audit_path)
print("Total confirmed pairs:", len(confirmed_pairs))
print("Unconfirmed pairs:", (~confirmed_pairs["visual_confirmed"]).sum())
print("\nClass distribution:")
print(confirmed_pairs["train_class"].value_counts())

In [ ]:
# Build connected components using all 137 visually confirmed pairs

parent = {}

def find(x):
    if x not in parent:
        parent[x] = x
    if parent[x] != x:
        parent[x] = find(parent[x])
    return parent[x]

def union(a, b):
    root_a = find(a)
    root_b = find(b)

    if root_a != root_b:
        parent[root_b] = root_a

for _, row in confirmed_pairs.iterrows():
    union(row["train_image"], row["val_image"])

# Collect connected components
components = {}

for image in parent:
    root = find(image)
    components.setdefault(root, []).append(image)

component_summary = pd.DataFrame([
    {
        "group_id": f"PG_{i+1:03d}",
        "number_of_images": len(images),
        "images": images
    }
    for i, images in enumerate(
        sorted(components.values(), key=lambda x: (-len(x), sorted(x)[0]))
    )
])

print("=== PERCEPTUAL DUPLICATE GROUP AUDIT ===")
print("Confirmed pair relationships:", len(confirmed_pairs))
print("Unique image nodes:", len(parent))
print("Connected duplicate groups:", len(component_summary))
print("Largest group size:", component_summary["number_of_images"].max())
print("\nGroup size distribution:")
print(component_summary["number_of_images"].value_counts().sort_index())
print("\nLargest groups:")
display(
    component_summary[
        ["group_id", "number_of_images"]
    ].head(20)
)

In [ ]:
# Verify perceptual groups against existing manifests

combined_manifest = pd.concat(
    [
        train_manifest.assign(original_split="train"),
        val_manifest.assign(original_split="validation")
    ],
    ignore_index=True
)

node_rows = []

for _, row in combined_manifest.iterrows():
    node_rows.append({
        "image_name": row["image_name"],
        "class": row["class"],
        "original_split": row["original_split"]
    })

manifest_lookup = pd.DataFrame(node_rows)

group_mapping = []

for group in component_summary.itertuples(index=False):
    for image in group.images:
        group_mapping.append({
            "image_name": image,
            "group_id": group.group_id
        })

group_mapping_df = pd.DataFrame(group_mapping)

verification = group_mapping_df.merge(
    manifest_lookup,
    on="image_name",
    how="left"
)

print("=== GROUP MEMBERSHIP VERIFICATION ===")
print("Graph image nodes:", len(group_mapping_df))
print("Matched manifest records:", verification["class"].notna().sum())
print("Unmatched image nodes:", verification["class"].isna().sum())
print("Duplicate image names in manifests:",
      manifest_lookup["image_name"].duplicated().sum())

print("\nOriginal split distribution:")
print(verification["original_split"].value_counts(dropna=False))

print("\nClass distribution:")
print(verification["class"].value_counts(dropna=False))

print("\nGroups containing multiple classes:")
print(
    verification.groupby("group_id")["class"]
    .nunique()
    .gt(1)
    .sum()
)

In [ ]:
# Prepare group IDs for every image in the clean training dataset

all_train_val = pd.concat(
    [train_manifest, val_manifest],
    ignore_index=True
).copy()

# Map the 269 visually confirmed image nodes to their graph groups
perceptual_lookup = dict(
    zip(group_mapping_df["image_name"], group_mapping_df["group_id"])
)

all_train_val["group_id"] = all_train_val["image_name"].map(
    perceptual_lookup
)

# Give every remaining image an independent singleton group
missing_group = all_train_val["group_id"].isna()

all_train_val.loc[missing_group, "group_id"] = (
    "SINGLE_" +
    all_train_val.loc[missing_group, "image_name"].astype(str)
)

print("=== COMPLETE GROUP MAPPING ===")
print("Total images:", len(all_train_val))
print("Perceptually grouped images:", (~missing_group).sum())
print("Independent singleton images:", missing_group.sum())
print("Total groups:", all_train_val["group_id"].nunique())
print("Missing group IDs:", all_train_val["group_id"].isna().sum())

print("\nClass distribution:")
print(all_train_val["class"].value_counts())

print("\nGroup size distribution (largest groups):")
print(
    all_train_val["group_id"]
    .value_counts()
    .value_counts()
    .sort_index()
    .tail(10)
)

In [ ]:
from sklearn.model_selection import StratifiedGroupKFold
import numpy as np

X = np.zeros(len(all_train_val))
y = all_train_val["class"].values
groups = all_train_val["group_id"].values

sgkf = StratifiedGroupKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

fold_results = []

for fold_number, (train_idx, val_idx) in enumerate(
    sgkf.split(X, y, groups), start=1
):
    train_part = all_train_val.iloc[train_idx]
    val_part = all_train_val.iloc[val_idx]

    overlap = set(train_part["group_id"]) & set(val_part["group_id"])

    fold_results.append({
        "fold": fold_number,
        "train_images": len(train_part),
        "validation_images": len(val_part),
        "validation_percentage": round(
            len(val_part) / len(all_train_val) * 100, 2
        ),
        "group_overlap": len(overlap),
        "validation_class_counts": val_part["class"].value_counts().to_dict()
    })

print("=== GROUP-AWARE SPLIT CANDIDATES ===")

for result in fold_results:
    print(result)

In [ ]:
sgkf = StratifiedGroupKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

for fold_number, (train_idx, val_idx) in enumerate(
    sgkf.split(X, y, groups), start=1
):
    if fold_number == 4:
        grouped_train = all_train_val.iloc[train_idx].copy().reset_index(drop=True)
        grouped_val = all_train_val.iloc[val_idx].copy().reset_index(drop=True)
        break

print("=== FOLD 4 VALIDATION ===")
print("Training images:", len(grouped_train))
print("Validation images:", len(grouped_val))

print("\nTraining class distribution:")
print(grouped_train["class"].value_counts().sort_index())

print("\nValidation class distribution:")
print(grouped_val["class"].value_counts().sort_index())

train_groups = set(grouped_train["group_id"])
val_groups = set(grouped_val["group_id"])

print("\nGroup overlap:", len(train_groups & val_groups))
print("Missing training group IDs:", grouped_train["group_id"].isna().sum())
print("Missing validation group IDs:", grouped_val["group_id"].isna().sum())

In [ ]:
# Final integrity checks — no files are modified

train_hashes = set(grouped_train["hash"])
val_hashes = set(grouped_val["hash"])
test_hashes = set(test_manifest["hash"])

checks = {
    "Training + validation total": (
        len(grouped_train) + len(grouped_val) == 4114
    ),
    "No duplicate training filenames": (
        not grouped_train["image_name"].duplicated().any()
    ),
    "No duplicate validation filenames": (
        not grouped_val["image_name"].duplicated().any()
    ),
    "No exact-hash overlap: train vs validation": (
        len(train_hashes & val_hashes) == 0
    ),
    "No exact-hash overlap: train vs test": (
        len(train_hashes & test_hashes) == 0
    ),
    "No exact-hash overlap: validation vs test": (
        len(val_hashes & test_hashes) == 0
    ),
    "No perceptual group overlap": (
        len(set(grouped_train["group_id"]) &
            set(grouped_val["group_id"])) == 0
    ),
    "All four classes in training": (
        grouped_train["class"].nunique() == 4
    ),
    "All four classes in validation": (
        grouped_val["class"].nunique() == 4
    ),
    "No mixed-class perceptual groups": (
        all_train_val.groupby("group_id")["class"].nunique().max() == 1
    ),
    "No missing hashes": (
        not all_train_val["hash"].isna().any()
    ),
}

print("=== FINAL M1 INTEGRITY CHECK ===")

for description, passed in checks.items():
    print(f"{'PASS' if passed else 'FAIL'} | {description}")

print("\nOverall:", "ALL CHECKS PASSED" if all(checks.values()) else "REVIEW REQUIRED")

In [ ]:
print("Available columns:")
print(grouped_train.columns.tolist())

print("\nTraining split values:")
print(grouped_train["split"].value_counts(dropna=False))

print("\nValidation split values:")
print(grouped_val["split"].value_counts(dropna=False))

print("\nTraining ZIP availability:")
print(grouped_train["zip_file"].nunique(), "unique ZIP files")

print("\nValidation ZIP availability:")
print(grouped_val["zip_file"].nunique(), "unique ZIP files")

In [ ]:
print("=== ORIGINAL SPLIT LABELS VS ACTUAL SPLIT ===")

print(pd.crosstab(
    all_train_val["split"],
    all_train_val["actual_split"],
    margins=True
))

print("\nTraining candidate:")
print(pd.crosstab(grouped_train["split"], grouped_train["actual_split"]))

print("\nValidation candidate:")
print(pd.crosstab(grouped_val["split"], grouped_val["actual_split"]))

print("\nZIP filename examples:")
print(all_train_val["zip_file"].drop_duplicates().tolist())

In [ ]:
# Create versioned, group-aware manifests
import os

grouped_train_final = grouped_train.copy()
grouped_val_final = grouped_val.copy()

grouped_train_final["new_split"] = "Training"
grouped_val_final["new_split"] = "Validation"

train_path = os.path.join(
    RESULTS, "clean_train_manifest_grouped_v2.csv"
)
val_path = os.path.join(
    RESULTS, "clean_validation_manifest_grouped_v2.csv"
)
groups_path = os.path.join(
    RESULTS, "perceptual_group_mapping_v2.csv"
)
audit_path = os.path.join(
    RESULTS, "phash_137_visual_confirmation_audit.csv"
)

# Preserve all existing files
output_paths = [train_path, val_path, groups_path]

existing = [p for p in output_paths if os.path.exists(p)]

if existing:
    raise FileExistsError(
        "Files already exist; no files were overwritten:\n" +
        "\n".join(existing)
    )

# Save new manifests
grouped_train_final.to_csv(train_path, index=False)
grouped_val_final.to_csv(val_path, index=False)

# Save complete group mapping
group_mapping_full = all_train_val[
    ["image_name", "class", "group_id", "hash", "zip_file"]
].copy()

group_mapping_full.to_csv(groups_path, index=False)

print("=== NEW MANIFESTS SAVED ===")
for path in output_paths:
    print(path, "|", os.path.getsize(path), "bytes")

print("\nTraining:", len(grouped_train_final))
print("Validation:", len(grouped_val_final))
print("Perceptual groups:", len(components))
print("Independent singleton images:", missing_group.sum())
print("Original manifests preserved.")

In [ ]:
import os
import pandas as pd

ROOT = "/content/drive/MyDrive/Marine_Project"
RESULTS = os.path.join(ROOT, "results")
MODEL_DIR = os.path.join(ROOT, "models")
ZIP_DIR = os.path.join(ROOT, "dataset_zips")

# Load into NEW variables; preserve existing DataFrames
m2_train = pd.read_csv(
    os.path.join(RESULTS, "clean_train_manifest_grouped_v2.csv")
)

m2_val = pd.read_csv(
    os.path.join(RESULTS, "clean_validation_manifest_grouped_v2.csv")
)

m2_test = pd.read_csv(
    os.path.join(RESULTS, "clean_test_manifest.csv")
)

print("=== M2 MANIFEST VERIFICATION ===")

print("Training:", m2_train.shape)
print("Validation:", m2_val.shape)
print("Test:", m2_test.shape)

print("\nTraining classes:")
print(m2_train["class"].value_counts().sort_index())

print("\nValidation classes:")
print(m2_val["class"].value_counts().sort_index())

print("\nTest classes:")
print(m2_test["class"].value_counts().sort_index())

print("\nNew split labels:")
print(m2_train["new_split"].unique())
print(m2_val["new_split"].unique())

print("\nExact hash overlap:")
print("Train vs validation:", len(
    set(m2_train["hash"]) & set(m2_val["hash"])
))
print("Train vs test:", len(
    set(m2_train["hash"]) & set(m2_test["hash"])
))
print("Validation vs test:", len(
    set(m2_val["hash"]) & set(m2_test["hash"])
))

print("\nZIP archives available:", len([
    f for f in os.listdir(ZIP_DIR) if f.endswith(".zip")
]))

print("\nManifest files loaded successfully.")

In [ ]:
import zipfile
import os

manifests = {
    "Training": m2_train,
    "Validation": m2_val,
    "Test": m2_test
}

zip_file_cache = {}
missing_files = []
missing_archives = []

for split_name, df in manifests.items():
    for zip_name, group in df.groupby("zip_file"):

        zip_path = os.path.join(ZIP_DIR, zip_name)

        if not os.path.exists(zip_path):
            missing_archives.append(zip_name)
            continue

        if zip_name not in zip_file_cache:
            with zipfile.ZipFile(zip_path, "r") as zf:
                zip_file_cache[zip_name] = set(zf.namelist())

        archive_files = zip_file_cache[zip_name]

        for image_path in group["image_name"]:
            expected = image_path.replace("\\", "/")

            matches = [
                name for name in archive_files
                if name.endswith("/" + expected)
                or name == expected
            ]

            if not matches:
                missing_files.append({
                    "split": split_name,
                    "zip_file": zip_name,
                    "image_name": image_path
                })

print("=== ZIP IMAGE VERIFICATION ===")
print("Total manifest records:", sum(len(df) for df in manifests.values()))
print("Missing ZIP archives:", len(missing_archives))
print("Missing images:", len(missing_files))

if missing_archives:
    print("\nMissing archives:", missing_archives)

if missing_files:
    print("\nFirst missing images:")
    print(missing_files[:10])

print("\nResult:",
      "PASS" if not missing_archives and not missing_files
      else "REVIEW REQUIRED")

In [ ]:
import torch
import torchvision
import numpy as np
import pandas as pd
import zipfile
from PIL import Image
from io import BytesIO
import os

print("=== M2 ENVIRONMENT CHECK ===")
print("PyTorch:", torch.__version__)
print("Torchvision:", torchvision.__version__)
print("NumPy:", np.__version__)
print("Pandas:", pd.__version__)

print("\nCUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("CUDA version:", torch.version.cuda)
    print("GPU memory (GB):",
          round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2))
else:
    print("GPU not detected. Do not start training yet.")

print("\nManifest variables loaded:",
      all(name in globals() for name in ["m2_train", "m2_val", "m2_test"]))

In [ ]:
import zipfile
from PIL import Image
from io import BytesIO
import os

class_mapping = {
    "Amphibolis": 0,
    "Background": 1,
    "Halophila": 2,
    "Posidonia": 3
}

class MarineZipDataset(torch.utils.data.Dataset):
    def __init__(self, dataframe, zip_dir, transform=None):
        self.dataframe = dataframe.reset_index(drop=True).copy()
        self.zip_dir = zip_dir
        self.transform = transform
        self.zip_handles = {}

        unknown = set(self.dataframe["class"]) - set(class_mapping)
        if unknown:
            raise ValueError(f"Unknown classes: {unknown}")

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, index):
        row = self.dataframe.iloc[index]

        zip_path = os.path.join(self.zip_dir, row["zip_file"])

        if zip_path not in self.zip_handles:
            self.zip_handles[zip_path] = zipfile.ZipFile(zip_path, "r")

        zf = self.zip_handles[zip_path]
        expected = str(row["image_name"]).replace("\\", "/")

        matches = [
            name for name in zf.namelist()
            if name == expected or name.endswith("/" + expected)
        ]

        if len(matches) != 1:
            raise FileNotFoundError(
                f"Expected one archive match, found {len(matches)}: "
                f"{expected} in {row['zip_file']}"
            )

        with zf.open(matches[0]) as file:
            image = Image.open(BytesIO(file.read())).convert("RGB")

        if self.transform:
            image = self.transform(image)

        label = class_mapping[row["class"]]
        return image, label


# Test one image without transformations
m2_train_dataset = MarineZipDataset(m2_train, ZIP_DIR)

sample_image, sample_label = m2_train_dataset[0]

print("=== DATASET SAMPLE TEST ===")
print("Dataset size:", len(m2_train_dataset))
print("Image type:", type(sample_image))
print("Image dimensions:", sample_image.size)
print("Image mode:", sample_image.mode)
print("Label index:", sample_label)
print("Class:", list(class_mapping.keys())[sample_label])
print("Result:", "PASS" if sample_image.mode == "RGB" else "REVIEW")

In [ ]:
print("=== DATASET / DATALOADER STATUS ===")

for name in [
    "train_dataset", "val_dataset", "test_dataset",
    "train_loader", "val_loader", "test_loader",
    "train_transform", "val_transform", "test_transform",
    "MarineZipDataset"
]:
    obj = globals().get(name)
    print(f"{name}:",
          type(obj).__name__ if obj is not None else "NOT FOUND")

In [ ]:
import inspect

print("=== DATASET CLASS INSPECTION ===")
print("Class:", MarineZipDataset)
print("Constructor signature:", inspect.signature(MarineZipDataset))
print("Methods:", [
    name for name, value in inspect.getmembers(MarineZipDataset)
    if callable(value) and not name.startswith("__")
])
print("Class attributes:", [
    name for name in vars(MarineZipDataset)
    if not name.startswith("__")
])

In [ ]:
print("=== MANIFEST ZIP PATH EXAMPLES ===")

for split_name, df in {
    "Training": m2_train,
    "Validation": m2_val,
    "Test": m2_test
}.items():
    print(f"\n{split_name}:")
    print(df[["zip_file", "image_name", "class"]].head(3).to_string(index=False))

In [ ]:
import zipfile
from PIL import Image
from io import BytesIO
from torch.utils.data import Dataset

class MarineZipDatasetV2(Dataset):
    def __init__(self, dataframe, zip_dir, transform=None):
        self.dataframe = dataframe.reset_index(drop=True).copy()
        self.zip_dir = zip_dir
        self.transform = transform
        self.class_mapping = {
            name: idx for idx, name in enumerate(
                sorted(self.dataframe["class"].unique())
            )
        }
        self.zip_handles = {}

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, index):
        row = self.dataframe.iloc[index]

        zip_name = row["zip_file"]
        image_path = row["image_name"]
        zip_path = os.path.join(self.zip_dir, zip_name)

        if zip_name not in self.zip_handles:
            self.zip_handles[zip_name] = zipfile.ZipFile(
                zip_path, "r"
            )

        with self.zip_handles[zip_name].open(image_path) as file:
            image = Image.open(BytesIO(file.read())).convert("RGB")

        label = self.class_mapping[row["class"]]

        if self.transform:
            image = self.transform(image)

        return image, label

print("MarineZipDatasetV2 defined successfully.")
print("Class mapping:", MarineZipDatasetV2(m2_train, ZIP_DIR).class_mapping)

In [ ]:
from torchvision import transforms

IMAGE_SIZE = 224

imagenet_mean = [0.485, 0.456, 0.406]
imagenet_std = [0.229, 0.224, 0.225]

train_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(imagenet_mean, imagenet_std)
])

val_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(imagenet_mean, imagenet_std)
])

test_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(imagenet_mean, imagenet_std)
])

print("Transforms defined successfully.")
print("Image size:", IMAGE_SIZE)
print("Training augmentation: None (baseline)")
print("Validation/test preprocessing: Deterministic")

In [ ]:
from torch.utils.data import DataLoader

BATCH_SIZE = 32

train_dataset = MarineZipDatasetV2(m2_train, ZIP_DIR, train_transform)
val_dataset = MarineZipDatasetV2(m2_val, ZIP_DIR, val_transform)
test_dataset = MarineZipDatasetV2(m2_test, ZIP_DIR, test_transform)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0,
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=True
)

print("=== DATALOADER VERIFICATION ===")
print("Training:", len(train_dataset), "images |", len(train_loader), "batches")
print("Validation:", len(val_dataset), "images |", len(val_loader), "batches")
print("Test:", len(test_dataset), "images |", len(test_loader), "batches")
print("Batch size:", BATCH_SIZE)
print("Class mapping:", train_dataset.class_mapping)
print("Test sampler:", type(test_loader.sampler).__name__)

In [ ]:
print("=== BATCH VALIDATION ===")

for split_name, loader in [
    ("Training", train_loader),
    ("Validation", val_loader),
    ("Test", test_loader)
]:
    images, labels = next(iter(loader))

    print(f"\n{split_name}:")
    print("Image tensor shape:", tuple(images.shape))
    print("Label tensor shape:", tuple(labels.shape))
    print("Image dtype:", images.dtype)
    print("Label dtype:", labels.dtype)
    print("Pixel values finite:", torch.isfinite(images).all().item())
    print("Label range:", labels.min().item(), "to", labels.max().item())

print("\nResult: PASS")

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

grouped_v2_model = grouped_v2_model.to(device)

print("=== MODEL VERIFICATION ===")
print("Architecture: ResNet-18")
print("Pretrained weights: ImageNet")
print("Output classes:", grouped_v2_model.fc.out_features)
print("Device:", next(grouped_v2_model.parameters()).device)
print("Total parameters:", sum(p.numel() for p in grouped_v2_model.parameters()))
print("Trainable parameters:", sum(
    p.numel() for p in grouped_v2_model.parameters() if p.requires_grad
))
print("Class mapping:", train_dataset.class_mapping)

In [ ]:
import torch.nn as nn
from torchvision.models import resnet18, ResNet18_Weights

# Reproducibility
SEED = 42
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# Fresh ImageNet-pretrained ResNet-18
grouped_v2_model = resnet18(weights=ResNet18_Weights.DEFAULT)

# Replace final layer for four classes
grouped_v2_model.fc = nn.Linear(
    grouped_v2_model.fc.in_features,
    len(train_dataset.class_mapping)
)

grouped_v2_model = grouped_v2_model.to(device)

print("=== MODEL VERIFICATION ===")
print("Architecture: ResNet-18")
print("Pretrained weights: ImageNet")
print("Output classes:", grouped_v2_model.fc.out_features)
print("Device:", next(grouped_v2_model.parameters()).device)
print("Total parameters:", sum(p.numel() for p in grouped_v2_model.parameters()))
print("Trainable parameters:", sum(
    p.numel() for p in grouped_v2_model.parameters() if p.requires_grad
))
print("Class mapping:", train_dataset.class_mapping)

In [ ]:
import torch.nn as nn
import torch.optim as optim
import os

LEARNING_RATE = 1e-4
MAX_EPOCHS = 10
PATIENCE = 3
BATCH_SIZE = 32

criterion = nn.CrossEntropyLoss()

optimizer = optim.Adam(
    grouped_v2_model.parameters(),
    lr=LEARNING_RATE
)

use_amp = device.type == "cuda"
scaler = torch.amp.GradScaler("cuda", enabled=use_amp)

BEST_MODEL_PATH = os.path.join(
    MODEL_DIR, "resnet18_grouped_v2_baseline_best.pth"
)

HISTORY_PATH = os.path.join(
    RESULTS, "resnet18_grouped_v2_baseline_history.csv"
)

print("=== TRAINING CONFIGURATION ===")
print("Optimizer: Adam")
print("Learning rate:", LEARNING_RATE)
print("Loss: CrossEntropyLoss")
print("Batch size:", BATCH_SIZE)
print("Maximum epochs:", MAX_EPOCHS)
print("Early stopping patience:", PATIENCE)
print("AMP enabled:", use_amp)
print("Best checkpoint:", BEST_MODEL_PATH)
print("History file:", HISTORY_PATH)
print("Checkpoint already exists:", os.path.exists(BEST_MODEL_PATH))
print("History file already exists:", os.path.exists(HISTORY_PATH))

In [ ]:
from google.colab import drive

drive.mount('/content/drive')

In [ ]:
import os

MODEL_DIR = "/content/drive/MyDrive/Marine_Project/models"
RESULTS = "/content/drive/MyDrive/Marine_Project/results"

files_to_check = {
    "Best checkpoint": os.path.join(
        MODEL_DIR, "resnet18_grouped_v2_baseline_best.pth"
    ),
    "Training history": os.path.join(
        RESULTS, "resnet18_grouped_v2_baseline_history.csv"
    )
}

print("=== TRAINING RECOVERY CHECK ===")

for name, path in files_to_check.items():
    exists = os.path.exists(path)
    print(f"\n{name}: {'FOUND' if exists else 'NOT FOUND'}")

    if exists:
        print("Size (MB):", round(os.path.getsize(path) / (1024 * 1024), 2))

In [ ]:
import os

ROOT = "/content/drive/MyDrive/Marine_Project"

print("Google Drive project accessible:", os.path.isdir(ROOT))

for folder in ["models", "results", "dataset_zips"]:
    path = os.path.join(ROOT, folder)
    print(f"{folder}:",
          "AVAILABLE" if os.path.isdir(path) else "NOT FOUND")

In [ ]:
import torch
import os

checkpoint_path = os.path.join(
    "/content/drive/MyDrive/Marine_Project/models",
    "resnet18_grouped_v2_baseline_best.pth"
)

checkpoint = torch.load(
    checkpoint_path,
    map_location="cpu",
    weights_only=True
)

print("=== CHECKPOINT RECOVERY ===")
print("Saved epoch:", checkpoint.get("epoch"))
print("Best validation loss:", checkpoint.get("best_val_loss"))
print("Class mapping:", checkpoint.get("class_mapping"))
print("Checkpoint keys:", list(checkpoint.keys()))
print("Model state available:",
      "model_state_dict" in checkpoint)
print("Optimizer state available:",
      "optimizer_state_dict" in checkpoint)

In [ ]:
print("=== SAVED TRAINING CONFIGURATION ===")

print("Configuration:")
for key, value in checkpoint["training_config"].items():
    print(f"{key}: {value}")

print("\nSaved best epoch:", checkpoint["epoch"])
print("Saved best validation loss:", checkpoint["best_val_loss"])
print("Optimizer state:", "Available" if checkpoint.get("optimizer_state_dict") else "Missing")

In [ ]:
print("=== RUNTIME RECOVERY STATUS ===")

required = [
    "torch", "device",
    "m2_train", "m2_val", "m2_test",
    "train_loader", "val_loader", "test_loader",
    "train_transform", "val_transform", "test_transform",
    "grouped_v2_model", "optimizer", "criterion", "scaler"
]

for name in required:
    print(f"{name}: {'AVAILABLE' if name in globals() else 'MISSING'}")

print("\nCUDA available:", torch.cuda.is_available())
print("Checkpoint available:", os.path.exists(checkpoint_path))

In [ ]:
import os
import pandas as pd
import torch

ROOT = "/content/drive/MyDrive/Marine_Project"
RESULTS = os.path.join(ROOT, "results")
MODEL_DIR = os.path.join(ROOT, "models")
ZIP_DIR = os.path.join(ROOT, "dataset_zips")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

m2_train = pd.read_csv(
    os.path.join(RESULTS, "clean_train_manifest_grouped_v2.csv")
)
m2_val = pd.read_csv(
    os.path.join(RESULTS, "clean_validation_manifest_grouped_v2.csv")
)
m2_test = pd.read_csv(
    os.path.join(RESULTS, "clean_test_manifest.csv")
)

print("=== MANIFEST RECOVERY ===")
print("Training:", len(m2_train))
print("Validation:", len(m2_val))
print("Test:", len(m2_test))
print("Device:", device)
print("ZIP directory accessible:", os.path.isdir(ZIP_DIR))
print("Checkpoint accessible:", os.path.exists(checkpoint_path))

In [ ]:
import zipfile
from PIL import Image
from io import BytesIO
from torch.utils.data import Dataset

class MarineZipDatasetV2(Dataset):
    def __init__(self, dataframe, zip_dir, transform=None):
        self.dataframe = dataframe.reset_index(drop=True).copy()
        self.zip_dir = zip_dir
        self.transform = transform
        self.class_mapping = {
            name: idx for idx, name in enumerate(
                sorted(self.dataframe["class"].unique())
            )
        }
        self.zip_handles = {}

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, index):
        row = self.dataframe.iloc[index]
        zip_name = row["zip_file"]
        image_path = row["image_name"]
        zip_path = os.path.join(self.zip_dir, zip_name)

        if zip_name not in self.zip_handles:
            self.zip_handles[zip_name] = zipfile.ZipFile(
                zip_path, "r"
            )

        with self.zip_handles[zip_name].open(image_path) as file:
            image = Image.open(BytesIO(file.read())).convert("RGB")

        label = self.class_mapping[row["class"]]

        if self.transform:
            image = self.transform(image)

        return image, label

print("MarineZipDatasetV2 restored.")
print("Training class mapping:",
      MarineZipDatasetV2(m2_train, ZIP_DIR).class_mapping)

In [ ]:
from torchvision import transforms

IMAGE_SIZE = 224

imagenet_mean = [0.485, 0.456, 0.406]
imagenet_std = [0.229, 0.224, 0.225]

train_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(imagenet_mean, imagenet_std)
])

val_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(imagenet_mean, imagenet_std)
])

test_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(imagenet_mean, imagenet_std)
])

print("Preprocessing restored.")
print("Image size:", IMAGE_SIZE)
print("Training augmentation: None")
print("Validation/test: Deterministic")

In [ ]:
from torch.utils.data import DataLoader

BATCH_SIZE = 32

train_dataset = MarineZipDatasetV2(m2_train, ZIP_DIR, train_transform)
val_dataset = MarineZipDatasetV2(m2_val, ZIP_DIR, val_transform)
test_dataset = MarineZipDatasetV2(m2_test, ZIP_DIR, test_transform)

train_loader = DataLoader(
    train_dataset, batch_size=BATCH_SIZE,
    shuffle=True, num_workers=0, pin_memory=True
)

val_loader = DataLoader(
    val_dataset, batch_size=BATCH_SIZE,
    shuffle=False, num_workers=0, pin_memory=True
)

test_loader = DataLoader(
    test_dataset, batch_size=BATCH_SIZE,
    shuffle=False, num_workers=0, pin_memory=True
)

print("=== DATALOADER RECOVERY ===")
print("Training:", len(train_dataset), "| batches:", len(train_loader))
print("Validation:", len(val_dataset), "| batches:", len(val_loader))
print("Test:", len(test_dataset), "| batches:", len(test_loader))
print("Class mapping:", train_dataset.class_mapping)
print("Test sampler:", type(test_loader.sampler).__name__)

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision.models import resnet18
import os

checkpoint_path = os.path.join(
    MODEL_DIR, "resnet18_grouped_v2_baseline_best.pth"
)

checkpoint = torch.load(
    checkpoint_path,
    map_location="cpu",
    weights_only=True
)

# Reconstruct architecture without downloading pretrained weights
grouped_v2_model = resnet18(weights=None)
grouped_v2_model.fc = nn.Linear(
    grouped_v2_model.fc.in_features, 4
)

grouped_v2_model.load_state_dict(checkpoint["model_state_dict"])
grouped_v2_model = grouped_v2_model.to(device)

# Restore optimizer and its saved state
LEARNING_RATE = checkpoint["training_config"]["learning_rate"]

optimizer = optim.Adam(
    grouped_v2_model.parameters(),
    lr=LEARNING_RATE
)
optimizer.load_state_dict(checkpoint["optimizer_state_dict"])

criterion = nn.CrossEntropyLoss()

use_amp = device.type == "cuda"
scaler = torch.amp.GradScaler("cuda", enabled=use_amp)

print("=== MODEL RECOVERY ===")
print("Restored epoch:", checkpoint["epoch"])
print("Best validation loss:", checkpoint["best_val_loss"])
print("Model device:", next(grouped_v2_model.parameters()).device)
print("Optimizer state restored:", len(optimizer.state) > 0)
print("AMP enabled:", use_amp)
print("Class mapping matches:",
      checkpoint["class_mapping"] == train_dataset.class_mapping)

In [ ]:
LATEST_MODEL_PATH = os.path.join(
    MODEL_DIR, "resnet18_grouped_v2_baseline_latest.pth"
)

BEST_MODEL_PATH = os.path.join(
    MODEL_DIR, "resnet18_grouped_v2_baseline_best.pth"
)

HISTORY_PATH = os.path.join(
    RESULTS, "resnet18_grouped_v2_baseline_recovery_history.csv"
)

START_EPOCH = checkpoint["epoch"] + 1
MAX_EPOCHS = checkpoint["training_config"]["max_epochs"]
PATIENCE = checkpoint["training_config"]["patience"]

best_val_loss = checkpoint["best_val_loss"]
best_epoch = checkpoint["epoch"]
epochs_without_improvement = 0

print("=== SAFE TRAINING SETUP ===")
print("Resume from epoch:", START_EPOCH)
print("Maximum epochs:", MAX_EPOCHS)
print("Remaining epochs:", MAX_EPOCHS - START_EPOCH + 1)
print("Recovered best epoch:", best_epoch)
print("Recovered best validation loss:", best_val_loss)
print("Latest checkpoint exists:", os.path.exists(LATEST_MODEL_PATH))
print("Best checkpoint preserved:", os.path.exists(BEST_MODEL_PATH))
print("Recovery history exists:", os.path.exists(HISTORY_PATH))
print("Early stopping patience:", PATIENCE)

In [ ]:
import copy
import pandas as pd
import torch
import os

# Seed the latest checkpoint from the recovered epoch-3 state
recovery_checkpoint = copy.deepcopy(checkpoint)

recovery_checkpoint["last_completed_epoch"] = checkpoint["epoch"]
recovery_checkpoint["best_epoch"] = checkpoint["epoch"]
recovery_checkpoint["epochs_without_improvement"] = 0
recovery_checkpoint["history"] = []

torch.save(recovery_checkpoint, LATEST_MODEL_PATH)

# Create a separate recovery history file
history_columns = [
    "epoch",
    "train_loss",
    "train_accuracy",
    "val_loss",
    "val_accuracy",
    "time_seconds"
]

pd.DataFrame(columns=history_columns).to_csv(
    HISTORY_PATH, index=False
)

print("=== INITIAL RECOVERY POINT ===")
print("Latest checkpoint saved:", os.path.exists(LATEST_MODEL_PATH))
print("Best checkpoint preserved:", os.path.exists(BEST_MODEL_PATH))
print("Recovery history created:", os.path.exists(HISTORY_PATH))
print("Recovery starts after epoch:", checkpoint["epoch"])
print("Next epoch:", START_EPOCH)

In [ ]:
import time
import pandas as pd
import torch
from torch.amp import autocast

history = recovery_checkpoint.get("history", []).copy()

for epoch in range(START_EPOCH, MAX_EPOCHS + 1):
    epoch_start = time.time()

    # ---- Training ----
    grouped_v2_model.train()
    train_loss_sum = 0.0
    train_correct = 0
    train_total = 0

    for images, labels in train_loader:
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        with autocast(device_type=device.type, enabled=use_amp):
            outputs = grouped_v2_model(images)
            loss = criterion(outputs, labels)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        train_loss_sum += loss.item() * labels.size(0)
        train_correct += (outputs.argmax(1) == labels).sum().item()
        train_total += labels.size(0)

    train_loss = train_loss_sum / train_total
    train_accuracy = train_correct / train_total

    # ---- Validation ----
    grouped_v2_model.eval()
    val_loss_sum = 0.0
    val_correct = 0
    val_total = 0

    with torch.no_grad():
        for images, labels in val_loader:
            images = images.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)

            with autocast(device_type=device.type, enabled=use_amp):
                outputs = grouped_v2_model(images)
                loss = criterion(outputs, labels)

            val_loss_sum += loss.item() * labels.size(0)
            val_correct += (outputs.argmax(1) == labels).sum().item()
            val_total += labels.size(0)

    val_loss = val_loss_sum / val_total
    val_accuracy = val_correct / val_total
    elapsed = time.time() - epoch_start

    epoch_record = {
        "epoch": epoch,
        "train_loss": train_loss,
        "train_accuracy": train_accuracy,
        "val_loss": val_loss,
        "val_accuracy": val_accuracy,
        "time_seconds": elapsed
    }
    history.append(epoch_record)

    # ---- Update best model tracking ----
    improved = val_loss < best_val_loss

    if improved:
        best_val_loss = val_loss
        best_epoch = epoch
        epochs_without_improvement = 0
    else:
        epochs_without_improvement += 1

    # ---- Build recovery checkpoint ----
    save_data = {
        "epoch": epoch,
        "last_completed_epoch": epoch,
        "model_state_dict": grouped_v2_model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "scaler_state_dict": scaler.state_dict(),
        "best_val_loss": best_val_loss,
        "best_epoch": best_epoch,
        "epochs_without_improvement": epochs_without_improvement,
        "class_mapping": train_dataset.class_mapping,
        "training_config": checkpoint["training_config"],
        "history": history
    }

    # Save latest checkpoint safely
    temp_latest = LATEST_MODEL_PATH + ".tmp"
    torch.save(save_data, temp_latest)
    os.replace(temp_latest, LATEST_MODEL_PATH)

    # Save best checkpoint only when improved
    if improved:
        temp_best = BEST_MODEL_PATH + ".tmp"
        torch.save(save_data, temp_best)
        os.replace(temp_best, BEST_MODEL_PATH)

    # Save history after every completed epoch
    history_df = pd.DataFrame(history)
    temp_history = HISTORY_PATH + ".tmp"
    history_df.to_csv(temp_history, index=False)
    os.replace(temp_history, HISTORY_PATH)

    print(
        f"Epoch {epoch}/{MAX_EPOCHS} | "
        f"Train Loss: {train_loss:.5f} | "
        f"Train Acc: {train_accuracy*100:.2f}% | "
        f"Val Loss: {val_loss:.5f} | "
        f"Val Acc: {val_accuracy*100:.2f}% | "
        f"Time: {elapsed:.1f}s"
    )

    print(
        f"  Latest checkpoint saved | "
        f"Best epoch: {best_epoch} | "
        f"Best val loss: {best_val_loss:.5f}"
    )

    if epochs_without_improvement >= PATIENCE:
        print("\nEarly stopping triggered.")
        break

print("\n=== RECOVERY TRAINING SUMMARY ===")
print("Completed recovery epochs:", len(history))
print("Last completed epoch:", history[-1]["epoch"])
print("Best epoch:", best_epoch)
print("Best validation loss:", best_val_loss)
print("Latest checkpoint:", os.path.exists(LATEST_MODEL_PATH))
print("Best checkpoint:", os.path.exists(BEST_MODEL_PATH))
print("History CSV:", os.path.exists(HISTORY_PATH))

In [ ]:
import os
import pandas as pd
import torch

print("=== FINAL TRAINING ARTIFACT AUDIT ===")

for name, path in [
    ("Best model", BEST_MODEL_PATH),
    ("Latest model", LATEST_MODEL_PATH),
    ("Recovery history", HISTORY_PATH)
]:
    print(f"\n{name}:")
    print("Exists:", os.path.exists(path))
    if os.path.exists(path):
        print("Size (MB):", round(os.path.getsize(path) / 1024**2, 2))

saved_history = pd.read_csv(HISTORY_PATH)

print("\n=== RECOVERY HISTORY ===")
print(saved_history.to_string(index=False))

best_ckpt = torch.load(
    BEST_MODEL_PATH, map_location="cpu", weights_only=True
)
latest_ckpt = torch.load(
    LATEST_MODEL_PATH, map_location="cpu", weights_only=True
)

print("\nBest checkpoint epoch:", best_ckpt["epoch"])
print("Latest checkpoint epoch:", latest_ckpt["epoch"])
print("Best validation loss:", best_ckpt["best_val_loss"])
print("Latest validation loss:", latest_ckpt["history"][-1]["val_loss"])

In [ ]:
best_checkpoint = torch.load(
    BEST_MODEL_PATH,
    map_location=device,
    weights_only=True
)

grouped_v2_model.load_state_dict(
    best_checkpoint["model_state_dict"]
)

grouped_v2_model.eval()

print("=== BEST MODEL RESTORED ===")
print("Selected epoch:", best_checkpoint["epoch"])
print("Validation loss:", best_checkpoint["best_val_loss"])
print("Model device:", next(grouped_v2_model.parameters()).device)
print("Evaluation mode:", not grouped_v2_model.training)
print("Class mapping matches:",
      best_checkpoint["class_mapping"] == train_dataset.class_mapping)

In [ ]:
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    accuracy_score
)
import numpy as np
import pandas as pd
import torch

grouped_v2_model.eval()

all_labels = []
all_predictions = []
all_probabilities = []

with torch.no_grad():
    for images, labels in val_loader:
        images = images.to(device)
        labels = labels.to(device)

        with torch.amp.autocast(
            device_type="cuda",
            enabled=(device.type == "cuda")
        ):
            outputs = grouped_v2_model(images)

        probabilities = torch.softmax(outputs.float(), dim=1)
        predictions = probabilities.argmax(dim=1)

        all_labels.extend(labels.cpu().numpy())
        all_predictions.extend(predictions.cpu().numpy())
        all_probabilities.extend(probabilities.cpu().numpy())

all_labels = np.array(all_labels)
all_predictions = np.array(all_predictions)
all_probabilities = np.array(all_probabilities)

class_names = [
    name for name, index in sorted(
        train_dataset.class_mapping.items(),
        key=lambda item: item[1]
    )
]

print("=== GROUPED-V2 VALIDATION RESULTS ===")
print("Images evaluated:", len(all_labels))
print("Accuracy:", round(accuracy_score(all_labels, all_predictions) * 100, 4), "%")

print("\n=== CLASSIFICATION REPORT ===")
print(classification_report(
    all_labels,
    all_predictions,
    target_names=class_names,
    digits=4
))

print("=== CONFUSION MATRIX ===")
print(pd.DataFrame(
    confusion_matrix(all_labels, all_predictions),
    index=class_names,
    columns=class_names
))

In [ ]:
import os
import pandas as pd
import numpy as np
from sklearn.metrics import classification_report, confusion_matrix

results_dir = "/content/drive/MyDrive/Marine_Project/results"
os.makedirs(results_dir, exist_ok=True)

# Confirm prediction order matches the validation manifest
val_df = val_loader.dataset.dataframe.reset_index(drop=True)

assert len(val_df) == len(all_labels) == 824
assert list(val_df["class"]) == [
    class_names[label] for label in all_labels
], "Validation order or labels do not match!"

# Classification report
report = classification_report(
    all_labels,
    all_predictions,
    target_names=class_names,
    output_dict=True,
    zero_division=0
)
report_df = pd.DataFrame(report).transpose()

# Confusion matrix
cm_df = pd.DataFrame(
    confusion_matrix(all_labels, all_predictions),
    index=class_names,
    columns=class_names
)

# Individual predictions
predictions_df = pd.DataFrame({
    "image_name": val_df["image_name"],
    "actual_class": [class_names[x] for x in all_labels],
    "predicted_class": [class_names[x] for x in all_predictions],
    "confidence": all_probabilities.max(axis=1),
})

for i, class_name in enumerate(class_names):
    predictions_df[f"prob_{class_name}"] = all_probabilities[:, i]

# Save using new filenames
artifacts = {
    "report": os.path.join(
        results_dir, "grouped_v2_baseline_validation_report.csv"
    ),
    "confusion": os.path.join(
        results_dir, "grouped_v2_baseline_validation_confusion_matrix.csv"
    ),
    "predictions": os.path.join(
        results_dir, "grouped_v2_baseline_validation_predictions.csv"
    )
}

for key, dataframe in [
    ("report", report_df),
    ("confusion", cm_df),
    ("predictions", predictions_df)
]:
    path = artifacts[key]
    assert not os.path.exists(path), f"File already exists: {path}"

    temp_path = path + ".tmp"
    dataframe.to_csv(temp_path)
    os.replace(temp_path, path)

print("=== VALIDATION ARTIFACTS SAVED ===")
for name, path in artifacts.items():
    print(name, "| Exists:", os.path.exists(path),
          "| Size:", round(os.path.getsize(path) / 1024, 2), "KB")

In [ ]:
misclassified_df = predictions_df[
    predictions_df["actual_class"] != predictions_df["predicted_class"]
].copy()

print("=== VALIDATION MISCLASSIFICATIONS ===")
print("Total errors:", len(misclassified_df))

display(
    misclassified_df[
        ["image_name", "actual_class", "predicted_class", "confidence"]
    ].reset_index(drop=True)
)

print("\n=== ERROR DISTRIBUTION ===")
print(
    misclassified_df.groupby(
        ["actual_class", "predicted_class"]
    ).size().reset_index(name="count")
)

In [ ]:
import zipfile
import io
import cv2
import numpy as np
import pandas as pd
from PIL import Image

error_details = []

for _, row in misclassified_df.iterrows():
    zip_path = os.path.join(
        ZIP_DIR,
        row["image_name"].split("/")[0] + "_training.zip"
    )

    # Use the exact archive recorded in the validation manifest
    matching = val_df[val_df["image_name"] == row["image_name"]]
    zip_name = matching.iloc[0]["zip_file"]
    zip_path = os.path.join(ZIP_DIR, zip_name)

    with zipfile.ZipFile(zip_path, "r") as z:
        image_bytes = z.read(row["image_name"])

    image = Image.open(io.BytesIO(image_bytes)).convert("RGB")
    arr = np.array(image)
    gray = cv2.cvtColor(arr, cv2.COLOR_RGB2GRAY)

    error_details.append({
        "image_name": row["image_name"],
        "actual_class": row["actual_class"],
        "predicted_class": row["predicted_class"],
        "width": image.width,
        "height": image.height,
        "sharpness": cv2.Laplacian(gray, cv2.CV_64F).var(),
        "contrast": gray.std(),
        "brightness": gray.mean(),
        "saturation": cv2.cvtColor(
            arr, cv2.COLOR_RGB2HSV
        )[:, :, 1].mean()
    })

error_quality_df = pd.DataFrame(error_details)

print("=== MISCLASSIFIED IMAGE QUALITY ===")
display(error_quality_df.round(3))

In [ ]:
import numpy as np
import pandas as pd
import torch
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix
)

grouped_v2_model.eval()

test_labels = []
test_predictions = []
test_probabilities = []

with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device)
        labels = labels.to(device)

        with torch.amp.autocast(
            device_type="cuda",
            enabled=(device.type == "cuda")
        ):
            outputs = grouped_v2_model(images)

        probabilities = torch.softmax(outputs.float(), dim=1)
        predictions = probabilities.argmax(dim=1)

        test_labels.extend(labels.cpu().numpy())
        test_predictions.extend(predictions.cpu().numpy())
        test_probabilities.extend(probabilities.cpu().numpy())

test_labels = np.array(test_labels)
test_predictions = np.array(test_predictions)
test_probabilities = np.array(test_probabilities)

print("=== GROUPED-V2 CLEAN TEST RESULTS ===")
print("Images evaluated:", len(test_labels))
print(
    "Accuracy:",
    round(accuracy_score(test_labels, test_predictions) * 100, 4),
    "%"
)

print("\n=== CLASSIFICATION REPORT ===")
print(classification_report(
    test_labels,
    test_predictions,
    target_names=class_names,
    digits=4
))

print("=== CONFUSION MATRIX ===")
print(pd.DataFrame(
    confusion_matrix(test_labels, test_predictions),
    index=class_names,
    columns=class_names
))

In [ ]:
import os
import pandas as pd
import numpy as np
from sklearn.metrics import classification_report, confusion_matrix

results_dir = "/content/drive/MyDrive/Marine_Project/results"

test_df = test_loader.dataset.dataframe.reset_index(drop=True)

# Verify prediction order and labels
assert len(test_df) == len(test_labels) == 1002
assert list(test_df["class"]) == [
    class_names[label] for label in test_labels
], "Test order or labels do not match!"

# Classification report
test_report_df = pd.DataFrame(
    classification_report(
        test_labels,
        test_predictions,
        target_names=class_names,
        output_dict=True,
        zero_division=0
    )
).transpose()

# Confusion matrix
test_cm_df = pd.DataFrame(
    confusion_matrix(test_labels, test_predictions),
    index=class_names,
    columns=class_names
)

# Individual predictions
test_predictions_df = pd.DataFrame({
    "image_name": test_df["image_name"],
    "actual_class": [class_names[x] for x in test_labels],
    "predicted_class": [class_names[x] for x in test_predictions],
    "confidence": test_probabilities.max(axis=1)
})

for i, class_name in enumerate(class_names):
    test_predictions_df[f"prob_{class_name}"] = test_probabilities[:, i]

artifacts = {
    "report": os.path.join(
        results_dir, "grouped_v2_baseline_clean_test_report.csv"
    ),
    "confusion_matrix": os.path.join(
        results_dir, "grouped_v2_baseline_clean_test_confusion_matrix.csv"
    ),
    "predictions": os.path.join(
        results_dir, "grouped_v2_baseline_clean_test_predictions.csv"
    )
}

dataframes = {
    "report": test_report_df,
    "confusion_matrix": test_cm_df,
    "predictions": test_predictions_df
}

for name, path in artifacts.items():
    assert not os.path.exists(path), f"File already exists: {path}"

    temp_path = path + ".tmp"
    dataframes[name].to_csv(temp_path)
    os.replace(temp_path, path)

print("=== CLEAN TEST ARTIFACT AUDIT ===")

for name, path in artifacts.items():
    print(
        name,
        "| Exists:", os.path.exists(path),
        "| Size:", round(os.path.getsize(path) / 1024, 2), "KB"
    )

print("\nTest images:", len(test_predictions_df))
print(
    "Correct:",
    (test_predictions_df["actual_class"] ==
     test_predictions_df["predicted_class"]).sum()
)
print(
    "Incorrect:",
    (test_predictions_df["actual_class"] !=
     test_predictions_df["predicted_class"]).sum()
)

In [ ]:
# Separate correct and incorrect test predictions
test_errors_df = test_predictions_df[
    test_predictions_df["actual_class"] !=
    test_predictions_df["predicted_class"]
].copy()

test_correct_df = test_predictions_df[
    test_predictions_df["actual_class"] ==
    test_predictions_df["predicted_class"]
].copy()

print("=== TEST ERROR DISTRIBUTION ===")
error_distribution = (
    test_errors_df
    .groupby(["actual_class", "predicted_class"])
    .size()
    .reset_index(name="count")
    .sort_values("count", ascending=False)
)

display(error_distribution.reset_index(drop=True))

print("\n=== ERRORS BY ACTUAL CLASS ===")
display(
    test_errors_df.groupby("actual_class")
    .size()
    .reindex(class_names, fill_value=0)
    .rename("error_count")
    .to_frame()
)

print("\n=== CONFIDENCE COMPARISON ===")
print("Correct predictions:", len(test_correct_df))
print("Incorrect predictions:", len(test_errors_df))
print(
    "Mean confidence — correct:",
    round(test_correct_df["confidence"].mean(), 6)
)
print(
    "Mean confidence — incorrect:",
    round(test_errors_df["confidence"].mean(), 6)
)

print("\n=== INCORRECT PREDICTIONS WITH HIGHEST CONFIDENCE ===")
display(
    test_errors_df.sort_values(
        "confidence", ascending=False
    )[
        ["image_name", "actual_class", "predicted_class", "confidence"]
    ].head(10).reset_index(drop=True)
)

In [ ]:
import os
import pandas as pd
import numpy as np

error_path = os.path.join(
    results_dir,
    "grouped_v2_baseline_clean_test_error_analysis.csv"
)

confidence_path = os.path.join(
    results_dir,
    "grouped_v2_baseline_clean_test_confidence_summary.csv"
)

assert not os.path.exists(error_path)
assert not os.path.exists(confidence_path)

# Error distribution
error_distribution = (
    test_errors_df
    .groupby(["actual_class", "predicted_class"])
    .size()
    .reset_index(name="count")
)

error_distribution.to_csv(error_path, index=False)

# Confidence summary
confidence_summary = pd.DataFrame({
    "prediction_status": ["Correct", "Incorrect"],
    "count": [len(test_correct_df), len(test_errors_df)],
    "mean_confidence": [
        test_correct_df["confidence"].mean(),
        test_errors_df["confidence"].mean()
    ],
    "median_confidence": [
        test_correct_df["confidence"].median(),
        test_errors_df["confidence"].median()
    ],
    "minimum_confidence": [
        test_correct_df["confidence"].min(),
        test_errors_df["confidence"].min()
    ],
    "maximum_confidence": [
        test_correct_df["confidence"].max(),
        test_errors_df["confidence"].max()
    ]
})

confidence_summary.to_csv(confidence_path, index=False)

print("=== ERROR ANALYSIS FILE AUDIT ===")

for path in [error_path, confidence_path]:
    print(
        os.path.basename(path),
        "| Exists:", os.path.exists(path),
        "| Size:", round(os.path.getsize(path) / 1024, 3), "KB"
    )

print("\n=== CONFIDENCE SUMMARY ===")
display(confidence_summary.round(6))

In [ ]:
import os
import torch
import pandas as pd
import numpy as np

print("=== FINAL M2 CHECKPOINT AUDIT ===")

best_ckpt = torch.load(
    BEST_MODEL_PATH, map_location="cpu", weights_only=True
)

latest_ckpt = torch.load(
    LATEST_MODEL_PATH, map_location="cpu", weights_only=True
)

print("\nBest checkpoint epoch:", best_ckpt["epoch"])
print("Latest checkpoint epoch:", latest_ckpt["epoch"])
print("Best validation loss:", best_ckpt["best_val_loss"])

print("\n=== TRAINING CONFIGURATION ===")
print(best_ckpt.get("training_config", "Configuration unavailable"))

print("\n=== CLASS MAPPING ===")
print(best_ckpt["class_mapping"])

print("\n=== RESULT FILE AUDIT ===")

required_files = [
    BEST_MODEL_PATH,
    LATEST_MODEL_PATH,
    HISTORY_PATH,
    os.path.join(results_dir, "grouped_v2_baseline_validation_report.csv"),
    os.path.join(results_dir, "grouped_v2_baseline_validation_confusion_matrix.csv"),
    os.path.join(results_dir, "grouped_v2_baseline_validation_predictions.csv"),
    os.path.join(results_dir, "grouped_v2_baseline_clean_test_report.csv"),
    os.path.join(results_dir, "grouped_v2_baseline_clean_test_confusion_matrix.csv"),
    os.path.join(results_dir, "grouped_v2_baseline_clean_test_predictions.csv"),
    error_path,
    confidence_path
]

for path in required_files:
    print(
        os.path.basename(path),
        "| Exists:", os.path.exists(path),
        "| Size (MB):",
        round(os.path.getsize(path) / 1024**2, 4)
        if os.path.exists(path) else "N/A"
    )

print("\n=== CONFIDENCE DATA INTEGRITY ===")
print("Missing confidence values:", test_predictions_df["confidence"].isna().sum())
print("All confidence values finite:", np.isfinite(test_predictions_df["confidence"]).all())
print("Incorrect prediction maximum confidence:",
      test_errors_df["confidence"].max())

print("\n=== FINAL COUNTS ===")
print("Validation images:", len(all_labels))
print("Test images:", len(test_labels))
print("Test correct:", len(test_correct_df))
print("Test incorrect:", len(test_errors_df))

**M 3 - Segmentation and Localization**

In [ ]:
import os
import zipfile
from collections import Counter

zip_dir = "/content/drive/MyDrive/Marine_Project/dataset_zips"

annotation_keywords = [
    "mask", "annotation", "segment", "label",
    "ground_truth", "groundtruth", "polygon"
]

print("=== M3: ANNOTATION AVAILABILITY AUDIT ===")

for filename in sorted(os.listdir(zip_dir)):
    if not filename.lower().endswith(".zip"):
        continue

    path = os.path.join(zip_dir, filename)

    with zipfile.ZipFile(path, "r") as archive:
        files = [
            name for name in archive.namelist()
            if not name.endswith("/")
        ]

        extensions = Counter(
            os.path.splitext(name)[1].lower()
            for name in files
        )

        candidates = [
            name for name in files
            if any(keyword in name.lower()
                   for keyword in annotation_keywords)
        ]

        print(f"\nArchive: {filename}")
        print("Total files:", len(files))
        print("File extensions:", dict(extensions))
        print("Possible annotation files:", len(candidates))

        for candidate in candidates[:10]:
            print("  ", candidate)

print("\n=== AUDIT COMPLETE ===")

In [ ]:
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import cv2

# Select 3 correct vegetation examples and 3 vegetation -> Background errors
selected_indices = []

for class_name in ["Amphibolis", "Halophila", "Posidonia"]:
    correct_candidates = test_predictions_df[
        (test_predictions_df["actual_class"] == class_name) &
        (test_predictions_df["predicted_class"] == class_name)
    ]

    error_candidates = test_predictions_df[
        (test_predictions_df["actual_class"] == class_name) &
        (test_predictions_df["predicted_class"] == "Background")
    ].sort_values("confidence", ascending=False)

    if len(correct_candidates) == 0 or len(error_candidates) == 0:
        raise ValueError(f"Missing required examples for {class_name}")

    selected_indices.append(
        (correct_candidates.index[0], "Correct")
    )
    selected_indices.append(
        (error_candidates.index[0], "Vegetation predicted as Background")
    )

# Target convolutional layer in ResNet-18
target_layer = grouped_v2_model.layer4[-1]
activation_storage = {}

def save_activation(module, inputs, output):
    activation_storage["activation"] = output
    output.register_hook(
        lambda gradient: activation_storage.update(
            {"gradient": gradient}
        )
    )

hook = target_layer.register_forward_hook(save_activation)

mean = torch.tensor(
    [0.485, 0.456, 0.406]
).view(3, 1, 1)

std = torch.tensor(
    [0.229, 0.224, 0.225]
).view(3, 1, 1)

cam_results = []

grouped_v2_model.eval()

try:
    for index, selection_type in selected_indices:
        image_tensor, true_label = test_loader.dataset[index]

        input_tensor = image_tensor.unsqueeze(0).to(device)

        grouped_v2_model.zero_grad(set_to_none=True)
        activation_storage.clear()

        output = grouped_v2_model(input_tensor)
        predicted_label = output.argmax(dim=1).item()

        # Explain the model's predicted class
        target_score = output[0, predicted_label]
        target_score.backward()

        activation = activation_storage["activation"].detach()
        gradient = activation_storage["gradient"].detach()

        weights = gradient.mean(dim=(2, 3), keepdim=True)
        cam = torch.relu(
            (weights * activation).sum(dim=1)
        )[0].cpu().numpy()

        cam = cv2.resize(
            cam,
            (224, 224),
            interpolation=cv2.INTER_LINEAR
        )

        if cam.max() > cam.min():
            cam = (cam - cam.min()) / (cam.max() - cam.min())
        else:
            cam = np.zeros_like(cam)

        original = (
            image_tensor.cpu() * std + mean
        ).permute(1, 2, 0).numpy()

        original = np.clip(original, 0, 1)

        heatmap = cv2.applyColorMap(
            np.uint8(255 * cam),
            cv2.COLORMAP_JET
        )
        heatmap = cv2.cvtColor(
            heatmap, cv2.COLOR_BGR2RGB
        ) / 255.0

        overlay = np.clip(
            0.55 * original + 0.45 * heatmap,
            0, 1
        )

        cam_results.append({
            "index": index,
            "selection": selection_type,
            "actual": class_names[true_label],
            "predicted": class_names[predicted_label],
            "original": original,
            "cam": cam,
            "overlay": overlay
        })

finally:
    hook.remove()

# Display original images and Grad-CAM overlays
fig, axes = plt.subplots(
    len(cam_results), 2,
    figsize=(10, 4 * len(cam_results))
)

for row, result in enumerate(cam_results):
    axes[row, 0].imshow(result["original"])
    axes[row, 0].set_title(
        f'{result["selection"]}\n'
        f'Actual: {result["actual"]} | '
        f'Predicted: {result["predicted"]}'
    )
    axes[row, 0].axis("off")

    axes[row, 1].imshow(result["overlay"])
    axes[row, 1].set_title("Grad-CAM overlay")
    axes[row, 1].axis("off")

plt.tight_layout()
plt.show()

print("\nGrad-CAM examples generated:", len(cam_results))
print("Target layer: ResNet18 layer4[-1]")

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
import cv2

target_layer = grouped_v2_model.layer4[-1]
activation_storage = {}

def save_activation(module, inputs, output):
    activation_storage["activation"] = output
    output.register_hook(
        lambda gradient: activation_storage.update(
            {"gradient": gradient}
        )
    )

hook = target_layer.register_forward_hook(save_activation)

mean = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
std = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)

error_indices = [
    index for index, selection in selected_indices
    if selection != "Correct"
]

actual_class_cam_results = []

try:
    for index in error_indices:
        image_tensor, true_label = test_loader.dataset[index]
        input_tensor = image_tensor.unsqueeze(0).to(device)

        grouped_v2_model.zero_grad(set_to_none=True)
        activation_storage.clear()

        output = grouped_v2_model(input_tensor)

        predicted_label = output.argmax(dim=1).item()

        # Explain the actual vegetation class, not Background
        target_score = output[0, true_label]
        target_score.backward()

        activation = activation_storage["activation"].detach()
        gradient = activation_storage["gradient"].detach()

        weights = gradient.mean(dim=(2, 3), keepdim=True)

        cam = torch.relu(
            (weights * activation).sum(dim=1)
        )[0].cpu().numpy()

        cam = cv2.resize(cam, (224, 224))

        if cam.max() > cam.min():
            cam = (cam - cam.min()) / (cam.max() - cam.min())

        original = (
            image_tensor.cpu() * std + mean
        ).permute(1, 2, 0).numpy()

        original = np.clip(original, 0, 1)

        heatmap = cv2.applyColorMap(
            np.uint8(255 * cam),
            cv2.COLORMAP_JET
        )
        heatmap = cv2.cvtColor(
            heatmap, cv2.COLOR_BGR2RGB
        ) / 255.0

        overlay = np.clip(
            0.55 * original + 0.45 * heatmap,
            0, 1
        )

        actual_class_cam_results.append({
            "actual": class_names[true_label],
            "predicted": class_names[predicted_label],
            "original": original,
            "overlay": overlay
        })

finally:
    hook.remove()

fig, axes = plt.subplots(
    len(actual_class_cam_results), 2,
    figsize=(10, 4 * len(actual_class_cam_results))
)

for row, result in enumerate(actual_class_cam_results):
    axes[row, 0].imshow(result["original"])
    axes[row, 0].set_title(
        f'Actual: {result["actual"]} | '
        f'Predicted: {result["predicted"]}'
    )
    axes[row, 0].axis("off")

    axes[row, 1].imshow(result["overlay"])
    axes[row, 1].set_title(
        f'Grad-CAM for actual class: {result["actual"]}'
    )
    axes[row, 1].axis("off")

plt.tight_layout()
plt.show()

In [ ]:
import torch
import numpy as np
import pandas as pd

target_layer = grouped_v2_model.layer4[-1]
activation_storage = {}
diagnostic_rows = []

def capture_activation(module, inputs, output):
    activation_storage["activation"] = output
    output.register_hook(
        lambda gradient: activation_storage.update(
            {"gradient": gradient}
        )
    )

hook = target_layer.register_forward_hook(capture_activation)

try:
    for index in error_indices:
        image_tensor, true_label = test_loader.dataset[index]
        input_tensor = image_tensor.unsqueeze(0).to(device)

        grouped_v2_model.zero_grad(set_to_none=True)
        activation_storage.clear()

        output = grouped_v2_model(input_tensor)

        predicted_label = output.argmax(dim=1).item()
        actual_score = output[0, true_label]
        predicted_score = output[0, predicted_label]

        actual_score.backward()

        activation = activation_storage["activation"].detach()
        gradient = activation_storage["gradient"].detach()

        weights = gradient.mean(dim=(2, 3), keepdim=True)

        raw_cam = torch.relu(
            (weights * activation).sum(dim=1)
        )[0]

        diagnostic_rows.append({
            "actual_class": class_names[true_label],
            "predicted_class": class_names[predicted_label],
            "actual_class_logit": actual_score.item(),
            "predicted_class_logit": predicted_score.item(),
            "raw_cam_min": raw_cam.min().item(),
            "raw_cam_max": raw_cam.max().item(),
            "raw_cam_mean": raw_cam.mean().item(),
            "raw_cam_std": raw_cam.std().item(),
            "gradient_norm": gradient.norm().item()
        })

finally:
    hook.remove()

gradcam_diagnostic_df = pd.DataFrame(diagnostic_rows)

print("=== RAW GRAD-CAM DIAGNOSTICS ===")
display(gradcam_diagnostic_df.round(6))

In [ ]:
import torch
import numpy as np
import pandas as pd

target_layer = grouped_v2_model.layer4[-1]
activation_storage = {}
correct_cam_diagnostics = []

def capture_activation(module, inputs, output):
    activation_storage["activation"] = output
    output.register_hook(
        lambda gradient: activation_storage.update(
            {"gradient": gradient}
        )
    )

hook = target_layer.register_forward_hook(capture_activation)

try:
    for index, selection in selected_indices:
        if selection != "Correct":
            continue

        image_tensor, true_label = test_loader.dataset[index]
        input_tensor = image_tensor.unsqueeze(0).to(device)

        grouped_v2_model.zero_grad(set_to_none=True)
        activation_storage.clear()

        output = grouped_v2_model(input_tensor)
        predicted_label = output.argmax(dim=1).item()

        output[0, true_label].backward()

        activation = activation_storage["activation"].detach()
        gradient = activation_storage["gradient"].detach()

        weights = gradient.mean(dim=(2, 3), keepdim=True)

        raw_cam = torch.relu(
            (weights * activation).sum(dim=1)
        )[0]

        correct_cam_diagnostics.append({
            "actual_class": class_names[true_label],
            "predicted_class": class_names[predicted_label],
            "raw_cam_min": raw_cam.min().item(),
            "raw_cam_max": raw_cam.max().item(),
            "raw_cam_mean": raw_cam.mean().item(),
            "raw_cam_std": raw_cam.std().item(),
            "positive_pixel_fraction": (
                raw_cam > 0
            ).float().mean().item()
        })

finally:
    hook.remove()

correct_cam_df = pd.DataFrame(correct_cam_diagnostics)

print("=== CORRECTLY CLASSIFIED VEGETATION: GRAD-CAM DIAGNOSTICS ===")
display(correct_cam_df.round(6))

In [ ]:
import torch
import numpy as np
import pandas as pd

target_layer = grouped_v2_model.layer4[-1]
activation_storage = {}
validation_cam_rows = []

def capture_activation(module, inputs, output):
    activation_storage["activation"] = output
    output.register_hook(
        lambda gradient: activation_storage.update(
            {"gradient": gradient}
        )
    )

hook = target_layer.register_forward_hook(capture_activation)

try:
    for class_name in ["Amphibolis", "Halophila", "Posidonia"]:

        class_id = train_dataset.class_mapping[class_name]

        # Correctly classified validation examples
        candidates = np.where(
            (all_labels == class_id) &
            (all_predictions == class_id)
        )[0][:20]

        print(class_name, "images selected:", len(candidates))

        for index in candidates:
            image_tensor, true_label = val_loader.dataset[index]
            input_tensor = image_tensor.unsqueeze(0).to(device)

            grouped_v2_model.zero_grad(set_to_none=True)
            activation_storage.clear()

            output = grouped_v2_model(input_tensor)
            output[0, true_label].backward()

            activation = activation_storage["activation"].detach()
            gradient = activation_storage["gradient"].detach()

            weights = gradient.mean(dim=(2, 3), keepdim=True)

            raw_cam = torch.relu(
                (weights * activation).sum(dim=1)
            )[0]

            validation_cam_rows.append({
                "class": class_name,
                "index": int(index),
                "cam_max": raw_cam.max().item(),
                "cam_mean": raw_cam.mean().item(),
                "cam_std": raw_cam.std().item(),
                "positive_pixel_fraction": (
                    raw_cam > 0
                ).float().mean().item()
            })

finally:
    hook.remove()

validation_cam_df = pd.DataFrame(validation_cam_rows)

print("\n=== VALIDATION GRAD-CAM SUMMARY ===")

summary = validation_cam_df.groupby("class").agg(
    image_count=("index", "count"),
    mean_cam_max=("cam_max", "mean"),
    mean_cam_mean=("cam_mean", "mean"),
    mean_cam_std=("cam_std", "mean"),
    mean_positive_fraction=("positive_pixel_fraction", "mean"),
    zero_cam_images=("cam_max", lambda x: (x == 0).sum())
)

display(summary.round(6))

In [ ]:
 import torch
import numpy as np
import pandas as pd

target_layer = grouped_v2_model.layer4[-1]
activation_storage = {}
concentration_rows = []

def capture_activation(module, inputs, output):
    activation_storage["activation"] = output
    output.register_hook(
        lambda gradient: activation_storage.update(
            {"gradient": gradient}
        )
    )

hook = target_layer.register_forward_hook(capture_activation)

try:
    for index in range(len(val_loader.dataset)):
        true_label = int(all_labels[index])
        predicted_label = int(all_predictions[index])

        if true_label != predicted_label:
            continue

        class_name = class_names[true_label]

        if class_name == "Background":
            continue

        if sum(
            row["class"] == class_name
            for row in concentration_rows
        ) >= 20:
            continue

        image_tensor, _ = val_loader.dataset[index]
        input_tensor = image_tensor.unsqueeze(0).to(device)

        grouped_v2_model.zero_grad(set_to_none=True)
        activation_storage.clear()

        output = grouped_v2_model(input_tensor)
        output[0, true_label].backward()

        activation = activation_storage["activation"].detach()
        gradient = activation_storage["gradient"].detach()

        weights = gradient.mean(dim=(2, 3), keepdim=True)

        cam = torch.relu(
            (weights * activation).sum(dim=1)
        )[0].cpu().numpy().flatten()

        total_energy = cam.sum()

        if total_energy > 0:
            normalized = cam / total_energy
            sorted_values = np.sort(normalized)[::-1]

            top_count = int(np.ceil(0.20 * len(sorted_values)))
            top20_energy = sorted_values[:top_count].sum()

            entropy = -np.sum(
                normalized[normalized > 0] *
                np.log(normalized[normalized > 0])
            ) / np.log(len(normalized))

        else:
            top20_energy = 0.0
            entropy = 0.0

        concentration_rows.append({
            "class": class_name,
            "validation_index": index,
            "top20_energy_fraction": top20_energy,
            "normalized_entropy": entropy
        })

finally:
    hook.remove()

concentration_df = pd.DataFrame(concentration_rows)

print("=== GRAD-CAM SPATIAL CONCENTRATION ===")
display(
    concentration_df.groupby("class").agg(
        image_count=("validation_index", "count"),
        mean_top20_energy=("top20_energy_fraction", "mean"),
        mean_entropy=("normalized_entropy", "mean")
    ).round(4)
)

In [ ]:
import os

localization_files = {
    "validation_cam": (
        validation_cam_df,
        "grouped_v2_gradcam_validation_diagnostics.csv"
    ),
    "concentration": (
        concentration_df,
        "grouped_v2_gradcam_spatial_concentration.csv"
    ),
    "correct_vs_error": (
        pd.concat([
            correct_cam_df.assign(selection="Correct"),
            gradcam_diagnostic_df.assign(selection="Misclassified")
        ], ignore_index=True, sort=False),
        "grouped_v2_gradcam_correct_vs_error.csv"
    )
}

print("=== M3 LOCALIZATION ARTIFACT SAVE ===")

for name, (dataframe, filename) in localization_files.items():
    path = os.path.join(results_dir, filename)

    assert not os.path.exists(path), f"File already exists: {path}"

    temp_path = path + ".tmp"
    dataframe.to_csv(temp_path, index=False)
    os.replace(temp_path, path)

    print(
        name,
        "| Exists:", os.path.exists(path),
        "| Rows:", len(dataframe),
        "| Size (KB):", round(os.path.getsize(path) / 1024, 3)
    )

In [ ]:
import os
import pandas as pd

m3_files = [
    "grouped_v2_gradcam_validation_diagnostics.csv",
    "grouped_v2_gradcam_spatial_concentration.csv",
    "grouped_v2_gradcam_correct_vs_error.csv"
]

print("=== M3 FINAL ARTIFACT AUDIT ===")

all_passed = True

for filename in m3_files:
    path = os.path.join(results_dir, filename)

    exists = os.path.exists(path)
    print(f"\n{filename}")
    print("Exists:", exists)

    if exists:
        df = pd.read_csv(path)
        print("Rows:", len(df))
        print("Columns:", len(df.columns))
    else:
        all_passed = False

print("\nM3 artifact audit passed:", all_passed)

In [ ]:
import os
from pathlib import Path

project_dir = Path("/content/drive/MyDrive/Marine_Project")

folders = {
    "Models": project_dir / "models",
    "Results": project_dir / "results",
    "Outputs": project_dir / "outputs",
}

print("=== PROJECT BACKUP PREFLIGHT ===")
print("Project folder exists:", project_dir.exists())

for folder_name, folder_path in folders.items():
    print(f"\n--- {folder_name} ---")

    if not folder_path.exists():
        print("Folder not found")
        continue

    files = [p for p in folder_path.rglob("*") if p.is_file()]
    total_size = sum(p.stat().st_size for p in files)

    print("File count:", len(files))
    print("Total size (GB):", round(total_size / (1024**3), 3))

    for file in files:
        print(f"{file.relative_to(project_dir)} | "
              f"{file.stat().st_size / (1024**2):.2f} MB")

print("\n--- CURRENT COLAB CODE HISTORY ---")
try:
    print("Executed input entries:", len(In) - 1)
except Exception as e:
    print("Code history unavailable:", str(e))

In [ ]:
from pathlib import Path

drive_root = Path("/content/drive/MyDrive")
project_dir = drive_root / "Marine_Project"

print("=== NOTEBOOK SEARCH ===")

notebooks = list(project_dir.rglob("*.ipynb"))

print("\nNotebooks inside Marine_Project:", len(notebooks))

for nb in notebooks:
    print(nb)
    print("Size:", round(nb.stat().st_size / (1024**2), 2), "MB")

print("\n=== EXISTING BACKUP FOLDERS ===")

for item in sorted(project_dir.iterdir()):
    if item.is_dir() and item.name.startswith("backup_"):
        print(item.name)

In [ ]:
from pathlib import Path

drive_root = Path("/content/drive/MyDrive")

print("Searching Google Drive for notebooks...")

notebooks = list(drive_root.rglob("*.ipynb"))

print("\nTotal notebooks found:", len(notebooks))

for nb in notebooks:
    print(f"{nb} | {nb.stat().st_size / (1024**2):.2f} MB")

In [ ]:
import json
from pathlib import Path

drive_root = Path("/content/drive/MyDrive")
notebooks = list(drive_root.rglob("*.ipynb"))

print("=== NOTEBOOK CELL COUNT AUDIT ===")

for path in notebooks:
    try:
        with open(path, "r", encoding="utf-8") as f:
            nb = json.load(f)

        cells = nb.get("cells", [])
        if len(cells) >= 50:
            code_cells = sum(c.get("cell_type") == "code" for c in cells)
            markdown_cells = sum(c.get("cell_type") == "markdown" for c in cells)

            print(f"\nNotebook: {path}")
            print("Total cells:", len(cells))
            print("Code cells:", code_cells)
            print("Markdown cells:", markdown_cells)
            print("Size (MB):", round(path.stat().st_size / (1024**2), 2))

    except Exception as e:
        print("Could not inspect:", path.name, "|", str(e))

In [ ]:
import os
import json
import shutil
import hashlib
from pathlib import Path
from datetime import datetime

project_dir = Path("/content/drive/MyDrive/Marine_Project")
notebook_source = Path("/content/drive/MyDrive/Colab Notebooks/Mini Project.ipynb")

# Create a unique backup folder
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
backup_dir = project_dir / f"backup_M3_complete_{timestamp}"
backup_dir.mkdir(parents=True, exist_ok=False)

print("Creating backup:", backup_dir)

# Copy the complete notebook
shutil.copy2(notebook_source, backup_dir / "Mini Project.ipynb")

# Copy all project artifacts
for folder in ["models", "results", "outputs"]:
    source = project_dir / folder
    destination = backup_dir / folder

    if source.exists():
        shutil.copytree(source, destination)
        print("Copied:", folder)

# Create a backup inventory and SHA256 checksums
inventory = []

for file in sorted(backup_dir.rglob("*")):
    if file.is_file():
        sha256 = hashlib.sha256()

        with open(file, "rb") as f:
            for chunk in iter(lambda: f.read(1024 * 1024), b""):
                sha256.update(chunk)

        inventory.append({
            "file": str(file.relative_to(backup_dir)),
            "size_bytes": file.stat().st_size,
            "sha256": sha256.hexdigest()
        })

# Read notebook cell count from the backed-up copy
with open(backup_dir / "Mini Project.ipynb", "r", encoding="utf-8") as f:
    saved_notebook = json.load(f)

cell_count = len(saved_notebook.get("cells", []))
code_count = sum(c.get("cell_type") == "code" for c in saved_notebook["cells"])
markdown_count = sum(c.get("cell_type") == "markdown" for c in saved_notebook["cells"])

backup_info = {
    "backup_created": timestamp,
    "notebook": "Mini Project.ipynb",
    "notebook_cells": cell_count,
    "code_cells": code_count,
    "markdown_cells": markdown_count,
    "artifact_files": len(inventory),
    "files": inventory
}

with open(backup_dir / "BACKUP_MANIFEST.json", "w", encoding="utf-8") as f:
    json.dump(backup_info, f, indent=2)

total_bytes = sum(item["size_bytes"] for item in inventory)

print("\n=== BACKUP CREATION SUMMARY ===")
print("Backup folder:", backup_dir)
print("Notebook cells:", cell_count)
print("Code cells:", code_count)
print("Markdown cells:", markdown_count)
print("Copied artifact files:", len(inventory))
print("Copied size (GB):", round(total_bytes / (1024**3), 3))
print("Manifest created:", (backup_dir / "BACKUP_MANIFEST.json").exists())
print("\nBackup creation finished. Verification is the next step.")

In [ ]:
import hashlib
import json
from pathlib import Path

project_dir = Path("/content/drive/MyDrive/Marine_Project")
backup_dir = project_dir / "backup_M3_complete_20261114_091542"

def sha256_file(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

print("=== INDEPENDENT BACKUP VERIFICATION ===")

checks = []

# Notebook verification
original_nb = Path("/content/drive/MyDrive/Colab Notebooks/Mini Project.ipynb")
backup_nb = backup_dir / "Mini Project.ipynb"

notebook_match = (
    original_nb.exists()
    and backup_nb.exists()
    and sha256_file(original_nb) == sha256_file(backup_nb)
)

checks.append(notebook_match)
print("\nNotebook checksum match:", notebook_match)

# Artifact verification
for folder in ["models", "results", "outputs"]:
    source = project_dir / folder
    copied = backup_dir / folder

    source_files = sorted(
        p.relative_to(source) for p in source.rglob("*") if p.is_file()
    )
    copied_files = sorted(
        p.relative_to(copied) for p in copied.rglob("*") if p.is_file()
    )

    same_names = source_files == copied_files

    checksum_matches = 0
    for relative_path in source_files:
        source_file = source / relative_path
        copied_file = copied / relative_path

        if copied_file.exists() and sha256_file(source_file) == sha256_file(copied_file):
            checksum_matches += 1

    folder_pass = same_names and checksum_matches == len(source_files)
    checks.append(folder_pass)

    print(f"\n{folder}")
    print("Original files:", len(source_files))
    print("Backup files:", len(copied_files))
    print("Matching checksums:", checksum_matches)
    print("Folder verification:", folder_pass)

# Notebook cell count
with open(backup_nb, "r", encoding="utf-8") as f:
    saved_nb = json.load(f)

print("\nSaved notebook cell count:", len(saved_nb["cells"]))

print("\n=== FINAL VERIFICATION ===")
print("Passed checks:", sum(checks), "/", len(checks))
print("BACKUP VERIFIED:", all(checks))

In [ ]:
import json
import hashlib
from pathlib import Path
from datetime import datetime

original = Path("/content/drive/MyDrive/Colab Notebooks/Mini Project.ipynb")
backup = Path("/content/drive/MyDrive/Marine_Project/backup_M3_complete_20261114_091542/Mini Project.ipynb")

def file_hash(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

for name, path in [("ORIGINAL", original), ("BACKUP", backup)]:
    with open(path, "r", encoding="utf-8") as f:
        nb = json.load(f)

    cells = nb.get("cells", [])

    print(f"\n=== {name} NOTEBOOK ===")
    print("Size (MB):", round(path.stat().st_size / (1024**2), 3))
    print("Last modified:", datetime.fromtimestamp(path.stat().st_mtime))
    print("Total cells:", len(cells))
    print("Code cells:", sum(c.get("cell_type") == "code" for c in cells))
    print("Markdown cells:", sum(c.get("cell_type") == "markdown" for c in cells))
    print("SHA256:", file_hash(path))

print("\nNotebook files are only being inspected. Nothing modified.")

In [ ]:
import shutil
import hashlib
import json
from pathlib import Path

original = Path("/content/drive/MyDrive/Colab Notebooks/Mini Project.ipynb")

backup_dir = Path(
    "/content/drive/MyDrive/Marine_Project/backup_M3_complete_20261114_091542"
)

snapshot_dir = backup_dir / "latest_notebook_snapshot"
snapshot_dir.mkdir(exist_ok=True)

snapshot = snapshot_dir / "Mini Project_latest.ipynb"

# Refuse to overwrite an existing snapshot
if snapshot.exists():
    raise FileExistsError(f"Snapshot already exists: {snapshot}")

shutil.copy2(original, snapshot)

def sha256_file(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

original_hash = sha256_file(original)
snapshot_hash = sha256_file(snapshot)

with open(snapshot, "r", encoding="utf-8") as f:
    nb = json.load(f)

print("=== LATEST NOTEBOOK SNAPSHOT ===")
print("Snapshot:", snapshot)
print("Total cells:", len(nb["cells"]))
print("Code cells:", sum(c["cell_type"] == "code" for c in nb["cells"]))
print("Markdown cells:", sum(c["cell_type"] == "markdown" for c in nb["cells"]))
print("Original SHA256:", original_hash)
print("Snapshot SHA256:", snapshot_hash)
print("Checksum match:", original_hash == snapshot_hash)

In [ ]:
import hashlib
import json
from pathlib import Path

project = Path("/content/drive/MyDrive/Marine_Project")
backup = project / "backup_M3_complete_20261114_091542"

def sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

checks = {}

# Latest notebook
original_nb = Path("/content/drive/MyDrive/Colab Notebooks/Mini Project.ipynb")
snapshot_nb = backup / "latest_notebook_snapshot/Mini Project_latest.ipynb"

checks["Latest notebook"] = (
    original_nb.exists()
    and snapshot_nb.exists()
    and sha256(original_nb) == sha256(snapshot_nb)
)

with open(snapshot_nb, "r", encoding="utf-8") as f:
    latest = json.load(f)

print("Latest notebook cells:", len(latest["cells"]))
print("Latest notebook code cells:",
      sum(c["cell_type"] == "code" for c in latest["cells"]))

# Artifact folders
for folder in ["models", "results", "outputs"]:
    source = project / folder
    destination = backup / folder

    source_files = {
        p.relative_to(source): p
        for p in source.rglob("*") if p.is_file()
    }
    backup_files = {
        p.relative_to(destination): p
        for p in destination.rglob("*") if p.is_file()
    }

    same_files = source_files.keys() == backup_files.keys()

    same_hashes = same_files and all(
        sha256(source_files[name]) == sha256(backup_files[name])
        for name in source_files
    )

    checks[folder] = same_hashes

# Original historical notebook copy is preserved
checks["Historical notebook copy exists"] = (
    (backup / "Mini Project.ipynb").exists()
)

# Backup manifest
manifest_path = backup / "BACKUP_MANIFEST.json"
checks["Backup manifest exists"] = manifest_path.exists()

print("\n=== FINAL BACKUP AUDIT ===")
for name, passed in checks.items():
    print(f"{name}: {'PASS' if passed else 'FAIL'}")

print("\nPassed:", sum(checks.values()), "/", len(checks))
print("FINAL BACKUP STATUS:", "VERIFIED" if all(checks.values()) else "NOT VERIFIED")
print("Backup location:", backup)

In [ ]:

# ============================================================
# MARINE VEGETATION RESEARCH PROJECT
# MASTER RESTORE CELL — AFTER RUNTIME RESTART
# ============================================================
# Project: Fine-Grained Discrimination of Invasive and Native
# Marine Vegetation Using a Hybrid Deep Learning Model
# ============================================================
# PURPOSE:
# 1. Mount Google Drive
# 2. Restore project paths and ZIP dataset access
# 3. Load M1 manifests
# 4. Restore M2 ResNet18 best checkpoint
# 5. Recreate datasets and DataLoaders
# 6. Load M2/M3 result artifacts
# 7. Verify the M3 backup
#
# This cell DOES NOT retrain the model or modify old results.
# ============================================================

import os
import sys
import json
import hashlib
import zipfile
import io
import gc
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd
from PIL import Image

import torch
import torch.nn as nn
import torchvision
import torchvision.transforms as transforms
from torchvision import models
from torch.utils.data import Dataset, DataLoader

# ------------------------------------------------------------
# 1. GOOGLE DRIVE CONNECTION
# ------------------------------------------------------------

print("=" * 65)
print("STEP 1 — CONNECTING GOOGLE DRIVE")
print("=" * 65)

if not Path("/content/drive/MyDrive").exists():
    from google.colab import drive
    drive.mount("/content/drive")
else:
    print("Google Drive is already accessible.")

# ------------------------------------------------------------
# 2. RESTORE PROJECT DIRECTORIES
# ------------------------------------------------------------

PROJECT_DIR = Path("/content/drive/MyDrive/Marine_Project")

ZIP_DIR = PROJECT_DIR / "dataset_zips"
MODELS_DIR = PROJECT_DIR / "models"
RESULTS_DIR = PROJECT_DIR / "results"
OUTPUTS_DIR = PROJECT_DIR / "outputs"

BACKUP_DIR = (
    PROJECT_DIR /
    "backup_M3_complete_20261114_091542"
)

NOTEBOOK_PATH = (
    Path("/content/drive/MyDrive/Colab Notebooks")
    / "Mini Project.ipynb"
)

assert PROJECT_DIR.exists(), "Marine_Project folder not found!"
assert ZIP_DIR.exists(), "Dataset ZIP directory not found!"
assert MODELS_DIR.exists(), "Models directory not found!"
assert RESULTS_DIR.exists(), "Results directory not found!"

print("Project directory:", PROJECT_DIR)
print("ZIP directory:", ZIP_DIR)
print("Models directory:", MODELS_DIR)
print("Results directory:", RESULTS_DIR)
print("Outputs directory:", OUTPUTS_DIR)

# ------------------------------------------------------------
# 3. VERIFY ALL EIGHT DATASET ARCHIVES
# ------------------------------------------------------------

print("\n" + "=" * 65)
print("STEP 2 — DATASET ZIP RESTORATION")
print("=" * 65)

EXPECTED_ZIPS = [
    "ECU-MSS-2 Amphibolis_training.zip",
    "ECU-MSS-2 Amphibolis_testing.zip",
    "ECU-MSS-2 Background_training.zip",
    "ECU-MSS-2 Background_testing.zip",
    "ECU-MSS-2 Halophila_training.zip",
    "ECU-MSS-2 Halophila_testing.zip",
    "ECU-MSS-2 Posidonia_training.zip",
    "ECU-MSS-2 Posidonia_testing.zip"
]

ZIP_PATHS = {}

for filename in EXPECTED_ZIPS:
    path = ZIP_DIR / filename
    assert path.exists(), f"Missing ZIP: {filename}"
    ZIP_PATHS[filename] = path
    print(f"FOUND: {filename}")

print("\nTotal ZIP archives:", len(ZIP_PATHS))

# ------------------------------------------------------------
# 4. RESTORE M1 MANIFESTS
# ------------------------------------------------------------

print("\n" + "=" * 65)
print("STEP 3 — RESTORING M1 MANIFESTS")
print("=" * 65)

TRAIN_MANIFEST_PATH = (
    RESULTS_DIR / "clean_train_manifest_grouped_v2.csv"
)

VAL_MANIFEST_PATH = (
    RESULTS_DIR / "clean_validation_manifest_grouped_v2.csv"
)

TEST_MANIFEST_PATH = (
    RESULTS_DIR / "clean_test_manifest.csv"
)

for path in [
    TRAIN_MANIFEST_PATH,
    VAL_MANIFEST_PATH,
    TEST_MANIFEST_PATH
]:
    assert path.exists(), f"Missing manifest: {path.name}"

train_df = pd.read_csv(TRAIN_MANIFEST_PATH)
val_df = pd.read_csv(VAL_MANIFEST_PATH)
test_df = pd.read_csv(TEST_MANIFEST_PATH)

print("Training:", train_df.shape)
print("Validation:", val_df.shape)
print("Clean test:", test_df.shape)

print("\nTraining class counts:")
print(train_df["class"].value_counts())

print("\nValidation class counts:")
print(val_df["class"].value_counts())

print("\nTest class counts:")
print(test_df["class"].value_counts())

# ------------------------------------------------------------
# 5. VERIFY ZIP REFERENCES IN MANIFESTS
# ------------------------------------------------------------

print("\n" + "=" * 65)
print("STEP 4 — VERIFYING MANIFEST ZIP REFERENCES")
print("=" * 65)

for name, df in [
    ("Training", train_df),
    ("Validation", val_df),
    ("Test", test_df)
]:
    missing = [
        z for z in df["zip_file"].unique()
        if not (ZIP_DIR / z).exists()
    ]

    print(name, "| Records:", len(df),
          "| Missing ZIP references:", len(missing))

    assert len(missing) == 0, f"{name} ZIP references missing!"

# ------------------------------------------------------------
# 6. RESTORE CLASS MAPPING AND PREPROCESSING
# ------------------------------------------------------------

print("\n" + "=" * 65)
print("STEP 5 — RESTORING MODEL CONFIGURATION")
print("=" * 65)

class_mapping = {
    "Amphibolis": 0,
    "Background": 1,
    "Halophila": 2,
    "Posidonia": 3
}

idx_to_class = {
    value: key for key, value in class_mapping.items()
}

IMAGE_SIZE = 224
BATCH_SIZE = 32

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

print("Device:", device)
print("Image size:", IMAGE_SIZE)
print("Batch size:", BATCH_SIZE)
print("Class mapping:", class_mapping)

# ------------------------------------------------------------
# 7. RESTORE ZIP-BASED DATASET CLASS
# ------------------------------------------------------------

print("\n" + "=" * 65)
print("STEP 6 — RESTORING ZIP DATASET LOADER")
print("=" * 65)

class MarineZipDatasetV2(Dataset):

    def __init__(self, dataframe, zip_dir, transform=None):
        self.df = dataframe.reset_index(drop=True)
        self.zip_dir = Path(zip_dir)
        self.transform = transform
        self.zip_handles = {}

    def __len__(self):
        return len(self.df)

    def __getitem__(self, index):

        row = self.df.iloc[index]

        zip_name = row["zip_file"]
        image_name = row["image_name"]

        zip_path = self.zip_dir / zip_name

        if zip_name not in self.zip_handles:
            self.zip_handles[zip_name] = zipfile.ZipFile(
                zip_path, "r"
            )

        archive = self.zip_handles[zip_name]

        try:
            image_bytes = archive.read(image_name)
        except KeyError:
            raise FileNotFoundError(
                f"Image {image_name} not found in {zip_name}"
            )

        image = Image.open(
            io.BytesIO(image_bytes)
        ).convert("RGB")

        if self.transform:
            image = self.transform(image)

        label = class_mapping[row["class"]]

        return image, label

    def close(self):
        for archive in self.zip_handles.values():
            archive.close()
        self.zip_handles.clear()

# ------------------------------------------------------------
# 8. RECREATE DATASETS AND DATALOADERS
# ------------------------------------------------------------

print("\n" + "=" * 65)
print("STEP 7 — RECREATING DATALOADERS")
print("=" * 65)

train_dataset = MarineZipDatasetV2(
    train_df, ZIP_DIR, transform
)

val_dataset = MarineZipDatasetV2(
    val_df, ZIP_DIR, transform
)

test_dataset = MarineZipDatasetV2(
    test_df, ZIP_DIR, transform
)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0,
    pin_memory=torch.cuda.is_available()
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=torch.cuda.is_available()
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=torch.cuda.is_available()
)

print("Training batches:", len(train_loader))
print("Validation batches:", len(val_loader))
print("Test batches:", len(test_loader))

# ------------------------------------------------------------
# 9. RESTORE GROUPED-V2 RESNET18 BEST CHECKPOINT
# ------------------------------------------------------------

print("\n" + "=" * 65)
print("STEP 8 — RESTORING BEST RESNET18 CHECKPOINT")
print("=" * 65)

BEST_MODEL_PATH = (
    MODELS_DIR /
    "resnet18_grouped_v2_baseline_best.pth"
)

LATEST_MODEL_PATH = (
    MODELS_DIR /
    "resnet18_grouped_v2_baseline_latest.pth"
)

assert BEST_MODEL_PATH.exists(), "Best checkpoint missing!"
assert LATEST_MODEL_PATH.exists(), "Latest checkpoint missing!"

checkpoint = torch.load(
    BEST_MODEL_PATH,
    map_location=device,
    weights_only=False
)

assert "model_state_dict" in checkpoint
assert checkpoint["class_mapping"] == class_mapping

model = models.resnet18(weights=None)
model.fc = nn.Linear(
    model.fc.in_features,
    len(class_mapping)
)

model.load_state_dict(
    checkpoint["model_state_dict"],
    strict=True
)

model = model.to(device)
model.eval()

print("Best checkpoint loaded successfully.")
print("Selected epoch:", checkpoint["epoch"])
print("Best validation loss:", checkpoint["best_val_loss"])
print("Model:", "ResNet18")
print("Device:", device)

# ------------------------------------------------------------
# 10. RESTORE M2 AND M3 RESULT TABLES
# ------------------------------------------------------------

print("\n" + "=" * 65)
print("STEP 9 — RESTORING RESEARCH RESULTS")
print("=" * 65)

result_files = list(RESULTS_DIR.glob("*.csv"))

results_tables = {}

for path in result_files:
    try:
        results_tables[path.stem] = pd.read_csv(path)
    except Exception as e:
        print("Could not load:", path.name, "|", str(e))

print("Loaded CSV result tables:", len(results_tables))

# Convenient M3 references
gradcam_validation_df = results_tables.get(
    "grouped_v2_gradcam_validation_diagnostics"
)

gradcam_spatial_df = results_tables.get(
    "grouped_v2_gradcam_spatial_concentration"
)

gradcam_correct_error_df = results_tables.get(
    "grouped_v2_gradcam_correct_vs_error"
)

# Convenient M2 references
test_report_df = results_tables.get(
    "grouped_v2_baseline_clean_test_report"
)

test_predictions_df = results_tables.get(
    "grouped_v2_baseline_clean_test_predictions"
)

test_confusion_df = results_tables.get(
    "grouped_v2_baseline_clean_test_confusion_matrix"
)

# ------------------------------------------------------------
# 11. VERIFY M3 BACKUP AND LATEST NOTEBOOK
# ------------------------------------------------------------

print("\n" + "=" * 65)
print("STEP 10 — BACKUP AND NOTEBOOK VERIFICATION")
print("=" * 65)

snapshot_path = (
    BACKUP_DIR /
    "latest_notebook_snapshot" /
    "Mini Project_latest.ipynb"
)

def calculate_sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(
            lambda: f.read(1024 * 1024), b""
        ):
            h.update(chunk)
    return h.hexdigest()

backup_checks = {}

backup_checks["Backup folder"] = BACKUP_DIR.exists()
backup_checks["Notebook snapshot"] = snapshot_path.exists()

if snapshot_path.exists():
    backup_checks["Notebook checksum"] = (
        NOTEBOOK_PATH.exists()
        and calculate_sha256(snapshot_path)
        == calculate_sha256(NOTEBOOK_PATH)
    )

for folder in ["models", "results", "outputs"]:
    backup_checks[folder] = (
        (BACKUP_DIR / folder).exists()
    )

for name, status in backup_checks.items():
    print(name, ":", "PASS" if status else "FAIL")

# ------------------------------------------------------------
# 12. VERIFY THREE M3 ARTIFACTS
# ------------------------------------------------------------

print("\n" + "=" * 65)
print("STEP 11 — M3 ARTIFACT VERIFICATION")
print("=" * 65)

m3_files = [
    "grouped_v2_gradcam_validation_diagnostics.csv",
    "grouped_v2_gradcam_spatial_concentration.csv",
    "grouped_v2_gradcam_correct_vs_error.csv"
]

m3_status = True

for filename in m3_files:
    path = RESULTS_DIR / filename
    exists = path.exists()

    print(filename, ":", exists)

    if not exists:
        m3_status = False

# ------------------------------------------------------------
# 13. SAMPLE IMAGE DECODING CHECK
# ------------------------------------------------------------

print("\n" + "=" * 65)
print("STEP 12 — DATASET RESTORATION TEST")
print("=" * 65)

for name, dataset in [
    ("Training", train_dataset),
    ("Validation", val_dataset),
    ("Test", test_dataset)
]:
    sample_image, sample_label = dataset[0]

    print(
        name,
        "| Tensor:", tuple(sample_image.shape),
        "| Label:", idx_to_class[sample_label],
        "| Finite:", torch.isfinite(sample_image).all().item()
    )

# ------------------------------------------------------------
# 14. FINAL RESTORE SUMMARY
# ------------------------------------------------------------

print("\n" + "=" * 65)
print("MARINE PROJECT RESTORE SUMMARY")
print("=" * 65)

print("M1 — Data Audit: COMPLETE")
print("M2 — Four-Class Baseline: COMPLETE")
print("M3 — Exploratory Localization: COMPLETE")
print("M4 — Invasive/Native Dataset: NOT STARTED")
print("M5 — Invasive/Native Classification: PENDING")
print("M6 — Selective Removal Decision System: PENDING")

print("\nTraining images:", len(train_dataset))
print("Validation images:", len(val_dataset))
print("Clean test images:", len(test_dataset))

print("Best model epoch:", checkpoint["epoch"])
print("Model evaluation mode:", not model.training)

print("\nM3 artifacts available:", m3_status)
print("Backup checks passed:", sum(backup_checks.values()),
      "/", len(backup_checks))

print("\nRESTORE CELL EXECUTION FINISHED")
print("No model retraining performed.")
print("No existing research results overwritten.")
print("Next milestone: M4 — Western Australian ecological dataset.")
print("=" * 65)


In [ ]:
import hashlib
import json
from pathlib import Path
from datetime import datetime

live = Path(NOTEBOOK_PATH)
snapshot = BACKUP_DIR / "latest_notebook_snapshot" / "Mini Project_latest.ipynb"

def sha256_file(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

for label, path in [("LIVE NOTEBOOK", live), ("BACKUP SNAPSHOT", snapshot)]:
    print(f"\n{label}")
    print("Exists:", path.exists())

    if path.exists():
        data = json.loads(path.read_text(encoding="utf-8"))
        cells = data.get("cells", [])

        print("Size (MB):", round(path.stat().st_size / (1024 * 1024), 3))
        print("Total cells:", len(cells))
        print("Code cells:", sum(c.get("cell_type") == "code" for c in cells))
        print("Markdown cells:", sum(c.get("cell_type") == "markdown" for c in cells))
        print("Modified:", datetime.fromtimestamp(path.stat().st_mtime))
        print("SHA256:", sha256_file(path))

print("\nBYTE-FOR-BYTE MATCH:", sha256_file(live) == sha256_file(snapshot))

In [ ]:
import shutil
import hashlib
from pathlib import Path

live = Path(NOTEBOOK_PATH)
snapshot_dir = BACKUP_DIR / "latest_notebook_snapshot"

new_backup = snapshot_dir / "Mini_Project_live_20261114_144847.ipynb"

if new_backup.exists():
    raise FileExistsError(f"Backup already exists: {new_backup}")

shutil.copy2(live, new_backup)

def sha256_file(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

live_hash = sha256_file(live)
backup_hash = sha256_file(new_backup)

print("New notebook snapshot:", new_backup.name)
print("Live notebook SHA256:", live_hash)
print("Backup SHA256:", backup_hash)
print("Checksum match:", live_hash == backup_hash)
print("Backup exists:", new_backup.exists())

In [ ]:
import requests

url = "https://zenodo.org/api/records/14185867"

response = requests.get(url, timeout=30)
print("HTTP status:", response.status_code)

if response.status_code == 200:
    record = response.json()

    print("\nDataset:", record["metadata"]["title"])
    print("DOI:", record["metadata"].get("doi"))
    print("License:", record["metadata"].get("license", {}).get("id", "Check metadata"))

    print("\nAvailable files:")
    for file in record.get("files", []):
        print(
            file["key"],
            "| Size (GB):",
            round(file["size"] / (1024**3), 2)
        )
else:
    print(response.text[:500])

In [ ]:
print("DESCRIPTION:\n")
print(record["metadata"].get("description", "Not available"))

print("\n\nKEYWORDS:")
print(record["metadata"].get("keywords", []))

print("\n\nRELATED IDENTIFIERS:")
for item in record["metadata"].get("related_identifiers", []):
    print(item)

In [ ]:
import requests

file_info = record["files"][0]
file_url = file_info["links"]["self"]

response = requests.head(file_url, allow_redirects=True, timeout=30)

print("File:", file_info["key"])
print("Size (GB):", round(file_info["size"] / (1024**3), 2))
print("HTTP status:", response.status_code)
print("Accept-Ranges:", response.headers.get("Accept-Ranges"))
print("Content-Length:", response.headers.get("Content-Length"))
print("Final URL available:", bool(response.url))

In [ ]:
import requests

response = requests.get(
    file_url,
    headers={"Range": "bytes=0-1023"},
    stream=True,
    timeout=30
)

print("HTTP status:", response.status_code)
print("Content-Range:", response.headers.get("Content-Range"))
print("Bytes received:", len(next(response.iter_content(1024), b"")))

response.close()

In [ ]:
import requests
import zipfile
import io

class RemoteZipReader:
    def __init__(self, url, size):
        self.url = url
        self.size = size
        self.pos = 0
        self.session = requests.Session()

    def seekable(self):
        return True

    def tell(self):
        return self.pos

    def seek(self, offset, whence=0):
        if whence == 0:
            self.pos = offset
        elif whence == 1:
            self.pos += offset
        elif whence == 2:
            self.pos = self.size + offset
        return self.pos

    def read(self, n=-1):
        if n < 0:
            n = self.size - self.pos

        end = min(self.pos + n, self.size)
        if end <= self.pos:
            return b""

        response = self.session.get(
            self.url,
            headers={"Range": f"bytes={self.pos}-{end-1}"},
            timeout=60
        )
        response.raise_for_status()

        if response.status_code != 206:
            raise RuntimeError("Server did not return partial ZIP data.")

        data = response.content
        self.pos += len(data)
        return data

remote_file = RemoteZipReader(file_url, file_info["size"])

with zipfile.ZipFile(remote_file) as z:
    entries = z.infolist()

    print("ZIP directory read successfully")
    print("Total entries:", len(entries))
    print("\nFirst 40 entries:")

    for item in entries[:40]:
        print(item.filename, "|", round(item.file_size / 1024, 1), "KB")

    print("\nFile types:")
    from collections import Counter
    print(Counter(
        item.filename.rsplit(".", 1)[-1].lower()
        for item in entries if "." in item.filename
    ))

In [ ]:
print("CSV FILES:")
for item in entries:
    if item.filename.lower().endswith(".csv"):
        print(item.filename, "|", round(item.file_size / 1024, 2), "KB")

print("\nEXCEL FILES:")
for item in entries:
    if item.filename.lower().endswith((".xlsx", ".xls")):
        print(item.filename, "|", round(item.file_size / 1024, 2), "KB")

print("\nPNG IMAGE PATH EXAMPLES:")
png_files = [x for x in entries if x.filename.lower().endswith(".png")]

for item in png_files[:20]:
    print(item.filename)

print("\nTotal PNG images:", len(png_files))

In [ ]:
import pandas as pd
import io

with zipfile.ZipFile(remote_file) as z:
    for filename in [
        "dataset_comprimir/Tables/Classes.csv",
        "dataset_comprimir/Tables/Labels.csv"
    ]:
        print("\n" + "="*60)
        print(filename)
        print("="*60)

        data = z.read(filename)
        df = pd.read_csv(io.BytesIO(data))

        print("Shape:", df.shape)
        print("Columns:", df.columns.tolist())
        display(df.head(10))

In [ ]:
with zipfile.ZipFile(remote_file) as z:
    classes_df = pd.read_csv(
        io.BytesIO(z.read("dataset_comprimir/Tables/Classes.csv")),
        sep=";"
    )

    labels_df = pd.read_csv(
        io.BytesIO(z.read("dataset_comprimir/Tables/Labels.csv")),
        sep=";"
    )

species_summary = (
    labels_df.groupby("Sp_id")
    .size()
    .reset_index(name="image_count")
    .merge(classes_df, on="Sp_id", how="left")
    .sort_values("image_count", ascending=False)
)

print("Classes columns:", classes_df.columns.tolist())
print("Labels columns:", labels_df.columns.tolist())
print("\nTotal labelled images:", len(labels_df))
print("\nSpecies-wise image counts:")

display(species_summary)

In [ ]:
split_summary = (
    labels_df.merge(
        classes_df[["Sp_id", "Especies", "General"]],
        on="Sp_id",
        how="left"
    )
    .groupby(["Dataset_type", "Especies"])
    .size()
    .reset_index(name="images")
)

print("Dataset split values:")
print(labels_df["Dataset_type"].value_counts(dropna=False))

print("\nSpecies distribution by split:")
display(split_summary)

In [ ]:
# Check whether the same orthomosaic occurs in both splits

train_orth = set(
    labels_df.loc[labels_df["Dataset_type"] == "Train", "Orth_id"].dropna()
)

val_orth = set(
    labels_df.loc[labels_df["Dataset_type"] == "Validation", "Orth_id"].dropna()
)

overlap = train_orth.intersection(val_orth)

print("Unique training orthomosaics:", len(train_orth))
print("Unique validation orthomosaics:", len(val_orth))
print("Orthomosaic overlap:", len(overlap))

print("\nOverlapping Orth_id values:")
print(sorted(overlap)[:30])

print("\nUnique orthomosaics per species and split:")

orth_summary = (
    labels_df.merge(
        classes_df[["Sp_id", "Especies"]],
        on="Sp_id",
        how="left"
    )
    .groupby(["Dataset_type", "Especies"])["Orth_id"]
    .nunique()
    .reset_index(name="unique_orthomosaics")
)

display(orth_summary)

In [ ]:
# Reopen the remote ZIP archive
z = zipfile.ZipFile(remote_file, "r")

# Inspect metadata tables without extracting the full dataset
for filename in [
    "dataset_comprimir/Tables/Orthoimages.csv",
    "dataset_comprimir/Tables/Flights.csv",
    "dataset_comprimir/Tables/Locations.csv"
]:
    df = pd.read_csv(
        io.BytesIO(z.read(filename)),
        sep=";"
    )

    print("\n" + "=" * 65)
    print(filename)
    print("Shape:", df.shape)
    print("Columns:", df.columns.tolist())
    display(df.head(5))

In [ ]:
# Standardize metadata join columns

labels_df["Orth_id"] = pd.to_numeric(labels_df["Orth_id"], errors="coerce")
ortho_df["Orth_id"] = pd.to_numeric(ortho_df["Orth_id"], errors="coerce")
ortho_df["F_id"] = pd.to_numeric(ortho_df["F_id"], errors="coerce")
flights_df["F_id"] = pd.to_numeric(flights_df["F_id"], errors="coerce")
flights_df["L_id"] = pd.to_numeric(flights_df["L_id"], errors="coerce")
locations_df["L_id"] = pd.to_numeric(locations_df["L_id"], errors="coerce")

# Merge metadata
metadata_df = (
    labels_df
    .merge(ortho_df, on="Orth_id", how="left")
    .merge(flights_df, on="F_id", how="left")
    .merge(locations_df, on="L_id", how="left")
)

print("Images without location:", metadata_df["Location"].isna().sum())

print("\nImages per split and location:")
display(
    metadata_df.groupby(["Dataset_type", "Location"])
    .size()
    .reset_index(name="image_count")
)

print("\nImages per split and flight date:")
display(
    metadata_df.groupby(["Dataset_type", "Date"])
    .size()
    .reset_index(name="image_count")
)

print("\nUnique flights per split:")
display(
    metadata_df.groupby("Dataset_type")["F_id"]
    .nunique()
    .reset_index(name="unique_flights")
)

In [ ]:
# Check shared flight IDs between Train and Validation

train_flights = set(
    metadata_df.loc[
        metadata_df["Dataset_type"] == "Train", "F_id"
    ].dropna()
)

val_flights = set(
    metadata_df.loc[
        metadata_df["Dataset_type"] == "Validation", "F_id"
    ].dropna()
)

shared_flights = train_flights.intersection(val_flights)

print("Training flight IDs:", sorted(train_flights))
print("Validation flight IDs:", sorted(val_flights))
print("Shared flight IDs:", sorted(shared_flights))
print("Number of shared flights:", len(shared_flights))

print("\nOrthomosaics belonging to shared flights:")
display(
    metadata_df[
        metadata_df["F_id"].isin(shared_flights)
    ].groupby(["Dataset_type", "F_id", "Date"])["Orth_id"]
    .nunique()
    .reset_index(name="orthomosaic_count")
)

In [ ]:
print("Available columns:")
print(metadata_df.columns.tolist())

# Flight and location mapping
flight_audit = (
    metadata_df.groupby(
        ["F_id", "Date", "Location", "Dataset_type"]
    )
    .agg(
        images=("Im_id", "count"),
        orthomosaics=("Orth_id", "nunique")
    )
    .reset_index()
    .sort_values(["F_id", "Dataset_type"])
)

display(flight_audit)

In [ ]:
# Species distribution across all 9 flights

species_flight = (
    metadata_df
    .merge(
        classes_df[["Sp_id", "Especies"]],
        on="Sp_id",
        how="left"
    )
    .pivot_table(
        index="Especies",
        columns="F_id",
        values="Im_id",
        aggfunc="count",
        fill_value=0
    )
)

display(species_flight)

print("\nTotal images per flight:")
display(
    metadata_df.groupby("F_id")
    .agg(
        date=("Date", "first"),
        location=("Location", "first"),
        images=("Im_id", "count"),
        orthomosaics=("Orth_id", "nunique")
    )
    .reset_index()
)

In [ ]:
# Proposed flight-grouped split: Flights 1 and 6 for validation

grouped_df = metadata_df.merge(
    classes_df[["Sp_id", "Especies"]],
    on="Sp_id",
    how="left"
)

grouped_df["New_split"] = grouped_df["F_id"].apply(
    lambda x: "Validation" if x in [1, 6] else "Train"
)

split_summary = (
    grouped_df.groupby(["Especies", "New_split"])
    .size()
    .unstack(fill_value=0)
)

if "Train" not in split_summary.columns:
    split_summary["Train"] = 0

if "Validation" not in split_summary.columns:
    split_summary["Validation"] = 0

split_summary["Total"] = (
    split_summary["Train"] + split_summary["Validation"]
)

print("Proposed split totals:")
print(grouped_df["New_split"].value_counts())

print("\nSpecies distribution:")
display(split_summary.sort_values("Total", ascending=False))

print("\nInvasive candidate counts:")
display(
    grouped_df[
        grouped_df["Especies"].isin([
            "Asparagopsis armata",
            "Sargassum muticum"
        ])
    ].groupby(["Especies", "New_split"])
    .size()
    .unstack(fill_value=0)
)

In [ ]:
# Test alternative flight-grouped split

validation_flights = [1, 6, 9]

grouped_df["New_split_2"] = grouped_df["F_id"].apply(
    lambda x: "Validation" if x in validation_flights else "Train"
)

print("Split totals:")
print(grouped_df["New_split_2"].value_counts())

print("\nInvasive candidate distribution:")
display(
    grouped_df[
        grouped_df["Especies"].isin([
            "Asparagopsis armata",
            "Sargassum muticum"
        ])
    ].groupby(["Especies", "New_split_2"])
    .size()
    .unstack(fill_value=0)
)

print("\nNative candidate distribution:")
display(
    grouped_df[
        grouped_df["Especies"].isin([
            "Ericaria selaginoides",
            "Bifurcaria bifurcata",
            "Saccorhiza polyschides",
            "Fucus spp.",
            "Himanthalia elongata"
        ])
    ].groupby(["Especies", "New_split_2"])
    .size()
    .unstack(fill_value=0)
)

In [ ]:
# Proposed flight-grouped split
validation_flights = [2, 3, 4, 5, 8]

grouped_df["Final_candidate"] = grouped_df["F_id"].apply(
    lambda x: "Validation" if x in validation_flights else "Train"
)

print("Split totals:")
print(grouped_df["Final_candidate"].value_counts())

print("\nInvasive candidate counts:")
display(
    grouped_df[
        grouped_df["Especies"].isin([
            "Asparagopsis armata",
            "Sargassum muticum"
        ])
    ].groupby(["Especies", "Final_candidate"])
    .size()
    .unstack(fill_value=0)
)

print("\nLocation distribution:")
display(
    grouped_df.groupby(["Final_candidate", "Location"])
    .size()
    .reset_index(name="images")
)

print("\nFlight overlap check:")
train_ids = set(
    grouped_df.loc[
        grouped_df["Final_candidate"] == "Train", "F_id"
    ].dropna()
)

val_ids = set(
    grouped_df.loc[
        grouped_df["Final_candidate"] == "Validation", "F_id"
    ].dropna()
)

print("Shared flights:", train_ids.intersection(val_ids))

In [ ]:

# ============================================================
# MARINE VEGETATION RESEARCH PROJECT
# MASTER RESTORE CELL — AFTER RUNTIME RESTART
# ============================================================
# Project: Fine-Grained Discrimination of Invasive and Native
# Marine Vegetation Using a Hybrid Deep Learning Model
# ============================================================
# PURPOSE:
# 1. Mount Google Drive
# 2. Restore project paths and ZIP dataset access
# 3. Load M1 manifests
# 4. Restore M2 ResNet18 best checkpoint
# 5. Recreate datasets and DataLoaders
# 6. Load M2/M3 result artifacts
# 7. Verify the M3 backup
#
# This cell DOES NOT retrain the model or modify old results.
# ============================================================

import os
import sys
import json
import hashlib
import zipfile
import io
import gc
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd
from PIL import Image

import torch
import torch.nn as nn
import torchvision
import torchvision.transforms as transforms
from torchvision import models
from torch.utils.data import Dataset, DataLoader

# ------------------------------------------------------------
# 1. GOOGLE DRIVE CONNECTION
# ------------------------------------------------------------

print("=" * 65)
print("STEP 1 — CONNECTING GOOGLE DRIVE")
print("=" * 65)

if not Path("/content/drive/MyDrive").exists():
    from google.colab import drive
    drive.mount("/content/drive")
else:
    print("Google Drive is already accessible.")

# ------------------------------------------------------------
# 2. RESTORE PROJECT DIRECTORIES
# ------------------------------------------------------------

PROJECT_DIR = Path("/content/drive/MyDrive/Marine_Project")

ZIP_DIR = PROJECT_DIR / "dataset_zips"
MODELS_DIR = PROJECT_DIR / "models"
RESULTS_DIR = PROJECT_DIR / "results"
OUTPUTS_DIR = PROJECT_DIR / "outputs"

BACKUP_DIR = (
    PROJECT_DIR /
    "backup_M3_complete_20261114_091542"
)

NOTEBOOK_PATH = (
    Path("/content/drive/MyDrive/Colab Notebooks")
    / "Mini Project.ipynb"
)

assert PROJECT_DIR.exists(), "Marine_Project folder not found!"
assert ZIP_DIR.exists(), "Dataset ZIP directory not found!"
assert MODELS_DIR.exists(), "Models directory not found!"
assert RESULTS_DIR.exists(), "Results directory not found!"

print("Project directory:", PROJECT_DIR)
print("ZIP directory:", ZIP_DIR)
print("Models directory:", MODELS_DIR)
print("Results directory:", RESULTS_DIR)
print("Outputs directory:", OUTPUTS_DIR)

# ------------------------------------------------------------
# 3. VERIFY ALL EIGHT DATASET ARCHIVES
# ------------------------------------------------------------

print("\n" + "=" * 65)
print("STEP 2 — DATASET ZIP RESTORATION")
print("=" * 65)

EXPECTED_ZIPS = [
    "ECU-MSS-2 Amphibolis_training.zip",
    "ECU-MSS-2 Amphibolis_testing.zip",
    "ECU-MSS-2 Background_training.zip",
    "ECU-MSS-2 Background_testing.zip",
    "ECU-MSS-2 Halophila_training.zip",
    "ECU-MSS-2 Halophila_testing.zip",
    "ECU-MSS-2 Posidonia_training.zip",
    "ECU-MSS-2 Posidonia_testing.zip"
]

ZIP_PATHS = {}

for filename in EXPECTED_ZIPS:
    path = ZIP_DIR / filename
    assert path.exists(), f"Missing ZIP: {filename}"
    ZIP_PATHS[filename] = path
    print(f"FOUND: {filename}")

print("\nTotal ZIP archives:", len(ZIP_PATHS))

# ------------------------------------------------------------
# 4. RESTORE M1 MANIFESTS
# ------------------------------------------------------------

print("\n" + "=" * 65)
print("STEP 3 — RESTORING M1 MANIFESTS")
print("=" * 65)

TRAIN_MANIFEST_PATH = (
    RESULTS_DIR / "clean_train_manifest_grouped_v2.csv"
)

VAL_MANIFEST_PATH = (
    RESULTS_DIR / "clean_validation_manifest_grouped_v2.csv"
)

TEST_MANIFEST_PATH = (
    RESULTS_DIR / "clean_test_manifest.csv"
)

for path in [
    TRAIN_MANIFEST_PATH,
    VAL_MANIFEST_PATH,
    TEST_MANIFEST_PATH
]:
    assert path.exists(), f"Missing manifest: {path.name}"

train_df = pd.read_csv(TRAIN_MANIFEST_PATH)
val_df = pd.read_csv(VAL_MANIFEST_PATH)
test_df = pd.read_csv(TEST_MANIFEST_PATH)

print("Training:", train_df.shape)
print("Validation:", val_df.shape)
print("Clean test:", test_df.shape)

print("\nTraining class counts:")
print(train_df["class"].value_counts())

print("\nValidation class counts:")
print(val_df["class"].value_counts())

print("\nTest class counts:")
print(test_df["class"].value_counts())

# ------------------------------------------------------------
# 5. VERIFY ZIP REFERENCES IN MANIFESTS
# ------------------------------------------------------------

print("\n" + "=" * 65)
print("STEP 4 — VERIFYING MANIFEST ZIP REFERENCES")
print("=" * 65)

for name, df in [
    ("Training", train_df),
    ("Validation", val_df),
    ("Test", test_df)
]:
    missing = [
        z for z in df["zip_file"].unique()
        if not (ZIP_DIR / z).exists()
    ]

    print(name, "| Records:", len(df),
          "| Missing ZIP references:", len(missing))

    assert len(missing) == 0, f"{name} ZIP references missing!"

# ------------------------------------------------------------
# 6. RESTORE CLASS MAPPING AND PREPROCESSING
# ------------------------------------------------------------

print("\n" + "=" * 65)
print("STEP 5 — RESTORING MODEL CONFIGURATION")
print("=" * 65)

class_mapping = {
    "Amphibolis": 0,
    "Background": 1,
    "Halophila": 2,
    "Posidonia": 3
}

idx_to_class = {
    value: key for key, value in class_mapping.items()
}

IMAGE_SIZE = 224
BATCH_SIZE = 32

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

print("Device:", device)
print("Image size:", IMAGE_SIZE)
print("Batch size:", BATCH_SIZE)
print("Class mapping:", class_mapping)

# ------------------------------------------------------------
# 7. RESTORE ZIP-BASED DATASET CLASS
# ------------------------------------------------------------

print("\n" + "=" * 65)
print("STEP 6 — RESTORING ZIP DATASET LOADER")
print("=" * 65)

class MarineZipDatasetV2(Dataset):

    def __init__(self, dataframe, zip_dir, transform=None):
        self.df = dataframe.reset_index(drop=True)
        self.zip_dir = Path(zip_dir)
        self.transform = transform
        self.zip_handles = {}

    def __len__(self):
        return len(self.df)

    def __getitem__(self, index):

        row = self.df.iloc[index]

        zip_name = row["zip_file"]
        image_name = row["image_name"]

        zip_path = self.zip_dir / zip_name

        if zip_name not in self.zip_handles:
            self.zip_handles[zip_name] = zipfile.ZipFile(
                zip_path, "r"
            )

        archive = self.zip_handles[zip_name]

        try:
            image_bytes = archive.read(image_name)
        except KeyError:
            raise FileNotFoundError(
                f"Image {image_name} not found in {zip_name}"
            )

        image = Image.open(
            io.BytesIO(image_bytes)
        ).convert("RGB")

        if self.transform:
            image = self.transform(image)

        label = class_mapping[row["class"]]

        return image, label

    def close(self):
        for archive in self.zip_handles.values():
            archive.close()
        self.zip_handles.clear()

# ------------------------------------------------------------
# 8. RECREATE DATASETS AND DATALOADERS
# ------------------------------------------------------------

print("\n" + "=" * 65)
print("STEP 7 — RECREATING DATALOADERS")
print("=" * 65)

train_dataset = MarineZipDatasetV2(
    train_df, ZIP_DIR, transform
)

val_dataset = MarineZipDatasetV2(
    val_df, ZIP_DIR, transform
)

test_dataset = MarineZipDatasetV2(
    test_df, ZIP_DIR, transform
)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0,
    pin_memory=torch.cuda.is_available()
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=torch.cuda.is_available()
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=torch.cuda.is_available()
)

print("Training batches:", len(train_loader))
print("Validation batches:", len(val_loader))
print("Test batches:", len(test_loader))

# ------------------------------------------------------------
# 9. RESTORE GROUPED-V2 RESNET18 BEST CHECKPOINT
# ------------------------------------------------------------

print("\n" + "=" * 65)
print("STEP 8 — RESTORING BEST RESNET18 CHECKPOINT")
print("=" * 65)

BEST_MODEL_PATH = (
    MODELS_DIR /
    "resnet18_grouped_v2_baseline_best.pth"
)

LATEST_MODEL_PATH = (
    MODELS_DIR /
    "resnet18_grouped_v2_baseline_latest.pth"
)

assert BEST_MODEL_PATH.exists(), "Best checkpoint missing!"
assert LATEST_MODEL_PATH.exists(), "Latest checkpoint missing!"

checkpoint = torch.load(
    BEST_MODEL_PATH,
    map_location=device,
    weights_only=False
)

assert "model_state_dict" in checkpoint
assert checkpoint["class_mapping"] == class_mapping

model = models.resnet18(weights=None)
model.fc = nn.Linear(
    model.fc.in_features,
    len(class_mapping)
)

model.load_state_dict(
    checkpoint["model_state_dict"],
    strict=True
)

model = model.to(device)
model.eval()

print("Best checkpoint loaded successfully.")
print("Selected epoch:", checkpoint["epoch"])
print("Best validation loss:", checkpoint["best_val_loss"])
print("Model:", "ResNet18")
print("Device:", device)

# ------------------------------------------------------------
# 10. RESTORE M2 AND M3 RESULT TABLES
# ------------------------------------------------------------

print("\n" + "=" * 65)
print("STEP 9 — RESTORING RESEARCH RESULTS")
print("=" * 65)

result_files = list(RESULTS_DIR.glob("*.csv"))

results_tables = {}

for path in result_files:
    try:
        results_tables[path.stem] = pd.read_csv(path)
    except Exception as e:
        print("Could not load:", path.name, "|", str(e))

print("Loaded CSV result tables:", len(results_tables))

# Convenient M3 references
gradcam_validation_df = results_tables.get(
    "grouped_v2_gradcam_validation_diagnostics"
)

gradcam_spatial_df = results_tables.get(
    "grouped_v2_gradcam_spatial_concentration"
)

gradcam_correct_error_df = results_tables.get(
    "grouped_v2_gradcam_correct_vs_error"
)

# Convenient M2 references
test_report_df = results_tables.get(
    "grouped_v2_baseline_clean_test_report"
)

test_predictions_df = results_tables.get(
    "grouped_v2_baseline_clean_test_predictions"
)

test_confusion_df = results_tables.get(
    "grouped_v2_baseline_clean_test_confusion_matrix"
)

# ------------------------------------------------------------
# 11. VERIFY M3 BACKUP AND LATEST NOTEBOOK
# ------------------------------------------------------------

print("\n" + "=" * 65)
print("STEP 10 — BACKUP AND NOTEBOOK VERIFICATION")
print("=" * 65)

snapshot_path = (
    BACKUP_DIR /
    "latest_notebook_snapshot" /
    "Mini Project_latest.ipynb"
)

def calculate_sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(
            lambda: f.read(1024 * 1024), b""
        ):
            h.update(chunk)
    return h.hexdigest()

backup_checks = {}

backup_checks["Backup folder"] = BACKUP_DIR.exists()
backup_checks["Notebook snapshot"] = snapshot_path.exists()

if snapshot_path.exists():
    backup_checks["Notebook checksum"] = (
        NOTEBOOK_PATH.exists()
        and calculate_sha256(snapshot_path)
        == calculate_sha256(NOTEBOOK_PATH)
    )

for folder in ["models", "results", "outputs"]:
    backup_checks[folder] = (
        (BACKUP_DIR / folder).exists()
    )

for name, status in backup_checks.items():
    print(name, ":", "PASS" if status else "FAIL")

# ------------------------------------------------------------
# 12. VERIFY THREE M3 ARTIFACTS
# ------------------------------------------------------------

print("\n" + "=" * 65)
print("STEP 11 — M3 ARTIFACT VERIFICATION")
print("=" * 65)

m3_files = [
    "grouped_v2_gradcam_validation_diagnostics.csv",
    "grouped_v2_gradcam_spatial_concentration.csv",
    "grouped_v2_gradcam_correct_vs_error.csv"
]

m3_status = True

for filename in m3_files:
    path = RESULTS_DIR / filename
    exists = path.exists()

    print(filename, ":", exists)

    if not exists:
        m3_status = False

# ------------------------------------------------------------
# 13. SAMPLE IMAGE DECODING CHECK
# ------------------------------------------------------------

print("\n" + "=" * 65)
print("STEP 12 — DATASET RESTORATION TEST")
print("=" * 65)

for name, dataset in [
    ("Training", train_dataset),
    ("Validation", val_dataset),
    ("Test", test_dataset)
]:
    sample_image, sample_label = dataset[0]

    print(
        name,
        "| Tensor:", tuple(sample_image.shape),
        "| Label:", idx_to_class[sample_label],
        "| Finite:", torch.isfinite(sample_image).all().item()
    )

# ------------------------------------------------------------
# 14. FINAL RESTORE SUMMARY
# ------------------------------------------------------------

print("\n" + "=" * 65)
print("MARINE PROJECT RESTORE SUMMARY")
print("=" * 65)

print("M1 — Data Audit: COMPLETE")
print("M2 — Four-Class Baseline: COMPLETE")
print("M3 — Exploratory Localization: COMPLETE")
print("M4 — Invasive/Native Dataset: NOT STARTED")
print("M5 — Invasive/Native Classification: PENDING")
print("M6 — Selective Removal Decision System: PENDING")

print("\nTraining images:", len(train_dataset))
print("Validation images:", len(val_dataset))
print("Clean test images:", len(test_dataset))

print("Best model epoch:", checkpoint["epoch"])
print("Model evaluation mode:", not model.training)

print("\nM3 artifacts available:", m3_status)
print("Backup checks passed:", sum(backup_checks.values()),
      "/", len(backup_checks))

print("\nRESTORE CELL EXECUTION FINISHED")
print("No model retraining performed.")
print("No existing research results overwritten.")
print("Next milestone: M4 — Western Australian ecological dataset.")
print("=" * 65)


In [ ]:
# Check whether M4 variables survived restoration

required_vars = [
    "remote_file",
    "z",
    "entries",
    "classes_df",
    "labels_df",
    "ortho_df",
    "flights_df",
    "locations_df",
    "metadata_df",
    "grouped_df"
]

for name in required_vars:
    print(f"{name}: {'Available' if name in globals() else 'Not available'}")

In [ ]:
print("File information:")
print(file_info)

print("\nAvailable link keys:")
print(file_info.get("links", {}).keys())

In [ ]:
# Use the correct Zenodo download link
file_url = file_info["links"]["self"]

# Check remote range access (only 1 byte)
response = requests.get(
    file_url,
    headers={"Range": "bytes=0-0"},
    timeout=60
)

print("HTTP status:", response.status_code)
print("Content-Range:", response.headers.get("Content-Range"))

if response.status_code == 206:
    total_size = int(response.headers["Content-Range"].split("/")[-1])
    print("Remote ZIP size:", round(total_size / (1024**3), 2), "GB")
    print("Remote access confirmed. No full download performed.")
else:
    print("Range access not confirmed. Stop here.")

In [ ]:
class RemoteZipReader:
    def __init__(self, url, size):
        self.url = url
        self.size = size
        self.position = 0
        self.session = requests.Session()

    def seek(self, offset, whence=0):
        if whence == 0:
            self.position = offset
        elif whence == 1:
            self.position += offset
        elif whence == 2:
            self.position = self.size + offset
        return self.position

    def tell(self):
        return self.position

    def read(self, size=-1):
        if size is None or size < 0:
            size = self.size - self.position

        end = min(self.position + size, self.size)

        if end <= self.position:
            return b""

        response = self.session.get(
            self.url,
            headers={"Range": f"bytes={self.position}-{end-1}"},
            timeout=120
        )

        if response.status_code != 206:
            raise RuntimeError(
                f"Range request failed: HTTP {response.status_code}"
            )

        data = response.content
        self.position += len(data)
        return data

    def close(self):
        self.session.close()

    def seekable(self):
        return True

    def readable(self):
        return True


remote_zip = RemoteZipReader(file_url, total_size)
z = zipfile.ZipFile(remote_zip)

print("Remote ZIP opened successfully.")
print("Total archive entries:", len(z.namelist()))

csv_files = [
    name for name in z.namelist()
    if name.lower().endswith(".csv")
]

print("\nCSV metadata files:")
for name in csv_files:
    print(name)

In [ ]:
def read_imo_csv(filename):
    path = f"dataset_comprimir/Tables/{filename}"
    return pd.read_csv(io.BytesIO(z.read(path)), sep=";")

classes_df = read_imo_csv("Classes.csv")
labels_df = read_imo_csv("Labels.csv")
flights_df = read_imo_csv("Flights.csv")
locations_df = read_imo_csv("Locations.csv")
ortho_df = read_imo_csv("Orthoimages.csv")

tables = {
    "Classes": classes_df,
    "Labels": labels_df,
    "Flights": flights_df,
    "Locations": locations_df,
    "Orthoimages": ortho_df
}

for name, df in tables.items():
    print(f"\n{'='*50}")
    print(f"{name}: {df.shape}")
    print("Columns:", df.columns.tolist())
    print(df.head(3).to_string(index=False))

In [ ]:
# Standardise join-key data types
for df, columns in [
    (labels_df, ["Sp_id", "Orth_id"]),
    (classes_df, ["Sp_id"]),
    (ortho_df, ["Orth_id", "F_id"]),
    (flights_df, ["F_id", "L_id"]),
    (locations_df, ["L_id"])
]:
    for col in columns:
        df[col] = pd.to_numeric(df[col], errors="coerce").astype("Int64")

# Connect each image to its species, flight and location
metadata_df = (
    labels_df
    .merge(classes_df, on="Sp_id", how="left")
    .merge(ortho_df[["Orth_id", "F_id"]], on="Orth_id", how="left")
    .merge(flights_df, on="F_id", how="left")
    .merge(locations_df[["L_id", "Location"]], on="L_id", how="left")
)

print("Combined metadata shape:", metadata_df.shape)
print("\nMissing metadata:")
print(metadata_df[["Especies", "F_id", "Location"]].isna().sum())

print("\nOriginal dataset split:")
print(metadata_df["Dataset_type"].value_counts())

print("\nImages per flight and split:")
print(pd.crosstab(metadata_df["F_id"], metadata_df["Dataset_type"]))

train_flights = set(metadata_df.loc[
    metadata_df["Dataset_type"] == "Train", "F_id"
].dropna())

val_flights = set(metadata_df.loc[
    metadata_df["Dataset_type"] == "Validation", "F_id"
].dropna())

train_orthos = set(metadata_df.loc[
    metadata_df["Dataset_type"] == "Train", "Orth_id"
].dropna())

val_orthos = set(metadata_df.loc[
    metadata_df["Dataset_type"] == "Validation", "Orth_id"
].dropna())

print("\nShared flights:", sorted(train_flights & val_flights))
print("Number of shared flights:", len(train_flights & val_flights))

print("\nShared orthomosaics:", sorted(train_orthos & val_orthos))
print("Number of shared orthomosaics:", len(train_orthos & val_orthos))

In [ ]:
# Species counts by flight
species_flight = pd.crosstab(
    metadata_df["Especies"],
    metadata_df["F_id"]
)

# Species counts in the supplied train/validation split
species_split = pd.crosstab(
    metadata_df["Especies"],
    metadata_df["Dataset_type"]
)

print("INVASIVE CANDIDATE DISTRIBUTION")
for species in ["Sargassum muticum", "Asparagopsis armata"]:
    if species in species_flight.index:
        print(f"\n{species}")
        print("Total images:", int(species_flight.loc[species].sum()))
        print(species_flight.loc[species].to_string())
        print("Original split:")
        print(species_split.loc[species].to_string())

print("\n\nALL SPECIES — IMAGE COUNTS BY ORIGINAL SPLIT")
print(species_split.to_string())

print("\n\nNUMBER OF FLIGHTS REPRESENTED PER SPECIES")
print(
    (species_flight > 0).sum(axis=1)
    .sort_values(ascending=False)
    .to_string()
)

In [ ]:
# Flight-level distribution of invasive candidates
candidate_counts = pd.crosstab(
    metadata_df["F_id"],
    metadata_df["Especies"]
)

candidate_counts = candidate_counts.reindex(
    columns=["Asparagopsis armata", "Sargassum muticum"],
    fill_value=0
)

flight_summary = (
    flights_df
    .merge(locations_df[["L_id", "Location"]], on="L_id", how="left")
    .merge(
        metadata_df.groupby("F_id").size().rename("Total_images"),
        on="F_id",
        how="left"
    )
    .merge(candidate_counts, on="F_id", how="left")
)

print("FLIGHT-LEVEL GEOGRAPHICAL DISTRIBUTION")
print(
    flight_summary[
        ["F_id", "Date", "Location", "Total_images",
         "Asparagopsis armata", "Sargassum muticum"]
    ].to_string(index=False)
)

print("\nINVASIVE CANDIDATE TOTALS BY LOCATION")
print(
    metadata_df[
        metadata_df["Especies"].isin(
            ["Asparagopsis armata", "Sargassum muticum"]
        )
    ].groupby(["Location", "Especies"], dropna=False)
    .size()
    .unstack(fill_value=0)
    .to_string()
)

In [ ]:
# Keep only records identified as Macroalgae
macroalgae_df = metadata_df[
    metadata_df["General"].astype(str).str.strip().str.lower()
    == "macroalgae"
].copy()

location_species = pd.crosstab(
    macroalgae_df["Especies"],
    macroalgae_df["Location"]
)

print("MACROALGAE SPECIES BY LOCATION")
print(location_species.to_string())

print("\nTOTAL MACROALGAE IMAGES BY LOCATION")
print(macroalgae_df["Location"].value_counts(dropna=False))

print("\nTOTAL MACROALGAE IMAGES BY SPECIES")
print(
    macroalgae_df["Especies"]
    .value_counts(dropna=False)
    .to_string()
)

print("\nCHECK: NON-MACROALGAE / UNCLASSIFIED LABEL COUNTS")
print(
    metadata_df["General"]
    .value_counts(dropna=False)
    .to_string()
)

In [ ]:
# Compare species availability across both locations
location_presence = (location_species > 0)

shared_species = location_presence[
    location_presence.all(axis=1)
].index.tolist()

baiona_only = location_presence[
    location_presence["Baiona"]
    & ~location_presence["Illa de Arousa"]
].index.tolist()

illa_only = location_presence[
    location_presence["Illa de Arousa"]
    & ~location_presence["Baiona"]
].index.tolist()

print("Species present at BOTH locations:")
for species in shared_species:
    print(f"- {species}")

print("\nSpecies found ONLY at Baiona:")
for species in baiona_only:
    print(f"- {species}")

print("\nSpecies found ONLY at Illa de Arousa:")
for species in illa_only:
    print(f"- {species}")

print("\nSummary:")
print("Shared species:", len(shared_species))
print("Baiona-only species:", len(baiona_only))
print("Illa-only species:", len(illa_only))

In [ ]:
# Conservative species-to-ecological-status mapping
ecological_labels = {
    "Asparagopsis armata": "Invasive",
    "Sargassum muticum": "Invasive",
    "Bifurcaria bifurcata": "Native",
    "Codium spp.": "Native",
    "Ericaria selaginoides": "Native",
    "Saccorhiza polyschides": "Native"
}

# Keep only verified species
m4_binary_df = metadata_df[
    metadata_df["Especies"].isin(ecological_labels.keys())
].copy()

m4_binary_df["Ecological_Status"] = (
    m4_binary_df["Especies"].map(ecological_labels)
)

print("Selected dataset shape:", m4_binary_df.shape)

print("\nBinary class distribution:")
print(m4_binary_df["Ecological_Status"].value_counts())

print("\nSpecies distribution:")
print(pd.crosstab(
    m4_binary_df["Especies"],
    m4_binary_df["Ecological_Status"]
))

print("\nSpecies distribution by location:")
print(pd.crosstab(
    [m4_binary_df["Especies"], m4_binary_df["Ecological_Status"]],
    m4_binary_df["Location"]
))

print("\nBinary class distribution by location:")
print(pd.crosstab(
    m4_binary_df["Location"],
    m4_binary_df["Ecological_Status"]
))

print("\nImages with missing flight metadata:",
      m4_binary_df["F_id"].isna().sum())

In [ ]:
# Proposed independent test flights
test_flights = [1, 6]

train_candidate = m4_binary_df[
    ~m4_binary_df["F_id"].isin(test_flights)
].copy()

test_candidate = m4_binary_df[
    m4_binary_df["F_id"].isin(test_flights)
].copy()

print("PROPOSED SPLIT: TEST FLIGHTS 1 AND 6")
print("\nImage counts:")
print("Training:", len(train_candidate))
print("Testing:", len(test_candidate))

print("\nBinary class distribution — TRAIN")
print(train_candidate["Ecological_Status"].value_counts())

print("\nBinary class distribution — TEST")
print(test_candidate["Ecological_Status"].value_counts())

print("\nSpecies distribution — TRAIN")
print(train_candidate["Especies"].value_counts())

print("\nSpecies distribution — TEST")
print(test_candidate["Especies"].value_counts())

print("\nLocation distribution — TRAIN")
print(pd.crosstab(
    train_candidate["Location"],
    train_candidate["Ecological_Status"]
))

print("\nLocation distribution — TEST")
print(pd.crosstab(
    test_candidate["Location"],
    test_candidate["Ecological_Status"]
))

print("\nFlight overlap:",
      set(train_candidate["F_id"]) & set(test_candidate["F_id"]))

In [ ]:
# Freeze the independent test split
M4_TEST_FLIGHTS = [1, 6]

m4_test_df = m4_binary_df[
    m4_binary_df["F_id"].isin(M4_TEST_FLIGHTS)
].copy()

m4_development_df = m4_binary_df[
    ~m4_binary_df["F_id"].isin(M4_TEST_FLIGHTS)
].copy()

# Verify flight separation
assert set(m4_test_df["F_id"]).isdisjoint(
    set(m4_development_df["F_id"])
)

# Verify both classes exist in both partitions
assert set(m4_test_df["Ecological_Status"]) == {"Native", "Invasive"}
assert set(m4_development_df["Ecological_Status"]) == {"Native", "Invasive"}

print("INDEPENDENT TEST SPLIT LOCKED")
print("Test flights:", M4_TEST_FLIGHTS)
print("Development images:", len(m4_development_df))
print("Test images:", len(m4_test_df))

print("\nFlight overlap:",
      set(m4_development_df["F_id"]) & set(m4_test_df["F_id"]))

print("\nTest class distribution:")
print(m4_test_df["Ecological_Status"].value_counts())

print("\nDevelopment class distribution:")
print(m4_development_df["Ecological_Status"].value_counts())

print("\nTest species distribution:")
print(m4_test_df["Especies"].value_counts())

In [ ]:
# Inspect remaining development flights
development_flight_summary = (
    m4_development_df
    .groupby(["F_id", "Location", "Ecological_Status"])
    .size()
    .unstack(fill_value=0)
)

print("DEVELOPMENT FLIGHT DISTRIBUTION")
print(development_flight_summary.to_string())

print("\nDEVELOPMENT SPECIES BY FLIGHT")
print(
    pd.crosstab(
        m4_development_df["Especies"],
        m4_development_df["F_id"]
    ).to_string()
)

print("\nAVAILABLE DEVELOPMENT FLIGHTS:")
print(sorted(m4_development_df["F_id"].dropna().unique().tolist()))

In [ ]:
from itertools import combinations

candidate_species = set(ecological_labels.keys())
split_options = []

for pair in combinations(
    sorted(m4_development_df["F_id"].dropna().unique()),
    2
):
    val_df = m4_development_df[
        m4_development_df["F_id"].isin(pair)
    ]

    train_df = m4_development_df[
        ~m4_development_df["F_id"].isin(pair)
    ]

    val_species = set(val_df["Especies"].dropna())
    train_species = set(train_df["Especies"].dropna())

    val_classes = set(val_df["Ecological_Status"])
    train_classes = set(train_df["Ecological_Status"])

    locations = set(val_df["Location"].dropna())

    split_options.append({
        "Validation_Flights": pair,
        "Val_Images": len(val_df),
        "Val_Invasive": int((val_df["Ecological_Status"] == "Invasive").sum()),
        "Val_Native": int((val_df["Ecological_Status"] == "Native").sum()),
        "Train_Invasive": int((train_df["Ecological_Status"] == "Invasive").sum()),
        "Train_Native": int((train_df["Ecological_Status"] == "Native").sum()),
        "Val_Species": len(val_species),
        "Train_Species": len(train_species),
        "Both_Locations": len(locations) == 2,
        "Both_Classes": (
            val_classes == {"Native", "Invasive"}
            and train_classes == {"Native", "Invasive"}
        ),
        "All_Six_Species_Val": candidate_species.issubset(val_species),
        "All_Six_Species_Train": candidate_species.issubset(train_species)
    })

split_comparison = pd.DataFrame(split_options)

print("POSSIBLE FLIGHT-LEVEL VALIDATION SPLITS")
print(
    split_comparison.sort_values(
        ["All_Six_Species_Val", "Both_Locations", "Val_Invasive"],
        ascending=[False, False, False]
    ).to_string(index=False)
)

In [ ]:
# Inspect original split within development data
m4_train_candidate = m4_development_df[
    m4_development_df["Dataset_type"] == "Train"
].copy()

m4_val_candidate = m4_development_df[
    m4_development_df["Dataset_type"] == "Validation"
].copy()

train_orthos = set(m4_train_candidate["Orth_id"].dropna())
val_orthos = set(m4_val_candidate["Orth_id"].dropna())

print("ORTHOMOSAIC-GROUPED DEVELOPMENT SPLIT")
print("Training images:", len(m4_train_candidate))
print("Validation images:", len(m4_val_candidate))

print("\nTraining classes:")
print(m4_train_candidate["Ecological_Status"].value_counts())

print("\nValidation classes:")
print(m4_val_candidate["Ecological_Status"].value_counts())

print("\nTraining species:")
print(m4_train_candidate["Especies"].value_counts())

print("\nValidation species:")
print(m4_val_candidate["Especies"].value_counts())

print("\nShared orthomosaics:", train_orthos & val_orthos)
print("Shared flight IDs:",
      set(m4_train_candidate["F_id"].dropna())
      & set(m4_val_candidate["F_id"].dropna()))

In [ ]:
# Species counts within each development orthomosaic
ortho_species = pd.crosstab(
    m4_development_df["Orth_id"],
    m4_development_df["Especies"]
)

# Add flight and location information
ortho_info = (
    m4_development_df[
        ["Orth_id", "F_id", "Location", "Dataset_type"]
    ]
    .drop_duplicates()
    .sort_values(["F_id", "Orth_id"])
)

ortho_audit = ortho_info.merge(
    ortho_species,
    on="Orth_id",
    how="left"
)

print("DEVELOPMENT ORTHOMOSAIC SPECIES AUDIT")
print(ortho_audit.to_string(index=False))

print("\nTotal development orthomosaics:", len(ortho_species))
print(
    "\nAsparagopsis-containing orthomosaics:",
    int((ortho_species.get("Asparagopsis armata", 0) > 0).sum())
)

In [ ]:
# Lock orthomosaic-grouped development split
m4_train_df = m4_train_candidate.copy()
m4_val_df = m4_val_candidate.copy()

# Verify no orthomosaic overlap
train_groups = set(m4_train_df["Orth_id"].dropna())
val_groups = set(m4_val_df["Orth_id"].dropna())

assert train_groups.isdisjoint(val_groups)

# Verify the independent test remains separate by flight
assert set(m4_train_df["F_id"]).isdisjoint(set(m4_test_df["F_id"]))
assert set(m4_val_df["F_id"]).isdisjoint(set(m4_test_df["F_id"]))

print("M4 DEVELOPMENT SPLIT LOCKED")
print("\nTraining:", len(m4_train_df))
print("Validation:", len(m4_val_df))
print("Independent test:", len(m4_test_df))

print("\nShared train-validation orthomosaics:",
      train_groups & val_groups)

print("\nShared train-test flights:",
      set(m4_train_df["F_id"]) & set(m4_test_df["F_id"]))

print("\nShared validation-test flights:",
      set(m4_val_df["F_id"]) & set(m4_test_df["F_id"]))

print("\nValidation invasive species:")
print(
    m4_val_df[m4_val_df["Ecological_Status"] == "Invasive"]
    ["Especies"].value_counts()
)

print("\nAll split checks passed.")

In [ ]:
from pathlib import Path
from datetime import datetime

# Project results directory
results_dir = Path(
    "/content/drive/MyDrive/Marine_Project/results"
)
results_dir.mkdir(parents=True, exist_ok=True)

# Timestamped filenames to avoid overwriting earlier results
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

manifest_columns = [
    "Im_id", "Filename", "Sp_id", "Especies",
    "Ecological_Status", "Orth_id", "F_id",
    "Location", "Dataset_type"
]

manifests = {
    "train": m4_train_df[manifest_columns].copy(),
    "validation": m4_val_df[manifest_columns].copy(),
    "independent_test": m4_test_df[manifest_columns].copy()
}

saved_files = []

for split_name, dataframe in manifests.items():
    output_path = results_dir / (
        f"IMO_M4_{split_name}_manifest_{timestamp}.csv"
    )

    if output_path.exists():
        raise FileExistsError(
            f"Refusing to overwrite existing file: {output_path}"
        )

    dataframe.to_csv(output_path, index=False)
    saved_files.append(output_path)

    print(f"Saved {split_name}: {len(dataframe)} rows")
    print(output_path)

print("\nAll three manifests saved.")
print("Total images:", sum(len(df) for df in manifests.values()))

In [ ]:
# Build an index of PNG files inside the remote ZIP
archive_pngs = [
    name for name in z.namelist()
    if name.lower().endswith(".png")
]

# Match manifest filenames against archive filenames
archive_basenames = {}

for path in archive_pngs:
    basename = path.split("/")[-1]
    archive_basenames.setdefault(basename, []).append(path)

all_manifest_names = pd.concat(
    [df["Filename"] for df in manifests.values()],
    ignore_index=True
).astype(str)

matched = all_manifest_names[
    all_manifest_names.isin(archive_basenames)
]

missing = all_manifest_names[
    ~all_manifest_names.isin(archive_basenames)
]

print("IMAGE FILE VERIFICATION")
print("Manifest image records:", len(all_manifest_names))
print("Matched filenames:", len(matched))
print("Missing filenames:", len(missing))
print("PNG files in archive:", len(archive_pngs))

print("\nExample matched archive paths:")
for filename in matched.head(5):
    print(filename, "->", archive_basenames[filename][0])

if len(missing):
    print("\nFirst missing filenames:")
    print(missing.head(10).to_string(index=False))
else:
    print("\nAll manifest filenames found in the remote ZIP.")

In [ ]:
from PIL import Image
from io import BytesIO

# Select two deterministic samples from each species
sample_df = (
    pd.concat(list(manifests.values()), ignore_index=True)
    .groupby("Especies", group_keys=False)
    .head(2)
)

print("M4 IMAGE INTEGRITY CHECK")
print("Images selected:", len(sample_df))

integrity_results = []

for _, row in sample_df.iterrows():
    filename = str(row["Filename"])
    archive_path = archive_basenames[filename][0]

    try:
        image_bytes = z.read(archive_path)

        with Image.open(BytesIO(image_bytes)) as img:
            img.verify()

        with Image.open(BytesIO(image_bytes)) as img:
            integrity_results.append({
                "Species": row["Especies"],
                "Status": row["Ecological_Status"],
                "Filename": filename,
                "Format": img.format,
                "Dimensions": img.size,
                "Mode": img.mode,
                "File_Size_KB": round(len(image_bytes) / 1024, 2),
                "Valid": True
            })

    except Exception as error:
        integrity_results.append({
            "Species": row["Especies"],
            "Status": row["Ecological_Status"],
            "Filename": filename,
            "Valid": False,
            "Error": str(error)
        })

integrity_df = pd.DataFrame(integrity_results)

display(integrity_df)

print("\nValid images:", integrity_df["Valid"].sum())
print("Failed images:", (~integrity_df["Valid"]).sum())

In [ ]:
import requests

test_response = requests.get(
    file_url,
    headers={"Range": "bytes=0-23"},
    timeout=30
)

print("HTTP status:", test_response.status_code)
print("Content-Range:", test_response.headers.get("Content-Range"))
print("Response size:", len(test_response.content))

In [ ]:
check_names = [
    "z",
    "metadata_df",
    "m4_train_df",
    "m4_val_df",
    "m4_test_df",
    "manifests",
    "archive_basenames",
    "m4_dimension_df"
]

print("RUNTIME STATUS\n")

for name in check_names:
    exists = name in globals()
    print(f"{name}: {'AVAILABLE' if exists else 'NOT AVAILABLE'}")

In [ ]:
print("AVAILABLE IMAGE METADATA COLUMNS")
print(metadata_df.columns.tolist())

print("\nPOSSIBLE IMAGE DIMENSION COLUMNS")
dimension_columns = [
    col for col in metadata_df.columns
    if any(term in col.lower() for term in ["width", "height", "dimension", "size", "pixel"])
]

print(dimension_columns if dimension_columns else "No image dimension columns found.")

In [ ]:
print("IMAGE ACQUISITION METADATA AUDIT")

print("\n1. Locations:")
print(metadata_df["Location"].value_counts(dropna=False))

print("\n2. Drone types:")
print(metadata_df["Drone"].value_counts(dropna=False))

print("\n3. Ground Sampling Distance (GSD):")
print(metadata_df.groupby("Location")["GSD"].describe().round(4))

print("\n4. Flight heights:")
print(metadata_df.groupby("Location")["flight_height"].describe().round(2))

print("\n5. Unique acquisition flights:")
print(
    metadata_df[["F_id", "Date", "Location", "Drone", "GSD", "flight_height"]]
    .drop_duplicates()
    .sort_values("F_id")
    .to_string(index=False)
)

In [ ]:
print("M4 SPECIES × LOCATION DISTRIBUTION")

species_location_table = pd.crosstab(
    m4_binary_df["Especies"],
    m4_binary_df["Location"],
    margins=True
)

print(species_location_table.to_string())

print("\nECOLOGICAL STATUS × LOCATION")

status_location_table = pd.crosstab(
    m4_binary_df["Ecological_Status"],
    m4_binary_df["Location"],
    margins=True
)

print(status_location_table.to_string())

In [ ]:
species_location_percent = pd.crosstab(
    m4_binary_df["Especies"],
    m4_binary_df["Location"],
    normalize="index"
) * 100

print("SPECIES DISTRIBUTION BY LOCATION (%)")
print(species_location_percent.round(2).to_string())

print("\nECOLOGICAL STATUS DISTRIBUTION BY LOCATION (%)")
status_location_percent = pd.crosstab(
    m4_binary_df["Ecological_Status"],
    m4_binary_df["Location"],
    normalize="index"
) * 100

print(status_location_percent.round(2).to_string())

print("\nLOCATION SHARE OF TOTAL DATASET (%)")
print(
    (m4_binary_df["Location"].value_counts(normalize=True) * 100)
    .round(2)
    .to_string()
)

In [ ]:
print("M4 SPLIT × LOCATION × ECOLOGICAL STATUS")

combined_split_df = pd.concat([
    m4_train_df.assign(Split="Training"),
    m4_val_df.assign(Split="Validation"),
    m4_test_df.assign(Split="Independent Test")
])

split_location_table = pd.crosstab(
    [
        combined_split_df["Split"],
        combined_split_df["Location"]
    ],
    combined_split_df["Ecological_Status"],
    margins=False
)

print(split_location_table.to_string())

print("\nSPECIES DISTRIBUTION IN EACH SPLIT")

species_split_table = pd.crosstab(
    combined_split_df["Especies"],
    combined_split_df["Split"]
)

print(species_split_table.to_string())

In [ ]:
from sklearn.utils.class_weight import compute_class_weight
import numpy as np

class_names = ["Native", "Invasive"]

training_labels = m4_train_df["Ecological_Status"].values

class_weights = compute_class_weight(
    class_weight="balanced",
    classes=np.array(class_names),
    y=training_labels
)

class_weight_dict = dict(zip(class_names, class_weights))

print("M4 TRAINING CLASS BALANCE")
print(m4_train_df["Ecological_Status"].value_counts())

print("\nCALCULATED CLASS WEIGHTS")
for class_name, weight in class_weight_dict.items():
    print(f"{class_name}: {weight:.4f}")

print("\nThese weights are calculated from training data only.")

In [ ]:
from pathlib import Path
from datetime import datetime
import json

results_dir = Path("/content/drive/MyDrive/Marine_Project/results")

audit_report = {
    "dataset": "IMO Macroalgae Dataset",
    "doi": "10.5281/zenodo.14185867",
    "selected_images": 3199,
    "training_images": 1528,
    "validation_images": 1249,
    "independent_test_images": 422,
    "selected_species": 6,
    "native_images": 2751,
    "invasive_images": 448,
    "training_native": 1409,
    "training_invasive": 119,
    "class_weights": class_weight_dict,
    "image_filename_verification": "3199/3199 matched",
    "sample_integrity_check": "12/12 valid",
    "geographic_confounding": True,
    "train_validation_orthomosaic_overlap": 0,
    "independent_test_flight_overlap": 0,
    "validation_asparagopsis_images": 1,
    "independent_test_codium_images": 0,
    "image_domain": "Intertidal UAV imagery",
    "limitations": [
        "Ecological status is associated with geographic location.",
        "Flight and acquisition conditions vary.",
        "Validation has only one Asparagopsis image.",
        "Independent test contains no Codium images.",
        "Dataset does not directly validate underwater removal."
    ]
}

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
report_path = results_dir / f"IMO_M4_dataset_audit_{timestamp}.json"

if report_path.exists():
    raise FileExistsError(f"Refusing to overwrite: {report_path}")

with open(report_path, "w") as f:
    json.dump(audit_report, f, indent=4)

print("M4 AUDIT REPORT SAVED")
print(report_path)
print("Selected images:", audit_report["selected_images"])
print("Limitations documented:", len(audit_report["limitations"]))

In [ ]:
import shutil

paths_to_check = [
    "/content",
    "/content/drive/MyDrive/Marine_Project"
]

print("STORAGE AVAILABILITY\n")

for path in paths_to_check:
    total, used, free = shutil.disk_usage(path)

    print(f"Path: {path}")
    print(f"Total: {total / (1024**3):.2f} GB")
    print(f"Used:  {used / (1024**3):.2f} GB")
    print(f"Free:  {free / (1024**3):.2f} GB")
    print()

In [ ]:
selected_filenames = set(
    pd.concat(
        [df["Filename"] for df in manifests.values()],
        ignore_index=True
    ).astype(str)
)

selected_archive_entries = [
    info for info in z.infolist()
    if info.filename.lower().endswith(".png")
    and info.filename.split("/")[-1] in selected_filenames
]

selected_compressed_bytes = sum(
    info.compress_size for info in selected_archive_entries
)

selected_uncompressed_bytes = sum(
    info.file_size for info in selected_archive_entries
)

print("SELECTED IMAGE STORAGE ESTIMATE")
print("Expected images:", len(selected_filenames))
print("Matched ZIP entries:", len(selected_archive_entries))
print(f"Compressed size: {selected_compressed_bytes / (1024**3):.2f} GB")
print(f"Extracted size: {selected_uncompressed_bytes / (1024**3):.2f} GB")

assert len(selected_archive_entries) == len(selected_filenames)
print("\nAll selected images have matching ZIP entries.")

In [ ]:
from pathlib import Path

check_dirs = [
    Path("/content"),
    Path("/content/drive/MyDrive/Marine_Project")
]

print("EXISTING IMAGE FOLDERS\n")

for directory in check_dirs:
    print(f"\nDirectory: {directory}")

    for item in directory.iterdir():
        if item.is_dir():
            image_count = sum(
                1 for p in item.rglob("*")
                if p.suffix.lower() in [".png", ".jpg", ".jpeg"]
            )
            if image_count > 0:
                print(f"{item.name}: {image_count} images")

In [ ]:
import requests

try:
    response = requests.get(
        file_url,
        headers={"Range": "bytes=0-23"},
        timeout=30
    )

    print("HTTP status:", response.status_code)
    print("Content-Range:", response.headers.get("Content-Range"))
    print("Response size:", len(response.content))

    if response.status_code == 206:
        print("\nRemote ZIP access is available.")
    elif response.status_code == 429:
        print("\nZenodo rate limit is still active. Do not extract yet.")
    else:
        print("\nUnexpected response. Do not extract yet.")

except Exception as error:
    print("Connection error:", str(error))

In [ ]:
from pathlib import Path
import shutil

m4_image_root = Path("/content/IMO_M4_selected_images")

split_order = ["train", "validation", "independent_test"]

extraction_queue = []

for split_name in split_order:
    dataframe = manifests[split_name]

    for _, row in dataframe.iterrows():
        filename = str(row["Filename"])
        archive_path = archive_basenames[filename][0]

        extraction_queue.append({
            "split": split_name,
            "filename": filename,
            "archive_path": archive_path
        })

batch = extraction_queue[:100]

extracted_count = 0
failed_files = []

for item in batch:
    output_dir = m4_image_root / item["split"]
    output_dir.mkdir(parents=True, exist_ok=True)

    output_path = output_dir / item["filename"]

    if output_path.exists():
        print("Already exists, skipped:", item["filename"])
        continue

    temporary_path = output_path.with_suffix(".png.part")

    try:
        with z.open(item["archive_path"]) as source:
            with open(temporary_path, "xb") as destination:
                shutil.copyfileobj(source, destination)

        temporary_path.rename(output_path)
        extracted_count += 1

    except Exception as error:
        if temporary_path.exists():
            temporary_path.unlink()
        failed_files.append((item["filename"], str(error)))
        print("Extraction stopped:", error)
        break

print("\nM4 EXTRACTION BATCH 1")
print("Images in batch:", len(batch))
print("Successfully extracted:", extracted_count)
print("Failed files:", len(failed_files))
print("Output folder:", m4_image_root)

In [ ]:
print("REMOTE ZIP READER DIAGNOSTICS\n")

print("ZIP object:", type(z).__name__)
print("Reader object:", type(z.fp).__name__)
print("Reader module:", type(z.fp).__module__)

print("\nReader attributes:")
print(list(vars(z.fp).keys()) if hasattr(z.fp, "__dict__") else "No instance attributes")

print("\nRelevant global objects:")
for name in ["RemoteZipReader", "requests", "session", "file_url"]:
    print(f"{name}: {'AVAILABLE' if name in globals() else 'NOT AVAILABLE'}")

print("\nRemote reader methods:")
print([
    name for name in dir(z.fp)
    if not name.startswith("__")
])

In [ ]:
reader_session = z.fp.session

print("HTTP SESSION DIAGNOSTICS\n")

print("Session type:", type(reader_session).__name__)

print("\nSession headers:")
for key, value in reader_session.headers.items():
    print(f"{key}: {value}")

print("\nMounted connection adapters:")
for prefix, adapter in reader_session.adapters.items():
    print(prefix)
    print("  Retry configuration:", adapter.max_retries)
    print("  Connection pool size:", adapter._pool_connections)
    print("  Maximum pool connections:", adapter._pool_maxsize)

print("\nRemote reader position:", z.fp.position)
print("Remote ZIP size:", round(z.fp.size / (1024**3), 2), "GB")

print("\nNo network requests were made.")

In [ ]:
from pathlib import Path
from PIL import Image

print("M4 EXTRACTION STATUS\n")

image_root = Path("/content/IMO_M4_selected_images")

valid_count = 0
invalid_files = []

for path in image_root.rglob("*.png"):
    try:
        with Image.open(path) as img:
            img.verify()
        valid_count += 1
        print("Valid:", path.name)
    except Exception as error:
        invalid_files.append((path.name, str(error)))
        print("Invalid:", path.name, error)

partial_files = list(image_root.rglob("*.part"))

print("\nSUMMARY")
print("Valid extracted images:", valid_count)
print("Invalid images:", len(invalid_files))
print("Incomplete temporary files:", len(partial_files))

In [ ]:
# ============================================================
# MARINE VEGETATION RESEARCH PROJECT
# MASTER RESTORE CELL V2 — M1 THROUGH M4 STEP 48
# ============================================================
# Project:
# Fine-Grained Discrimination of Invasive and Native Marine
# Vegetation Using a Hybrid Deep Learning Model
#
# RESTORES:
# 1. Google Drive and project directories
# 2. Original ECU-MSS-2 M1 manifests and ZIP-based DataLoaders
# 3. M2 ResNet18 best checkpoint and result tables
# 4. M3 Grad-CAM artifacts and backup checks
# 5. M4 IMO manifests, audit report, split dataframes and class weights
# 6. M4 extracted-image folder and extraction status
#
# SAFETY:
# - Does NOT retrain any model.
# - Does NOT overwrite existing project files.
# - Does NOT delete or modify existing checkpoints/results.
# - Does NOT download/extract the 29 GB IMO ZIP.
# - Does NOT contact Zenodo by default.
# - Remote IMO ZIP access is OPTIONAL and disabled by default.
#
# Current stopping point:
# M4 Step 48 — one selected IMO image extracted and verified;
# subsequent remote ZIP requests received HTTP 429.
# ============================================================

import os
import io
import gc
import json
import zipfile
import hashlib
import struct
import requests
from pathlib import Path
from datetime import datetime
from collections import Counter

import numpy as np
import pandas as pd
from PIL import Image

import torch
import torch.nn as nn
import torchvision
import torchvision.transforms as transforms
from torchvision import models
from torch.utils.data import Dataset, DataLoader

from sklearn.utils.class_weight import compute_class_weight

# ------------------------------------------------------------
# USER CONTROL
# ------------------------------------------------------------

# Keep False during ordinary restoration.
# Set True ONLY when you intentionally want to test/open the
# remote IMO ZIP. This can make HTTP range requests to Zenodo.
ENABLE_REMOTE_IMO_ZIP = False

# ------------------------------------------------------------
# 1. GOOGLE DRIVE
# ------------------------------------------------------------

print("=" * 72)
print("MASTER RESTORE CELL V2 — M1 THROUGH M4 STEP 48")
print("=" * 72)
print("\nSTEP 1 — GOOGLE DRIVE")

if not Path("/content/drive/MyDrive").exists():
    from google.colab import drive
    drive.mount("/content/drive")
else:
    print("Google Drive is already accessible.")

# ------------------------------------------------------------
# 2. PROJECT DIRECTORIES
# ------------------------------------------------------------

PROJECT_DIR = Path("/content/drive/MyDrive/Marine_Project")
ZIP_DIR = PROJECT_DIR / "dataset_zips"
MODELS_DIR = PROJECT_DIR / "models"
RESULTS_DIR = PROJECT_DIR / "results"
OUTPUTS_DIR = PROJECT_DIR / "outputs"

BACKUP_DIR = PROJECT_DIR / "backup_M3_complete_20261114_091542"
NOTEBOOK_DIR = Path("/content/drive/MyDrive/Colab Notebooks")
NOTEBOOK_PATH = NOTEBOOK_DIR / "Mini Project.ipynb"

M4_IMAGE_ROOT = Path("/content/IMO_M4_selected_images")

for required_dir in [PROJECT_DIR, ZIP_DIR, MODELS_DIR, RESULTS_DIR]:
    assert required_dir.exists(), f"Required project path missing: {required_dir}"

OUTPUTS_DIR.mkdir(parents=True, exist_ok=True)

print("Project:", PROJECT_DIR)
print("Dataset ZIPs:", ZIP_DIR)
print("Models:", MODELS_DIR)
print("Results:", RESULTS_DIR)
print("Outputs:", OUTPUTS_DIR)
print("M4 temporary image folder:", M4_IMAGE_ROOT)

# ------------------------------------------------------------
# 3. VERIFY ORIGINAL ECU-MSS-2 ARCHIVES
# ------------------------------------------------------------

print("\nSTEP 2 — M1 ECU-MSS-2 ARCHIVES")

EXPECTED_ZIPS = [
    "ECU-MSS-2 Amphibolis_training.zip",
    "ECU-MSS-2 Amphibolis_testing.zip",
    "ECU-MSS-2 Background_training.zip",
    "ECU-MSS-2 Background_testing.zip",
    "ECU-MSS-2 Halophila_training.zip",
    "ECU-MSS-2 Halophila_testing.zip",
    "ECU-MSS-2 Posidonia_training.zip",
    "ECU-MSS-2 Posidonia_testing.zip",
]

ZIP_PATHS = {}
for filename in EXPECTED_ZIPS:
    path = ZIP_DIR / filename
    assert path.exists(), f"Missing ECU-MSS-2 archive: {filename}"
    ZIP_PATHS[filename] = path
    print("FOUND:", filename)

print("Total ECU-MSS-2 archives:", len(ZIP_PATHS))

# ------------------------------------------------------------
# 4. RESTORE M1 MANIFESTS
# ------------------------------------------------------------

print("\nSTEP 3 — M1 MANIFESTS")

TRAIN_MANIFEST_PATH = RESULTS_DIR / "clean_train_manifest_grouped_v2.csv"
VAL_MANIFEST_PATH = RESULTS_DIR / "clean_validation_manifest_grouped_v2.csv"
TEST_MANIFEST_PATH = RESULTS_DIR / "clean_test_manifest.csv"

for path in [TRAIN_MANIFEST_PATH, VAL_MANIFEST_PATH, TEST_MANIFEST_PATH]:
    assert path.exists(), f"Missing M1 manifest: {path}"

train_df = pd.read_csv(TRAIN_MANIFEST_PATH)
val_df = pd.read_csv(VAL_MANIFEST_PATH)
test_df = pd.read_csv(TEST_MANIFEST_PATH)

print("M1 training:", train_df.shape)
print("M1 validation:", val_df.shape)
print("M1 clean test:", test_df.shape)

for split_name, dataframe in [
    ("M1 training", train_df),
    ("M1 validation", val_df),
    ("M1 clean test", test_df),
]:
    missing_zip_refs = [
        name for name in dataframe["zip_file"].dropna().unique()
        if not (ZIP_DIR / name).exists()
    ]
    print(split_name, "| rows:", len(dataframe),
          "| missing ZIP references:", len(missing_zip_refs))
    assert not missing_zip_refs, f"{split_name} has missing ZIP references."

# ------------------------------------------------------------
# 5. M1 CLASS MAPPING, TRANSFORMS AND ZIP DATASET
# ------------------------------------------------------------

print("\nSTEP 4 — M1 DATASET CONFIGURATION")

class_mapping = {
    "Amphibolis": 0,
    "Background": 1,
    "Halophila": 2,
    "Posidonia": 3,
}
idx_to_class = {value: key for key, value in class_mapping.items()}

IMAGE_SIZE = 224
BATCH_SIZE = 32

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225],
    ),
])

class MarineZipDatasetV2(Dataset):
    def __init__(self, dataframe, zip_dir, transform=None):
        self.df = dataframe.reset_index(drop=True)
        self.zip_dir = Path(zip_dir)
        self.transform = transform
        self.zip_handles = {}

    def __len__(self):
        return len(self.df)

    def __getitem__(self, index):
        row = self.df.iloc[index]
        zip_name = row["zip_file"]
        image_name = row["image_name"]
        zip_path = self.zip_dir / zip_name

        if zip_name not in self.zip_handles:
            self.zip_handles[zip_name] = zipfile.ZipFile(zip_path, "r")

        archive = self.zip_handles[zip_name]
        try:
            image_bytes = archive.read(image_name)
        except KeyError:
            raise FileNotFoundError(
                f"Image {image_name} not found in {zip_name}"
            )

        with Image.open(io.BytesIO(image_bytes)) as opened:
            image = opened.convert("RGB")

        if self.transform:
            image = self.transform(image)

        label = class_mapping[row["class"]]
        return image, label

    def close(self):
        for archive in self.zip_handles.values():
            archive.close()
        self.zip_handles.clear()


train_dataset = MarineZipDatasetV2(train_df, ZIP_DIR, transform)
val_dataset = MarineZipDatasetV2(val_df, ZIP_DIR, transform)
test_dataset = MarineZipDatasetV2(test_df, ZIP_DIR, transform)

train_loader = DataLoader(
    train_dataset, batch_size=BATCH_SIZE, shuffle=True,
    num_workers=0, pin_memory=torch.cuda.is_available()
)
val_loader = DataLoader(
    val_dataset, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=0, pin_memory=torch.cuda.is_available()
)
test_loader = DataLoader(
    test_dataset, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=0, pin_memory=torch.cuda.is_available()
)

print("Device:", device)
print("M1 train/validation/test:", len(train_dataset), len(val_dataset), len(test_dataset))
print("M1 batches:", len(train_loader), len(val_loader), len(test_loader))

# ------------------------------------------------------------
# 6. RESTORE M2 RESNET18 CHECKPOINT
# ------------------------------------------------------------

print("\nSTEP 5 — M2 RESNET18 CHECKPOINT")

BEST_MODEL_PATH = MODELS_DIR / "resnet18_grouped_v2_baseline_best.pth"
LATEST_MODEL_PATH = MODELS_DIR / "resnet18_grouped_v2_baseline_latest.pth"

assert BEST_MODEL_PATH.exists(), "M2 best checkpoint is missing."
assert LATEST_MODEL_PATH.exists(), "M2 latest checkpoint is missing."

try:
    checkpoint = torch.load(
        BEST_MODEL_PATH, map_location=device, weights_only=False
    )
except TypeError:
    checkpoint = torch.load(BEST_MODEL_PATH, map_location=device)

assert "model_state_dict" in checkpoint
assert checkpoint["class_mapping"] == class_mapping

model = models.resnet18(weights=None)
model.fc = nn.Linear(model.fc.in_features, len(class_mapping))
model.load_state_dict(checkpoint["model_state_dict"], strict=True)
model = model.to(device)
model.eval()

print("M2 best checkpoint loaded.")
print("Best epoch:", checkpoint.get("epoch"))
print("Best validation loss:", checkpoint.get("best_val_loss"))
print("Model:", "ResNet18")
print("Evaluation mode:", not model.training)

# ------------------------------------------------------------
# 7. RESTORE M2/M3 RESULT TABLES
# ------------------------------------------------------------

print("\nSTEP 6 — M2/M3 RESULT TABLES")

result_files = sorted(RESULTS_DIR.glob("*.csv"))
results_tables = {}

for path in result_files:
    try:
        results_tables[path.stem] = pd.read_csv(path)
    except Exception as error:
        print("Could not read result CSV:", path.name, "|", error)

print("CSV result tables loaded:", len(results_tables))

gradcam_validation_df = results_tables.get(
    "grouped_v2_gradcam_validation_diagnostics"
)
gradcam_spatial_df = results_tables.get(
    "grouped_v2_gradcam_spatial_concentration"
)
gradcam_correct_error_df = results_tables.get(
    "grouped_v2_gradcam_correct_vs_error"
)

test_report_df = results_tables.get(
    "grouped_v2_baseline_clean_test_report"
)
test_predictions_df = results_tables.get(
    "grouped_v2_baseline_clean_test_predictions"
)
test_confusion_df = results_tables.get(
    "grouped_v2_baseline_clean_test_confusion_matrix"
)

# ------------------------------------------------------------
# 8. VERIFY M3 BACKUP WITHOUT ASSUMING NOTEBOOK HASHES MATCH
# ------------------------------------------------------------

print("\nSTEP 7 — M3 BACKUP VERIFICATION")

ORIGINAL_NOTEBOOK_SNAPSHOT = (
    BACKUP_DIR / "latest_notebook_snapshot" / "Mini Project_latest.ipynb"
)
LIVE_NOTEBOOK_SNAPSHOT = (
    BACKUP_DIR / "latest_notebook_snapshot"
    / "Mini_Project_live_20261114_144847.ipynb"
)

def calculate_sha256(path):
    digest = hashlib.sha256()
    with open(path, "rb") as file:
        for chunk in iter(lambda: file.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

backup_checks = {
    "M3 backup folder": BACKUP_DIR.exists(),
    "Original notebook snapshot": ORIGINAL_NOTEBOOK_SNAPSHOT.exists(),
    "Live notebook snapshot": LIVE_NOTEBOOK_SNAPSHOT.exists(),
    "Backup models folder": (BACKUP_DIR / "models").exists(),
    "Backup results folder": (BACKUP_DIR / "results").exists(),
    "Backup outputs folder": (BACKUP_DIR / "outputs").exists(),
}

for name, status in backup_checks.items():
    print(name, ":", "PASS" if status else "NOT FOUND")

if NOTEBOOK_PATH.exists():
    print("Current notebook SHA256:", calculate_sha256(NOTEBOOK_PATH))
else:
    print("Current notebook file not found at expected path.")

if LIVE_NOTEBOOK_SNAPSHOT.exists():
    print("Live snapshot SHA256:", calculate_sha256(LIVE_NOTEBOOK_SNAPSHOT))

print("Note: the older notebook snapshot is not expected to have the same")
print("checksum as the later live snapshot.")

M3_FILES = [
    "grouped_v2_gradcam_validation_diagnostics.csv",
    "grouped_v2_gradcam_spatial_concentration.csv",
    "grouped_v2_gradcam_correct_vs_error.csv",
]
m3_status = True
for filename in M3_FILES:
    exists = (RESULTS_DIR / filename).exists()
    print("M3 artifact:", filename, ":", exists)
    m3_status = m3_status and exists

# ------------------------------------------------------------
# 9. RESTORE M4 MANIFESTS FROM A MATCHING TIMESTAMP SET
# ------------------------------------------------------------

print("\nSTEP 8 — M4 IMO MANIFEST RESTORATION")

def find_latest_common_m4_timestamp():
    split_prefixes = {
        "train": "IMO_M4_train_manifest_",
        "validation": "IMO_M4_validation_manifest_",
        "independent_test": "IMO_M4_independent_test_manifest_",
    }
    timestamp_sets = {}

    for split_name, prefix in split_prefixes.items():
        timestamp_sets[split_name] = {
            path.stem[len(prefix):]
            for path in RESULTS_DIR.glob(prefix + "*.csv")
        }

    common = set.intersection(*timestamp_sets.values())
    if not common:
        raise FileNotFoundError(
            "Could not find a matching timestamp for all three M4 manifests."
        )
    return sorted(common)[-1]

M4_MANIFEST_TIMESTAMP = find_latest_common_m4_timestamp()

M4_MANIFEST_PATHS = {
    "train": RESULTS_DIR / f"IMO_M4_train_manifest_{M4_MANIFEST_TIMESTAMP}.csv",
    "validation": RESULTS_DIR / f"IMO_M4_validation_manifest_{M4_MANIFEST_TIMESTAMP}.csv",
    "independent_test": RESULTS_DIR / f"IMO_M4_independent_test_manifest_{M4_MANIFEST_TIMESTAMP}.csv",
}

for split_name, path in M4_MANIFEST_PATHS.items():
    assert path.exists(), f"Missing M4 {split_name} manifest: {path}"
    print(split_name, "manifest:", path.name)

m4_train_df = pd.read_csv(M4_MANIFEST_PATHS["train"])
m4_val_df = pd.read_csv(M4_MANIFEST_PATHS["validation"])
m4_test_df = pd.read_csv(M4_MANIFEST_PATHS["independent_test"])

M4_REQUIRED_COLUMNS = [
    "Im_id", "Filename", "Sp_id", "Especies",
    "Ecological_Status", "Orth_id", "F_id",
    "Location", "Dataset_type",
]

for split_name, dataframe in [
    ("Training", m4_train_df),
    ("Validation", m4_val_df),
    ("Independent test", m4_test_df),
]:
    missing_columns = [
        column for column in M4_REQUIRED_COLUMNS
        if column not in dataframe.columns
    ]
    assert not missing_columns, (
        f"{split_name} manifest missing columns: {missing_columns}"
    )

print("M4 split sizes:", len(m4_train_df), len(m4_val_df), len(m4_test_df))
assert len(m4_train_df) == 1528
assert len(m4_val_df) == 1249
assert len(m4_test_df) == 422
assert len(m4_train_df) + len(m4_val_df) + len(m4_test_df) == 3199

m4_development_df = pd.concat(
    [m4_train_df, m4_val_df], ignore_index=True
)
m4_binary_df = pd.concat(
    [m4_train_df, m4_val_df, m4_test_df], ignore_index=True
)

# Verify the locked grouping constraints from saved manifests.
train_orthos = set(m4_train_df["Orth_id"].dropna().astype(int))
val_orthos = set(m4_val_df["Orth_id"].dropna().astype(int))
train_flights = set(m4_train_df["F_id"].dropna().astype(int))
val_flights = set(m4_val_df["F_id"].dropna().astype(int))
test_flights = set(m4_test_df["F_id"].dropna().astype(int))

assert train_orthos.isdisjoint(val_orthos), (
    "M4 train/validation orthomosaic overlap detected."
)
assert train_flights.isdisjoint(test_flights), (
    "M4 train/test flight overlap detected."
)
assert val_flights.isdisjoint(test_flights), (
    "M4 validation/test flight overlap detected."
)

print("Train-validation shared orthomosaics:", len(train_orthos & val_orthos))
print("Train-test shared flights:", len(train_flights & test_flights))
print("Validation-test shared flights:", len(val_flights & test_flights))

# ------------------------------------------------------------
# 10. RESTORE M4 ECOLOGICAL LABELS AND CLASS WEIGHTS
# ------------------------------------------------------------

print("\nSTEP 9 — M4 ECOLOGICAL LABELS AND CLASS WEIGHTS")

ecological_labels = {
    "Asparagopsis armata": "Invasive",
    "Sargassum muticum": "Invasive",
    "Bifurcaria bifurcata": "Native",
    "Codium spp.": "Native",
    "Ericaria selaginoides": "Native",
    "Saccorhiza polyschides": "Native",
}

m4_class_names = ["Native", "Invasive"]

assert set(m4_binary_df["Ecological_Status"].unique()).issubset(
    set(m4_class_names)
)

m4_training_labels = m4_train_df["Ecological_Status"].values
m4_class_weight_values = compute_class_weight(
    class_weight="balanced",
    classes=np.array(m4_class_names),
    y=m4_training_labels,
)
m4_class_weight_dict = dict(
    zip(m4_class_names, m4_class_weight_values)
)

print("M4 selected images:", len(m4_binary_df))
print("M4 training class counts:")
print(m4_train_df["Ecological_Status"].value_counts())
print("M4 class weights:", m4_class_weight_dict)

# ------------------------------------------------------------
# 11. LOAD M4 AUDIT JSON
# ------------------------------------------------------------

print("\nSTEP 10 — M4 AUDIT REPORT")

audit_files = sorted(RESULTS_DIR.glob("IMO_M4_dataset_audit_*.json"))
m4_audit_report = None
M4_AUDIT_PATH = None

if audit_files:
    M4_AUDIT_PATH = audit_files[-1]
    with open(M4_AUDIT_PATH, "r") as file:
        m4_audit_report = json.load(file)
    print("Loaded:", M4_AUDIT_PATH.name)
    print("Selected images in audit:", m4_audit_report.get("selected_images"))
else:
    print("M4 audit JSON not found. Manifest restoration is still available.")

# ------------------------------------------------------------
# 12. RESTORE OR SAVE AVAILABLE M4 FULL METADATA
# ------------------------------------------------------------

print("\nSTEP 11 — M4 METADATA SNAPSHOT")

# If the full metadata dataframe is still in memory (for example, if this
# cell is run before a runtime restart), save a timestamped copy without
# overwriting anything. On a later restart, load the latest saved copy.
M4_METADATA_PATH = None

if "metadata_df" in globals() and isinstance(globals()["metadata_df"], pd.DataFrame):
    metadata_snapshot_time = datetime.now().strftime("%Y%m%d_%H%M%S")
    candidate_path = RESULTS_DIR / f"IMO_M4_full_metadata_{metadata_snapshot_time}.csv"
    if not candidate_path.exists():
        metadata_df.to_csv(candidate_path, index=False)
        M4_METADATA_PATH = candidate_path
        print("Saved current in-memory metadata snapshot:", candidate_path.name)
    else:
        print("Metadata snapshot already exists; not overwriting.")
elif list(RESULTS_DIR.glob("IMO_M4_full_metadata_*.csv")):
    M4_METADATA_PATH = sorted(
        RESULTS_DIR.glob("IMO_M4_full_metadata_*.csv")
    )[-1]
    metadata_df = pd.read_csv(M4_METADATA_PATH)
    print("Loaded saved full metadata:", M4_METADATA_PATH.name)
else:
    metadata_df = None
    print("Full 7,685-row metadata table is not available locally.")
    print("The three locked M4 manifests are restored and usable.")

# Convenient aliases for the selected M4 splits.
m4_train_manifest = m4_train_df
m4_validation_manifest = m4_val_df
m4_independent_test_manifest = m4_test_df

# ------------------------------------------------------------
# 13. RESTORE M4 IMAGE EXTRACTION STATUS
# ------------------------------------------------------------

print("\nSTEP 12 — M4 EXTRACTED IMAGE STATUS")

M4_IMAGE_ROOT.mkdir(parents=True, exist_ok=True)

m4_extracted_files = sorted(M4_IMAGE_ROOT.rglob("*.png"))
m4_extracted_valid = []
m4_extracted_invalid = []

for image_path in m4_extracted_files:
    try:
        with Image.open(image_path) as image:
            image.verify()
        m4_extracted_valid.append(image_path)
    except Exception as error:
        m4_extracted_invalid.append((image_path, str(error)))

print("M4 extraction folder:", M4_IMAGE_ROOT)
print("PNG files present:", len(m4_extracted_files))
print("Valid PNG files:", len(m4_extracted_valid))
print("Invalid PNG files:", len(m4_extracted_invalid))
print("Last known successful extraction: 1 image")
print("Last known remote extraction issue: HTTP 429")

# Rebuild a simple filename-to-split lookup from the saved manifests.
m4_filename_to_split = {}
for split_name, dataframe in [
    ("train", m4_train_df),
    ("validation", m4_val_df),
    ("independent_test", m4_test_df),
]:
    for filename in dataframe["Filename"].astype(str):
        m4_filename_to_split[filename] = split_name

# ------------------------------------------------------------
# 14. OPTIONAL REMOTE IMO ZIP SUPPORT — DISABLED BY DEFAULT
# ------------------------------------------------------------

print("\nSTEP 13 — OPTIONAL REMOTE IMO ZIP")

IMO_RECORD_API = "https://zenodo.org/api/records/14185867"
IMO_ZIP_NAME = "IMO_dataset.zip"

class RemoteZipReader:
    """Seekable HTTP Range reader for a remote ZIP; no automatic retries."""

    def __init__(self, url, size, session=None):
        self.url = url
        self.size = int(size)
        self.position = 0
        self.session = session or requests.Session()
        self.session.headers.update({
            "User-Agent": "Mozilla/5.0 (compatible; research-data-access/1.0)"
        })

    def seek(self, offset, whence=0):
        if whence == 0:
            new_position = offset
        elif whence == 1:
            new_position = self.position + offset
        elif whence == 2:
            new_position = self.size + offset
        else:
            raise ValueError("Invalid whence value.")
        if new_position < 0:
            raise ValueError("Negative seek position.")
        self.position = min(int(new_position), self.size)
        return self.position

    def tell(self):
        return self.position

    def readable(self):
        return True

    def seekable(self):
        return True

    def read(self, size=-1):
        if size is None or size < 0:
            size = self.size - self.position
        if size == 0 or self.position >= self.size:
            return b""

        start = self.position
        end = min(start + int(size) - 1, self.size - 1)

        response = self.session.get(
            self.url,
            headers={"Range": f"bytes={start}-{end}"},
            timeout=(20, 90),
        )

        if response.status_code == 429:
            raise RuntimeError(
                "Zenodo returned HTTP 429. Stop remote access and try later."
            )
        if response.status_code != 206:
            raise RuntimeError(
                f"Range request failed: HTTP {response.status_code}"
            )

        content_range = response.headers.get("Content-Range", "")
        expected_prefix = f"bytes {start}-{end}/"
        if not content_range.startswith(expected_prefix):
            raise RuntimeError(
                f"Unexpected Content-Range: {content_range}"
            )

        data = response.content
        expected_length = end - start + 1
        if len(data) != expected_length:
            raise RuntimeError(
                f"Incomplete range response: expected {expected_length}, "
                f"received {len(data)} bytes."
            )

        self.position = end + 1
        return data

    def close(self):
        self.session.close()

imo_remote_reader = None
imo_zip = None
file_url = None
total_size = None
archive_basenames = {}

if ENABLE_REMOTE_IMO_ZIP:
    print("Remote ZIP access explicitly enabled.")
    try:
        api_response = requests.get(IMO_RECORD_API, timeout=30)
        api_response.raise_for_status()
        record_data = api_response.json()

        imo_file_info = next(
            item for item in record_data["files"]
            if item["key"] == IMO_ZIP_NAME
        )
        file_url = imo_file_info["links"]["self"]
        total_size = int(imo_file_info["size"])

        imo_remote_reader = RemoteZipReader(file_url, total_size)
        imo_zip = zipfile.ZipFile(imo_remote_reader, "r")

        print("Remote IMO ZIP opened.")
        print("ZIP size:", round(total_size / (1024**3), 2), "GB")
        print("Archive entries:", len(imo_zip.namelist()))

        # Index PNG basenames only after remote ZIP opening succeeds.
        for member_name in imo_zip.namelist():
            if member_name.lower().endswith(".png"):
                basename = member_name.rsplit("/", 1)[-1]
                archive_basenames.setdefault(basename, []).append(member_name)

        print("PNG basename index entries:", len(archive_basenames))

    except Exception as error:
        print("Remote IMO ZIP was not opened:", str(error))
        print("M4 manifests and audit remain restored.")
        imo_zip = None
        archive_basenames = {}
else:
    print("Remote IMO ZIP access is disabled.")
    print("No Zenodo requests were made by this restore cell.")
    print("Use the saved M4 manifests; enable remote access only deliberately.")

# ------------------------------------------------------------
# 15. REBUILD M4 EXTRACTION QUEUE WITHOUT NETWORK ACCESS
# ------------------------------------------------------------

print("\nSTEP 14 — M4 EXTRACTION QUEUE")

m4_extraction_queue = []

for split_name, dataframe in [
    ("train", m4_train_df),
    ("validation", m4_val_df),
    ("independent_test", m4_test_df),
]:
    for _, row in dataframe.iterrows():
        m4_extraction_queue.append({
            "split": split_name,
            "filename": str(row["Filename"]),
            "archive_path": (
                "dataset_comprimir/Imagery/Images/Train/"
                + str(row["Filename"])
            ),
        })

print("Queued selected images:", len(m4_extraction_queue))
print("Queue built from saved manifests; no image requests made.")

# ------------------------------------------------------------
# 16. OPTIONAL M1 SAMPLE DECODING CHECK
# ------------------------------------------------------------

print("\nSTEP 15 — M1 RESTORATION SAMPLE CHECK")

try:
    for split_name, dataset in [
        ("M1 training", train_dataset),
        ("M1 validation", val_dataset),
        ("M1 clean test", test_dataset),
    ]:
        sample_image, sample_label = dataset[0]
        print(
            split_name,
            "| tensor:", tuple(sample_image.shape),
            "| class:", idx_to_class[sample_label],
            "| finite:", bool(torch.isfinite(sample_image).all().item()),
        )
except Exception as error:
    print("M1 sample decoding check failed:", str(error))
    print("Manifests and model checkpoint remain loaded.")

# ------------------------------------------------------------
# 17. FINAL RESTORE SUMMARY
# ------------------------------------------------------------

print("\n" + "=" * 72)
print("MASTER RESTORE V2 — SUMMARY")
print("=" * 72)

print("M1 — ECU-MSS-2 data audit and grouped manifests: RESTORED")
print("M2 — ResNet18 baseline checkpoint and results: RESTORED")
print("M3 — Grad-CAM exploratory artifacts: RESTORED" if m3_status
      else "M3 — One or more Grad-CAM artifacts need checking")
print("M4 — IMO manifests: RESTORED")
print("M4 — IMO audit JSON:", "AVAILABLE" if m4_audit_report else "NOT FOUND")
print("M4 — Full metadata table:", "AVAILABLE" if metadata_df is not None else "NOT LOCALLY SAVED")
print("M4 — Extracted valid images:", len(m4_extracted_valid))
print("M4 — Remote ZIP:", "OPEN" if imo_zip is not None else "NOT OPEN")
print("M4 — Remote access default:", ENABLE_REMOTE_IMO_ZIP)

print("\nM4 split sizes:")
print("Training:", len(m4_train_df))
print("Validation:", len(m4_val_df))
print("Independent test:", len(m4_test_df))
print("Total:", len(m4_binary_df))

print("\nM4 training class weights:")
print(m4_class_weight_dict)

print("\nCURRENT PROJECT POSITION")
print("M1: COMPLETE")
print("M2: COMPLETE")
print("M3: COMPLETE")
print("M4: IN PROGRESS — image extraction / preprocessing")
print("M5: NOT STARTED")
print("M6: NOT STARTED")

print("\nLAST COMPLETED M4 WORK:")
print("- 3,199 selected filenames matched the remote ZIP.")
print("- 12 sampled images passed image integrity checks.")
print("- Three timestamped manifests saved.")
print("- M4 dataset audit JSON saved.")
print("- Acquisition, geography, species and split audits completed.")
print("- Training class weights calculated: Native 0.5422, Invasive 6.4202.")
print("- 1 selected image extracted and verified.")
print("- Subsequent remote extraction request returned HTTP 429.")
print("- Full extraction and M4 preprocessing are still pending.")

print("\nNo model retraining performed.")
print("No existing checkpoint or result file overwritten.")
print("No full ZIP download or extraction performed.")
print("=" * 72)


In [ ]:

# ============================================================
# M4 — STEP 49: EXTRACTION ENVIRONMENT CHECK
# Read-only inspection; no downloads or file modifications.
# ============================================================

from pathlib import Path
import shutil
import pandas as pd

EXTRACT_DIR = Path("/content/IMO_M4_selected_images")
PROJECT_DIR = Path("/content/drive/MyDrive/Marine_Project")
RESULTS_DIR = PROJECT_DIR / "results"

print("=" * 65)
print("M4 STEP 49 — EXTRACTION ENVIRONMENT CHECK")
print("=" * 65)

# 1. Disk space
disk = shutil.disk_usage("/content")
print("\n1. AVAILABLE DISK SPACE")
print(f"Total:     {disk.total / (1024**3):.2f} GB")
print(f"Used:      {disk.used / (1024**3):.2f} GB")
print(f"Available: {disk.free / (1024**3):.2f} GB")

# 2. Extraction folder
print("\n2. EXTRACTION FOLDER")
print("Exists:", EXTRACT_DIR.exists())

if EXTRACT_DIR.exists():
    png_files = list(EXTRACT_DIR.rglob("*.png"))
    other_files = [
        p for p in EXTRACT_DIR.rglob("*")
        if p.is_file() and p.suffix.lower() != ".png"
    ]
    print("PNG files:", len(png_files))
    print("Other files:", len(other_files))
    print("PNG size:", f"{sum(p.stat().st_size for p in png_files)/(1024**3):.4f} GB")
else:
    print("Extraction folder is currently absent.")

# 3. Locked manifests
print("\n3. LOCKED M4 MANIFESTS")

manifest_names = [
    "IMO_M4_train_manifest_20261119_184551.csv",
    "IMO_M4_validation_manifest_20261119_184551.csv",
    "IMO_M4_independent_test_manifest_20261119_184551.csv"
]

for name in manifest_names:
    path = RESULTS_DIR / name
    if path.exists():
        df = pd.read_csv(path)
        print(f"{name}: {len(df)} rows | columns: {list(df.columns)}")
    else:
        print(f"MISSING: {name}")

# 4. Extraction queue variable
print("\n4. RESTORED EXTRACTION QUEUE")

queue_candidates = [
    "m4_extraction_queue",
    "extraction_queue",
    "m4_queue_df",
    "m4_selected_queue"
]

queue_found = False

for variable_name in queue_candidates:
    if variable_name in globals():
        queue = globals()[variable_name]
        queue_found = True
        print("Queue variable:", variable_name)
        print("Type:", type(queue).__name__)

        if isinstance(queue, pd.DataFrame):
            print("Rows:", len(queue))
            print("Columns:", list(queue.columns))
            print(queue.head(3).to_string(index=False))
        elif isinstance(queue, (list, tuple)):
            print("Queue entries:", len(queue))
            print("First entry:", queue[0] if queue else "Empty")
        else:
            print("Queue preview:", repr(queue)[:500])
        break

if not queue_found:
    print("No recognized queue variable found.")
    print("We can rebuild it from the saved manifests if necessary.")

# 5. Remote access state — inspect only, do not connect
print("\n5. REMOTE ACCESS")
print("Remote ZIP access remains disabled unless deliberately enabled.")
print("No network requests were made.")

print("\n" + "=" * 65)
print("STEP 49 CHECK COMPLETE")
print("Waiting for inspection before extraction.")
print("=" * 65)


In [ ]:

# ============================================================
# M4 — STEP 50: REMOTE ARCHIVE READINESS
# Read-only runtime inspection. No network requests.
# ============================================================

print("=" * 65)
print("M4 STEP 50 — REMOTE ARCHIVE READINESS")
print("=" * 65)

# Check remote archive information
for name in ["file_url", "file_info", "RemoteZipReader", "z"]:
    exists = name in globals()
    print(f"{name}: {'AVAILABLE' if exists else 'NOT AVAILABLE'}")

# Inspect saved archive URL without printing credentials or query strings
if "file_url" in globals():
    from urllib.parse import urlparse
    parsed = urlparse(file_url)
    print("\nArchive host:", parsed.netloc)
    print("Archive path:", parsed.path)

# Inspect remote ZIP reader class
if "RemoteZipReader" in globals():
    print("\nRemoteZipReader class is defined.")
else:
    print("\nRemoteZipReader class needs to be restored.")

# Inspect any existing reader object
if "z" in globals():
    print("\nExisting archive reader type:", type(z).__name__)
else:
    print("\nNo active remote archive reader.")

# Inspect extraction queue integrity
print("\nExtraction queue entries:", len(m4_extraction_queue))
print("First queued archive path:")
print(m4_extraction_queue[0]["archive_path"])

print("\n" + "=" * 65)
print("STEP 50 COMPLETE — NO NETWORK REQUESTS MADE")
print("=" * 65)


In [ ]:

# ============================================================
# M4 — STEP 51: REMOTE CONNECTION DIAGNOSTICS
# Read-only. Does not contact Zenodo.
# ============================================================

print("=" * 65)
print("M4 STEP 51 — REMOTE CONNECTION DIAGNOSTICS")
print("=" * 65)

# 1. Inspect URL variable
print("\n1. FILE URL")
print("Type:", type(file_url).__name__)
print("Value representation:", repr(file_url)[:300])
print("Is non-empty:", bool(file_url))

# 2. Inspect reader class constructor
print("\n2. REMOTE ZIP READER CLASS")
import inspect

print("Class:", RemoteZipReader)
try:
    print("Constructor:", inspect.signature(RemoteZipReader))
except Exception as e:
    print("Constructor inspection unavailable:", str(e))

# 3. Inspect class methods
print("\n3. AVAILABLE READER METHODS")
print([
    name for name in dir(RemoteZipReader)
    if not name.startswith("_")
])

# 4. Inspect likely connection variables
print("\n4. RELATED RUNTIME VARIABLES")
for name in [
    "session", "remote_session", "zenodo_url",
    "archive_url", "total_size", "archive_size",
    "file_size", "REMOTE_IMO_ZIP_URL"
]:
    if name in globals():
        value = globals()[name]
        print(name, "| type:", type(value).__name__,
              "| value:", repr(value)[:160])

print("\n" + "=" * 65)
print("STEP 51 COMPLETE — NO NETWORK REQUESTS MADE")
print("=" * 65)


In [ ]:

# ============================================================
# M4 — STEP 52: RESTORE PUBLIC ZENODO FILE METADATA
# Metadata request only. No archive download.
# ============================================================

import requests

ZENODO_RECORD_URL = "https://zenodo.org/api/records/14185867"

print("=" * 65)
print("M4 STEP 52 — ZENODO METADATA REQUEST")
print("=" * 65)

response = requests.get(
    ZENODO_RECORD_URL,
    timeout=30
)

print("HTTP status:", response.status_code)

response.raise_for_status()

record = response.json()

files = record.get("files", [])

print("\nRecord title:", record.get("metadata", {}).get("title"))
print("Number of record files:", len(files))

imo_file = None

for item in files:
    key = item.get("key", "")
    size = item.get("size", 0)

    print(f"\nFile: {key}")
    print(f"Size: {size / (1024**3):.2f} GB")

    if key == "IMO_dataset.zip":
        imo_file = item

if imo_file is None:
    raise RuntimeError("IMO_dataset.zip was not found in record metadata.")

# Restore connection information in memory only
file_info = imo_file
file_url = imo_file.get("links", {}).get("self")
total_size = imo_file.get("size")

print("\nRESTORED ARCHIVE INFORMATION")
print("URL available:", bool(file_url))
print("Total size:", f"{total_size / (1024**3):.2f} GB")

print("\nNo archive image data downloaded.")
print("=" * 65)
print("STEP 52 COMPLETE")
print("=" * 65)


In [ ]:

# ============================================================
# M4 — STEP 53 RETRY: RUNTIME CLASS INSPECTION
# No network requests or file modifications.
# ============================================================

print("=" * 65)
print("M4 STEP 53 — RUNTIME CLASS INSPECTION")
print("=" * 65)

print("\n1. CLASS MODULE")
print(RemoteZipReader.__module__)

print("\n2. CLASS DICTIONARY")
print(list(RemoteZipReader.__dict__.keys()))

print("\n3. INITIALIZER")
print(RemoteZipReader.__init__)

print("\n4. READ METHOD")
print(RemoteZipReader.read)

print("\n5. SEEK METHOD")
print(RemoteZipReader.seek)

print("\n6. READER OBJECT STATUS")
print("Active reader z:", "z" in globals())

print("\n7. CONNECTION DETAILS")
print("URL available:", bool(file_url))
print("Archive size:", total_size)

print("\n" + "=" * 65)
print("STEP 53 RETRY COMPLETE")
print("=" * 65)


In [ ]:

# ============================================================
# M4 — STEP 54: SINGLE 1 KB RANGE REQUEST
# Exactly one request. No automatic retries.
# ============================================================

import requests

print("=" * 65)
print("M4 STEP 54 — REMOTE RANGE TEST")
print("=" * 65)

test_session = requests.Session()
test_session.mount(
    "https://",
    requests.adapters.HTTPAdapter(max_retries=0)
)

try:
    response = test_session.get(
        file_url,
        headers={"Range": "bytes=0-1023"},
        timeout=30,
        stream=True
    )

    print("HTTP status:", response.status_code)
    print("Content-Range:", response.headers.get("Content-Range"))
    print("Content-Length:", response.headers.get("Content-Length"))
    print("Accept-Ranges:", response.headers.get("Accept-Ranges"))

    if response.status_code == 206:
        print("\nRESULT: Range access successful.")
    elif response.status_code == 429:
        print("\nRESULT: Zenodo rate limit. Stop remote requests.")
    else:
        print("\nRESULT: Unexpected response. Do not extract yet.")

    response.close()

except requests.RequestException as error:
    print("Request error:", repr(error))

finally:
    test_session.close()

print("\nNo archive images extracted.")
print("=" * 65)
print("STEP 54 COMPLETE")
print("=" * 65)


In [ ]:

# ============================================================
# M4 — STEP 55: ZIP CENTRAL DIRECTORY ACCESS TEST
# One 64 KB range request. No retries.
# ============================================================

import requests

print("=" * 65)
print("M4 STEP 55 — ZIP DIRECTORY ACCESS TEST")
print("=" * 65)

range_size = 64 * 1024
range_start = total_size - range_size
range_end = total_size - 1

session = requests.Session()
session.mount(
    "https://",
    requests.adapters.HTTPAdapter(max_retries=0)
)

try:
    response = session.get(
        file_url,
        headers={
            "Range": f"bytes={range_start}-{range_end}"
        },
        timeout=45
    )

    print("HTTP status:", response.status_code)
    print("Content-Range:", response.headers.get("Content-Range"))
    print("Received bytes:", len(response.content))

    if response.status_code == 206:
        print("\nRESULT: ZIP ending successfully retrieved.")
        print("ZIP directory inspection can be considered next.")
    elif response.status_code == 429:
        print("\nRESULT: Rate limited. Stop remote requests.")
    else:
        print("\nRESULT: Unexpected response. Do not extract.")

except requests.RequestException as error:
    print("Request error:", repr(error))

finally:
    session.close()

print("\nNo images extracted.")
print("=" * 65)
print("STEP 55 COMPLETE")
print("=" * 65)


In [ ]:

# ============================================================
# M4 — STEP 56: READ REMOTE ZIP DIRECTORY
# Metadata inspection only. No image extraction.
# ============================================================

import zipfile
import requests

print("=" * 65)
print("M4 STEP 56 — REMOTE ZIP DIRECTORY")
print("=" * 65)

remote_session = requests.Session()
remote_session.mount(
    "https://",
    requests.adapters.HTTPAdapter(max_retries=0)
)

remote_reader = None
remote_zip = None

try:
    remote_reader = RemoteZipReader(
        file_url,
        total_size,
        session=remote_session
    )

    remote_zip = zipfile.ZipFile(remote_reader, mode="r")
    archive_entries = remote_zip.infolist()

    print("ZIP directory successfully read.")
    print("Total archive entries:", len(archive_entries))

    extension_counts = {}

    for entry in archive_entries:
        if not entry.is_dir():
            ext = entry.filename.rsplit(".", 1)[-1].lower()
            extension_counts[ext] = extension_counts.get(ext, 0) + 1

    print("\nFile types:")
    for ext, count in sorted(
        extension_counts.items(),
        key=lambda item: item[1],
        reverse=True
    )[:12]:
        print(f".{ext}: {count}")

    # Confirm that selected manifest images exist in the archive
    archive_name_set = {
        entry.filename.rsplit("/", 1)[-1]
        for entry in archive_entries
        if not entry.is_dir()
    }

    selected_names = {
        item["filename"] for item in m4_extraction_queue
    }

    matched = len(selected_names.intersection(archive_name_set))

    print("\nSelected filenames:", len(selected_names))
    print("Matched in archive:", matched)
    print("Missing from archive:", len(selected_names) - matched)

    print("\nRESULT: Directory inspection completed.")

except Exception as error:
    print("\nDIRECTORY INSPECTION FAILED")
    print(type(error).__name__, str(error)[:500])
    print("Do not retry automatically.")

finally:
    if remote_zip is not None:
        remote_zip.close()
    elif remote_reader is not None:
        remote_reader.close()

    remote_session.close()

print("\nNo image files extracted.")
print("=" * 65)
print("STEP 56 COMPLETE")
print("=" * 65)


In [ ]:

# ============================================================
# M4 — STEP 57: SINGLE IMAGE EXTRACTION TEST
# One selected PNG only. No automatic retries.
# ============================================================

import zipfile
import requests
from pathlib import Path
from io import BytesIO
from PIL import Image

print("=" * 65)
print("M4 STEP 57 — SINGLE IMAGE EXTRACTION")
print("=" * 65)

EXTRACT_DIR = Path("/content/IMO_M4_selected_images")
EXTRACT_DIR.mkdir(parents=True, exist_ok=True)

first_item = m4_extraction_queue[0]
archive_path = first_item["archive_path"]
filename = first_item["filename"]
destination = EXTRACT_DIR / filename

print("Selected filename:", filename)
print("Archive path:", archive_path)

if destination.exists():
    raise FileExistsError(
        f"Destination already exists; refusing to overwrite: {destination}"
    )

session = requests.Session()
session.mount(
    "https://",
    requests.adapters.HTTPAdapter(max_retries=0)
)

reader = None
archive = None

try:
    reader = RemoteZipReader(file_url, total_size, session=session)
    archive = zipfile.ZipFile(reader, mode="r")

    # Confirm exact archive member
    if archive_path not in archive.namelist():
        raise FileNotFoundError(
            "Selected archive member was not found."
        )

    print("\nReading one selected image...")
    image_bytes = archive.read(archive_path)

    print("Bytes received:", len(image_bytes))

    # Validate before saving
    with Image.open(BytesIO(image_bytes)) as image:
        image.verify()

    with Image.open(BytesIO(image_bytes)) as image:
        image_format = image.format
        image_size = image.size
        image_mode = image.mode

    if image_format != "PNG":
        raise ValueError(f"Unexpected image format: {image_format}")

    # Exclusive creation prevents accidental overwrite
    with open(destination, "xb") as output_file:
        output_file.write(image_bytes)

    # Verify saved file
    with Image.open(destination) as saved_image:
        saved_image.verify()

    print("\nEXTRACTION SUCCESSFUL")
    print("Saved:", destination)
    print("Format:", image_format)
    print("Dimensions:", image_size)
    print("Mode:", image_mode)
    print("File size:", f"{destination.stat().st_size / 1024:.2f} KB")

except Exception as error:
    print("\nEXTRACTION FAILED")
    print(type(error).__name__, str(error)[:500])

    if destination.exists():
        destination.unlink()
        print("Incomplete test output removed.")

finally:
    if archive is not None:
        archive.close()
    elif reader is not None:
        reader.close()

    session.close()

print("\nPNG files currently present:", len(list(EXTRACT_DIR.glob("*.png"))))
print("=" * 65)
print("STEP 57 COMPLETE")
print("=" * 65)


In [ ]:

# ============================================================
# M4 — STEP 58A: VERIFY LOCAL EXTRACTION AFTER HTTP 500
# Local-only inspection. No network requests.
# ============================================================

from pathlib import Path
from PIL import Image

EXTRACT_DIR = Path("/content/IMO_M4_selected_images")

print("=" * 65)
print("M4 STEP 58A — LOCAL FILE SAFETY CHECK")
print("=" * 65)

png_files = list(EXTRACT_DIR.glob("*.png"))
other_files = [
    p for p in EXTRACT_DIR.iterdir()
    if p.is_file() and p.suffix.lower() != ".png"
]

valid = 0
invalid = []

for path in png_files:
    try:
        with Image.open(path) as img:
            img.verify()
        valid += 1
    except Exception as error:
        invalid.append((path.name, str(error)[:150]))

print("PNG files present:", len(png_files))
print("Valid PNG files:", valid)
print("Invalid PNG files:", len(invalid))
print("Other files:", len(other_files))

print("\nExtracted filenames:")
for path in png_files:
    print("-", path.name)

if invalid:
    print("\nInvalid files:")
    for name, error in invalid:
        print(name, "|", error)

print("\nNo files deleted or modified.")
print("=" * 65)
print("STEP 58A COMPLETE")
print("=" * 65)


In [ ]:

# ============================================================
# M4 — STEP 59: RESUMABLE EXTRACTION TRACKER
# Local-only. No downloads. No file modifications.
# ============================================================

from pathlib import Path
from PIL import Image

EXTRACT_DIR = Path("/content/IMO_M4_selected_images")

verified_filenames = set()
invalid_filenames = set()

for path in EXTRACT_DIR.glob("*.png"):
    try:
        with Image.open(path) as img:
            img.verify()
        verified_filenames.add(path.name)
    except Exception:
        invalid_filenames.add(path.name)

# Rebuild pending queue without changing the original queue
m4_verified_images = verified_filenames.copy()

m4_pending_queue = [
    item for item in m4_extraction_queue
    if item["filename"] not in m4_verified_images
]

m4_invalid_images = invalid_filenames.copy()

print("=" * 65)
print("M4 STEP 59 — RESUMABLE EXTRACTION TRACKER")
print("=" * 65)

print("Total selected images:", len(m4_extraction_queue))
print("Verified extracted images:", len(m4_verified_images))
print("Invalid extracted images:", len(m4_invalid_images))
print("Pending images:", len(m4_pending_queue))

print(
    "Completion:",
    f"{len(m4_verified_images) / len(m4_extraction_queue) * 100:.3f}%"
)

print("\nNext pending image:")
if m4_pending_queue:
    print(m4_pending_queue[0])
else:
    print("No pending images.")

print("\nOriginal extraction queue preserved:",
      len(m4_extraction_queue) == 3199)

print("=" * 65)
print("STEP 59 COMPLETE")
print("=" * 65)


In [ ]:

# ============================================================
# M4 — STEP 60: SAVE PERSISTENT EXTRACTION CHECKPOINT
# ============================================================

from pathlib import Path
from datetime import datetime
import pandas as pd

# Confirm Google Drive results folder
RESULTS_DIR = Path(
    "/content/drive/MyDrive/Marine_Project/results"
)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

# Build checkpoint from the original extraction queue
checkpoint_rows = []

for item in m4_extraction_queue:
    filename = item["filename"]

    if filename in m4_verified_images:
        status = "verified"
    elif filename in m4_invalid_images:
        status = "invalid_existing"
    else:
        status = "pending"

    checkpoint_rows.append({
        "split": item["split"],
        "filename": filename,
        "archive_path": item["archive_path"],
        "status": status
    })

checkpoint_df = pd.DataFrame(checkpoint_rows)

# Create a unique filename; never overwrite
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
checkpoint_path = (
    RESULTS_DIR /
    f"IMO_M4_extraction_checkpoint_{timestamp}.csv"
)

counter = 1
while checkpoint_path.exists():
    checkpoint_path = (
        RESULTS_DIR /
        f"IMO_M4_extraction_checkpoint_{timestamp}_{counter}.csv"
    )
    counter += 1

# Exclusive creation protects existing files
with open(checkpoint_path, "x", encoding="utf-8", newline="") as f:
    checkpoint_df.to_csv(f, index=False)

# Verify saved checkpoint
saved_checkpoint = pd.read_csv(checkpoint_path)

print("=" * 65)
print("M4 STEP 60 — PERSISTENT CHECKPOINT")
print("=" * 65)

print("Saved file:", checkpoint_path)
print("Rows saved:", len(saved_checkpoint))
print("\nStatus counts:")
print(saved_checkpoint["status"].value_counts().to_string())

print("\nVerification:")
print("File exists:", checkpoint_path.exists())
print("Saved rows match:", len(saved_checkpoint) == len(m4_extraction_queue))
print(
    "Verified image count:",
    (saved_checkpoint["status"] == "verified").sum()
)
print(
    "Pending image count:",
    (saved_checkpoint["status"] == "pending").sum()
)

print("\nSTEP 60 COMPLETE")
print("=" * 65)


In [ ]:

# ============================================================
# M4 — STEP 61A: SAFE EXTRACTION FUNCTION
# Preparation only — no network requests.
# ============================================================

from pathlib import Path
from PIL import Image
import io

EXTRACT_DIR = Path("/content/IMO_M4_selected_images")
EXTRACT_DIR.mkdir(parents=True, exist_ok=True)

m4_extraction_success = []
m4_extraction_failures = []

def safe_save_m4_image(filename, image_bytes):
    """
    Save one PNG safely.
    Never overwrite an existing file.
    Verify image bytes before saving.
    """

    destination = EXTRACT_DIR / Path(filename).name

    # Skip existing files
    if destination.exists():
        return {
            "filename": filename,
            "status": "skipped_existing",
            "reason": "File already exists"
        }

    try:
        # Verify image data before writing
        with Image.open(io.BytesIO(image_bytes)) as img:
            img.verify()

        # Exclusive creation: fails if file already exists
        with open(destination, "xb") as f:
            f.write(image_bytes)

        # Verify saved image
        with Image.open(destination) as img:
            img.verify()

        result = {
            "filename": filename,
            "status": "verified",
            "reason": "Saved and verified successfully"
        }

        m4_extraction_success.append(result)
        return result

    except Exception as e:
        result = {
            "filename": filename,
            "status": "failed",
            "reason": str(e)
        }

        m4_extraction_failures.append(result)
        return result

print("M4 STEP 61A — SAFE EXTRACTION FUNCTION")
print("Function created:", callable(safe_save_m4_image))
print("Extraction folder:", EXTRACT_DIR)
print("Existing verified images:", len(m4_verified_images))
print("Network requests made: 0")
print("Existing images modified: 0")
print("STEP 61A COMPLETE")


In [ ]:

# ============================================================
# M4 — STEP 61B: LOCAL SAFETY TEST
# Uses the existing image only. No downloads.
# ============================================================

from pathlib import Path

test_filename = "AR12m-230705-0-1-20240405105222class6.png"
test_path = EXTRACT_DIR / test_filename

assert test_path.exists(), "Previously extracted test image is missing."

# Read existing image bytes
with open(test_path, "rb") as f:
    test_bytes = f.read()

# Test 1: Existing filename must be skipped
result_existing = safe_save_m4_image(test_filename, test_bytes)

print("Existing-file test:", result_existing["status"])
print("Reason:", result_existing["reason"])

# Test 2: Ensure original file remains intact
with Image.open(test_path) as img:
    img.verify()

print("Original image still valid: True")
print("Original image size:", test_path.stat().st_size, "bytes")

assert result_existing["status"] == "skipped_existing"

print("\nSTEP 61B COMPLETE")


In [ ]:

# ============================================================
# M4 — STEP 61C: INVALID IMAGE SAFETY TEST
# No network requests. No existing files modified.
# ============================================================

test_invalid_filename = "M4_invalid_safety_test.png"
test_invalid_path = EXTRACT_DIR / test_invalid_filename

assert not test_invalid_path.exists(), (
    "Test filename already exists. Stopping to avoid overwriting."
)

invalid_bytes = b"This is not a valid PNG image."

result_invalid = safe_save_m4_image(
    test_invalid_filename,
    invalid_bytes
)

print("Invalid image test:", result_invalid["status"])
print("Error detected:", bool(result_invalid["reason"]))
print("Invalid file created:", test_invalid_path.exists())

assert result_invalid["status"] == "failed"
assert not test_invalid_path.exists()

print("\nSuccessful extraction records:", len(m4_extraction_success))
print("Failed extraction records:", len(m4_extraction_failures))
print("Network requests made: 0")
print("STEP 61C COMPLETE")


In [ ]:

# ============================================================
# M4 — STEP 61D: EXTRACTION RESULT LOGGER
# No downloads. No existing files overwritten.
# ============================================================

from datetime import datetime
import pandas as pd

def save_m4_extraction_log():
    log_rows = []

    for item in m4_extraction_success:
        log_rows.append({
            "filename": item["filename"],
            "status": item["status"],
            "reason": item["reason"]
        })

    for item in m4_extraction_failures:
        log_rows.append({
            "filename": item["filename"],
            "status": item["status"],
            "reason": item["reason"]
        })

    log_df = pd.DataFrame(
        log_rows,
        columns=["filename", "status", "reason"]
    )

    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    log_path = RESULTS_DIR / f"IMO_M4_extraction_log_{timestamp}.csv"

    counter = 1
    while log_path.exists():
        log_path = RESULTS_DIR / (
            f"IMO_M4_extraction_log_{timestamp}_{counter}.csv"
        )
        counter += 1

    with open(log_path, "x", encoding="utf-8", newline="") as f:
        log_df.to_csv(f, index=False)

    # Verify the saved log
    check_df = pd.read_csv(log_path)

    return log_path, check_df


print("Logger function created:", callable(save_m4_extraction_log))
print("Successful records in memory:", len(m4_extraction_success))
print("Failed records in memory:", len(m4_extraction_failures))
print("Network requests made: 0")
print("STEP 61D COMPLETE")


In [ ]:

# ============================================================
# M4 — STEP 61E: LOGGER VERIFICATION
# Saves only the current test record.
# ============================================================

log_path, check_df = save_m4_extraction_log()

print("=" * 60)
print("M4 EXTRACTION LOGGER TEST")
print("=" * 60)

print("Saved log:", log_path)
print("File exists:", log_path.exists())
print("Rows saved:", len(check_df))

print("\nSaved records:")
print(check_df.to_string(index=False))

assert log_path.exists()
assert len(check_df) == len(m4_extraction_failures) + len(m4_extraction_success)
assert "M4_invalid_safety_test.png" in check_df["filename"].values

print("\nCSV reopened successfully: True")
print("Existing checkpoint untouched: True")
print("STEP 61E COMPLETE")


In [ ]:

# ============================================================
# M4 — STEP 62: CONTROLLED ZENODO CONNECTION CHECK
# One lightweight request. No image downloads.
# ============================================================

import requests

test_url = file_url

print("Checking Zenodo archive endpoint...")
print("Archive URL available:", bool(test_url))

try:
    response = requests.get(
        test_url,
        headers={"Range": "bytes=0-1023"},
        stream=True,
        timeout=(15, 30)
    )

    print("HTTP status:", response.status_code)
    print("Content-Range:", response.headers.get("Content-Range"))
    print("Server response received:", response.status_code in (200, 206))

    response.close()

except Exception as e:
    print("Connection check failed:", type(e).__name__, str(e))

print("\nNo images downloaded.")
print("No local files modified.")
print("STEP 62 CHECK COMPLETE")


In [ ]:

# ============================================================
# M4 — STEP 63: RECOVER EXTRACTION CHECKPOINT
# Local Google Drive read only. No network requests.
# ============================================================

import pandas as pd
from pathlib import Path

checkpoint_path = Path(
    "/content/drive/MyDrive/Marine_Project/results/"
    "IMO_M4_extraction_checkpoint_20261121_144333.csv"
)

assert checkpoint_path.exists(), "Persistent checkpoint not found."

recovered_checkpoint = pd.read_csv(checkpoint_path)

recovered_verified = recovered_checkpoint[
    recovered_checkpoint["status"] == "verified"
]["filename"].tolist()

recovered_pending = recovered_checkpoint[
    recovered_checkpoint["status"] == "pending"
]["filename"].tolist()

recovered_invalid = recovered_checkpoint[
    recovered_checkpoint["status"] == "invalid_existing"
]["filename"].tolist()

print("=" * 60)
print("M4 STEP 63 — CHECKPOINT RECOVERY")
print("=" * 60)

print("Checkpoint recovered:", checkpoint_path.exists())
print("Total records:", len(recovered_checkpoint))
print("Verified:", len(recovered_verified))
print("Pending:", len(recovered_pending))
print("Invalid existing:", len(recovered_invalid))

print("\nFirst verified image:", recovered_verified[0] if recovered_verified else "None")
print("First pending image:", recovered_pending[0] if recovered_pending else "None")

assert len(recovered_checkpoint) == 3199
assert len(recovered_verified) == 1
assert len(recovered_pending) == 3198

print("\nRecovery verification: PASSED")
print("Original checkpoint unchanged.")
print("Network requests made: 0")
print("STEP 63 COMPLETE")


In [ ]:

# ============================================================
# M4 — STEP 64: LOCAL FILE AND CHECKPOINT CONSISTENCY
# Read-only verification. No network requests.
# ============================================================

from pathlib import Path
from PIL import Image

verified_on_disk = []
missing_on_disk = []
invalid_on_disk = []

for filename in recovered_verified:
    image_path = EXTRACT_DIR / filename

    if not image_path.exists():
        missing_on_disk.append(filename)
        continue

    try:
        with Image.open(image_path) as img:
            img.verify()
        verified_on_disk.append(filename)
    except Exception:
        invalid_on_disk.append(filename)

print("=" * 60)
print("M4 STEP 64 — LOCAL FILE VERIFICATION")
print("=" * 60)

print("Checkpoint verified entries:", len(recovered_verified))
print("Valid files on disk:", len(verified_on_disk))
print("Missing files:", len(missing_on_disk))
print("Corrupted files:", len(invalid_on_disk))

print("\nVerified filenames:")
for filename in verified_on_disk:
    print("-", filename)

assert len(missing_on_disk) == 0
assert len(invalid_on_disk) == 0

print("\nCHECKPOINT AND DISK CONSISTENCY: PASSED")
print("No files modified.")
print("Network requests made: 0")
print("STEP 64 COMPLETE")


In [ ]:

# ============================================================
# M4 — STEP 65: PREPARE NEXT 5-IMAGE BATCH
# No downloads. No file modifications.
# ============================================================

import pandas as pd

BATCH_SIZE = 5

pending_df = recovered_checkpoint[
    recovered_checkpoint["status"] == "pending"
].copy()

next_batch_df = pending_df.head(BATCH_SIZE).copy()

print("=" * 60)
print("M4 STEP 65 — NEXT EXTRACTION BATCH")
print("=" * 60)

print("Batch size:", len(next_batch_df))
print("\nSelected images:")

for _, row in next_batch_df.iterrows():
    print("-", row["filename"])
    print("  Split:", row["split"])
    print("  Archive:", row["archive_path"])

print("\nFilename uniqueness:",
      next_batch_df["filename"].is_unique)

print("Archive path uniqueness:",
      next_batch_df["archive_path"].is_unique)

print("All selected entries pending:",
      (next_batch_df["status"] == "pending").all())

assert len(next_batch_df) == BATCH_SIZE
assert next_batch_df["filename"].is_unique
assert next_batch_df["archive_path"].is_unique
assert (next_batch_df["status"] == "pending").all()

m4_next_batch = next_batch_df.to_dict("records")

print("\nBatch prepared successfully.")
print("Network requests made: 0")
print("Existing files modified: 0")
print("STEP 65 COMPLETE")


In [ ]:

# ============================================================
# M4 — STEP 66: SAVE NEXT BATCH MANIFEST
# Drive write only. No downloads or overwrites.
# ============================================================

from datetime import datetime
from pathlib import Path
import pandas as pd

batch_df = pd.DataFrame(m4_next_batch)

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
batch_path = RESULTS_DIR / f"IMO_M4_next_batch_{timestamp}.csv"

counter = 1
while batch_path.exists():
    batch_path = RESULTS_DIR / (
        f"IMO_M4_next_batch_{timestamp}_{counter}.csv"
    )
    counter += 1

with open(batch_path, "x", encoding="utf-8", newline="") as f:
    batch_df.to_csv(f, index=False)

# Reopen and verify
saved_batch = pd.read_csv(batch_path)

print("=" * 60)
print("M4 STEP 66 — BATCH MANIFEST")
print("=" * 60)

print("Saved path:", batch_path)
print("File exists:", batch_path.exists())
print("Rows saved:", len(saved_batch))
print("Filenames unique:", saved_batch["filename"].is_unique)
print("Archive paths unique:", saved_batch["archive_path"].is_unique)

assert len(saved_batch) == 5
assert saved_batch["filename"].is_unique
assert saved_batch["archive_path"].is_unique

print("\nBATCH MANIFEST VERIFIED")
print("No network requests made.")
print("No existing files overwritten.")
print("STEP 66 COMPLETE")


In [ ]:

# ============================================================
# M4 — STEP 67: ONE-TIME ZENODO AVAILABILITY CHECK
# One request only. No retries. No image extraction.
# ============================================================

import requests

print("Starting one-time Zenodo check...")

try:
    with requests.get(
        file_url,
        headers={"Range": "bytes=0-1023"},
        stream=True,
        timeout=(10, 15)
    ) as response:

        print("HTTP status:", response.status_code)
        print("Content-Range:", response.headers.get("Content-Range"))

        if response.status_code == 206:
            print("Result: Range access available.")
        elif response.status_code == 429:
            print("Result: Server rate limit. Stop requests.")
        elif response.status_code >= 500:
            print("Result: Server error. Stop requests.")
        else:
            print("Result: Unexpected response. Do not extract.")

except requests.exceptions.Timeout:
    print("Result: Connection timed out. Stop requests.")

except requests.exceptions.RequestException as e:
    print("Result: Connection error:", type(e).__name__)

print("\nRetries: 0")
print("Images downloaded: 0")
print("Local files modified: 0")
print("STEP 67 CHECK COMPLETE")


In [ ]:

# M4 STEP 68 — CHECK STORAGE BEFORE FULL DATASET DOWNLOAD

import shutil
from pathlib import Path

disk = shutil.disk_usage("/content")

archive_size_gb = 29.15
free_gb = disk.free / (1024 ** 3)

print("Available Colab storage:", round(free_gb, 2), "GB")
print("IMO dataset archive size:", archive_size_gb, "GB")
print("Space available for download:", free_gb > archive_size_gb)

print("\nExisting project ZIP files:")
for p in Path("/content/drive/MyDrive/Marine_Project/dataset_zips").glob("*.zip"):
    print("-", p.name)

print("\nSTEP 68 COMPLETE")


In [ ]:

# ============================================================
# M4 STEP 69 — FULL IMO DATASET DOWNLOAD
# Resumable download. Does not extract or delete anything.
# ============================================================

import subprocess
import shutil
from pathlib import Path

archive_dir = Path("/content")
expected_size = 31_302_217_484

assert file_url and "zenodo.org" in file_url, "Zenodo download URL unavailable."
assert shutil.which("wget"), "wget is not available."

print("Starting full IMO dataset download...")
print("Destination: /content")
print("Expected size: approximately 29.15 GB")
print("Resume support: enabled")
print("Existing project files: preserved")

result = subprocess.run([
    "wget",
    "-c",
    "--content-disposition",
    "--directory-prefix=/content",
    "--tries=3",
    "--timeout=60",
    "--waitretry=10",
    "--progress=bar:force:noscroll",
    file_url
])

print("\nWget exit code:", result.returncode)

# Inspect downloaded archive without modifying it
archive_candidates = [
    p for p in archive_dir.glob("IMO_dataset.zip")
    if p.is_file()
]

if archive_candidates:
    archive_path = archive_candidates[0]
    actual_size = archive_path.stat().st_size

    print("Archive found:", archive_path)
    print("Downloaded size (GB):", round(actual_size / (1024**3), 2))
    print("Expected size match:", actual_size == expected_size)
    print("Download complete:", actual_size == expected_size)
else:
    print("IMO_dataset.zip not found yet.")
    print("If interrupted, rerun this cell to resume.")


In [ ]:

# ============================================================
# M4 STEP 70A — INSPECT AVAILABLE ZENODO LINKS
# Read-only diagnostic. No network requests.
# ============================================================

print("File metadata keys:")
print(list(file_info.keys()))

print("\nAvailable links:")
for key, value in file_info.get("links", {}).items():
    print(f"{key}: {value}")

print("\nCurrent file_url:")
print(file_url)

print("\nExpected archive size:")
print(file_info.get("size"), "bytes")

print("\nSTEP 70A COMPLETE")


In [ ]:

# ============================================================
# M4 STEP 70B — FULL IMO DOWNLOAD WITH PROGRESS BAR
# Correct Zenodo URL + resume support
# ============================================================

import requests
from pathlib import Path
from tqdm.auto import tqdm

download_url = file_info["links"]["self"]

partial_path = Path("/content/IMO_dataset.zip.part")
final_path = Path("/content/IMO_dataset.zip")
expected_size = file_info["size"]

if final_path.exists():
    raise FileExistsError(
        "Final archive already exists. Stopping without overwriting."
    )

existing_size = partial_path.stat().st_size if partial_path.exists() else 0

if existing_size > expected_size:
    raise RuntimeError("Partial file is larger than expected. Stopping.")

headers = {}
if existing_size:
    headers["Range"] = f"bytes={existing_size}-"

print("IMO DATASET DOWNLOAD")
print("URL:", download_url)
print("Expected size:", round(expected_size / (1024**3), 2), "GB")
print("Already downloaded:", round(existing_size / (1024**3), 2), "GB")
print("Starting download...\n")

try:
    with requests.get(
        download_url,
        headers=headers,
        stream=True,
        timeout=(30, 180)
    ) as response:

        print("HTTP status:", response.status_code)

        if existing_size and response.status_code != 206:
            raise RuntimeError(
                "Server did not support resume. Partial file preserved."
            )

        if not existing_size and response.status_code != 200:
            raise RuntimeError(
                f"Unexpected HTTP response: {response.status_code}"
            )

        if existing_size:
            content_range = response.headers.get("Content-Range", "")
            if not content_range.startswith(f"bytes {existing_size}-"):
                raise RuntimeError(
                    "Content-Range mismatch. Partial file preserved."
                )

        response.raise_for_status()

        with open(partial_path, "ab" if existing_size else "wb") as f:
            with tqdm(
                total=expected_size,
                initial=existing_size,
                unit="B",
                unit_scale=True,
                unit_divisor=1024,
                desc="IMO ZIP",
                dynamic_ncols=True
            ) as bar:

                for chunk in response.iter_content(
                    chunk_size=8 * 1024 * 1024
                ):
                    if chunk:
                        f.write(chunk)
                        bar.update(len(chunk))

    actual_size = partial_path.stat().st_size

    if actual_size == expected_size:
        partial_path.rename(final_path)
        print("\nDOWNLOAD COMPLETE!")
        print("Archive:", final_path)
        print("Size:", actual_size, "bytes")
    else:
        print("\nDownload incomplete.")
        print("Downloaded:", actual_size, "bytes")
        print("Run this cell again to resume.")

except Exception as e:
    print("\nDOWNLOAD INTERRUPTED:", type(e).__name__, str(e))
    if partial_path.exists():
        print("Partial download preserved.")
        print("Current size (GB):",
              round(partial_path.stat().st_size / (1024**3), 3))
    else:
        print("No partial archive was created.")


In [ ]:

# ============================================================
# M4 STEP 72 — DIAGNOSE ARCHIVE PATH MISMATCH
# Read-only. No extraction or file modifications.
# ============================================================

import zipfile
from pathlib import PurePosixPath
from collections import Counter

with zipfile.ZipFile("/content/IMO_dataset.zip", "r") as zf:

    archive_files = [
        name for name in zf.namelist()
        if not name.endswith("/")
    ]

    archive_basenames = Counter(
        PurePosixPath(name).name for name in archive_files
    )

    missing_details = []

    for item in m4_extraction_queue:
        path = item["archive_path"]

        if path not in set(archive_files):
            basename = PurePosixPath(path).name
            missing_details.append({
                "filename": basename,
                "expected_path": path,
                "basename_matches": archive_basenames[basename]
            })

print("=" * 65)
print("M4 STEP 72 — PATH DIAGNOSIS")
print("=" * 65)

print("Total ZIP files:", len(archive_files))
print("Exact path matches:", 3199 - len(missing_details))
print("Exact path mismatches:", len(missing_details))

print("\nFirst 10 mismatched entries:")
for item in missing_details[:10]:
    print("\nFilename:", item["filename"])
    print("Expected:", item["expected_path"])
    print("Basename matches in ZIP:", item["basename_matches"])

print("\nArchive image path examples:")
for name in archive_files[:10]:
    print("-", name)

print("\nSTEP 72 DIAGNOSTIC COMPLETE")
print("No files extracted or modified.")


In [ ]:

# ============================================================
# M4 STEP 73 — MAP EXPECTED PATHS TO ACTUAL ZIP PATHS
# Read-only. No extraction or file modifications.
# ============================================================

import zipfile
from pathlib import PurePosixPath
from collections import defaultdict

with zipfile.ZipFile("/content/IMO_dataset.zip", "r") as zf:

    actual_paths = defaultdict(list)

    for path in zf.namelist():
        if path.lower().endswith(".png"):
            actual_paths[PurePosixPath(path).name].append(path)

    mismatch_rows = []

    for item in m4_extraction_queue:
        expected = item["archive_path"]
        filename = PurePosixPath(expected).name

        if expected not in zf.namelist():
            matches = actual_paths.get(filename, [])
            mismatch_rows.append((filename, expected, matches))

print("Mismatched manifest entries:", len(mismatch_rows))
print("\nFirst 20 actual path mappings:\n")

for filename, expected, matches in mismatch_rows[:20]:
    print("Filename:", filename)
    print("Expected:", expected)
    for actual in matches:
        print("ACTUAL:  ", actual)
    print("-" * 65)

print("\nActual folder distribution among mismatches:")
folder_counts = defaultdict(int)

for _, _, matches in mismatch_rows:
    for path in matches:
        folder = str(PurePosixPath(path).parent)
        folder_counts[folder] += 1

for folder, count in sorted(folder_counts.items()):
    print(folder, ":", count)

print("\nSTEP 73 COMPLETE — READ ONLY")


In [ ]:

# ============================================================
# M4 STEP 74 — BUILD CORRECTED ARCHIVE PATH MANIFEST
# Read-only verification. No extraction.
# ============================================================

import zipfile
from pathlib import PurePosixPath
from collections import defaultdict

with zipfile.ZipFile("/content/IMO_dataset.zip", "r") as zf:

    archive_names = set(zf.namelist())

    basename_to_paths = defaultdict(list)

    for path in archive_names:
        if path.lower().endswith(".png"):
            basename_to_paths[PurePosixPath(path).name].append(path)

    corrected_rows = []
    unresolved = []
    ambiguous = []

    for item in m4_extraction_queue:

        filename = PurePosixPath(item["archive_path"]).name
        matches = basename_to_paths.get(filename, [])

        if len(matches) == 1:
            corrected_item = dict(item)
            corrected_item["archive_path"] = matches[0]
            corrected_rows.append(corrected_item)

        elif len(matches) == 0:
            unresolved.append(filename)

        else:
            ambiguous.append((filename, matches))

print("=" * 65)
print("M4 STEP 74 — CORRECTED PATH VERIFICATION")
print("=" * 65)

print("Original selected images:", len(m4_extraction_queue))
print("Corrected unique paths:", len(corrected_rows))
print("Unresolved filenames:", len(unresolved))
print("Ambiguous filenames:", len(ambiguous))

if unresolved:
    print("\nUnresolved examples:", unresolved[:5])

if ambiguous:
    print("\nAmbiguous examples:", ambiguous[:5])

assert len(corrected_rows) == len(m4_extraction_queue)
assert len(unresolved) == 0
assert len(ambiguous) == 0

m4_corrected_extraction_queue = corrected_rows

print("\nCORRECTED MANIFEST VERIFIED.")
print("Original manifests and split assignments remain unchanged.")


In [ ]:

# ============================================================
# M4 STEP 75 — EXTRACT FIRST 50 SELECTED IMAGES
# Uses local ZIP only. Does not overwrite existing files.
# ============================================================

import zipfile
from pathlib import Path
from PIL import Image
from io import BytesIO

extract_dir = Path("/content/IMO_M4_selected_images")
extract_dir.mkdir(parents=True, exist_ok=True)

# Find first 50 images not already extracted and verified
batch = []

for item in m4_corrected_extraction_queue:
    destination = extract_dir / item["filename"]

    if destination.exists():
        try:
            with Image.open(destination) as img:
                img.verify()
            continue
        except Exception:
            print("Existing invalid file; leaving untouched:", destination.name)
            continue

    batch.append(item)

    if len(batch) == 50:
        break

print("Images selected for this batch:", len(batch))

verified = 0
failed = []

with zipfile.ZipFile("/content/IMO_dataset.zip", "r") as zf:

    for index, item in enumerate(batch, start=1):

        destination = extract_dir / item["filename"]

        try:
            image_bytes = zf.read(item["archive_path"])

            with Image.open(BytesIO(image_bytes)) as img:
                img.verify()

            # Exclusive creation prevents overwriting.
            with open(destination, "xb") as f:
                f.write(image_bytes)

            with Image.open(destination) as img:
                img.verify()

            verified += 1

        except Exception as e:
            failed.append({
                "filename": item["filename"],
                "error": str(e)
            })

        if index % 10 == 0 or index == len(batch):
            print(f"Processed {index}/{len(batch)} | Verified: {verified} | Failed: {len(failed)}")

print("\nFIRST BATCH COMPLETE")
print("Verified images:", verified)
print("Failed images:", len(failed))
print("Extraction folder:", extract_dir)


In [ ]:

# ============================================================
# M4 STEP 76 — EXTRACT NEXT 200 SELECTED IMAGES
# Local ZIP only. Existing verified images are skipped.
# ============================================================

import zipfile
from pathlib import Path
from PIL import Image
from io import BytesIO

extract_dir = Path("/content/IMO_M4_selected_images")

batch = []

for item in m4_corrected_extraction_queue:
    destination = extract_dir / item["filename"]

    if destination.exists():
        try:
            with Image.open(destination) as img:
                img.verify()
            continue
        except Exception:
            print("Invalid existing file; skipped:", destination.name)
            continue

    batch.append(item)

    if len(batch) == 200:
        break

print("Images selected:", len(batch))

verified = 0
failed = []

with zipfile.ZipFile("/content/IMO_dataset.zip", "r") as zf:

    for index, item in enumerate(batch, start=1):

        destination = extract_dir / item["filename"]

        try:
            image_bytes = zf.read(item["archive_path"])

            with Image.open(BytesIO(image_bytes)) as img:
                img.verify()

            with open(destination, "xb") as f:
                f.write(image_bytes)

            with Image.open(destination) as img:
                img.verify()

            verified += 1

        except Exception as e:
            failed.append({
                "filename": item["filename"],
                "error": str(e)
            })

        if index % 25 == 0 or index == len(batch):
            print(
                f"Processed {index}/{len(batch)} | "
                f"Verified: {verified} | Failed: {len(failed)}"
            )

print("\nBATCH COMPLETE")
print("Verified:", verified)
print("Failed:", len(failed))
print("Total selected images:", len(m4_corrected_extraction_queue))


In [ ]:

# ============================================================
# M4 STEP 77 — EXTRACT NEXT 300 IMAGES
# Local ZIP | No overwriting | PNG verification
# ============================================================

import zipfile
from pathlib import Path
from PIL import Image
from io import BytesIO

extract_dir = Path("/content/IMO_M4_selected_images")

batch = []

for item in m4_corrected_extraction_queue:
    destination = extract_dir / item["filename"]

    if destination.exists():
        try:
            with Image.open(destination) as img:
                img.verify()
            continue
        except Exception:
            continue

    batch.append(item)

    if len(batch) == 300:
        break

print("Images selected:", len(batch))

verified = 0
failed = []

with zipfile.ZipFile("/content/IMO_dataset.zip", "r") as zf:

    for index, item in enumerate(batch, start=1):
        destination = extract_dir / item["filename"]

        try:
            image_bytes = zf.read(item["archive_path"])

            with Image.open(BytesIO(image_bytes)) as img:
                img.verify()

            with open(destination, "xb") as f:
                f.write(image_bytes)

            with Image.open(destination) as img:
                img.verify()

            verified += 1

        except Exception as e:
            failed.append((item["filename"], str(e)))

        if index % 50 == 0 or index == len(batch):
            print(
                f"Processed {index}/{len(batch)} | "
                f"Verified: {verified} | Failed: {len(failed)}"
            )

print("\nBATCH COMPLETE")
print("Verified:", verified)
print("Failed:", len(failed))
print("Total target:", len(m4_corrected_extraction_queue))


In [ ]:

# ============================================================
# M4 STEP 78 — EXTRACT NEXT 400 IMAGES
# Local ZIP | No overwriting | PNG verification
# ============================================================

import zipfile
from pathlib import Path
from PIL import Image
from io import BytesIO

extract_dir = Path("/content/IMO_M4_selected_images")

batch = []

for item in m4_corrected_extraction_queue:
    destination = extract_dir / item["filename"]

    if destination.exists():
        try:
            with Image.open(destination) as img:
                img.verify()
            continue
        except Exception:
            continue

    batch.append(item)

    if len(batch) == 400:
        break

print("Images selected:", len(batch))

verified = 0
failed = []

with zipfile.ZipFile("/content/IMO_dataset.zip", "r") as zf:

    for index, item in enumerate(batch, start=1):
        destination = extract_dir / item["filename"]

        try:
            image_bytes = zf.read(item["archive_path"])

            with Image.open(BytesIO(image_bytes)) as img:
                img.verify()

            with open(destination, "xb") as f:
                f.write(image_bytes)

            with Image.open(destination) as img:
                img.verify()

            verified += 1

        except Exception as e:
            failed.append((item["filename"], str(e)))

        if index % 50 == 0 or index == len(batch):
            print(
                f"Processed {index}/{len(batch)} | "
                f"Verified: {verified} | Failed: {len(failed)}"
            )

print("\nBATCH COMPLETE")
print("Verified:", verified)
print("Failed:", len(failed))
print("Total target:", len(m4_corrected_extraction_queue))


In [ ]:

# ============================================================
# M4 STEP 79 — EXTRACT NEXT 500 IMAGES
# Local ZIP | No overwriting | PNG verification
# ============================================================

import zipfile
from pathlib import Path
from PIL import Image
from io import BytesIO

extract_dir = Path("/content/IMO_M4_selected_images")

batch = []

for item in m4_corrected_extraction_queue:
    destination = extract_dir / item["filename"]

    if destination.exists():
        try:
            with Image.open(destination) as img:
                img.verify()
            continue
        except Exception:
            continue

    batch.append(item)

    if len(batch) == 500:
        break

print("Images selected:", len(batch))

verified = 0
failed = []

with zipfile.ZipFile("/content/IMO_dataset.zip", "r") as zf:

    for index, item in enumerate(batch, start=1):
        destination = extract_dir / item["filename"]

        try:
            image_bytes = zf.read(item["archive_path"])

            with Image.open(BytesIO(image_bytes)) as img:
                img.verify()

            with open(destination, "xb") as f:
                f.write(image_bytes)

            with Image.open(destination) as img:
                img.verify()

            verified += 1

        except Exception as e:
            failed.append((item["filename"], str(e)))

        if index % 50 == 0 or index == len(batch):
            print(
                f"Processed {index}/{len(batch)} | "
                f"Verified: {verified} | Failed: {len(failed)}"
            )

print("\nBATCH COMPLETE")
print("Verified:", verified)
print("Failed:", len(failed))
print("Total target:", len(m4_corrected_extraction_queue))


In [ ]:

# ============================================================
# M4 STEP 80 — EXTRACT NEXT 500 IMAGES
# Local ZIP | No overwriting | PNG verification
# ============================================================

import zipfile
from pathlib import Path
from PIL import Image
from io import BytesIO

extract_dir = Path("/content/IMO_M4_selected_images")

batch = []

for item in m4_corrected_extraction_queue:
    destination = extract_dir / item["filename"]

    if destination.exists():
        try:
            with Image.open(destination) as img:
                img.verify()
            continue
        except Exception:
            continue

    batch.append(item)

    if len(batch) == 500:
        break

print("Images selected:", len(batch))

verified = 0
failed = []

with zipfile.ZipFile("/content/IMO_dataset.zip", "r") as zf:

    for index, item in enumerate(batch, start=1):
        destination = extract_dir / item["filename"]

        try:
            image_bytes = zf.read(item["archive_path"])

            with Image.open(BytesIO(image_bytes)) as img:
                img.verify()

            with open(destination, "xb") as f:
                f.write(image_bytes)

            with Image.open(destination) as img:
                img.verify()

            verified += 1

        except Exception as e:
            failed.append((item["filename"], str(e)))

        if index % 50 == 0 or index == len(batch):
            print(
                f"Processed {index}/{len(batch)} | "
                f"Verified: {verified} | Failed: {len(failed)}"
            )

print("\nBATCH COMPLETE")
print("Verified:", verified)
print("Failed:", len(failed))
print("Total target:", len(m4_corrected_extraction_queue))


In [ ]:

# ============================================================
# M4 STEP 81 — EXTRACT NEXT 500 IMAGES
# Local ZIP | No overwriting | PNG verification
# ============================================================

import zipfile
from pathlib import Path
from PIL import Image
from io import BytesIO

extract_dir = Path("/content/IMO_M4_selected_images")

batch = []

for item in m4_corrected_extraction_queue:
    destination = extract_dir / item["filename"]

    if destination.exists():
        try:
            with Image.open(destination) as img:
                img.verify()
            continue
        except Exception:
            continue

    batch.append(item)

    if len(batch) == 500:
        break

print("Images selected:", len(batch))

verified = 0
failed = []

with zipfile.ZipFile("/content/IMO_dataset.zip", "r") as zf:

    for index, item in enumerate(batch, start=1):
        destination = extract_dir / item["filename"]

        try:
            image_bytes = zf.read(item["archive_path"])

            with Image.open(BytesIO(image_bytes)) as img:
                img.verify()

            with open(destination, "xb") as f:
                f.write(image_bytes)

            with Image.open(destination) as img:
                img.verify()

            verified += 1

        except Exception as e:
            failed.append((item["filename"], str(e)))

        if index % 50 == 0 or index == len(batch):
            print(
                f"Processed {index}/{len(batch)} | "
                f"Verified: {verified} | Failed: {len(failed)}"
            )

print("\nBATCH COMPLETE")
print("Verified:", verified)
print("Failed:", len(failed))
print("Total target:", len(m4_corrected_extraction_queue))


In [ ]:

# ============================================================
# M4 STEP 82 — EXTRACT NEXT 500 IMAGES
# Local ZIP | No overwriting | PNG verification
# ============================================================

import zipfile
from pathlib import Path
from PIL import Image
from io import BytesIO

extract_dir = Path("/content/IMO_M4_selected_images")

batch = []

for item in m4_corrected_extraction_queue:
    destination = extract_dir / item["filename"]

    if destination.exists():
        try:
            with Image.open(destination) as img:
                img.verify()
            continue
        except Exception:
            continue

    batch.append(item)

    if len(batch) == 500:
        break

print("Images selected:", len(batch))

verified = 0
failed = []

with zipfile.ZipFile("/content/IMO_dataset.zip", "r") as zf:

    for index, item in enumerate(batch, start=1):
        destination = extract_dir / item["filename"]

        try:
            image_bytes = zf.read(item["archive_path"])

            with Image.open(BytesIO(image_bytes)) as img:
                img.verify()

            with open(destination, "xb") as f:
                f.write(image_bytes)

            with Image.open(destination) as img:
                img.verify()

            verified += 1

        except Exception as e:
            failed.append((item["filename"], str(e)))

        if index % 50 == 0 or index == len(batch):
            print(
                f"Processed {index}/{len(batch)} | "
                f"Verified: {verified} | Failed: {len(failed)}"
            )

print("\nBATCH COMPLETE")
print("Verified:", verified)
print("Failed:", len(failed))
print("Total target:", len(m4_corrected_extraction_queue))


In [ ]:

# ============================================================
# M4 STEP 83 — EXTRACT REMAINING SELECTED IMAGES
# Local ZIP | No overwriting | PNG verification
# ============================================================

import zipfile
from pathlib import Path
from PIL import Image
from io import BytesIO

extract_dir = Path("/content/IMO_M4_selected_images")

batch = []

for item in m4_corrected_extraction_queue:
    destination = extract_dir / item["filename"]

    if destination.exists():
        try:
            with Image.open(destination) as img:
                img.verify()
            continue
        except Exception:
            continue

    batch.append(item)

print("Remaining images selected:", len(batch))

verified = 0
failed = []

with zipfile.ZipFile("/content/IMO_dataset.zip", "r") as zf:

    for index, item in enumerate(batch, start=1):
        destination = extract_dir / item["filename"]

        try:
            image_bytes = zf.read(item["archive_path"])

            with Image.open(BytesIO(image_bytes)) as img:
                img.verify()

            with open(destination, "xb") as f:
                f.write(image_bytes)

            with Image.open(destination) as img:
                img.verify()

            verified += 1

        except Exception as e:
            failed.append((item["filename"], str(e)))

        if index % 25 == 0 or index == len(batch):
            print(
                f"Processed {index}/{len(batch)} | "
                f"Verified: {verified} | Failed: {len(failed)}"
            )

print("\nFINAL EXTRACTION BATCH COMPLETE")
print("Verified:", verified)
print("Failed:", len(failed))
print("Total target:", len(m4_corrected_extraction_queue))


In [ ]:

# ============================================================
# M4 STEP 84 — FINAL AUDIT + PERSISTENT CHECKPOINT
# Does not overwrite existing files or manifests.
# ============================================================

import pandas as pd
from pathlib import Path
from PIL import Image
from datetime import datetime

extract_dir = Path("/content/IMO_M4_selected_images")
results_dir = Path("/content/drive/MyDrive/Marine_Project/results")
results_dir.mkdir(parents=True, exist_ok=True)

audit_rows = []
missing = []
invalid = []

for index, item in enumerate(m4_corrected_extraction_queue, start=1):

    filename = item["filename"]
    image_path = extract_dir / filename

    if not image_path.exists():
        missing.append(filename)
        continue

    try:
        file_size = image_path.stat().st_size

        if file_size == 0:
            invalid.append(filename)
            continue

        audit_rows.append({
            "filename": filename,
            "archive_path": item["archive_path"],
            "local_path": str(image_path),
            "file_size_bytes": file_size,
            "status": "present"
        })

    except Exception as e:
        invalid.append((filename, str(e)))

    if index % 500 == 0 or index == len(m4_corrected_extraction_queue):
        print(f"Audited {index}/{len(m4_corrected_extraction_queue)}")

audit_df = pd.DataFrame(audit_rows)

print("\n" + "=" * 60)
print("M4 FINAL EXTRACTION AUDIT")
print("=" * 60)

print("Manifest target:", len(m4_corrected_extraction_queue))
print("Files present:", len(audit_df))
print("Missing files:", len(missing))
print("Invalid/empty files:", len(invalid))
print("Total extracted size (GB):",
      round(audit_df["file_size_bytes"].sum() / (1024**3), 3)
      if len(audit_df) else 0)

assert len(audit_df) == 3199
assert len(missing) == 0
assert len(invalid) == 0

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
checkpoint_path = results_dir / f"IMO_M4_extraction_checkpoint_FINAL_{timestamp}.csv"

audit_df.to_csv(checkpoint_path, index=False)

print("\nFINAL CHECKPOINT SAVED:")
print(checkpoint_path)

print("\nM4 EXTRACTION AUDIT PASSED!")


In [ ]:

# ============================================================
# M4 STEP 85 — FULL IMAGE DECODING + DIMENSION AUDIT
# Read-only. Does not modify original images.
# ============================================================

from pathlib import Path
from PIL import Image
import pandas as pd
from collections import Counter

extract_dir = Path("/content/IMO_M4_selected_images")

dimension_counts = Counter()
format_counts = Counter()
mode_counts = Counter()

corrupted = []
zero_dimension = []
total_pixels = 0

for index, item in enumerate(m4_corrected_extraction_queue, start=1):

    path = extract_dir / item["filename"]

    try:
        with Image.open(path) as img:
            img.verify()

        with Image.open(path) as img:
            width, height = img.size

            if width <= 0 or height <= 0:
                zero_dimension.append(item["filename"])
                continue

            dimension_counts[(width, height)] += 1
            format_counts[img.format] += 1
            mode_counts[img.mode] += 1
            total_pixels += width * height

    except Exception as e:
        corrupted.append({
            "filename": item["filename"],
            "error": str(e)
        })

    if index % 500 == 0 or index == len(m4_corrected_extraction_queue):
        print(f"Checked {index}/{len(m4_corrected_extraction_queue)}")

print("\n" + "=" * 60)
print("M4 IMAGE INTEGRITY REPORT")
print("=" * 60)

print("Total selected:", len(m4_corrected_extraction_queue))
print("Successfully decoded:", sum(dimension_counts.values()))
print("Corrupted:", len(corrupted))
print("Invalid dimensions:", len(zero_dimension))

print("\nImage formats:", dict(format_counts))
print("Image modes:", dict(mode_counts))

print("\nUnique dimensions:", len(dimension_counts))
print("Most common dimensions:")

for dimensions, count in dimension_counts.most_common(15):
    print(f"{dimensions[0]} x {dimensions[1]} : {count}")

print("\nSmallest image dimensions:",
      min(dimension_counts) if dimension_counts else None)

print("Largest image dimensions:",
      max(dimension_counts, key=lambda x: x[0] * x[1])
      if dimension_counts else None)

print("Total pixels (billions):", round(total_pixels / 1e9, 3))

print("\nSTEP 85 COMPLETE")


In [ ]:

# ============================================================
# M4 STEP 86 — IMAGE SIZE DISTRIBUTION ANALYSIS
# Read-only. No filtering or resizing.
# ============================================================

from pathlib import Path
from PIL import Image
import pandas as pd
import numpy as np

extract_dir = Path("/content/IMO_M4_selected_images")

dimension_rows = []

for item in m4_corrected_extraction_queue:

    path = extract_dir / item["filename"]

    with Image.open(path) as img:
        width, height = img.size

    row = {
        "filename": item["filename"],
        "width": width,
        "height": height,
        "min_side": min(width, height),
        "max_side": max(width, height),
        "pixels": width * height
    }

    for key in ["Sp_id", "Especies", "ecological_label",
                "label", "split", "Dataset_type"]:
        if key in item:
            row[key] = item[key]

    dimension_rows.append(row)

dimension_df = pd.DataFrame(dimension_rows)

print("=" * 60)
print("M4 IMAGE SIZE DISTRIBUTION")
print("=" * 60)

print("\nDimension statistics:")
print(dimension_df[["width", "height", "min_side", "max_side"]].describe(
    percentiles=[0.05, 0.10, 0.25, 0.50, 0.75, 0.90, 0.95]
).round(2))

print("\nMinimum-side threshold counts:")

for threshold in [16, 24, 32, 48, 64, 96, 128, 160, 224]:
    count = (dimension_df["min_side"] < threshold).sum()
    print(
        f"Below {threshold:3} px: {count:4} images "
        f"({count / len(dimension_df) * 100:.2f}%)"
    )

print("\nImages with minimum side <= 32:")
print(
    dimension_df.nsmallest(15, "min_side")[
        ["filename", "width", "height", "min_side"]
    ].to_string(index=False)
)

print("\nQueue metadata fields:", list(m4_corrected_extraction_queue[0].keys()))

print("\nSTEP 86 COMPLETE — NO IMAGES MODIFIED")


In [ ]:

# ============================================================
# M4 STEP 87 — PERSIST SELECTED IMAGES + DIMENSION AUDIT
# Original images remain unchanged.
# ============================================================

from pathlib import Path
from zipfile import ZipFile, ZIP_STORED
from datetime import datetime
import pandas as pd
import os

extract_dir = Path("/content/IMO_M4_selected_images")
project_dir = Path("/content/drive/MyDrive/Marine_Project")
results_dir = project_dir / "results"

results_dir.mkdir(parents=True, exist_ok=True)

stamp = datetime.now().strftime("%Y%m%d_%H%M%S")

archive_path = project_dir / f"IMO_M4_selected_images_3199_{stamp}.zip"
dimension_csv = results_dir / f"IMO_M4_image_dimensions_{stamp}.csv"

# Confirm all expected files exist before creating the archive.
expected_files = [
    extract_dir / item["filename"]
    for item in m4_corrected_extraction_queue
]

missing = [str(p) for p in expected_files if not p.is_file()]

if missing:
    raise RuntimeError(f"Missing {len(missing)} images. Archive not created.")

if len(set(expected_files)) != len(expected_files):
    raise RuntimeError("Duplicate image paths detected. Archive not created.")

# Save selected images with split folders inside the ZIP.
with ZipFile(archive_path, mode="x", compression=ZIP_STORED) as zf:
    for index, item in enumerate(m4_corrected_extraction_queue, start=1):
        source = extract_dir / item["filename"]
        archive_name = f'{item["split"]}/{item["filename"]}'
        zf.write(source, arcname=archive_name)

        if index % 500 == 0 or index == len(expected_files):
            print(f"Archived {index}/{len(expected_files)} images")

# Verify archive contents and read every stored member for CRC checking.
with ZipFile(archive_path, "r") as zf:
    members = zf.namelist()
    bad_member = zf.testzip()

if len(members) != len(expected_files):
    raise RuntimeError(
        f"Archive count mismatch: {len(members)} vs {len(expected_files)}"
    )

if bad_member is not None:
    raise RuntimeError(f"Archive integrity failure: {bad_member}")

# Save the dimension audit separately.
dimension_df.to_csv(dimension_csv, index=False)

print("\n" + "=" * 60)
print("M4 PERSISTENCE REPORT")
print("=" * 60)
print("Selected images archived:", len(members))
print("Archive integrity:", "PASSED")
print("Archive size (GB):", round(archive_path.stat().st_size / 1e9, 3))
print("Persistent image archive:", archive_path)
print("Dimension audit CSV:", dimension_csv)
print("Dimension audit rows:", len(dimension_df))
print("\nSTEP 87 COMPLETE")


In [ ]:
# M4 STEP 88 — DIMENSION DISTRIBUTION BY SPECIES AND SPLIT

from pathlib import Path
import pandas as pd

project_dir = Path("/content/drive/MyDrive/Marine_Project")
results_dir = project_dir / "results"

dimension_file = results_dir / "IMO_M4_image_dimensions_20261121_181622.csv"

manifest_files = {
    "Train": results_dir / "IMO_M4_train_manifest_20261119_184551.csv",
    "Validation": results_dir / "IMO_M4_validation_manifest_20261119_184551.csv",
    "Test": results_dir / "IMO_M4_independent_test_manifest_20261119_184551.csv"
}

dimensions = pd.read_csv(dimension_file)

print("DIMENSION AUDIT COLUMNS:")
print(dimensions.columns.tolist())

print("\nMANIFEST COLUMNS:")
for split, path in manifest_files.items():
    df = pd.read_csv(path)
    print(f"{split}: {df.columns.tolist()}")

print("\nSTEP 88A COMPLETE — SCHEMA INSPECTION ONLY")

In [ ]:
# M4 STEP 88B — IMAGE SIZE ANALYSIS BY SPECIES AND SPLIT

import pandas as pd
from pathlib import Path

project_dir = Path("/content/drive/MyDrive/Marine_Project")
results_dir = project_dir / "results"

dimensions = pd.read_csv(
    results_dir / "IMO_M4_image_dimensions_20261121_181622.csv"
)

manifest_paths = {
    "Train": results_dir / "IMO_M4_train_manifest_20261119_184551.csv",
    "Validation": results_dir / "IMO_M4_validation_manifest_20261119_184551.csv",
    "Test": results_dir / "IMO_M4_independent_test_manifest_20261119_184551.csv"
}

manifest_list = []

for split, path in manifest_paths.items():
    df = pd.read_csv(path)
    df["Locked_Split"] = split
    manifest_list.append(df)

manifest = pd.concat(manifest_list, ignore_index=True)

# Match image dimensions to the locked manifests
analysis_df = manifest.merge(
    dimensions,
    left_on="Filename",
    right_on="filename",
    how="left",
    validate="one_to_one"
)

print("=" * 65)
print("M4 IMAGE SIZE ANALYSIS")
print("=" * 65)

print("\nMatching audit:")
print("Manifest images:", len(manifest))
print("Matched dimensions:", analysis_df["width"].notna().sum())
print("Missing dimensions:", analysis_df["width"].isna().sum())

print("\n1. IMAGE SIZE BY SPECIES")
species_summary = analysis_df.groupby("Especies").agg(
    Images=("Filename", "count"),
    Median_Min_Side=("min_side", "median"),
    Minimum=("min_side", "min"),
    Maximum=("min_side", "max"),
    Below_64=("min_side", lambda x: (x < 64).sum()),
    Below_128=("min_side", lambda x: (x < 128).sum())
).round(1)

print(species_summary.to_string())

print("\n2. IMAGE SIZE BY SPLIT")
split_summary = analysis_df.groupby("Locked_Split").agg(
    Images=("Filename", "count"),
    Median_Min_Side=("min_side", "median"),
    Below_64=("min_side", lambda x: (x < 64).sum()),
    Below_128=("min_side", lambda x: (x < 128).sum())
).round(1)

print(split_summary.to_string())

print("\n3. IMAGE SIZE BY ECOLOGICAL CLASS")
class_summary = analysis_df.groupby("Ecological_Status").agg(
    Images=("Filename", "count"),
    Median_Min_Side=("min_side", "median"),
    Below_64=("min_side", lambda x: (x < 64).sum()),
    Below_128=("min_side", lambda x: (x < 128).sum())
).round(1)

print(class_summary.to_string())

print("\nSTEP 88B COMPLETE — NO IMAGES MODIFIED")

In [ ]:
# M4 STEP 89 — FILTERING IMPACT ANALYSIS
# Read-only. No images will be removed.

thresholds = [32, 48, 64]

print("=" * 65)
print("IMAGE EXCLUSION COUNTS BY THRESHOLD")
print("=" * 65)

for threshold in thresholds:
    excluded = analysis_df[analysis_df["min_side"] < threshold]

    print(f"\nTHRESHOLD: {threshold} pixels")
    print("Excluded:", len(excluded))
    print("Percentage:", round(len(excluded) / len(analysis_df) * 100, 2), "%")

    print("\nBy ecological class:")
    print(excluded["Ecological_Status"].value_counts().to_string())

    print("\nBy species:")
    print(excluded["Especies"].value_counts().to_string())

    print("\nBy split:")
    print(excluded["Locked_Split"].value_counts().to_string())

print("\nSTEP 89 COMPLETE — NO IMAGES MODIFIED")

In [ ]:
# M4 STEP 90 — FINAL CLASS DISTRIBUTION AUDIT
# Read-only. No files or images modified.

import pandas as pd
from pathlib import Path

project_dir = Path("/content/drive/MyDrive/Marine_Project/results")

manifest_paths = {
    "Train": project_dir / "IMO_M4_train_manifest_20261119_184551.csv",
    "Validation": project_dir / "IMO_M4_validation_manifest_20261119_184551.csv",
    "Test": project_dir / "IMO_M4_independent_test_manifest_20261119_184551.csv"
}

for split, path in manifest_paths.items():

    df = pd.read_csv(path)

    print("\n" + "=" * 60)
    print(f"{split.upper()} — {len(df)} IMAGES")
    print("=" * 60)

    print("\nEcological class distribution:")
    print(
        df["Ecological_Status"]
        .value_counts()
        .to_string()
    )

    print("\nSpecies distribution:")
    print(
        df["Especies"]
        .value_counts()
        .to_string()
    )

    print("\nSpecies percentages:")
    print(
        (df["Especies"].value_counts(normalize=True) * 100)
        .round(2)
        .astype(str)
        .add("%")
        .to_string()
    )

print("\nSTEP 90 COMPLETE — SPLITS UNCHANGED")

In [ ]:
# M4 STEP 91B — INSPECT PERSISTENT ZIP ARCHIVE PATHS
# Read-only: no extraction or modification.

from zipfile import ZipFile
from pathlib import Path

project_dir = Path("/content/drive/MyDrive/Marine_Project")

archive_path = (
    project_dir /
    "IMO_M4_selected_images_3199_20261121_181622.zip"
)

with ZipFile(archive_path, "r") as archive:

    members = [
        name for name in archive.namelist()
        if name.lower().endswith(".png")
    ]

    print("PNG members in archive:", len(members))
    print("\nFirst 15 archive paths:")

    for name in members[:15]:
        print(name)

    print("\nChecking the previously missing filename:")

    target = "BA15m-230518-0-0-20240408161316class13.png"

    matches = [
        name for name in members
        if Path(name).name == target
    ]

    print("Filename:", target)
    print("Matching archive entries:", len(matches))

    for match in matches:
        print("Exact path:", match)

print("\nSTEP 91B COMPLETE — ARCHIVE UNCHANGED")

In [ ]:
# M4 STEP 91C — VISUAL INSPECTION WITH CORRECT ARCHIVE PATHS

from pathlib import Path
from zipfile import ZipFile
from PIL import Image
import pandas as pd
import matplotlib.pyplot as plt
import io
import numpy as np

project_dir = Path("/content/drive/MyDrive/Marine_Project")

archive_path = (
    project_dir /
    "IMO_M4_selected_images_3199_20261121_181622.zip"
)

results_dir = project_dir / "results"

manifest_paths = [
    results_dir / "IMO_M4_train_manifest_20261119_184551.csv",
    results_dir / "IMO_M4_validation_manifest_20261119_184551.csv",
    results_dir / "IMO_M4_independent_test_manifest_20261119_184551.csv"
]

manifest = pd.concat(
    [pd.read_csv(path) for path in manifest_paths],
    ignore_index=True
)

dimensions = pd.read_csv(
    results_dir / "IMO_M4_image_dimensions_20261121_181622.csv"
)

data = manifest.merge(
    dimensions,
    left_on="Filename",
    right_on="filename",
    validate="one_to_one"
)

species_list = sorted(data["Especies"].unique())

with ZipFile(archive_path, "r") as archive:

    png_members = [
        name for name in archive.namelist()
        if name.lower().endswith(".png")
    ]

    # Map each image filename to its exact ZIP path
    archive_lookup = {}

    for member in png_members:
        basename = Path(member).name

        if basename in archive_lookup:
            raise ValueError(f"Duplicate filename in ZIP: {basename}")

        archive_lookup[basename] = member

    missing = [
        name for name in data["Filename"]
        if name not in archive_lookup
    ]

    if missing:
        raise ValueError(f"Missing archive images: {len(missing)}")

    fig, axes = plt.subplots(
        len(species_list), 2,
        figsize=(10, 4 * len(species_list))
    )

    for row, species in enumerate(species_list):

        species_data = (
            data[data["Especies"] == species]
            .sort_values("min_side")
            .reset_index(drop=True)
        )

        smallest = species_data.iloc[0]

        median_position = int(np.argmin(
            np.abs(
                species_data["min_side"].to_numpy()
                - species_data["min_side"].median()
            )
        ))

        median_image = species_data.iloc[median_position]

        for col, sample in enumerate([smallest, median_image]):

            exact_path = archive_lookup[sample["Filename"]]
            image_bytes = archive.read(exact_path)

            with Image.open(io.BytesIO(image_bytes)) as img:
                axes[row, col].imshow(img.copy())

            axes[row, col].set_title(
                f"{species}\n"
                f"{int(sample['width'])} × {int(sample['height'])} px"
            )

            axes[row, col].axis("off")

plt.tight_layout()
plt.show()

print("Images inspected:", len(species_list) * 2)
print("Archive images available:", len(archive_lookup))
print("Missing images:", len(missing))
print("STEP 91C COMPLETE — ORIGINAL ARCHIVE UNCHANGED")

In [ ]:
# M4 STEP 92 — BLACK BACKGROUND ANALYSIS
# Read-only analysis using the persistent image archive.

from zipfile import ZipFile
from pathlib import Path
from PIL import Image
import pandas as pd
import numpy as np
import io

project_dir = Path("/content/drive/MyDrive/Marine_Project")

archive_path = (
    project_dir /
    "IMO_M4_selected_images_3199_20261121_181622.zip"
)

results_dir = project_dir / "results"

manifest_paths = [
    results_dir / "IMO_M4_train_manifest_20261119_184551.csv",
    results_dir / "IMO_M4_validation_manifest_20261119_184551.csv",
    results_dir / "IMO_M4_independent_test_manifest_20261119_184551.csv"
]

manifest = pd.concat(
    [pd.read_csv(path) for path in manifest_paths],
    ignore_index=True
)

records = []

with ZipFile(archive_path, "r") as archive:

    lookup = {
        Path(name).name: name
        for name in archive.namelist()
        if name.lower().endswith(".png")
    }

    for index, row in manifest.iterrows():

        image_bytes = archive.read(lookup[row["Filename"]])

        with Image.open(io.BytesIO(image_bytes)) as img:
            image = np.asarray(img.convert("RGB"))

        # Percentage of pixels that are nearly black
        black_pixels = np.all(image <= 10, axis=2)

        black_percentage = (
            black_pixels.sum() / black_pixels.size
        ) * 100

        records.append({
            "Filename": row["Filename"],
            "Species": row["Especies"],
            "Ecological_Status": row["Ecological_Status"],
            "Split": (
                "Train" if row["Filename"] in set(
                    pd.read_csv(manifest_paths[0])["Filename"]
                )
                else "Validation" if row["Filename"] in set(
                    pd.read_csv(manifest_paths[1])["Filename"]
                )
                else "Test"
            ),
            "Black_Percentage": black_percentage
        })

background_df = pd.DataFrame(records)

print("=" * 60)
print("BLACK BACKGROUND ANALYSIS")
print("=" * 60)

print("\nOverall statistics:")
print(background_df["Black_Percentage"].describe().round(2))

print("\nBy species:")
print(
    background_df.groupby("Species")["Black_Percentage"]
    .agg(["count", "median", "mean"])
    .round(2)
    .to_string()
)

print("\nBy ecological class:")
print(
    background_df.groupby("Ecological_Status")["Black_Percentage"]
    .agg(["count", "median", "mean"])
    .round(2)
    .to_string()
)

print("\nSTEP 92 COMPLETE — NO IMAGES MODIFIED")

In [ ]:
# M4 STEP 93 — SAVE BLACK BACKGROUND AUDIT
# Creates new files; does not overwrite existing files.

from pathlib import Path
from datetime import datetime

results_dir = Path(
    "/content/drive/MyDrive/Marine_Project/results"
)

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

detail_path = results_dir / (
    f"IMO_M4_black_background_detail_{timestamp}.csv"
)

summary_path = results_dir / (
    f"IMO_M4_black_background_summary_{timestamp}.csv"
)

summary = (
    background_df.groupby(
        ["Ecological_Status", "Species", "Split"]
    )["Black_Percentage"]
    .agg(["count", "median", "mean", "std"])
    .reset_index()
)

# Exclusive creation prevents accidental overwriting.
with open(detail_path, "x", encoding="utf-8") as f:
    background_df.to_csv(f, index=False)

with open(summary_path, "x", encoding="utf-8") as f:
    summary.to_csv(f, index=False)

print("BACKGROUND AUDIT SAVED")
print("Detailed records:", len(background_df))
print("Summary records:", len(summary))
print("Detail file:", detail_path)
print("Summary file:", summary_path)
print("\nSTEP 93 COMPLETE")

In [ ]:
# M4 STEP 94 — TRAINING ENVIRONMENT CHECK

import torch
import torchvision
import platform

print("=" * 55)
print("M4 TRAINING ENVIRONMENT")
print("=" * 55)

print("Python:", platform.python_version())
print("PyTorch:", torch.__version__)
print("Torchvision:", torchvision.__version__)

print("\nCUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("GPU memory (GB):",
          round(torch.cuda.get_device_properties(0).total_memory / 1e9, 2))
else:
    print("GPU not detected. CPU training would be slower.")

print("\nImage input target: 224 × 224")
print("Image channels: RGB")
print("Classes: Native (0), Invasive (1)")
print("\nSTEP 94 COMPLETE — ENVIRONMENT CHECK ONLY")

In [ ]:
# M4 STEP 95 — PRETRAINED WEIGHT AND ARCHIVE CHECK

from pathlib import Path
from zipfile import ZipFile
import torch

project_dir = Path("/content/drive/MyDrive/Marine_Project")

archive_path = (
    project_dir /
    "IMO_M4_selected_images_3199_20261121_181622.zip"
)

checkpoint_dir = Path(torch.hub.get_dir()) / "checkpoints"

cached_weights = list(
    checkpoint_dir.glob("resnet18-f37072fd.pth")
)

print("=" * 55)
print("M4 MODEL PREPARATION CHECK")
print("=" * 55)

print("Archive exists:", archive_path.exists())

if archive_path.exists():
    print("Archive size (GB):",
          round(archive_path.stat().st_size / 1e9, 3))

    with ZipFile(archive_path, "r") as archive:
        members = archive.namelist()

        for split in ["train", "validation", "independent_test"]:
            count = sum(
                name.startswith(split + "/")
                and name.lower().endswith(".png")
                for name in members
            )
            print(f"{split}: {count} images")

print("\nPretrained ResNet18 weights cached:", bool(cached_weights))

if cached_weights:
    print("Cached file:", cached_weights[0])
else:
    print("No cached pretrained weights found.")

print("\nSTEP 95 COMPLETE — NO DOWNLOADS OR MODEL TRAINING")

In [ ]:
# M4 STEP 96 — DOWNLOAD PRETRAINED RESNET18 WEIGHTS
# Saves a verified copy in Google Drive without overwriting files.

from pathlib import Path
from datetime import datetime
import requests
import hashlib
from tqdm.auto import tqdm

models_dir = Path(
    "/content/drive/MyDrive/Marine_Project/models"
)
models_dir.mkdir(parents=True, exist_ok=True)

weight_path = models_dir / "resnet18_imagenet1k_v1.pth"

url = "https://download.pytorch.org/models/resnet18-f37072fd.pth"
expected_hash_prefix = "f37072fd"

def calculate_sha256(path):
    digest = hashlib.sha256()
    with open(path, "rb") as file:
        for chunk in iter(lambda: file.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

if weight_path.exists():
    existing_hash = calculate_sha256(weight_path)

    if existing_hash.startswith(expected_hash_prefix):
        print("Verified pretrained weights already exist.")
        print("File:", weight_path)
    else:
        raise RuntimeError(
            "Existing weight file failed checksum verification. "
            "It has not been modified."
        )

else:
    temporary_path = models_dir / (
        f"resnet18_download_{datetime.now().strftime('%Y%m%d_%H%M%S')}.part"
    )

    print("Downloading pretrained ResNet18 weights...")

    with requests.get(
        url, stream=True, timeout=(30, 180)
    ) as response:

        response.raise_for_status()
        total_size = int(response.headers.get("content-length", 0))

        with open(temporary_path, "xb") as file:
            with tqdm(
                total=total_size,
                unit="B",
                unit_scale=True,
                desc="ResNet18 weights"
            ) as progress:

                for chunk in response.iter_content(chunk_size=1024 * 1024):
                    if chunk:
                        file.write(chunk)
                        progress.update(len(chunk))

    downloaded_hash = calculate_sha256(temporary_path)

    if not downloaded_hash.startswith(expected_hash_prefix):
        raise RuntimeError(
            "Checksum mismatch. Temporary file preserved for inspection."
        )

    if weight_path.exists():
        raise FileExistsError(
            "Destination appeared during download; original file preserved."
        )

    temporary_path.rename(weight_path)

    print("\nDOWNLOAD AND CHECKSUM VERIFICATION PASSED!")

print("Weight file:", weight_path)
print("Size (MB):", round(weight_path.stat().st_size / 1e6, 2))
print("SHA256:", calculate_sha256(weight_path))
print("\nSTEP 96 COMPLETE")

In [ ]:
# M4 STEP 97 — RESNET18 CHECKPOINT COMPATIBILITY TEST

import torch
from torchvision.models import resnet18
from pathlib import Path

weight_path = Path(
    "/content/drive/MyDrive/Marine_Project/models/"
    "resnet18_imagenet1k_v1.pth"
)

# Load verified checkpoint
state_dict = torch.load(
    weight_path,
    map_location="cpu",
    weights_only=True
)

# Create standard ResNet18 architecture
model_check = resnet18(weights=None)

# Verify checkpoint compatibility
result = model_check.load_state_dict(
    state_dict,
    strict=True
)

print("=" * 55)
print("RESNET18 CHECKPOINT VERIFICATION")
print("=" * 55)

print("Architecture: ResNet18")
print("Checkpoint loaded:", True)
print("Strict loading:", True)
print("Missing keys:", len(result.missing_keys))
print("Unexpected keys:", len(result.unexpected_keys))
print("Parameter count:", sum(
    parameter.numel() for parameter in model_check.parameters()
))

print("\nSTEP 97 COMPLETE — NO TRAINING PERFORMED")

In [ ]:
# M4 STEP 98 — ZIP-BACKED PYTORCH DATASET

import io
import pandas as pd
import torch
from torch.utils.data import Dataset
from torchvision import transforms
from PIL import Image
from zipfile import ZipFile
from pathlib import Path

project_dir = Path("/content/drive/MyDrive/Marine_Project")

archive_path = (
    project_dir /
    "IMO_M4_selected_images_3199_20261121_181622.zip"
)

results_dir = project_dir / "results"

train_df = pd.read_csv(
    results_dir / "IMO_M4_train_manifest_20261119_184551.csv"
)

validation_df = pd.read_csv(
    results_dir / "IMO_M4_validation_manifest_20261119_184551.csv"
)

test_df = pd.read_csv(
    results_dir / "IMO_M4_independent_test_manifest_20261119_184551.csv"
)

# Explicit binary label mapping
label_map = {
    "Native": 0,
    "Invasive": 1
}

# ImageNet normalization
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

evaluation_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

class IMOBinaryDataset(Dataset):

    def __init__(self, dataframe, archive_path, transform=None):
        self.dataframe = dataframe.reset_index(drop=True)
        self.archive_path = str(archive_path)
        self.transform = transform
        self.archive = None
        self.archive_lookup = None

    def __len__(self):
        return len(self.dataframe)

    def _open_archive(self):
        if self.archive is None:
            self.archive = ZipFile(self.archive_path, "r")

            self.archive_lookup = {
                Path(name).name: name
                for name in self.archive.namelist()
                if name.lower().endswith(".png")
            }

    def __getitem__(self, index):
        self._open_archive()

        row = self.dataframe.iloc[index]

        filename = row["Filename"]
        archive_name = self.archive_lookup[filename]

        image_bytes = self.archive.read(archive_name)

        with Image.open(io.BytesIO(image_bytes)) as image:
            image = image.convert("RGB")

        if self.transform:
            image = self.transform(image)

        label = label_map[row["Ecological_Status"]]

        return image, torch.tensor(label, dtype=torch.long)

    def __getstate__(self):
        state = self.__dict__.copy()
        state["archive"] = None
        state["archive_lookup"] = None
        return state


# Create datasets
train_dataset = IMOBinaryDataset(
    train_df, archive_path, train_transform
)

validation_dataset = IMOBinaryDataset(
    validation_df, archive_path, evaluation_transform
)

test_dataset = IMOBinaryDataset(
    test_df, archive_path, evaluation_transform
)

print("ZIP-BACKED DATASETS CREATED")
print("Training:", len(train_dataset))
print("Validation:", len(validation_dataset))
print("Independent test:", len(test_dataset))
print("Input size: 224 × 224 × 3")
print("Native label: 0")
print("Invasive label: 1")
print("\nSTEP 98 COMPLETE — DATASETS DEFINED, NOT TRAINED")

In [ ]:
# M4 STEP 99 — DATASET SAMPLE VALIDATION

import torch

def inspect_dataset(name, dataset):
    print(f"\n{name}")

    for index in [0, 1]:
        image, label = dataset[index]

        print(
            f"Sample {index}: "
            f"Shape={tuple(image.shape)}, "
            f"Label={label.item()}, "
            f"Data type={image.dtype}, "
            f"Finite={torch.isfinite(image).all().item()}"
        )

        assert image.shape == (3, 224, 224)
        assert label.item() in [0, 1]
        assert torch.isfinite(image).all()

inspect_dataset("TRAINING DATASET", train_dataset)
inspect_dataset("VALIDATION DATASET", validation_dataset)
inspect_dataset("INDEPENDENT TEST DATASET", test_dataset)

print("\nALL SIX SAMPLE CHECKS PASSED")
print("STEP 99 COMPLETE — NO TRAINING PERFORMED")

In [ ]:
# M4 STEP 100 — CREATE DATALOADERS

from torch.utils.data import DataLoader
import torch

BATCH_SIZE = 32
NUM_WORKERS = 2

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=True,
    persistent_workers=True
)

validation_loader = DataLoader(
    validation_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True,
    persistent_workers=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True,
    persistent_workers=True
)

print("DATALOADERS CREATED")
print("Training batches:", len(train_loader))
print("Validation batches:", len(validation_loader))
print("Independent test batches:", len(test_loader))
print("Batch size:", BATCH_SIZE)
print("Workers:", NUM_WORKERS)
print("GPU available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

print("\nSTEP 100 COMPLETE — NO TRAINING PERFORMED")

In [ ]:
# M4 STEP 101 — MODEL AND LOSS CONFIGURATION

import torch
import torch.nn as nn
from torchvision.models import resnet18

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Load verified ImageNet checkpoint
state_dict = torch.load(
    "/content/drive/MyDrive/Marine_Project/models/resnet18_imagenet1k_v1.pth",
    map_location="cpu",
    weights_only=True
)

# Create ResNet18 and load pretrained weights
model = resnet18(weights=None)
model.load_state_dict(state_dict, strict=True)

# Replace final layer for binary classification
model.fc = nn.Linear(model.fc.in_features, 2)

# Move model to GPU
model = model.to(device)

# Calculate class weights from training manifest only
class_counts = train_df["Ecological_Status"].value_counts()

native_count = int(class_counts["Native"])
invasive_count = int(class_counts["Invasive"])
total_count = native_count + invasive_count

class_weights = torch.tensor(
    [
        total_count / (2 * native_count),
        total_count / (2 * invasive_count)
    ],
    dtype=torch.float32,
    device=device
)

criterion = nn.CrossEntropyLoss(weight=class_weights)

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.0001,
    weight_decay=0.0001
)

print("=" * 50)
print("M4 MODEL CONFIGURATION")
print("=" * 50)
print("Architecture: ResNet18")
print("Output classes: 2")
print("Class mapping: Native=0, Invasive=1")
print("Device:", device)
print("Training class counts:", native_count, invasive_count)
print("Class weights:", class_weights.tolist())
print("Loss: Weighted CrossEntropyLoss")
print("Optimizer: Adam")
print("Learning rate: 0.0001")
print("Weight decay: 0.0001")
print("Model parameters:", sum(p.numel() for p in model.parameters()))

print("\nSTEP 101 COMPLETE — MODEL CONFIGURED, NOT TRAINED")

In [ ]:
# M4 STEP 102A — ZIP ACCESS DIAGNOSTIC

from zipfile import ZipFile
from pathlib import Path

# Close any previously opened dataset archive handles
for dataset in [train_dataset, validation_dataset, test_dataset]:
    if dataset.archive is not None:
        dataset.archive.close()

    dataset.archive = None
    dataset.archive_lookup = None

print("Previous archive handles reset.")

# Open a fresh archive in the main process
archive_check = ZipFile(archive_path, "r")

print("ZIP opened successfully.")
print("Number of archive entries:", len(archive_check.namelist()))

# Test reading the first training image
sample_filename = train_df.iloc[0]["Filename"]

lookup = {
    Path(name).name: name
    for name in archive_check.namelist()
    if name.lower().endswith(".png")
}

member_name = lookup[sample_filename]
image_bytes = archive_check.read(member_name)

print("Sample filename:", sample_filename)
print("Image bytes:", len(image_bytes))
print("Sample image read successfully.")

archive_check.close()

print("\nSTEP 102A COMPLETE — DIAGNOSTIC FINISHED")

In [ ]:
# M4 STEP 102B — SAFE SINGLE-PROCESS DATALOADERS

from torch.utils.data import DataLoader

# Reset archive handles
for dataset in [train_dataset, validation_dataset, test_dataset]:
    if dataset.archive is not None:
        dataset.archive.close()

    dataset.archive = None
    dataset.archive_lookup = None

# Recreate loaders without multiprocessing
train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True,
    num_workers=0,
    pin_memory=True
)

validation_loader = DataLoader(
    validation_dataset,
    batch_size=32,
    shuffle=False,
    num_workers=0,
    pin_memory=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=32,
    shuffle=False,
    num_workers=0,
    pin_memory=True
)

print("SAFE DATALOADERS CREATED")
print("Training batches:", len(train_loader))
print("Validation batches:", len(validation_loader))
print("Test batches:", len(test_loader))
print("Workers:", 0)

print("\nSTEP 102B COMPLETE — NO TRAINING PERFORMED")

In [ ]:
# M4 STEP 102C — RETRY TRAINING SAFETY CHECK

import torch

# Verify GPU and model configuration
assert next(model.parameters()).device.type == "cuda"
assert model.fc.out_features == 2

# Load one batch using the safe DataLoader
images, labels = next(iter(train_loader))

images = images.to(device, non_blocking=True)
labels = labels.to(device, non_blocking=True)

# Forward-pass verification only
model.eval()

with torch.no_grad():
    outputs = model(images)

print("=" * 50)
print("M4 TRAINING SAFETY CHECK")
print("=" * 50)
print("GPU:", torch.cuda.get_device_name(0))
print("Input shape:", tuple(images.shape))
print("Label shape:", tuple(labels.shape))
print("Output shape:", tuple(outputs.shape))
print("Labels present:", torch.unique(labels).tolist())
print("Class weights:", class_weights.tolist())
print("Test set used for training:", False)

assert outputs.shape == (images.shape[0], 2)
assert torch.isfinite(outputs).all()

print("\nALL SAFETY CHECKS PASSED")
print("STEP 102C COMPLETE — NO TRAINING PERFORMED")

In [ ]:
# M4 STEP 103 — RESNET18 TRAINING LOOP

import os
import copy
import json
import time
import pandas as pd
import torch
import torch.nn as nn

from pathlib import Path

# Unique experiment folder
timestamp = time.strftime("%Y%m%d_%H%M%S")

experiment_dir = (
    Path("/content/drive/MyDrive/Marine_Project/models")
    / f"M4_ResNet18_experiment_{timestamp}"
)

experiment_dir.mkdir(parents=True, exist_ok=False)

best_checkpoint_path = experiment_dir / "best_model.pth"
latest_checkpoint_path = experiment_dir / "latest_model.pth"
history_path = experiment_dir / "training_history.csv"

EPOCHS = 10

# Reset model to training mode
model.train()

best_val_loss = float("inf")
history = []

print("Starting M4 training")
print("Epochs:", EPOCHS)
print("Experiment folder:", experiment_dir)

for epoch in range(EPOCHS):

    epoch_start = time.time()

    # -------------------------
    # TRAINING
    # -------------------------
    model.train()

    train_loss_total = 0.0
    train_correct = 0
    train_total = 0

    for images, labels in train_loader:

        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad(set_to_none=True)

        outputs = model(images)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        batch_size = labels.size(0)

        train_loss_total += loss.item() * batch_size
        predictions = outputs.argmax(dim=1)

        train_correct += (predictions == labels).sum().item()
        train_total += batch_size

    train_loss = train_loss_total / train_total
    train_accuracy = train_correct / train_total

    # -------------------------
    # VALIDATION
    # -------------------------
    model.eval()

    val_loss_total = 0.0
    val_correct = 0
    val_total = 0

    with torch.no_grad():

        for images, labels in validation_loader:

            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)
            loss = criterion(outputs, labels)

            batch_size = labels.size(0)

            val_loss_total += loss.item() * batch_size

            predictions = outputs.argmax(dim=1)

            val_correct += (predictions == labels).sum().item()
            val_total += batch_size

    val_loss = val_loss_total / val_total
    val_accuracy = val_correct / val_total

    epoch_time = time.time() - epoch_start

    record = {
        "epoch": epoch + 1,
        "train_loss": train_loss,
        "train_accuracy": train_accuracy,
        "validation_loss": val_loss,
        "validation_accuracy": val_accuracy,
        "epoch_seconds": epoch_time
    }

    history.append(record)

    # Save latest checkpoint after every epoch
    checkpoint = {
        "epoch": epoch + 1,
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "validation_loss": val_loss,
        "validation_accuracy": val_accuracy,
        "class_mapping": {"Native": 0, "Invasive": 1}
    }

    torch.save(checkpoint, latest_checkpoint_path)

    # Save best checkpoint
    if val_loss < best_val_loss:

        best_val_loss = val_loss

        torch.save(checkpoint, best_checkpoint_path)

        print("Best checkpoint updated.")

    # Persist metrics after every epoch
    pd.DataFrame(history).to_csv(history_path, index=False)

    print(
        f"Epoch {epoch + 1:02d}/{EPOCHS} | "
        f"Train Loss: {train_loss:.4f} | "
        f"Train Acc: {train_accuracy:.4f} | "
        f"Val Loss: {val_loss:.4f} | "
        f"Val Acc: {val_accuracy:.4f} | "
        f"Time: {epoch_time:.1f}s"
    )

print("\nM4 TRAINING RUN COMPLETE")
print("Best validation loss:", best_val_loss)
print("Best checkpoint:", best_checkpoint_path)
print("Latest checkpoint:", latest_checkpoint_path)
print("History:", history_path)

In [ ]:
# M4 STEP 104 — BEST CHECKPOINT VALIDATION EVALUATION

import torch
import pandas as pd
import json
import time

from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    accuracy_score,
    balanced_accuracy_score,
    f1_score
)

# Load selected best checkpoint
checkpoint = torch.load(
    best_checkpoint_path,
    map_location=device,
    weights_only=True
)

model.load_state_dict(checkpoint["model_state_dict"])
model.eval()

print("Selected checkpoint epoch:", checkpoint["epoch"])
print("Selected validation loss:", checkpoint["validation_loss"])

true_labels = []
predicted_labels = []
prediction_probabilities = []

# Validation evaluation only
with torch.no_grad():

    for images, labels in validation_loader:

        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images)
        probabilities = torch.softmax(outputs, dim=1)

        predictions = outputs.argmax(dim=1)

        true_labels.extend(labels.cpu().tolist())
        predicted_labels.extend(predictions.cpu().tolist())
        prediction_probabilities.extend(
            probabilities[:, 1].cpu().tolist()
        )

# Calculate metrics
accuracy = accuracy_score(true_labels, predicted_labels)
balanced_acc = balanced_accuracy_score(
    true_labels, predicted_labels
)
macro_f1 = f1_score(
    true_labels, predicted_labels, average="macro"
)

cm = confusion_matrix(
    true_labels, predicted_labels, labels=[0, 1]
)

report = classification_report(
    true_labels,
    predicted_labels,
    labels=[0, 1],
    target_names=["Native", "Invasive"],
    output_dict=True,
    zero_division=0
)

print("\nCONFUSION MATRIX")
print("Rows = Actual, Columns = Predicted")
print(cm)

print("\nCLASSIFICATION REPORT")
print(
    classification_report(
        true_labels,
        predicted_labels,
        labels=[0, 1],
        target_names=["Native", "Invasive"],
        zero_division=0
    )
)

print("Accuracy:", round(accuracy, 4))
print("Balanced accuracy:", round(balanced_acc, 4))
print("Macro F1:", round(macro_f1, 4))

# Save results without overwriting previous artifacts
evaluation_time = time.strftime("%Y%m%d_%H%M%S")

evaluation_dir = experiment_dir / f"validation_evaluation_{evaluation_time}"
evaluation_dir.mkdir(parents=True, exist_ok=False)

predictions_df = pd.DataFrame({
    "actual_label": true_labels,
    "predicted_label": predicted_labels,
    "invasive_probability": prediction_probabilities
})

predictions_df.to_csv(
    evaluation_dir / "validation_predictions.csv",
    index=False
)

with open(
    evaluation_dir / "validation_metrics.json",
    "w"
) as file:
    json.dump({
        "selected_epoch": int(checkpoint["epoch"]),
        "validation_loss": float(checkpoint["validation_loss"]),
        "accuracy": float(accuracy),
        "balanced_accuracy": float(balanced_acc),
        "macro_f1": float(macro_f1),
        "confusion_matrix": cm.tolist(),
        "classification_report": report
    }, file, indent=4)

print("\nValidation artifacts saved to:")
print(evaluation_dir)

print("\nSTEP 104 COMPLETE — INDEPENDENT TEST NOT USED")

In [ ]:
# M4 STEP 105 — LOCK SELECTED CHECKPOINT

import shutil
import hashlib
import json
import time
from pathlib import Path

# Original selected checkpoint
source_checkpoint = Path(best_checkpoint_path)

# Create a unique locked checkpoint name
lock_timestamp = time.strftime("%Y%m%d_%H%M%S")

locked_checkpoint = (
    experiment_dir /
    f"LOCKED_best_checkpoint_epoch2_{lock_timestamp}.pth"
)

# Preserve original checkpoint
assert source_checkpoint.exists()
assert not locked_checkpoint.exists()

shutil.copy2(
    source_checkpoint,
    locked_checkpoint
)

# Calculate SHA-256 checksum
def calculate_sha256(path):
    sha256 = hashlib.sha256()

    with open(path, "rb") as file:
        for chunk in iter(lambda: file.read(1024 * 1024), b""):
            sha256.update(chunk)

    return sha256.hexdigest()

checkpoint_hash = calculate_sha256(locked_checkpoint)

# Verify copied checkpoint
assert calculate_sha256(source_checkpoint) == checkpoint_hash

lock_record = {
    "selected_epoch": 2,
    "selection_metric": "Lowest validation loss",
    "validation_loss": float(checkpoint["validation_loss"]),
    "validation_accuracy": float(accuracy),
    "validation_macro_f1": float(macro_f1),
    "locked_checkpoint": str(locked_checkpoint),
    "sha256": checkpoint_hash,
    "independent_test_used_for_selection": False
}

lock_record_path = (
    experiment_dir /
    f"checkpoint_lock_record_{lock_timestamp}.json"
)

with open(lock_record_path, "x") as file:
    json.dump(lock_record, file, indent=4)

print("=" * 50)
print("CHECKPOINT LOCK VERIFICATION")
print("=" * 50)
print("Selected epoch:", lock_record["selected_epoch"])
print("Locked checkpoint:", locked_checkpoint)
print("SHA-256:", checkpoint_hash)
print("Copy verification: PASSED")
print("Original checkpoint preserved: YES")
print("Test set used for selection: NO")
print("Lock record:", lock_record_path)

print("\nSTEP 105 COMPLETE — CHECKPOINT LOCKED")

In [ ]:
# M4 STEP 106 — FINAL INDEPENDENT TEST EVALUATION

import torch
import pandas as pd
import json
import time

from pathlib import Path
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    precision_score,
    recall_score
)

# Load the locked checkpoint
locked_data = torch.load(
    locked_checkpoint,
    map_location=device,
    weights_only=True
)

model.load_state_dict(
    locked_data["model_state_dict"],
    strict=True
)

model.eval()

true_test = []
predicted_test = []
invasive_probabilities = []

print("Locked checkpoint loaded.")
print("Selected epoch:", locked_data["epoch"])
print("Independent test images:", len(test_dataset))

# Independent test evaluation
with torch.no_grad():

    for images, labels in test_loader:

        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images)
        probabilities = torch.softmax(outputs, dim=1)

        predictions = outputs.argmax(dim=1)

        true_test.extend(labels.cpu().tolist())
        predicted_test.extend(predictions.cpu().tolist())
        invasive_probabilities.extend(
            probabilities[:, 1].cpu().tolist()
        )

# Calculate metrics
test_accuracy = accuracy_score(true_test, predicted_test)

test_balanced_accuracy = balanced_accuracy_score(
    true_test, predicted_test
)

test_macro_f1 = f1_score(
    true_test, predicted_test, average="macro"
)

test_macro_precision = precision_score(
    true_test, predicted_test, average="macro",
    zero_division=0
)

test_macro_recall = recall_score(
    true_test, predicted_test, average="macro",
    zero_division=0
)

test_cm = confusion_matrix(
    true_test, predicted_test, labels=[0, 1]
)

test_report = classification_report(
    true_test,
    predicted_test,
    labels=[0, 1],
    target_names=["Native", "Invasive"],
    output_dict=True,
    zero_division=0
)

print("\nINDEPENDENT TEST CONFUSION MATRIX")
print("Rows = Actual, Columns = Predicted")
print(test_cm)

print("\nINDEPENDENT TEST CLASSIFICATION REPORT")
print(
    classification_report(
        true_test,
        predicted_test,
        labels=[0, 1],
        target_names=["Native", "Invasive"],
        zero_division=0
    )
)

print("Accuracy:", round(test_accuracy, 4))
print("Balanced accuracy:", round(test_balanced_accuracy, 4))
print("Macro precision:", round(test_macro_precision, 4))
print("Macro recall:", round(test_macro_recall, 4))
print("Macro F1:", round(test_macro_f1, 4))

# Save test evaluation artifacts
test_timestamp = time.strftime("%Y%m%d_%H%M%S")

test_output_dir = (
    experiment_dir /
    f"independent_test_evaluation_{test_timestamp}"
)

test_output_dir.mkdir(parents=True, exist_ok=False)

test_predictions_df = pd.DataFrame({
    "actual_label": true_test,
    "predicted_label": predicted_test,
    "invasive_probability": invasive_probabilities
})

test_predictions_df.to_csv(
    test_output_dir / "independent_test_predictions.csv",
    index=False
)

test_metrics = {
    "checkpoint_epoch": int(locked_data["epoch"]),
    "test_images": len(test_dataset),
    "accuracy": float(test_accuracy),
    "balanced_accuracy": float(test_balanced_accuracy),
    "macro_precision": float(test_macro_precision),
    "macro_recall": float(test_macro_recall),
    "macro_f1": float(test_macro_f1),
    "confusion_matrix": test_cm.tolist(),
    "classification_report": test_report,
    "test_used_for_model_selection": False
}

with open(
    test_output_dir / "independent_test_metrics.json",
    "x"
) as file:
    json.dump(test_metrics, file, indent=4)

print("\nTEST RESULTS SAVED TO:")
print(test_output_dir)

print("\nSTEP 106 COMPLETE — INDEPENDENT TEST EVALUATED")

In [ ]:
# M4 STEP 107 — SPECIES-WISE INDEPENDENT TEST ANALYSIS

import pandas as pd
import numpy as np
import time

# Reconstruct test prediction table with original metadata
species_test_df = test_df.reset_index(drop=True).copy()

assert len(species_test_df) == len(true_test)
assert len(predicted_test) == len(true_test)

species_test_df["actual_label"] = true_test
species_test_df["predicted_label"] = predicted_test

species_test_df["correct"] = (
    species_test_df["actual_label"] ==
    species_test_df["predicted_label"]
)

# Species-wise summary
species_summary = (
    species_test_df
    .groupby("Especies")
    .agg(
        images_tested=("Filename", "count"),
        correct_predictions=("correct", "sum"),
        accuracy=("correct", "mean")
    )
    .reset_index()
)

species_summary["incorrect_predictions"] = (
    species_summary["images_tested"] -
    species_summary["correct_predictions"]
)

species_summary["accuracy_percent"] = (
    species_summary["accuracy"] * 100
).round(2)

# Flight-wise summary
flight_summary = (
    species_test_df
    .groupby("F_id")
    .agg(
        images_tested=("Filename", "count"),
        correct_predictions=("correct", "sum"),
        accuracy=("correct", "mean")
    )
    .reset_index()
)

flight_summary["accuracy_percent"] = (
    flight_summary["accuracy"] * 100
).round(2)

print("=" * 55)
print("SPECIES-WISE INDEPENDENT TEST RESULTS")
print("=" * 55)

print(
    species_summary[
        [
            "Especies",
            "images_tested",
            "correct_predictions",
            "incorrect_predictions",
            "accuracy_percent"
        ]
    ].to_string(index=False)
)

print("\nFLIGHT-WISE INDEPENDENT TEST RESULTS")
print("=" * 55)

print(
    flight_summary[
        [
            "F_id",
            "images_tested",
            "correct_predictions",
            "accuracy_percent"
        ]
    ].to_string(index=False)
)

# Save separate descriptive analysis artifacts
analysis_timestamp = time.strftime("%Y%m%d_%H%M%S")

species_summary.to_csv(
    test_output_dir /
    f"species_wise_test_analysis_{analysis_timestamp}.csv",
    index=False
)

flight_summary.to_csv(
    test_output_dir /
    f"flight_wise_test_analysis_{analysis_timestamp}.csv",
    index=False
)

print("\nSTEP 107 COMPLETE")
print("Descriptive analysis only — no model tuning performed.")

In [ ]:
# M4 STEP 108 — SPECIES-LEVEL PREDICTION AUDIT

import pandas as pd
import time

audit_df = species_test_df.copy()

audit_df["invasive_probability"] = invasive_probabilities

# Add readable class names
class_names = {
    0: "Native",
    1: "Invasive"
}

audit_df["actual_class"] = audit_df["actual_label"].map(class_names)
audit_df["predicted_class"] = audit_df["predicted_label"].map(class_names)

# Summarize predictions for each species
species_audit = (
    audit_df.groupby("Especies")
    .agg(
        ecological_status=("Ecological_Status", "first"),
        total_images=("Filename", "count"),
        actual_native=("actual_label", lambda x: (x == 0).sum()),
        actual_invasive=("actual_label", lambda x: (x == 1).sum()),
        predicted_native=("predicted_label", lambda x: (x == 0).sum()),
        predicted_invasive=("predicted_label", lambda x: (x == 1).sum()),
        correct_predictions=("correct", "sum"),
        mean_invasive_probability=("invasive_probability", "mean")
    )
    .reset_index()
)

species_audit["accuracy_percent"] = (
    species_audit["correct_predictions"] /
    species_audit["total_images"] * 100
).round(2)

species_audit["mean_invasive_probability"] = (
    species_audit["mean_invasive_probability"].round(4)
)

print("=" * 65)
print("SPECIES-LEVEL ECOLOGICAL CLASSIFICATION AUDIT")
print("=" * 65)

print(species_audit.to_string(index=False))

# Save audit without overwriting existing results
audit_timestamp = time.strftime("%Y%m%d_%H%M%S")

audit_path = (
    test_output_dir /
    f"species_prediction_audit_{audit_timestamp}.csv"
)

species_audit.to_csv(audit_path, index=False)

print("\nAudit saved:", audit_path)
print("\nSTEP 108 COMPLETE — DESCRIPTIVE AUDIT ONLY")

In [ ]:
# M4 STEP 109 — SPECIES AND FLIGHT ERROR ANALYSIS

import pandas as pd

flight_species_audit = (
    audit_df.groupby(["Especies", "F_id", "Location"])
    .agg(
        images=("Filename", "count"),
        correct=("correct", "sum"),
        mean_invasive_probability=("invasive_probability", "mean")
    )
    .reset_index()
)

flight_species_audit["incorrect"] = (
    flight_species_audit["images"] -
    flight_species_audit["correct"]
)

flight_species_audit["accuracy_percent"] = (
    flight_species_audit["correct"] /
    flight_species_audit["images"] * 100
).round(2)

flight_species_audit["mean_invasive_probability"] = (
    flight_species_audit["mean_invasive_probability"].round(4)
)

print("=" * 75)
print("SPECIES × FLIGHT × LOCATION ANALYSIS")
print("=" * 75)

print(
    flight_species_audit.to_string(index=False)
)

# Save results as a separate artifact
import time

flight_species_path = (
    test_output_dir /
    f"species_flight_error_analysis_{time.strftime('%Y%m%d_%H%M%S')}.csv"
)

flight_species_audit.to_csv(
    flight_species_path,
    index=False
)

print("\nSaved:", flight_species_path)
print("\nSTEP 109 COMPLETE — NO MODEL CHANGES")

In [ ]:
# M4 STEP 110 — IMAGE AUDIT SCHEMA CHECK

import pandas as pd
from pathlib import Path

project_dir = Path(
    "/content/drive/MyDrive/Marine_Project"
)

dimension_path = (
    project_dir / "results" /
    "IMO_M4_image_dimensions_20261121_181622.csv"
)

background_path = (
    project_dir / "results" /
    "IMO_M4_black_background_detail_20261121_182939.csv"
)

dimension_df = pd.read_csv(dimension_path)
background_df = pd.read_csv(background_path)

print("=" * 60)
print("IMAGE DIMENSION AUDIT")
print("=" * 60)
print("Rows:", len(dimension_df))
print("Columns:", dimension_df.columns.tolist())
print(dimension_df.head(3).to_string(index=False))

print("\n" + "=" * 60)
print("BLACK BACKGROUND AUDIT")
print("=" * 60)
print("Rows:", len(background_df))
print("Columns:", background_df.columns.tolist())
print(background_df.head(3).to_string(index=False))

print("\nSTEP 110 COMPLETE — SCHEMA INSPECTION ONLY")

In [ ]:
# M4 STEP 111A — CHECK MERGE COLUMN NAMES

print("audit_df columns:")
print(audit_df.columns.tolist())

print("\ndimension_df columns:")
print(dimension_df.columns.tolist())

print("\nbackground_df columns:")
print(background_df.columns.tolist())

print("\nSTEP 111A COMPLETE — INSPECTION ONLY")

In [ ]:
# M4 STEP 111B — SPECIES × FLIGHT IMAGE CHARACTERISTICS

# Join dimensions using the correct column names
diagnostic_df = audit_df.merge(
    dimension_df[["filename", "min_side", "pixels"]],
    left_on="Filename",
    right_on="filename",
    how="left",
    validate="one_to_one"
)

# Join background characteristics
diagnostic_df = diagnostic_df.merge(
    background_df[["Filename", "Black_Percentage"]],
    on="Filename",
    how="left",
    validate="one_to_one"
)

# Verify matching
print("Test images:", len(diagnostic_df))
print("Missing resolution:", diagnostic_df["min_side"].isna().sum())
print("Missing background:", diagnostic_df["Black_Percentage"].isna().sum())

# Summarise characteristics by species, flight and location
summary = (
    diagnostic_df.groupby(["Especies", "F_id", "Location"])
    .agg(
        images=("Filename", "count"),
        accuracy=("correct", "mean"),
        median_min_side=("min_side", "median"),
        mean_min_side=("min_side", "mean"),
        median_black_percentage=("Black_Percentage", "median"),
        mean_black_percentage=("Black_Percentage", "mean")
    )
    .reset_index()
)

summary["accuracy_percent"] = (summary["accuracy"] * 100).round(2)

print("\n" + "=" * 90)
print("SPECIES × FLIGHT IMAGE CHARACTERISTICS")
print("=" * 90)

print(
    summary.drop(columns=["accuracy"]).round(2).to_string(index=False)
)

print("\nSTEP 111B COMPLETE — DIAGNOSTIC ANALYSIS ONLY")

In [ ]:
# M4 STEP 112 — ERROR VS IMAGE CHARACTERISTICS

import numpy as np

diagnostic_df["error"] = (
    ~diagnostic_df["correct"].astype(bool)
).astype(int)

# Compare correct and incorrect predictions
error_summary = (
    diagnostic_df.groupby("error")
    .agg(
        images=("Filename", "count"),
        median_resolution=("min_side", "median"),
        mean_resolution=("min_side", "mean"),
        median_black_percentage=("Black_Percentage", "median"),
        mean_black_percentage=("Black_Percentage", "mean")
    )
    .reset_index()
)

error_summary["prediction_status"] = (
    error_summary["error"].map({
        0: "Correct",
        1: "Incorrect"
    })
)

print("=" * 75)
print("IMAGE CHARACTERISTICS: CORRECT VS INCORRECT PREDICTIONS")
print("=" * 75)

print(
    error_summary[
        [
            "prediction_status",
            "images",
            "median_resolution",
            "mean_resolution",
            "median_black_percentage",
            "mean_black_percentage"
        ]
    ].round(2).to_string(index=False)
)

# Spearman correlations are descriptive, not causal
print("\nSPEARMAN CORRELATIONS WITH PREDICTION ERROR")

for column in ["min_side", "Black_Percentage"]:
    correlation = diagnostic_df[column].corr(
        diagnostic_df["error"],
        method="spearman"
    )

    print(f"{column}: {correlation:.4f}")

print("\nSTEP 112 COMPLETE — DESCRIPTIVE ANALYSIS ONLY")

In [ ]:
# M4 STEP 113 — SPECIES-LEVEL ERROR CONCENTRATION

species_errors = (
    diagnostic_df.groupby("Especies")
    .agg(
        total_images=("Filename", "count"),
        correct_predictions=("correct", "sum"),
        incorrect_predictions=("error", "sum")
    )
    .reset_index()
)

species_errors["error_rate_percent"] = (
    species_errors["incorrect_predictions"] /
    species_errors["total_images"] * 100
).round(2)

species_errors["share_of_all_errors_percent"] = (
    species_errors["incorrect_predictions"] /
    diagnostic_df["error"].sum() * 100
).round(2)

species_errors = species_errors.sort_values(
    "incorrect_predictions",
    ascending=False
)

print("=" * 85)
print("SPECIES-LEVEL ERROR DISTRIBUTION")
print("=" * 85)

print(species_errors.to_string(index=False))

print("\nTotal incorrect predictions:",
      diagnostic_df["error"].sum())

print("\nSTEP 113 COMPLETE — DIAGNOSTIC ANALYSIS ONLY")

In [ ]:
# M4 STEP 114 — SAVE FINAL EXPERIMENT SUMMARY

import json
import time
from pathlib import Path

m4_summary = {
    "milestone": "M4",
    "project_title": (
        "Fine-Grained Discrimination of Invasive and Native "
        "Marine Vegetation Using a Hybrid Deep Learning Model "
        "for Selective Underwater Plant Removal"
    ),
    "dataset": "IMO Macroalgae Dataset",
    "experiment": "ResNet18 binary ecological classification",
    "input_size": "224x224 RGB",
    "classes": {
        "0": "Native",
        "1": "Invasive"
    },
    "split_counts": {
        "train": 1528,
        "validation": 1249,
        "independent_test": 422
    },
    "locked_checkpoint": (
        "LOCKED_best_checkpoint_epoch2_20261121_184852.pth"
    ),
    "checkpoint_selection": "Lowest validation loss",
    "independent_test_metrics": {
        "accuracy": 0.8602,
        "balanced_accuracy": 0.8657,
        "macro_precision": 0.8708,
        "macro_recall": 0.8657,
        "macro_f1": 0.86,
        "incorrect_predictions": 59
    },
    "confusion_matrix": [
        [174, 50],
        [9, 189]
    ],
    "species_error_analysis": {
        "Saccorhiza polyschides": 49,
        "Sargassum muticum": 9,
        "Ericaria selaginoides": 1,
        "Asparagopsis armata": 0,
        "Bifurcaria bifurcata": 0
    },
    "diagnostic_findings": {
        "saccorhiza_baiona_accuracy_percent": 40.0,
        "saccorhiza_illa_accuracy_percent": 87.5,
        "resolution_error_spearman": 0.0483,
        "black_background_error_spearman": 0.0789,
        "saccorhiza_share_of_all_errors_percent": 83.05
    },
    "limitations": [
        "Intertidal imagery rather than underwater removal imagery",
        "Only six selected species used for ecological classification",
        "Codium absent from independent test",
        "Only one Asparagopsis image in validation",
        "Only two invasive candidate species in independent test",
        "High false invasive prediction rate for Saccorhiza",
        "Potential species, location and acquisition-domain confounding",
        "Classification does not demonstrate detection, segmentation or removal"
    ],
    "test_usage": (
        "Independent test evaluated once; subsequent work was "
        "descriptive diagnostics only. No model tuning performed."
    ),
    "status": "Experiment complete; scientific limitations documented",
    "created_at": time.strftime("%Y-%m-%d %H:%M:%S")
}

summary_path = (
    Path("/content/drive/MyDrive/Marine_Project/models")
    / "M4_ResNet18_experiment_20261121_183700"
    / "M4_FINAL_EXPERIMENT_SUMMARY.json"
)

with open(summary_path, "x", encoding="utf-8") as file:
    json.dump(m4_summary, file, indent=4)

print("Saved:", summary_path)
print("File exists:", summary_path.exists())
print("File size:", summary_path.stat().st_size, "bytes")
print("\nSTEP 114 COMPLETE — FINAL SUMMARY SAVED")

In [ ]:
print("INDEPENDENT TEST PREDICTION COLUMNS:")
print(m4_test_predictions_df.columns.tolist())

print("\nFIRST 5 PREDICTION ROWS:")
display(m4_test_predictions_df.head())

print("\nTEST MANIFEST COLUMNS:")
print(m4_test_df.columns.tolist())

print("\nPREDICTION DATA TYPES:")
print(m4_test_predictions_df.dtypes)

In [ ]:
# M4 RESTORATION FIX — VERIFY PREDICTION/MANIFEST ALIGNMENT
# Read-only: no training, evaluation, or file modifications.

label_mapping = {
    "Native": 0,
    "Invasive": 1
}

expected_labels = m4_test_df["Ecological_Status"].map(label_mapping)

print("Prediction rows:", len(m4_test_predictions_df))
print("Test manifest rows:", len(m4_test_df))

assert len(m4_test_predictions_df) == len(m4_test_df), (
    "Prediction and manifest row counts do not match."
)

assert expected_labels.notna().all(), (
    "Unknown ecological labels found in test manifest."
)

mismatch_count = (
    m4_test_predictions_df["actual_label"].reset_index(drop=True)
    != expected_labels.reset_index(drop=True)
).sum()

print("Actual-label mismatches:", int(mismatch_count))

if mismatch_count == 0:
    print("\nSUCCESS: Prediction order matches the test manifest.")
else:
    print("\nWARNING: Row alignment is not verified. Do not merge yet.")

In [ ]:
# M4 RESTORATION FIX — REBUILD DIAGNOSTIC DATAFRAME
# Uses existing saved predictions and test manifest.
# No training, evaluation, or file overwriting.

m4_test_predictions_df = m4_test_predictions_df.reset_index(drop=True).copy()
m4_test_manifest_aligned = m4_test_df.reset_index(drop=True).copy()

diagnostic_df = pd.concat(
    [
        m4_test_manifest_aligned,
        m4_test_predictions_df
    ],
    axis=1
)

# Add readable prediction labels
diagnostic_df["actual_prediction_label"] = (
    diagnostic_df["actual_label"].map({0: "Native", 1: "Invasive"})
)

diagnostic_df["model_prediction_label"] = (
    diagnostic_df["predicted_label"].map({0: "Native", 1: "Invasive"})
)

# Correctness indicator
diagnostic_df["correct"] = (
    diagnostic_df["actual_label"] == diagnostic_df["predicted_label"]
)

print("M4 diagnostic dataframe restored successfully.")
print("Rows:", len(diagnostic_df))
print("Columns:", diagnostic_df.columns.tolist())
print("\nPrediction correctness:")
print(diagnostic_df["correct"].value_counts())
print("\nSpecies-wise results:")
display(
    diagnostic_df.groupby("Especies")["correct"]
    .agg(["count", "sum", "mean"])
    .rename(columns={"count": "Total", "sum": "Correct", "mean": "Accuracy"})
)

In [ ]:
import os
import torch

PROJECT = "/content/drive/MyDrive/Marine_Project"
MODELS = os.path.join(PROJECT, "models")
RESULTS = os.path.join(PROJECT, "results")

M2_CHECKPOINT = os.path.join(
    MODELS, "resnet18_grouped_v2_baseline_best.pth"
)

M4_CHECKPOINT = os.path.join(
    MODELS,
    "M4_ResNet18_experiment_20261121_183700",
    "LOCKED_best_checkpoint_epoch2_20261121_184852.pth"
)

M4_ZIP = os.path.join(
    PROJECT, "IMO_M4_selected_images_3199_20261121_181622.zip"
)

checks = {
    "Google Drive": os.path.isdir("/content/drive/MyDrive"),
    "Project directory": os.path.isdir(PROJECT),
    "M2 best checkpoint": os.path.isfile(M2_CHECKPOINT),
    "M4 locked checkpoint": os.path.isfile(M4_CHECKPOINT),
    "M4 training history": len(m4_training_history_df) == 10,
    "M4 validation predictions": len(m4_validation_predictions_df) == 1249,
    "M4 test predictions": len(m4_test_predictions_df) == 422,
    "M4 diagnostic dataframe": len(diagnostic_df) == 422,
    "M4 selected image archive": os.path.isfile(M4_ZIP),
}

print("MASTER RESTORATION — FINAL VERIFICATION")
print("=" * 50)

for name, passed in checks.items():
    print(f"{'PASS' if passed else 'FAIL'} — {name}")

print("\nAll checks passed:", all(checks.values()))
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("Current device: CPU")

print("\nM4 diagnostic summary:")
print("Total:", len(diagnostic_df))
print("Correct:", int(diagnostic_df["correct"].sum()))
print("Incorrect:", int((~diagnostic_df["correct"]).sum()))

In [ ]:
# ============================================================
# M5 — STEP 1: SPECIES AND FLIGHT DISTRIBUTION AUDIT
# Read-only. Does not modify M1-M4 files or manifests.
# ============================================================

import pandas as pd

m5_source_df = pd.concat(
    [m4_train_df, m4_val_df, m4_test_df],
    ignore_index=True
).copy()

required_columns = [
    "Filename",
    "Especies",
    "Ecological_Status",
    "F_id",
    "Location",
    "Orth_id"
]

assert all(c in m5_source_df.columns for c in required_columns)
assert len(m5_source_df) == 3199
assert m5_source_df["Filename"].is_unique

print("M5 SOURCE DATA AUDIT")
print("=" * 55)
print("Total images:", len(m5_source_df))
print("Unique species:", m5_source_df["Especies"].nunique())
print("Flights:", sorted(m5_source_df["F_id"].dropna().unique().tolist()))
print("Locations:", m5_source_df["Location"].dropna().unique().tolist())

print("\nSPECIES DISTRIBUTION")
display(
    m5_source_df.groupby(
        ["Especies", "Ecological_Status"]
    ).size().reset_index(name="Images")
)

print("\nFLIGHT-WISE IMAGE DISTRIBUTION")
display(
    m5_source_df.groupby(
        ["F_id", "Location"]
    ).size().reset_index(name="Images")
)

print("\nSPECIES BY FLIGHT")
display(
    m5_source_df.pivot_table(
        index="Especies",
        columns="F_id",
        values="Filename",
        aggfunc="count",
        fill_value=0
    )
)

print("\nM4 ORIGINAL SPLITS — PRESERVED")
print(
    "Train:", len(m4_train_df),
    "| Validation:", len(m4_val_df),
    "| Test:", len(m4_test_df)
)

In [ ]:
# ============================================================
# M5 — STEP 2: DEVELOPMENT-ONLY FLIGHT AND SPECIES AUDIT
# M4 independent test set remains excluded.
# ============================================================

m5_development_df = pd.concat(
    [m4_train_df, m4_val_df],
    ignore_index=True
).copy()

m5_sealed_test_df = m4_test_df.copy()

print("M5 DEVELOPMENT DATA")
print("=" * 55)
print("Development images:", len(m5_development_df))
print("Sealed M4 test images:", len(m5_sealed_test_df))
print("Development flights:",
      sorted(m5_development_df["F_id"].dropna().unique().tolist()))

print("\nSPECIES COUNTS IN DEVELOPMENT DATA")
display(
    m5_development_df.groupby(
        ["Especies", "Ecological_Status"]
    ).size().reset_index(name="Images")
)

print("\nSPECIES BY DEVELOPMENT FLIGHT")
display(
    m5_development_df.pivot_table(
        index="Especies",
        columns="F_id",
        values="Filename",
        aggfunc="count",
        fill_value=0
    )
)

print("\nORTHOMOSAIC COVERAGE BY SPECIES")
display(
    m5_development_df.groupby("Especies").agg(
        Images=("Filename", "count"),
        Unique_Orthomosaics=("Orth_id", "nunique"),
        Unique_Flights=("F_id", "nunique"),
        Locations=("Location", "nunique")
    ).sort_values("Images", ascending=False)
)

print("\nM4 TEST ISOLATION CHECK")
development_flights = set(m5_development_df["F_id"].dropna())
sealed_test_flights = set(m5_sealed_test_df["F_id"].dropna())

print("Shared development/test flights:",
      sorted(development_flights & sealed_test_flights))
print("Development/test filename overlap:",
      len(set(m5_development_df["Filename"]) &
          set(m5_sealed_test_df["Filename"])))

assert len(m5_development_df) == 2777
assert len(m5_sealed_test_df) == 422
assert set(m5_development_df["Filename"]).isdisjoint(
    set(m5_sealed_test_df["Filename"])
)

In [ ]:
import pandas as pd

# Use only M5 development data. M4 test remains sealed.
m5_orthomosaic_audit = (
    m5_development_df
    .groupby(
        ["Especies", "Ecological_Status", "Orth_id", "F_id", "Location"],
        dropna=False
    )
    .size()
    .reset_index(name="Images")
    .sort_values(["Especies", "F_id", "Orth_id"])
)

print("=" * 75)
print("M5 ORTHOMOSAIC-LEVEL DISTRIBUTION")
print("=" * 75)

for species in sorted(m5_development_df["Especies"].dropna().unique()):
    species_df = m5_orthomosaic_audit[
        m5_orthomosaic_audit["Especies"] == species
    ]

    print(f"\n{species}")
    print(species_df[
        ["Orth_id", "F_id", "Location", "Images"]
    ].to_string(index=False))

print("\n" + "=" * 75)
print("ORTHOMOSAIC COUNTS PER SPECIES")
print("=" * 75)

print(
    m5_orthomosaic_audit
    .groupby("Especies")
    .agg(
        Images=("Images", "sum"),
        Orthomosaics=("Orth_id", "nunique"),
        Flights=("F_id", "nunique")
    )
    .to_string()
)

print("\nM4 TEST STILL SEALED:", len(m5_sealed_test_df) == 422)

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import GroupShuffleSplit

df = m5_development_df.copy().reset_index(drop=True)

# Composite grouping key: keep every image from an orthomosaic together.
df["M5_group"] = (
    df["F_id"].astype(str) + "_" +
    df["Orth_id"].astype(str)
)

total_images = len(df)
total_species = set(df["Especies"].unique())

# Keep the large Asparagopsis orthomosaic (20 images) in training.
protected_group = "2_5"

assert protected_group in set(df["M5_group"])

splitter = GroupShuffleSplit(
    n_splits=2000,
    test_size=0.20,
    random_state=42
)

candidates = []

for train_idx, val_idx in splitter.split(
    df,
    groups=df["M5_group"]
):
    train_df = df.iloc[train_idx]
    val_df = df.iloc[val_idx]

    # The 20-image Asparagopsis group must remain in training.
    if protected_group in set(val_df["M5_group"]):
        continue

    # Both ecological classes must appear in both partitions.
    if set(train_df["Ecological_Status"]) != {"Native", "Invasive"}:
        continue
    if set(val_df["Ecological_Status"]) != {"Native", "Invasive"}:
        continue

    # All six species must remain represented in training.
    if set(train_df["Especies"]) != total_species:
        continue

    # Prefer validation coverage of all six species where feasible.
    val_species = set(val_df["Especies"])
    missing_val_species = len(total_species - val_species)

    val_fraction = len(val_df) / total_images

    # Prefer approximately 20% validation and broader species coverage.
    score = (
        abs(val_fraction - 0.20)
        + 0.04 * missing_val_species
    )

    candidates.append({
        "score": score,
        "train_idx": train_idx,
        "val_idx": val_idx,
        "train_images": len(train_df),
        "validation_images": len(val_df),
        "validation_fraction": val_fraction,
        "validation_species": len(val_species),
        "missing_validation_species": sorted(total_species - val_species)
    })

print("=" * 70)
print("M5 ORTHOMOSAIC-GROUPED SPLIT FEASIBILITY")
print("=" * 70)
print("Candidates satisfying the constraints:", len(candidates))

if candidates:
    candidates = sorted(candidates, key=lambda x: x["score"])

    print("\nTOP 5 CANDIDATES")
    for rank, item in enumerate(candidates[:5], start=1):
        print(
            f"\nCandidate {rank}: "
            f"Train={item['train_images']}, "
            f"Validation={item['validation_images']}, "
            f"Validation={item['validation_fraction']:.1%}, "
            f"Validation species={item['validation_species']}/6"
        )
        print("Species absent from validation:",
              item["missing_validation_species"])

    best = candidates[0]
    best_train_df = df.iloc[best["train_idx"]].drop(
        columns=["M5_group"]
    ).copy()
    best_val_df = df.iloc[best["val_idx"]].drop(
        columns=["M5_group"]
    ).copy()

    print("\n" + "=" * 70)
    print("BEST CANDIDATE — SPECIES DISTRIBUTION")
    print("=" * 70)

    print(pd.crosstab(
        best_train_df["Especies"],
        best_train_df["Ecological_Status"]
    ).assign(Train=best_train_df.groupby("Especies").size()))

    print("\nValidation:")
    print(
        best_val_df.groupby(
            ["Especies", "Ecological_Status"]
        ).size().to_string()
    )

    print("\n" + "=" * 70)
    print("GROUP LEAKAGE CHECK")
    print("=" * 70)

    train_groups = set(
        best_train_df["F_id"].astype(str) + "_" +
        best_train_df["Orth_id"].astype(str)
    )
    val_groups = set(
        best_val_df["F_id"].astype(str) + "_" +
        best_val_df["Orth_id"].astype(str)
    )

    print("Shared orthomosaic groups:", len(train_groups & val_groups))
    print("Asparagopsis in training:",
          (best_train_df["Especies"] == "Asparagopsis armata").sum())
    print("Asparagopsis in validation:",
          (best_val_df["Especies"] == "Asparagopsis armata").sum())

else:
    print(
        "No candidate satisfied all constraints. "
        "We will reconsider the validation design rather than "
        "relaxing safeguards silently."
    )

print("\nM4 test remains sealed:", len(m5_sealed_test_df) == 422)

In [ ]:
import pandas as pd
from sklearn.model_selection import GroupShuffleSplit

df = m5_development_df.copy().reset_index(drop=True)

df["M5_group"] = (
    df["F_id"].astype(str) + "_" +
    df["Orth_id"].astype(str)
)

# Keep every Asparagopsis orthomosaic in training.
aspara_groups = set(
    df.loc[
        df["Especies"] == "Asparagopsis armata",
        "M5_group"
    ]
)

print("Asparagopsis groups protected in training:", sorted(aspara_groups))

splitter = GroupShuffleSplit(
    n_splits=3000,
    test_size=0.20,
    random_state=123
)

candidates = []

for train_idx, val_idx in splitter.split(df, groups=df["M5_group"]):
    train_df = df.iloc[train_idx]
    val_df = df.iloc[val_idx]

    train_groups = set(train_df["M5_group"])
    val_groups = set(val_df["M5_group"])

    if not aspara_groups.issubset(train_groups):
        continue

    if train_groups.intersection(val_groups):
        continue

    if set(train_df["Ecological_Status"]) != {"Native", "Invasive"}:
        continue

    if set(val_df["Ecological_Status"]) != {"Native", "Invasive"}:
        continue

    # All six species must be represented in training.
    if train_df["Especies"].nunique() != 6:
        continue

    score = abs(len(val_df) / len(df) - 0.20)

    candidates.append({
        "score": score,
        "train_idx": train_idx,
        "val_idx": val_idx
    })

print("\nFeasible candidates:", len(candidates))

if candidates:
    best = min(candidates, key=lambda x: x["score"])

    best_train_df = df.iloc[best["train_idx"]].drop(
        columns=["M5_group"]
    ).copy()

    best_val_df = df.iloc[best["val_idx"]].drop(
        columns=["M5_group"]
    ).copy()

    print("\nTRAIN IMAGES:", len(best_train_df))
    print("VALIDATION IMAGES:", len(best_val_df))
    print("VALIDATION PERCENTAGE:",
          round(100 * len(best_val_df) / len(df), 2))

    print("\nTRAIN SPECIES DISTRIBUTION:")
    print(
        best_train_df.groupby(
            ["Especies", "Ecological_Status"]
        ).size().to_string()
    )

    print("\nVALIDATION SPECIES DISTRIBUTION:")
    print(
        best_val_df.groupby(
            ["Especies", "Ecological_Status"]
        ).size().to_string()
    )

    print("\nASPARAGOPSIS TRAIN:",
          (best_train_df["Especies"] == "Asparagopsis armata").sum())

    print("ASPARAGOPSIS VALIDATION:",
          (best_val_df["Especies"] == "Asparagopsis armata").sum())

    train_groups = set(
        best_train_df["F_id"].astype(str) + "_" +
        best_train_df["Orth_id"].astype(str)
    )
    val_groups = set(
        best_val_df["F_id"].astype(str) + "_" +
        best_val_df["Orth_id"].astype(str)
    )

    print("\nSHARED ORTHOMOSAIC GROUPS:", len(train_groups & val_groups))

else:
    print("No feasible candidate found.")

print("\nM4 TEST STILL SEALED:", len(m5_sealed_test_df) == 422)

In [ ]:
def show_m5_group_allocation(split_df, split_name):
    audit = (
        split_df
        .groupby(
            ["Especies", "Ecological_Status", "F_id", "Orth_id"],
            dropna=False
        )
        .size()
        .reset_index(name="Images")
        .sort_values(["Especies", "F_id", "Orth_id"])
    )

    print(f"\n{split_name}")
    print(audit.to_string(index=False))


show_m5_group_allocation(best_train_df, "TRAIN ORTHOMOSAICS")
show_m5_group_allocation(best_val_df, "VALIDATION ORTHOMOSAICS")

print("\nVALIDATION SPECIES COVERAGE:")
print(
    best_val_df.groupby("Especies")
    .agg(Images=("Filename", "count"))
    .to_string()
)

print("\nM4 test remains sealed:", len(m5_sealed_test_df) == 422)

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import GroupShuffleSplit

df = m5_development_df.copy().reset_index(drop=True)

df["M5_group"] = (
    df["F_id"].astype(str) + "_" +
    df["Orth_id"].astype(str)
)

aspara_groups = set(
    df.loc[df["Especies"] == "Asparagopsis armata", "M5_group"]
)

species_list = sorted(df["Especies"].unique())
validation_species = [
    s for s in species_list if s != "Asparagopsis armata"
]

splitter = GroupShuffleSplit(
    n_splits=20000,
    test_size=0.20,
    random_state=2026
)

candidates = []

for train_idx, val_idx in splitter.split(df, groups=df["M5_group"]):
    train_df = df.iloc[train_idx]
    val_df = df.iloc[val_idx]

    train_groups = set(train_df["M5_group"])
    val_groups = set(val_df["M5_group"])

    if not aspara_groups.issubset(train_groups):
        continue

    if train_groups.intersection(val_groups):
        continue

    if train_df["Especies"].nunique() != 6:
        continue

    if not {"Native", "Invasive"}.issubset(
        set(val_df["Ecological_Status"])
    ):
        continue

    val_counts = val_df["Especies"].value_counts()

    # Require every other species to have at least 5 validation images.
    if any(val_counts.get(s, 0) < 5 for s in validation_species):
        continue

    # Compare each species' validation proportion with a 20% target.
    species_deviation = np.mean([
        abs(val_counts.get(s, 0) / (df["Especies"] == s).sum() - 0.20)
        for s in validation_species
    ])

    overall_deviation = abs(len(val_df) / len(df) - 0.20)

    score = species_deviation + 0.5 * overall_deviation

    candidates.append({
        "score": score,
        "train_idx": train_idx,
        "val_idx": val_idx,
        "train_images": len(train_df),
        "val_images": len(val_df),
        "species_deviation": species_deviation,
        "overall_deviation": overall_deviation
    })

print("FEASIBLE CANDIDATES:", len(candidates))

if candidates:
    candidates.sort(key=lambda x: x["score"])

    for rank, item in enumerate(candidates[:5], 1):
        print(
            f"\nCandidate {rank}: "
            f"Train={item['train_images']}, "
            f"Validation={item['val_images']}, "
            f"Validation={item['val_images']/len(df):.1%}, "
            f"Species deviation={item['species_deviation']:.4f}"
        )

    selected = candidates[0]

    best_m5_train_df = df.iloc[selected["train_idx"]].drop(
        columns=["M5_group"]
    ).copy()

    best_m5_val_df = df.iloc[selected["val_idx"]].drop(
        columns=["M5_group"]
    ).copy()

    print("\nSELECTED CANDIDATE — SPECIES DISTRIBUTION")
    distribution = pd.DataFrame({
        "Total": df["Especies"].value_counts(),
        "Train": best_m5_train_df["Especies"].value_counts(),
        "Validation": best_m5_val_df["Especies"].value_counts()
    }).fillna(0).astype(int)

    distribution["Validation_%"] = (
        100 * distribution["Validation"] / distribution["Total"]
    ).round(1)

    print(distribution.sort_index().to_string())

    train_groups = set(
        best_m5_train_df["F_id"].astype(str) + "_" +
        best_m5_train_df["Orth_id"].astype(str)
    )
    val_groups = set(
        best_m5_val_df["F_id"].astype(str) + "_" +
        best_m5_val_df["Orth_id"].astype(str)
    )

    print("\nTRAIN IMAGES:", len(best_m5_train_df))
    print("VALIDATION IMAGES:", len(best_m5_val_df))
    print("SHARED ORTHOMOSAIC GROUPS:", len(train_groups & val_groups))
    print("ASPARAGOPSIS TRAIN:",
          (best_m5_train_df["Especies"] == "Asparagopsis armata").sum())
    print("ASPARAGOPSIS VALIDATION:",
          (best_m5_val_df["Especies"] == "Asparagopsis armata").sum())

else:
    print("No candidate met all constraints. Do not relax safeguards yet.")

print("\nM4 TEST STILL SEALED:", len(m5_sealed_test_df) == 422)

In [ ]:
def inspect_split_groups(data, name):
    groups = (
        data.groupby(
            ["F_id", "Orth_id", "Location"],
            dropna=False
        )
        .agg(
            Images=("Filename", "count"),
            Species=("Especies", lambda x: ", ".join(sorted(set(x))))
        )
        .reset_index()
        .sort_values(["F_id", "Orth_id"])
    )

    print(f"\n{name}")
    print(groups.to_string(index=False))


inspect_split_groups(best_m5_train_df, "M5 TRAIN GROUPS")
inspect_split_groups(best_m5_val_df, "M5 VALIDATION GROUPS")

print("\nM4 TEST SEALED:", len(m5_sealed_test_df) == 422)

In [ ]:
import os
import json
from datetime import datetime
import pandas as pd

# Output directory
m5_results_dir = "/content/drive/MyDrive/Marine_Project/results"
os.makedirs(m5_results_dir, exist_ok=True)

# Final integrity checks
assert len(best_m5_train_df) == 2345
assert len(best_m5_val_df) == 432
assert len(m5_sealed_test_df) == 422

assert set(best_m5_train_df["Filename"]).isdisjoint(
    set(best_m5_val_df["Filename"])
)

assert set(best_m5_train_df["Filename"]).isdisjoint(
    set(m5_sealed_test_df["Filename"])
)

assert set(best_m5_val_df["Filename"]).isdisjoint(
    set(m5_sealed_test_df["Filename"])
)

def group_set(data):
    return set(
        data["F_id"].astype(str) + "_" +
        data["Orth_id"].astype(str)
    )

assert group_set(best_m5_train_df).isdisjoint(
    group_set(best_m5_val_df)
)

assert (
    best_m5_train_df["Especies"] == "Asparagopsis armata"
).sum() == 22

assert (
    best_m5_val_df["Especies"] == "Asparagopsis armata"
).sum() == 0

# Create unique timestamped filenames
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

train_path = os.path.join(
    m5_results_dir, f"M5_train_manifest_{timestamp}.csv"
)
val_path = os.path.join(
    m5_results_dir, f"M5_validation_manifest_{timestamp}.csv"
)
audit_path = os.path.join(
    m5_results_dir, f"M5_split_lock_record_{timestamp}.json"
)

# Save manifests
best_m5_train_df.to_csv(train_path, index=False)
best_m5_val_df.to_csv(val_path, index=False)

# Save split record
record = {
    "milestone": "M5",
    "split_status": "LOCKED",
    "split_method": "Orthomosaic-grouped",
    "train_images": len(best_m5_train_df),
    "validation_images": len(best_m5_val_df),
    "sealed_M4_test_images": len(m5_sealed_test_df),
    "train_orthomosaics": len(group_set(best_m5_train_df)),
    "validation_orthomosaics": len(group_set(best_m5_val_df)),
    "shared_orthomosaic_groups": 0,
    "asparagopsis_train_images": 22,
    "asparagopsis_validation_images": 0,
    "limitations": [
        "No Asparagopsis validation images",
        "Only 18 Codium validation images",
        "Only four validation orthomosaics",
        "Some flights occur in both training and validation",
        "Does not establish unseen-flight or unseen-location generalization"
    ],
    "M4_test_used_for_M5": False,
    "created_at": timestamp
}

with open(audit_path, "w") as f:
    json.dump(record, f, indent=4)

# Verify saved files
assert pd.read_csv(train_path).shape[0] == 2345
assert pd.read_csv(val_path).shape[0] == 432

print("M5 SPLIT LOCKED AND VERIFIED")
print("Training manifest:", train_path)
print("Validation manifest:", val_path)
print("Audit record:", audit_path)
print("Training images:", len(best_m5_train_df))
print("Validation images:", len(best_m5_val_df))
print("M4 test remains sealed:", len(m5_sealed_test_df) == 422)

In [ ]:
import os
import zipfile
import pandas as pd
from collections import Counter

# Locked M5 manifests
m5_train_path = (
    "/content/drive/MyDrive/Marine_Project/results/"
    "M5_train_manifest_20261122_152903.csv"
)

m5_val_path = (
    "/content/drive/MyDrive/Marine_Project/results/"
    "M5_validation_manifest_20261122_152903.csv"
)

# Existing selected-image archive
m5_image_zip = (
    "/content/drive/MyDrive/Marine_Project/"
    "IMO_M4_selected_images_3199_20261121_181622.zip"
)

assert os.path.exists(m5_train_path)
assert os.path.exists(m5_val_path)
assert os.path.exists(m5_image_zip)

m5_train_df = pd.read_csv(m5_train_path)
m5_val_df = pd.read_csv(m5_val_path)

# Six species labels
m5_species_to_id = {
    "Asparagopsis armata": 0,
    "Bifurcaria bifurcata": 1,
    "Codium spp.": 2,
    "Ericaria selaginoides": 3,
    "Saccorhiza polyschides": 4,
    "Sargassum muticum": 5
}

# Region-specific ecological mapping
m5_species_status = {
    "Asparagopsis armata": "Invasive",
    "Bifurcaria bifurcata": "Native",
    "Codium spp.": "Native",
    "Ericaria selaginoides": "Native",
    "Saccorhiza polyschides": "Native",
    "Sargassum muticum": "Invasive"
}

# Validate manifest labels
assert set(m5_train_df["Especies"]) == set(m5_species_to_id)
assert set(m5_val_df["Especies"]).issubset(set(m5_species_to_id))

assert all(
    m5_species_status[s] == status
    for s, status in zip(
        m5_train_df["Especies"],
        m5_train_df["Ecological_Status"]
    )
)

assert all(
    m5_species_status[s] == status
    for s, status in zip(
        m5_val_df["Especies"],
        m5_val_df["Ecological_Status"]
    )
)

# Build ZIP filename lookup
with zipfile.ZipFile(m5_image_zip, "r") as zf:
    image_members = [
        name for name in zf.namelist()
        if name.lower().endswith(".png")
    ]

    filename_lookup = {}
    duplicate_names = []

    for member in image_members:
        filename = os.path.basename(member)

        if filename in filename_lookup:
            duplicate_names.append(filename)
        else:
            filename_lookup[filename] = member

    train_missing = [
        f for f in m5_train_df["Filename"]
        if f not in filename_lookup
    ]

    val_missing = [
        f for f in m5_val_df["Filename"]
        if f not in filename_lookup
    ]

print("=" * 65)
print("M5 DATA PIPELINE VERIFICATION")
print("=" * 65)

print("Training manifest images:", len(m5_train_df))
print("Validation manifest images:", len(m5_val_df))
print("Images in selected ZIP:", len(image_members))
print("Duplicate ZIP filenames:", len(duplicate_names))
print("Missing training images:", len(train_missing))
print("Missing validation images:", len(val_missing))

print("\nSPECIES ID MAPPING:")
for species, class_id in m5_species_to_id.items():
    print(f"{class_id}: {species} -> {m5_species_status[species]}")

print("\nTRAINING SPECIES COUNTS:")
print(m5_train_df["Especies"].value_counts().sort_index().to_string())

print("\nVALIDATION SPECIES COUNTS:")
print(m5_val_df["Especies"].value_counts().sort_index().to_string())

assert len(duplicate_names) == 0
assert len(train_missing) == 0
assert len(val_missing) == 0

print("\nRESULT: ALL CHECKS PASSED")
print("M4 independent test remains excluded from M5.")

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import models
import os

# Reuse verified ImageNet weights from M4
m5_weights_path = (
    "/content/drive/MyDrive/Marine_Project/"
    "models/resnet18_imagenet1k_v1.pth"
)

assert os.path.exists(m5_weights_path), "ImageNet weights not found."

# CBAM channel attention
class ChannelAttention(nn.Module):
    def __init__(self, channels, reduction=16):
        super().__init__()

        self.shared_mlp = nn.Sequential(
            nn.Linear(channels, channels // reduction),
            nn.ReLU(inplace=True),
            nn.Linear(channels // reduction, channels)
        )

    def forward(self, x):
        avg_pool = F.adaptive_avg_pool2d(x, 1).flatten(1)
        max_pool = F.adaptive_max_pool2d(x, 1).flatten(1)

        attention = (
            self.shared_mlp(avg_pool) +
            self.shared_mlp(max_pool)
        )

        attention = torch.sigmoid(attention).unsqueeze(-1).unsqueeze(-1)

        return x * attention


# CBAM spatial attention
class SpatialAttention(nn.Module):
    def __init__(self, kernel_size=7):
        super().__init__()

        self.conv = nn.Conv2d(
            2, 1,
            kernel_size=kernel_size,
            padding=kernel_size // 2,
            bias=False
        )

    def forward(self, x):
        avg_map = torch.mean(x, dim=1, keepdim=True)
        max_map = torch.max(x, dim=1, keepdim=True)[0]

        attention = torch.sigmoid(
            self.conv(torch.cat([avg_map, max_map], dim=1))
        )

        return x * attention


# Complete CBAM block
class CBAM(nn.Module):
    def __init__(self, channels):
        super().__init__()

        self.channel_attention = ChannelAttention(channels)
        self.spatial_attention = SpatialAttention()

    def forward(self, x):
        x = self.channel_attention(x)
        x = self.spatial_attention(x)
        return x


# ResNet18 + CBAM + metric-learning embedding
class M5HybridModel(nn.Module):
    def __init__(self, embedding_dim=256, num_species=6):
        super().__init__()

        backbone = models.resnet18(weights=None)

        state = torch.load(
            m5_weights_path,
            map_location="cpu",
            weights_only=True
        )

        backbone.load_state_dict(state, strict=True)

        self.backbone = nn.Sequential(
            backbone.conv1,
            backbone.bn1,
            backbone.relu,
            backbone.maxpool,
            backbone.layer1,
            backbone.layer2,
            backbone.layer3,
            backbone.layer4
        )

        self.cbam = CBAM(512)
        self.pool = nn.AdaptiveAvgPool2d(1)

        self.embedding_head = nn.Sequential(
            nn.Linear(512, 512),
            nn.ReLU(inplace=True),
            nn.Dropout(0.3),
            nn.Linear(512, embedding_dim)
        )

        self.classifier = nn.Linear(embedding_dim, num_species)

    def forward(self, x):
        features = self.backbone(x)
        attended_features = self.cbam(features)

        pooled = self.pool(attended_features).flatten(1)

        embedding = self.embedding_head(pooled)

        # L2-normalized embedding for cosine similarity
        normalized_embedding = F.normalize(embedding, p=2, dim=1)

        logits = self.classifier(normalized_embedding)

        return {
            "embedding": normalized_embedding,
            "logits": logits
        }


# Instantiate model
m5_device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

m5_model = M5HybridModel(
    embedding_dim=256,
    num_species=6
).to(m5_device)

# Architecture safety test
m5_model.eval()

with torch.no_grad():
    dummy_input = torch.randn(4, 3, 224, 224).to(m5_device)
    output = m5_model(dummy_input)

print("M5 HYBRID ARCHITECTURE TEST")
print("=" * 55)
print("Device:", m5_device)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")
print("Input shape:", tuple(dummy_input.shape))
print("Embedding shape:", tuple(output["embedding"].shape))
print("Classifier output shape:", tuple(output["logits"].shape))
print("Embedding normalized:",
      torch.allclose(
          output["embedding"].norm(dim=1),
          torch.ones(4, device=m5_device),
          atol=1e-5
      ))
print("Total parameters:", sum(p.numel() for p in m5_model.parameters()))
print("Trainable parameters:", sum(p.numel() for p in m5_model.parameters() if p.requires_grad))

assert output["embedding"].shape == (4, 256)
assert output["logits"].shape == (4, 6)
assert torch.isfinite(output["embedding"]).all()
assert torch.isfinite(output["logits"]).all()

print("\nRESULT: ARCHITECTURE TEST PASSED")
print("No training performed.")

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

# Classification loss
m5_classification_loss = nn.CrossEntropyLoss()

# Batch-hard triplet loss
def m5_batch_hard_triplet_loss(embeddings, labels, margin=0.3):
    """
    Select hardest positive and hardest negative
    for each sample in the batch.
    """
    distances = torch.cdist(embeddings, embeddings, p=2)

    labels = labels.view(-1, 1)
    same_class = labels.eq(labels.T)

    # Exclude self-comparisons from positive pairs
    identity = torch.eye(
        len(labels),
        dtype=torch.bool,
        device=labels.device
    )

    positive_mask = same_class & ~identity
    negative_mask = ~same_class

    hardest_positive = distances.masked_fill(
        ~positive_mask, float("-inf")
    ).max(dim=1)[0]

    hardest_negative = distances.masked_fill(
        ~negative_mask, float("inf")
    ).min(dim=1)[0]

    valid = positive_mask.any(dim=1) & negative_mask.any(dim=1)

    if not valid.any():
        return embeddings.sum() * 0.0

    losses = F.relu(
        hardest_positive[valid] -
        hardest_negative[valid] +
        margin
    )

    return losses.mean()


# Synthetic safety test
torch.manual_seed(42)

test_embeddings = F.normalize(
    torch.randn(12, 256),
    p=2,
    dim=1
)

# Three examples per species, four species represented
test_labels = torch.tensor([
    0, 0, 0,
    1, 1, 1,
    2, 2, 2,
    3, 3, 3
])

test_logits = torch.randn(12, 6)

classification_value = m5_classification_loss(
    test_logits, test_labels
)

triplet_value = m5_batch_hard_triplet_loss(
    test_embeddings, test_labels
)

combined_value = classification_value + 0.5 * triplet_value

print("M5 METRIC-LEARNING LOSS TEST")
print("=" * 50)
print("Classification loss:", round(classification_value.item(), 5))
print("Batch-hard triplet loss:", round(triplet_value.item(), 5))
print("Combined loss:", round(combined_value.item(), 5))
print("Embedding shape:", tuple(test_embeddings.shape))
print("Labels shape:", tuple(test_labels.shape))

assert torch.isfinite(classification_value)
assert torch.isfinite(triplet_value)
assert torch.isfinite(combined_value)

print("\nRESULT: LOSS FUNCTIONS PASSED")
print("Synthetic data only — no model training performed.")

In [ ]:
import numpy as np
import pandas as pd
from collections import Counter, defaultdict
from torch.utils.data import Sampler

class SpeciesBalancedBatchSampler(Sampler):
    def __init__(
        self,
        dataframe,
        species_column="Especies",
        samples_per_species=4,
        batches_per_epoch=None,
        seed=42
    ):
        self.dataframe = dataframe.reset_index(drop=True)
        self.samples_per_species = samples_per_species
        self.seed = seed
        self.epoch = 0

        self.species_indices = {
            species: np.where(
                self.dataframe[species_column].values == species
            )[0].tolist()
            for species in sorted(
                self.dataframe[species_column].unique()
            )
        }

        self.species = list(self.species_indices.keys())

        if batches_per_epoch is None:
            batches_per_epoch = int(
                np.ceil(len(self.dataframe) /
                        (len(self.species) * samples_per_species))
            )

        self.batches_per_epoch = batches_per_epoch

    def __len__(self):
        return self.batches_per_epoch

    def __iter__(self):
        rng = np.random.default_rng(self.seed + self.epoch)
        self.epoch += 1

        for _ in range(self.batches_per_epoch):
            batch = []

            for species in self.species:
                available = self.species_indices[species]

                chosen = rng.choice(
                    available,
                    size=self.samples_per_species,
                    replace=len(available) < self.samples_per_species
                )

                batch.extend(chosen.tolist())

            rng.shuffle(batch)
            yield batch


# Build sampler from locked training manifest
m5_batch_sampler = SpeciesBalancedBatchSampler(
    m5_train_df,
    samples_per_species=4,
    seed=42
)

# Inspect first five batches
print("M5 SPECIES-BALANCED SAMPLER TEST")
print("=" * 60)
print("Training images:", len(m5_train_df))
print("Species per batch:", len(m5_species_to_id))
print("Samples per species:", 4)
print("Batch size:", 24)
print("Batches per epoch:", len(m5_batch_sampler))

for batch_number, batch_indices in enumerate(m5_batch_sampler):
    batch_species = m5_train_df.iloc[batch_indices]["Especies"]
    counts = batch_species.value_counts()

    print(f"\nBatch {batch_number + 1}:")
    print(counts.sort_index().to_dict())

    assert len(batch_indices) == 24
    assert len(counts) == 6
    assert all(counts == 4)

    if batch_number == 4:
        break

print("\nRESULT: SAMPLER TEST PASSED")
print("Training data only; validation and M4 test untouched.")

In [ ]:
import io
import zipfile
import torch
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

# ImageNet normalization for pretrained ResNet18
m5_mean = [0.485, 0.456, 0.406]
m5_std = [0.229, 0.224, 0.225]

# Training transformations
m5_train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ToTensor(),
    transforms.Normalize(mean=m5_mean, std=m5_std)
])

# Validation transformations: no random augmentation
m5_val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=m5_mean, std=m5_std)
])


class M5ZipImageDataset(Dataset):
    def __init__(
        self,
        dataframe,
        zip_path,
        filename_lookup,
        species_to_id,
        transform=None
    ):
        self.dataframe = dataframe.reset_index(drop=True)
        self.zip_path = zip_path
        self.filename_lookup = filename_lookup
        self.species_to_id = species_to_id
        self.transform = transform
        self.zip_file = None

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, index):
        row = self.dataframe.iloc[index]

        filename = row["Filename"]
        member = self.filename_lookup[filename]

        if self.zip_file is None:
            self.zip_file = zipfile.ZipFile(
                self.zip_path, "r"
            )

        image_bytes = self.zip_file.read(member)

        with Image.open(io.BytesIO(image_bytes)) as image:
            image = image.convert("RGB")

            if self.transform:
                image = self.transform(image)

        label = self.species_to_id[row["Especies"]]

        return image, torch.tensor(label, dtype=torch.long)


# Create independent train and validation datasets
m5_train_dataset = M5ZipImageDataset(
    dataframe=m5_train_df,
    zip_path=m5_image_zip,
    filename_lookup=filename_lookup,
    species_to_id=m5_species_to_id,
    transform=m5_train_transform
)

m5_val_dataset = M5ZipImageDataset(
    dataframe=m5_val_df,
    zip_path=m5_image_zip,
    filename_lookup=filename_lookup,
    species_to_id=m5_species_to_id,
    transform=m5_val_transform
)

# Training uses species-balanced batch sampler
m5_train_loader = DataLoader(
    m5_train_dataset,
    batch_sampler=m5_batch_sampler,
    num_workers=0,
    pin_memory=torch.cuda.is_available()
)

# Validation uses natural, unbalanced distribution
m5_val_loader = DataLoader(
    m5_val_dataset,
    batch_size=32,
    shuffle=False,
    num_workers=0,
    pin_memory=torch.cuda.is_available()
)

# Test one real batch from each loader
m5_train_images, m5_train_labels = next(iter(m5_train_loader))
m5_val_images, m5_val_labels = next(iter(m5_val_loader))

print("M5 ZIP DATASET AND DATALOADER TEST")
print("=" * 60)

print("Training dataset:", len(m5_train_dataset))
print("Validation dataset:", len(m5_val_dataset))

print("\nTraining batch:")
print("Image shape:", tuple(m5_train_images.shape))
print("Label shape:", tuple(m5_train_labels.shape))
print("Species counts:", torch.bincount(m5_train_labels, minlength=6).tolist())

print("\nValidation batch:")
print("Image shape:", tuple(m5_val_images.shape))
print("Label shape:", tuple(m5_val_labels.shape))
print("Species counts:", torch.bincount(m5_val_labels, minlength=6).tolist())

print("\nTensor checks:")
print("Training finite:", torch.isfinite(m5_train_images).all().item())
print("Validation finite:", torch.isfinite(m5_val_images).all().item())
print("Training dtype:", m5_train_images.dtype)

assert m5_train_images.shape == (24, 3, 224, 224)
assert m5_val_images.shape[1:] == (3, 224, 224)
assert len(m5_train_labels.unique()) == 6
assert torch.isfinite(m5_train_images).all()
assert torch.isfinite(m5_val_images).all()

print("\nRESULT: DATA PIPELINE PASSED")
print("M4 independent test has not been accessed.")

In [ ]:
import torch
import torch.nn.functional as F

def m5_supervised_contrastive_loss(
    embeddings,
    labels,
    temperature=0.07
):
    embeddings = F.normalize(embeddings, p=2, dim=1)
    labels = labels.view(-1, 1)

    similarity = torch.matmul(embeddings, embeddings.T)
    similarity = similarity / temperature

    batch_size = embeddings.shape[0]

    self_mask = torch.eye(
        batch_size,
        dtype=torch.bool,
        device=embeddings.device
    )

    positive_mask = labels.eq(labels.T) & ~self_mask

    logits = similarity.masked_fill(self_mask, float("-inf"))
    log_prob = logits - torch.logsumexp(
        logits, dim=1, keepdim=True
    )

    positive_count = positive_mask.sum(dim=1)
    valid = positive_count > 0

    if not valid.any():
        return embeddings.sum() * 0.0

    positive_log_prob = (
        log_prob.masked_fill(~positive_mask, 0.0).sum(dim=1)
        / positive_count.clamp(min=1)
    )

    return -positive_log_prob[valid].mean()


# Synthetic safety test — no training
torch.manual_seed(42)

test_embeddings = F.normalize(
    torch.randn(24, 256),
    p=2,
    dim=1
)

test_labels = torch.tensor(
    [0]*4 + [1]*4 + [2]*4 +
    [3]*4 + [4]*4 + [5]*4
)

test_ce = F.cross_entropy(
    torch.randn(24, 6),
    test_labels
)

test_triplet = m5_batch_hard_triplet_loss(
    test_embeddings,
    test_labels
)

test_contrastive = m5_supervised_contrastive_loss(
    test_embeddings,
    test_labels
)

test_combined = (
    test_ce
    + 0.5 * test_triplet
    + 0.1 * test_contrastive
)

print("M5 CONTRASTIVE LOSS TEST")
print("=" * 50)
print(f"Classification loss: {test_ce.item():.5f}")
print(f"Triplet loss:        {test_triplet.item():.5f}")
print(f"Contrastive loss:    {test_contrastive.item():.5f}")
print(f"Combined loss:       {test_combined.item():.5f}")

assert torch.isfinite(test_combined)
assert test_contrastive.item() >= 0

print("\nRESULT: ALL THREE LOSS FUNCTIONS PASSED")
print("Synthetic data only — no model training performed.")

In [ ]:
import os
import json
import torch
import torch.nn as nn
from datetime import datetime

# Create a separate M5 experiment directory
m5_experiment_time = datetime.now().strftime("%Y%m%d_%H%M%S")

m5_experiment_dir = os.path.join(
    "/content/drive/MyDrive/Marine_Project/models",
    f"M5_Hybrid_CBAM_MetricLearning_{m5_experiment_time}"
)

os.makedirs(m5_experiment_dir, exist_ok=False)

# Training configuration
m5_learning_rate = 0.0001
m5_weight_decay = 0.0001
m5_epochs = 20

m5_triplet_weight = 0.5
m5_contrastive_weight = 0.1

m5_optimizer = torch.optim.AdamW(
    m5_model.parameters(),
    lr=m5_learning_rate,
    weight_decay=m5_weight_decay
)

m5_scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    m5_optimizer,
    mode="min",
    factor=0.5,
    patience=2
)

m5_ce_loss = nn.CrossEntropyLoss()

# Record the experimental configuration
m5_config = {
    "experiment": "M5 Hybrid CBAM Metric Learning",
    "architecture": "ResNet18 + CBAM + 256D embedding + 6-species classifier",
    "input_size": [3, 224, 224],
    "species_mapping": m5_species_to_id,
    "training_images": len(m5_train_df),
    "validation_images": len(m5_val_df),
    "batch_size": 24,
    "samples_per_species": 4,
    "epochs": m5_epochs,
    "optimizer": "AdamW",
    "learning_rate": m5_learning_rate,
    "weight_decay": m5_weight_decay,
    "loss": "CrossEntropy + 0.5*Triplet + 0.1*SupervisedContrastive",
    "validation_selection": "Validation loss",
    "m4_independent_test_accessed": False
}

m5_config_path = os.path.join(
    m5_experiment_dir,
    "experiment_config.json"
)

with open(m5_config_path, "w") as f:
    json.dump(m5_config, f, indent=4)

print("M5 TRAINING CONFIGURATION")
print("=" * 55)
print("Experiment directory:", m5_experiment_dir)
print("Device:", m5_device)
print("Epochs:", m5_epochs)
print("Learning rate:", m5_learning_rate)
print("Optimizer: AdamW")
print("Scheduler: ReduceLROnPlateau")
print("Loss: CE + Triplet + Supervised Contrastive")
print("Training images:", len(m5_train_df))
print("Validation images:", len(m5_val_df))
print("Configuration saved:", os.path.exists(m5_config_path))

print("\nM4 checkpoint remains separate.")
print("RESULT: CONFIGURATION PREPARED")
print("No model training performed.")

In [ ]:
import os
import torch
import pandas as pd

print("M5 FINAL TRAINING SAFETY CHECK")
print("=" * 60)

# 1. Model and device
assert m5_device.type == "cuda"
assert next(m5_model.parameters()).device.type == "cuda"

print("GPU:", torch.cuda.get_device_name(0))
print("Model parameters:", sum(p.numel() for p in m5_model.parameters()))

# 2. Model output verification
m5_model.eval()

with torch.no_grad():
    safety_output = m5_model(
        m5_train_images[:4].to(m5_device)
    )

assert safety_output["embedding"].shape == (4, 256)
assert safety_output["logits"].shape == (4, 6)
assert torch.isfinite(safety_output["embedding"]).all()
assert torch.isfinite(safety_output["logits"]).all()

print("Embedding output:", tuple(safety_output["embedding"].shape))
print("Classifier output:", tuple(safety_output["logits"].shape))

# 3. Loss functions
assert callable(m5_batch_hard_triplet_loss)
assert callable(m5_supervised_contrastive_loss)
assert isinstance(m5_ce_loss, torch.nn.CrossEntropyLoss)

print("Classification loss: ready")
print("Triplet loss: ready")
print("Supervised contrastive loss: ready")

# 4. Optimizer and scheduler
assert isinstance(m5_optimizer, torch.optim.AdamW)
assert isinstance(
    m5_scheduler,
    torch.optim.lr_scheduler.ReduceLROnPlateau
)

print("Optimizer and scheduler: ready")

# 5. Dataset and sampler
assert len(m5_train_dataset) == 2345
assert len(m5_val_dataset) == 432
assert len(m5_batch_sampler) == 98

print("Training images:", len(m5_train_dataset))
print("Validation images:", len(m5_val_dataset))
print("Training batches per epoch:", len(m5_batch_sampler))

# 6. Verify train/validation filenames do not overlap
train_names = set(m5_train_df["Filename"])
val_names = set(m5_val_df["Filename"])

assert len(train_names & val_names) == 0

print("Train-validation filename overlap:", len(train_names & val_names))

# 7. Verify experiment directory is separate
assert os.path.isdir(m5_experiment_dir)
assert os.path.isfile(m5_config_path)

print("M5 experiment directory: verified")

# 8. Verify locked M4 test manifest is not part of M5 data
m4_test_names = set(
    pd.read_csv(
        "/content/drive/MyDrive/Marine_Project/results/"
        "IMO_M4_independent_test_manifest_20261119_184551.csv"
    )["Filename"]
)

assert len(train_names & m4_test_names) == 0
assert len(val_names & m4_test_names) == 0

print("M5 training / M4 test overlap:", len(train_names & m4_test_names))
print("M5 validation / M4 test overlap:", len(val_names & m4_test_names))

# Restore training mode after the check
m5_model.train()

print("\nRESULT: FINAL SAFETY CHECK PASSED")
print("No training performed in this cell.")

In [ ]:
import os
import time
import torch
import pandas as pd
import numpy as np
from sklearn.metrics import accuracy_score, f1_score

# Confirm checkpoint destinations are new
m5_best_path = os.path.join(m5_experiment_dir, "best_model.pth")
m5_latest_path = os.path.join(m5_experiment_dir, "latest_model.pth")
m5_history_path = os.path.join(m5_experiment_dir, "training_history.csv")

assert not os.path.exists(m5_best_path)
assert not os.path.exists(m5_latest_path)
assert not os.path.exists(m5_history_path)

m5_history = []
m5_best_val_loss = float("inf")
m5_best_epoch = None

print("M5 HYBRID TRAINING STARTED")
print("=" * 65)

for epoch in range(1, m5_epochs + 1):
    epoch_start = time.time()

    # ---------------- TRAINING ----------------
    m5_model.train()

    train_loss_total = 0.0
    train_ce_total = 0.0
    train_triplet_total = 0.0
    train_contrastive_total = 0.0

    train_predictions = []
    train_targets = []

    for images, labels in m5_train_loader:
        images = images.to(m5_device, non_blocking=True)
        labels = labels.to(m5_device, non_blocking=True)

        m5_optimizer.zero_grad(set_to_none=True)

        outputs = m5_model(images)

        ce_loss = m5_ce_loss(outputs["logits"], labels)

        triplet_loss = m5_batch_hard_triplet_loss(
            outputs["embedding"], labels
        )

        contrastive_loss = m5_supervised_contrastive_loss(
            outputs["embedding"], labels
        )

        loss = (
            ce_loss
            + m5_triplet_weight * triplet_loss
            + m5_contrastive_weight * contrastive_loss
        )

        if not torch.isfinite(loss):
            raise RuntimeError(
                f"Non-finite training loss at epoch {epoch}"
            )

        loss.backward()

        torch.nn.utils.clip_grad_norm_(
            m5_model.parameters(), max_norm=1.0
        )

        m5_optimizer.step()

        train_loss_total += loss.item()
        train_ce_total += ce_loss.item()
        train_triplet_total += triplet_loss.item()
        train_contrastive_total += contrastive_loss.item()

        train_predictions.extend(
            outputs["logits"].argmax(dim=1).detach().cpu().tolist()
        )
        train_targets.extend(labels.detach().cpu().tolist())

    number_train_batches = len(m5_train_loader)

    train_loss = train_loss_total / number_train_batches
    train_ce = train_ce_total / number_train_batches
    train_triplet = train_triplet_total / number_train_batches
    train_contrastive = train_contrastive_total / number_train_batches

    train_accuracy = accuracy_score(
        train_targets, train_predictions
    )

    # ---------------- VALIDATION ----------------
    m5_model.eval()

    val_loss_total = 0.0
    val_predictions = []
    val_targets = []

    with torch.no_grad():
        for images, labels in m5_val_loader:
            images = images.to(m5_device, non_blocking=True)
            labels = labels.to(m5_device, non_blocking=True)

            outputs = m5_model(images)
            loss = m5_ce_loss(outputs["logits"], labels)

            val_loss_total += loss.item() * images.size(0)

            val_predictions.extend(
                outputs["logits"].argmax(dim=1).cpu().tolist()
            )
            val_targets.extend(labels.cpu().tolist())

    val_loss = val_loss_total / len(m5_val_dataset)

    val_accuracy = accuracy_score(
        val_targets, val_predictions
    )

    val_macro_f1 = f1_score(
        val_targets,
        val_predictions,
        labels=list(range(6)),
        average="macro",
        zero_division=0
    )

    # Scheduler uses validation classification loss
    m5_scheduler.step(val_loss)

    current_lr = m5_optimizer.param_groups[0]["lr"]
    epoch_seconds = time.time() - epoch_start

    record = {
        "epoch": epoch,
        "train_combined_loss": train_loss,
        "train_ce_loss": train_ce,
        "train_triplet_loss": train_triplet,
        "train_contrastive_loss": train_contrastive,
        "train_accuracy": train_accuracy,
        "validation_ce_loss": val_loss,
        "validation_accuracy": val_accuracy,
        "validation_macro_f1": val_macro_f1,
        "learning_rate": current_lr,
        "epoch_seconds": epoch_seconds
    }

    m5_history.append(record)

    # Save latest checkpoint every epoch
    checkpoint = {
        "epoch": epoch,
        "model_state_dict": m5_model.state_dict(),
        "optimizer_state_dict": m5_optimizer.state_dict(),
        "scheduler_state_dict": m5_scheduler.state_dict(),
        "validation_loss": val_loss,
        "species_mapping": m5_species_to_id
    }

    torch.save(checkpoint, m5_latest_path)

    # Select best model using validation loss only
    if val_loss < m5_best_val_loss:
        m5_best_val_loss = val_loss
        m5_best_epoch = epoch

        torch.save(checkpoint, m5_best_path)

        best_status = "BEST CHECKPOINT SAVED"
    else:
        best_status = ""

    # Persist training history after each epoch
    pd.DataFrame(m5_history).to_csv(
        m5_history_path, index=False
    )

    print(f"\nEpoch {epoch:02d}/{m5_epochs}")
    print(f"Train combined loss : {train_loss:.4f}")
    print(f"  CE / Triplet / SupCon: "
          f"{train_ce:.4f} / {train_triplet:.4f} / "
          f"{train_contrastive:.4f}")
    print(f"Train accuracy      : {train_accuracy:.4f}")
    print(f"Validation CE loss  : {val_loss:.4f}")
    print(f"Validation accuracy: {val_accuracy:.4f}")
    print(f"Validation macro F1 : {val_macro_f1:.4f}")
    print(f"Learning rate       : {current_lr:.7f}")
    print(f"Duration            : {epoch_seconds:.1f} seconds")
    if best_status:
        print(best_status)

    # Stop if learning rate becomes extremely small
    if current_lr < 1e-7:
        print("\nStopping: learning rate reached minimum threshold.")
        break

print("\n" + "=" * 65)
print("M5 TRAINING FINISHED")
print("Epochs completed:", len(m5_history))
print("Best epoch:", m5_best_epoch)
print("Best validation loss:", round(m5_best_val_loss, 5))
print("Best checkpoint exists:", os.path.exists(m5_best_path))
print("Latest checkpoint exists:", os.path.exists(m5_latest_path))
print("Training history exists:", os.path.exists(m5_history_path))
print("M4 independent test was not accessed.")

In [ ]:
import os
import json
import torch
import pandas as pd
import numpy as np
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    accuracy_score,
    f1_score
)

# Load the best validation-selected checkpoint
m5_checkpoint = torch.load(
    m5_best_path,
    map_location=m5_device,
    weights_only=False
)

m5_model.load_state_dict(
    m5_checkpoint["model_state_dict"],
    strict=True
)

m5_model.eval()

m5_true_ids = []
m5_pred_ids = []
m5_probabilities = []
m5_filenames = []

# Validation evaluation only
with torch.no_grad():
    for batch_number, (images, labels) in enumerate(m5_val_loader):
        images = images.to(m5_device)

        outputs = m5_model(images)

        probabilities = torch.softmax(
            outputs["logits"], dim=1
        )

        predictions = probabilities.argmax(dim=1)

        start = batch_number * m5_val_loader.batch_size
        batch_filenames = m5_val_df.iloc[
            start:start + len(labels)
        ]["Filename"].tolist()

        m5_true_ids.extend(labels.tolist())
        m5_pred_ids.extend(predictions.cpu().tolist())
        m5_probabilities.extend(
            probabilities.cpu().numpy().tolist()
        )
        m5_filenames.extend(batch_filenames)

# Species mapping
m5_id_to_species = {
    value: key for key, value in m5_species_to_id.items()
}

m5_species_names = [
    m5_id_to_species[i] for i in range(6)
]

# Species-level metrics
m5_species_report = classification_report(
    m5_true_ids,
    m5_pred_ids,
    labels=list(range(6)),
    target_names=m5_species_names,
    output_dict=True,
    zero_division=0
)

m5_species_cm = confusion_matrix(
    m5_true_ids,
    m5_pred_ids,
    labels=list(range(6))
)

# Derived ecological classification
m5_true_ecology = [
    m5_species_status[m5_id_to_species[i]]
    for i in m5_true_ids
]

m5_pred_ecology = [
    m5_species_status[m5_id_to_species[i]]
    for i in m5_pred_ids
]

m5_ecology_report = classification_report(
    m5_true_ecology,
    m5_pred_ecology,
    labels=["Native", "Invasive"],
    output_dict=True,
    zero_division=0
)

m5_ecology_cm = confusion_matrix(
    m5_true_ecology,
    m5_pred_ecology,
    labels=["Native", "Invasive"]
)

# Save detailed validation predictions
m5_prediction_df = pd.DataFrame({
    "Filename": m5_filenames,
    "True_Species": [
        m5_id_to_species[i] for i in m5_true_ids
    ],
    "Predicted_Species": [
        m5_id_to_species[i] for i in m5_pred_ids
    ],
    "True_Ecological_Status": m5_true_ecology,
    "Predicted_Ecological_Status": m5_pred_ecology,
    "Correct_Species": [
        a == b for a, b in zip(m5_true_ids, m5_pred_ids)
    ],
    "Correct_Ecological_Status": [
        a == b for a, b in zip(
            m5_true_ecology, m5_pred_ecology
        )
    ]
})

for class_id, species in m5_id_to_species.items():
    m5_prediction_df[
        f"Probability_{species.replace(' ', '_')}"
    ] = [
        row[class_id] for row in m5_probabilities
    ]

m5_eval_dir = os.path.join(
    m5_experiment_dir, "validation_evaluation"
)
os.makedirs(m5_eval_dir, exist_ok=False)

m5_prediction_df.to_csv(
    os.path.join(m5_eval_dir, "validation_predictions.csv"),
    index=False
)

m5_eval_results = {
    "checkpoint_epoch": int(m5_checkpoint["epoch"]),
    "validation_images": len(m5_true_ids),
    "species_accuracy": accuracy_score(
        m5_true_ids, m5_pred_ids
    ),
    "species_macro_f1": f1_score(
        m5_true_ids,
        m5_pred_ids,
        labels=list(range(6)),
        average="macro",
        zero_division=0
    ),
    "species_confusion_matrix": m5_species_cm.tolist(),
    "species_classification_report": m5_species_report,
    "ecological_accuracy": accuracy_score(
        m5_true_ecology, m5_pred_ecology
    ),
    "ecological_macro_f1": f1_score(
        m5_true_ecology,
        m5_pred_ecology,
        labels=["Native", "Invasive"],
        average="macro",
        zero_division=0
    ),
    "ecological_confusion_matrix": m5_ecology_cm.tolist(),
    "ecological_classification_report": m5_ecology_report,
    "asparagopsis_validation_support": int(
        m5_true_ids.count(m5_species_to_id["Asparagopsis armata"])
    ),
    "m4_test_accessed": False
}

with open(
    os.path.join(m5_eval_dir, "validation_metrics.json"), "w"
) as f:
    json.dump(m5_eval_results, f, indent=4)

print("M5 VALIDATION EVALUATION")
print("=" * 60)
print("Checkpoint epoch:", m5_checkpoint["epoch"])
print("Validation images:", len(m5_true_ids))

print("\nSPECIES CLASSIFICATION")
print("Accuracy:", round(m5_eval_results["species_accuracy"], 4))
print("Macro F1:", round(m5_eval_results["species_macro_f1"], 4))
print("\nSpecies confusion matrix:")
print(m5_species_cm)
print("\nPer-species report:")
print(pd.DataFrame(m5_species_report).T.round(4))

print("\nECOLOGICAL CLASSIFICATION")
print("Accuracy:", round(m5_eval_results["ecological_accuracy"], 4))
print("Macro F1:", round(m5_eval_results["ecological_macro_f1"], 4))
print("\nEcological confusion matrix:")
print(m5_ecology_cm)
print("\nEcological report:")
print(pd.DataFrame(m5_ecology_report).T.round(4))

print("\nAsparagopsis validation support:",
      m5_eval_results["asparagopsis_validation_support"])

print("\nSaved predictions and metrics:", m5_eval_dir)
print("M4 independent test accessed: False")

In [ ]:
import os
import json
import shutil
import hashlib
from datetime import datetime

# Paths
m5_locked_path = os.path.join(
    m5_experiment_dir,
    "LOCKED_M5_best_checkpoint_epoch16.pth"
)

m5_lock_record_path = os.path.join(
    m5_experiment_dir,
    "M5_checkpoint_lock_record.json"
)

m5_final_summary_path = os.path.join(
    m5_experiment_dir,
    "M5_VALIDATION_SUMMARY.json"
)

# Do not overwrite existing artifacts
assert os.path.isfile(m5_best_path)
assert not os.path.exists(m5_locked_path)
assert not os.path.exists(m5_lock_record_path)
assert not os.path.exists(m5_final_summary_path)

# Copy best checkpoint
shutil.copy2(m5_best_path, m5_locked_path)

def m5_sha256(path):
    digest = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

original_hash = m5_sha256(m5_best_path)
locked_hash = m5_sha256(m5_locked_path)

assert original_hash == locked_hash

# Checkpoint lock record
lock_record = {
    "checkpoint": os.path.basename(m5_locked_path),
    "selected_epoch": int(m5_checkpoint["epoch"]),
    "selection_criterion": "Lowest validation classification loss",
    "sha256": locked_hash,
    "locked_at": datetime.now().isoformat(),
    "m4_test_accessed": False
}

with open(m5_lock_record_path, "w") as f:
    json.dump(lock_record, f, indent=4)

# Final validation summary
summary = {
    "project_stage": "M5",
    "model": "ResNet18 + CBAM + Metric Learning",
    "training_epochs": len(m5_history),
    "best_epoch": int(m5_checkpoint["epoch"]),
    "best_validation_loss": float(m5_best_val_loss),
    "validation_images": len(m5_true_ids),
    "species_accuracy": float(m5_eval_results["species_accuracy"]),
    "species_macro_f1": float(m5_eval_results["species_macro_f1"]),
    "ecological_accuracy": float(m5_eval_results["ecological_accuracy"]),
    "ecological_macro_f1": float(m5_eval_results["ecological_macro_f1"]),
    "asparagopsis_validation_support": 0,
    "limitations": [
        "No Asparagopsis images in validation",
        "Validation data is limited to selected orthomosaic groups",
        "Training-validation generalization gap is present",
        "Invasive recall is lower than native recall",
        "IMO imagery is intertidal imagery, not underwater removal imagery",
        "Independent external M5 test has not been performed"
    ],
    "m4_independent_test_accessed": False
}

with open(m5_final_summary_path, "w") as f:
    json.dump(summary, f, indent=4)

print("M5 CHECKPOINT LOCK")
print("=" * 55)
print("Selected epoch:", lock_record["selected_epoch"])
print("Original SHA256:", original_hash)
print("Locked SHA256:", locked_hash)
print("Hashes match:", original_hash == locked_hash)
print("Lock record saved:", os.path.exists(m5_lock_record_path))
print("Final summary saved:", os.path.exists(m5_final_summary_path))
print("M4 independent test accessed: False")

print("\nRESULT: M5 CHECKPOINT PRESERVED")

In [ ]:
print("M5 EMBEDDING ANALYSIS — ENVIRONMENT CHECK")
print("=" * 55)

required_objects = [
    "m5_model",
    "m5_train_dataset",
    "m5_val_dataset",
    "m5_train_loader",
    "m5_val_loader",
    "m5_device",
    "m5_experiment_dir",
    "m5_locked_path"
]

for name in required_objects:
    exists = name in globals()
    print(f"{name}: {'AVAILABLE' if exists else 'NOT FOUND'}")

if "m5_train_dataset" in globals():
    print("\nTRAIN DATASET")
    print("Type:", type(m5_train_dataset).__name__)
    print("Length:", len(m5_train_dataset))
    print("Attributes:", [
        a for a in vars(m5_train_dataset).keys()
        if not a.startswith("_")
    ])

if "m5_val_dataset" in globals():
    print("\nVALIDATION DATASET")
    print("Type:", type(m5_val_dataset).__name__)
    print("Length:", len(m5_val_dataset))
    print("Attributes:", [
        a for a in vars(m5_val_dataset).keys()
        if not a.startswith("_")
    ])

print("\nNo model training, checkpoint modification, or test-set access performed.")

In [ ]:
import torch
import inspect

print("M5 CHECKPOINT AND DATASET INSPECTION")
print("=" * 55)

# Inspect locked checkpoint without changing the model
checkpoint = torch.load(
    m5_locked_path,
    map_location="cpu",
    weights_only=False
)

print("\nCHECKPOINT")
print("Type:", type(checkpoint).__name__)
if isinstance(checkpoint, dict):
    print("Keys:", list(checkpoint.keys()))
    for key, value in checkpoint.items():
        if isinstance(value, dict):
            print(f"{key}: dictionary with {len(value)} entries")
        elif torch.is_tensor(value):
            print(f"{key}: tensor {tuple(value.shape)}")

# Inspect dataset output
print("\nTRAINING SAMPLE")
sample = m5_train_dataset[0]
print("Output type:", type(sample).__name__)

if isinstance(sample, (tuple, list)):
    print("Number of items:", len(sample))
    for i, item in enumerate(sample):
        if torch.is_tensor(item):
            print(f"Item {i}: tensor, shape={tuple(item.shape)}, dtype={item.dtype}")
        else:
            print(f"Item {i}: {type(item).__name__}, value={item}")

# Inspect deterministic validation transform
print("\nVALIDATION TRANSFORM")
print(m5_val_dataset.transform)

# Inspect dataset implementation
print("\nDATASET CLASS")
print(inspect.getsource(type(m5_train_dataset).__getitem__))

print("\nInspection only — no training, checkpoint changes, or test-set access.")

In [ ]:
import inspect

print("M5 EMBEDDING PREPARATION")
print("=" * 55)

print("\nLOCKED CHECKPOINT")
print("Epoch:", checkpoint["epoch"])
print("Validation loss:", checkpoint["validation_loss"])
print("Species mapping:")
for species, class_id in checkpoint["species_mapping"].items():
    print(f"  {class_id}: {species}")

print("\nDATASET CONSTRUCTOR")
print(inspect.signature(type(m5_train_dataset).__init__))

print("\nTRAINING TRANSFORM")
print(m5_train_dataset.transform)

print("\nVALIDATION TRANSFORM")
print(m5_val_dataset.transform)

print("\nTRAINING MANIFEST COLUMNS")
print(list(m5_train_dataset.dataframe.columns))

print("\nVALIDATION MANIFEST COLUMNS")
print(list(m5_val_dataset.dataframe.columns))

print("\nNo model weights or dataset files modified.")

In [ ]:
import os
import numpy as np
import torch
from torch.utils.data import DataLoader
from datetime import datetime

print("M5 EMBEDDING EXTRACTION")
print("=" * 55)

# Load the exact locked Epoch 16 weights into the existing model
m5_model.load_state_dict(
    checkpoint["model_state_dict"],
    strict=True
)
m5_model.to(m5_device)
m5_model.eval()

# Recreate training dataset with deterministic validation transforms
m5_train_eval_dataset = type(m5_train_dataset)(
    dataframe=m5_train_dataset.dataframe.copy(),
    zip_path=m5_train_dataset.zip_path,
    filename_lookup=m5_train_dataset.filename_lookup,
    species_to_id=m5_train_dataset.species_to_id,
    transform=m5_val_dataset.transform
)

# Deterministic loaders; no multiprocessing with ZIP files
m5_train_eval_loader = DataLoader(
    m5_train_eval_dataset,
    batch_size=32,
    shuffle=False,
    num_workers=0
)

m5_val_eval_loader = DataLoader(
    m5_val_dataset,
    batch_size=32,
    shuffle=False,
    num_workers=0
)

def extract_m5_embeddings(loader, split_name):
    embeddings = []
    labels = []

    with torch.inference_mode():
        for images, targets in loader:
            images = images.to(m5_device)
            outputs = m5_model(images)

            batch_embeddings = outputs["embedding"]

            embeddings.append(batch_embeddings.cpu().numpy())
            labels.append(targets.numpy())

    embeddings = np.concatenate(embeddings, axis=0)
    labels = np.concatenate(labels, axis=0)

    assert embeddings.shape == (len(loader.dataset), 256)
    assert labels.shape[0] == len(loader.dataset)
    assert np.isfinite(embeddings).all()

    print(
        f"{split_name}: {len(labels)} images | "
        f"Embedding shape: {embeddings.shape} | "
        f"Finite: {np.isfinite(embeddings).all()}"
    )

    return embeddings, labels

m5_train_embeddings, m5_train_labels = extract_m5_embeddings(
    m5_train_eval_loader, "TRAIN"
)

m5_val_embeddings, m5_val_labels = extract_m5_embeddings(
    m5_val_eval_loader, "VALIDATION"
)

# Confirm L2-normalized embeddings
print("\nEMBEDDING NORMS")
print("Train mean norm:", np.linalg.norm(m5_train_embeddings, axis=1).mean())
print("Validation mean norm:", np.linalg.norm(m5_val_embeddings, axis=1).mean())

# Save a separate, timestamped analysis artifact
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
m5_embedding_path = os.path.join(
    m5_experiment_dir,
    f"M5_embeddings_train_validation_{timestamp}.npz"
)

np.savez_compressed(
    m5_embedding_path,
    train_embeddings=m5_train_embeddings,
    train_labels=m5_train_labels,
    validation_embeddings=m5_val_embeddings,
    validation_labels=m5_val_labels
)

print("\nSaved embedding artifact:", m5_embedding_path)
print("Locked checkpoint file modified: False")
print("M4 independent test accessed: False")

In [ ]:
import numpy as np
import os

embedding_path = os.path.join(
    m5_experiment_dir,
    "M5_embeddings_train_validation_20261122_155659.npz"
)

assert os.path.isfile(embedding_path), "Embedding artifact not found."

data = np.load(embedding_path, allow_pickle=True)

print("M5 EMBEDDING ARTIFACT INSPECTION")
print("=" * 55)

print("\nSaved arrays:")
for key in data.files:
    arr = data[key]
    print(f"{key}: shape={arr.shape}, dtype={arr.dtype}")

print("\nTraining species counts:")
print(m5_train_dataset.dataframe["Especies"].value_counts().to_string())

print("\nValidation species counts:")
print(m5_val_dataset.dataframe["Especies"].value_counts().to_string())

print("\nSpecies mapping:")
for species, class_id in sorted(
    m5_train_dataset.species_to_id.items(),
    key=lambda item: item[1]
):
    print(f"{class_id}: {species}")

print("\nArtifact integrity:")
for key in data.files:
    if np.issubdtype(data[key].dtype, np.number):
        print(f"{key}: finite={np.isfinite(data[key]).all()}")

print("\nInspection only. No training or test-set access.")

In [ ]:
import numpy as np
import pandas as pd
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    classification_report
)

# Load saved embeddings
train_embeddings = data["train_embeddings"]
train_labels = data["train_labels"]
val_embeddings = data["validation_embeddings"]
val_labels = data["validation_labels"]

species_mapping = m5_train_dataset.species_to_id
num_classes = len(species_mapping)

# Normalize embeddings safely
def l2_normalize(x):
    return x / np.maximum(
        np.linalg.norm(x, axis=1, keepdims=True), 1e-12
    )

train_embeddings = l2_normalize(train_embeddings)
val_embeddings = l2_normalize(val_embeddings)

# Build prototypes using training embeddings only
prototypes = []

for class_id in range(num_classes):
    class_embeddings = train_embeddings[train_labels == class_id]
    assert len(class_embeddings) > 0

    centroid = class_embeddings.mean(axis=0)
    centroid = centroid / max(np.linalg.norm(centroid), 1e-12)
    prototypes.append(centroid)

prototypes = np.stack(prototypes)

# Cosine similarity: validation images vs training prototypes
similarities = val_embeddings @ prototypes.T

predictions = similarities.argmax(axis=1)
sorted_similarities = np.sort(similarities, axis=1)

best_similarity = sorted_similarities[:, -1]
second_best_similarity = sorted_similarities[:, -2]
similarity_margin = best_similarity - second_best_similarity

print("M5 COSINE PROTOTYPE ANALYSIS")
print("=" * 60)

print("\nPROTOTYPE COUNTS")
for species, class_id in sorted(
    species_mapping.items(), key=lambda x: x[1]
):
    print(
        f"{species}: "
        f"{np.sum(train_labels == class_id)} training embeddings"
    )

print("\nVALIDATION NEAREST-PROTOTYPE RESULTS")
print("Accuracy:", round(accuracy_score(val_labels, predictions), 4))
print("\nConfusion matrix:")
print(confusion_matrix(
    val_labels,
    predictions,
    labels=list(range(num_classes))
))

print("\nClassification report:")
print(classification_report(
    val_labels,
    predictions,
    labels=list(range(num_classes)),
    target_names=[
        species for species, _ in sorted(
            species_mapping.items(), key=lambda x: x[1]
        )
    ],
    zero_division=0
))

print("\nSIMILARITY SUMMARY")
print("Mean best cosine similarity:", round(float(best_similarity.mean()), 4))
print("Mean second-best similarity:", round(float(second_best_similarity.mean()), 4))
print("Mean similarity margin:", round(float(similarity_margin.mean()), 4))

# Per-species validation summary
rows = []

for class_id in range(num_classes):
    mask = val_labels == class_id
    support = int(mask.sum())

    rows.append({
        "Species": next(
            species for species, idx in species_mapping.items()
            if idx == class_id
        ),
        "Validation_support": support,
        "Correct": int(np.sum(predictions[mask] == class_id)),
        "Accuracy": (
            round(float(np.mean(predictions[mask] == class_id)), 4)
            if support else None
        ),
        "Mean_best_similarity": (
            round(float(best_similarity[mask].mean()), 4)
            if support else None
        ),
        "Mean_margin": (
            round(float(similarity_margin[mask].mean()), 4)
            if support else None
        )
    })

prototype_results = pd.DataFrame(rows)
print("\nPER-SPECIES SUMMARY")
display(prototype_results)

print("\nM4 independent test accessed: False")
print("No training or checkpoint modifications performed.")

In [ ]:
import numpy as np
import pandas as pd

# Similarity of each validation image to its actual species prototype
actual_similarity = similarities[
    np.arange(len(val_labels)), val_labels
]

# Highest similarity to any incorrect species prototype
other_similarities = similarities.copy()
other_similarities[
    np.arange(len(val_labels)), val_labels
] = -np.inf

strongest_other_similarity = other_similarities.max(axis=1)

# Positive margin means the correct prototype is closer
actual_margin = actual_similarity - strongest_other_similarity

correct_mask = predictions == val_labels

print("M5 EMBEDDING SEPARATION DIAGNOSTICS")
print("=" * 60)

print("\nCORRECT VS INCORRECT PREDICTIONS")
for name, mask in [
    ("Correct", correct_mask),
    ("Incorrect", ~correct_mask)
]:
    print(f"\n{name}: {int(mask.sum())} images")
    if mask.any():
        print("Mean actual-class similarity:",
              round(float(actual_similarity[mask].mean()), 4))
        print("Mean strongest-other similarity:",
              round(float(strongest_other_similarity[mask].mean()), 4))
        print("Mean actual-class margin:",
              round(float(actual_margin[mask].mean()), 4))

print("\nPER-SPECIES SEPARATION")
rows = []

for class_id in range(num_classes):
    mask = val_labels == class_id
    support = int(mask.sum())

    rows.append({
        "Species": next(
            s for s, idx in species_mapping.items()
            if idx == class_id
        ),
        "Support": support,
        "Correct": int(np.sum(correct_mask[mask])),
        "Mean_actual_similarity": (
            round(float(actual_similarity[mask].mean()), 4)
            if support else None
        ),
        "Mean_other_similarity": (
            round(float(strongest_other_similarity[mask].mean()), 4)
            if support else None
        ),
        "Mean_actual_margin": (
            round(float(actual_margin[mask].mean()), 4)
            if support else None
        )
    })

display(pd.DataFrame(rows))

print("\nPAIRWISE SPECIES PROTOTYPE SIMILARITY")
prototype_similarity = prototypes @ prototypes.T

species_names = [
    species for species, idx in sorted(
        species_mapping.items(), key=lambda x: x[1]
    )
]

display(pd.DataFrame(
    prototype_similarity,
    index=species_names,
    columns=species_names
).round(4))

print("\nNo model changes, retraining, or M4 test access.")

In [ ]:
import pandas as pd
import numpy as np

species_names = [
    species for species, idx in sorted(
        species_mapping.items(), key=lambda x: x[1]
    )
]

# Build a detailed table of incorrect validation predictions
error_mask = predictions != val_labels
error_rows = []

for i in np.where(error_mask)[0]:
    actual_id = int(val_labels[i])
    predicted_id = int(predictions[i])

    error_rows.append({
        "Actual_species": species_names[actual_id],
        "Predicted_species": species_names[predicted_id],
        "Actual_similarity": float(similarities[i, actual_id]),
        "Predicted_similarity": float(similarities[i, predicted_id]),
        "Similarity_gap": float(
            similarities[i, predicted_id] -
            similarities[i, actual_id]
        ),
        "Margin": float(actual_margin[i])
    })

errors_df = pd.DataFrame(error_rows)

print("M5 VALIDATION ERROR ANALYSIS")
print("=" * 60)
print("Total validation errors:", len(errors_df))

print("\nACTUAL → PREDICTED SPECIES CONFUSIONS")
display(
    errors_df.groupby(
        ["Actual_species", "Predicted_species"]
    ).size().reset_index(name="Error_count")
    .sort_values("Error_count", ascending=False)
)

print("\nERROR COUNTS BY ACTUAL SPECIES")
display(
    errors_df.groupby("Actual_species")
    .size()
    .reset_index(name="Error_count")
    .sort_values("Error_count", ascending=False)
)

print("\nSIMILARITY OF INCORRECT PREDICTIONS")
display(
    errors_df.groupby("Actual_species")[
        ["Actual_similarity", "Predicted_similarity", "Similarity_gap"]
    ].mean().round(4)
)

print("\nM4 independent test accessed: False")
print("No model training or checkpoint modification.")

In [ ]:
import os
import json
import pandas as pd
import numpy as np
from datetime import datetime

# Create a new diagnostic folder without overwriting anything
diagnostic_time = datetime.now().strftime("%Y%m%d_%H%M%S")
diagnostic_dir = os.path.join(
    m5_experiment_dir,
    f"M5_embedding_diagnostics_{diagnostic_time}"
)

assert not os.path.exists(diagnostic_dir), "Folder already exists."
os.makedirs(diagnostic_dir)

species_names = [
    species for species, idx in sorted(
        species_mapping.items(), key=lambda x: x[1]
    )
]

# Per-image validation diagnostics
prediction_df = pd.DataFrame({
    "Actual_species": [species_names[int(x)] for x in val_labels],
    "Predicted_species": [species_names[int(x)] for x in predictions],
    "Correct": correct_mask,
    "Actual_similarity": actual_similarity,
    "Strongest_other_similarity": strongest_other_similarity,
    "Actual_class_margin": actual_margin,
    "Best_similarity": best_similarity,
    "Best_vs_second_margin": similarity_margin
})

prediction_df.to_csv(
    os.path.join(diagnostic_dir, "validation_embedding_predictions.csv"),
    index=False
)

# Species-level summary
prototype_results.to_csv(
    os.path.join(diagnostic_dir, "species_prototype_performance.csv"),
    index=False
)

# Confusion analysis
errors_df.to_csv(
    os.path.join(diagnostic_dir, "validation_species_confusions.csv"),
    index=False
)

# Prototype similarity matrix
pd.DataFrame(
    prototype_similarity,
    index=species_names,
    columns=species_names
).to_csv(
    os.path.join(diagnostic_dir, "prototype_similarity_matrix.csv")
)

# Summary record
summary = {
    "experiment": "M5 ResNet18-CBAM Metric Learning",
    "selected_checkpoint_epoch": 16,
    "checkpoint_sha256": m5_sha256(m5_locked_path),
    "training_images": int(len(train_labels)),
    "validation_images": int(len(val_labels)),
    "embedding_dimension": int(train_embeddings.shape[1]),
    "validation_nearest_prototype_accuracy": float(
        accuracy_score(val_labels, predictions)
    ),
    "validation_correct": int(correct_mask.sum()),
    "validation_incorrect": int((~correct_mask).sum()),
    "mean_best_cosine_similarity": float(best_similarity.mean()),
    "mean_actual_class_similarity_correct": float(
        actual_similarity[correct_mask].mean()
    ),
    "mean_actual_class_similarity_incorrect": float(
        actual_similarity[~correct_mask].mean()
    ),
    "largest_error_species": {
        "Sargassum muticum": 23,
        "Saccorhiza polyschides": 20
    },
    "limitations": [
        "Asparagopsis armata has zero validation images.",
        "Validation flights overlap between training and validation.",
        "The independent M4 test set was not used.",
        "Cosine similarity is not a calibrated confidence score.",
        "The IMO imagery is intertidal/UAV and in-situ imagery, not a validated underwater removal dataset.",
        "Nearest-prototype agreement with the classifier is not independent proof of metric-learning improvement."
    ],
    "test_set_accessed": False
}

summary_path = os.path.join(
    diagnostic_dir, "M5_embedding_diagnostic_summary.json"
)

with open(summary_path, "w") as f:
    json.dump(summary, f, indent=4)

print("M5 DIAGNOSTIC ARTIFACTS SAVED")
print("=" * 55)
print("Folder:", diagnostic_dir)

for filename in sorted(os.listdir(diagnostic_dir)):
    path = os.path.join(diagnostic_dir, filename)
    print(f"{filename}: {os.path.getsize(path):,} bytes")

print("\nCheckpoint modified: False")
print("M4 independent test accessed: False")

In [ ]:
from pathlib import Path
import hashlib
import json

PROJECT = Path("/content/drive/MyDrive/Marine_Project")

M5_DIR = PROJECT / "models/M5_Hybrid_CBAM_MetricLearning_20261122_153321"

CHECKPOINT = M5_DIR / "LOCKED_M5_best_checkpoint_epoch16.pth"

FILES_TO_CHECK = {
    "Locked M5 checkpoint": CHECKPOINT,
    "M5 training history": M5_DIR / "training_history.csv",
    "M5 experiment config": M5_DIR / "experiment_config.json",
    "M5 validation predictions": M5_DIR / "validation_evaluation/validation_predictions.csv",
    "M5 validation metrics": M5_DIR / "validation_evaluation/validation_metrics.json",
    "M5 embedding archive": M5_DIR / "M5_embeddings_train_validation_20261122_155659.npz",
    "M5 final summary": M5_DIR / "M5_FINAL_EXPERIMENT_SUMMARY.json",
}

print("=" * 60)
print("M6 — STEP 1: RESOURCE VERIFICATION")
print("=" * 60)

for name, path in FILES_TO_CHECK.items():
    print(f"{'FOUND' if path.exists() else 'MISSING'} | {name}")
    if path.exists():
        print(f"         {path}")

print("\n--- EXISTING NOTEBOOK VARIABLES ---")

for name in [
    "m5_model",
    "M5HybridModel",
    "CBAM",
    "m5_train_loader",
    "m5_val_loader",
    "device",
    "species_to_id",
]:
    print(f"{name}: {'AVAILABLE' if name in globals() else 'NOT IN MEMORY'}")

print("\n--- CHECKPOINT INTEGRITY ---")

if CHECKPOINT.exists():
    sha256 = hashlib.sha256(CHECKPOINT.read_bytes()).hexdigest()
    print("Checkpoint SHA256:", sha256)
    print("Expected SHA256:  c5d0b8c5c65b5d878e6d87a4ac7ddc96fbc80fcc06c67cbdead85008825c4c27")
    print("Hash verified:", sha256 == "c5d0b8c5c65b5d878e6d87a4ac7ddc96fbc80fcc06c67cbdead85008825c4c27")
else:
    print("Checkpoint unavailable. No model loading attempted.")

print("\nM6 Step 1 verification finished.")

In [ ]:
import torch
import numpy as np
import json
from pathlib import Path

M5_DIR = Path(
    "/content/drive/MyDrive/Marine_Project/models/"
    "M5_Hybrid_CBAM_MetricLearning_20261122_153321"
)

CHECKPOINT = M5_DIR / "LOCKED_M5_best_checkpoint_epoch16.pth"
EMBEDDING_FILE = M5_DIR / "M5_embeddings_train_validation_20261122_155659.npz"

print("=" * 60)
print("M6 — STEP 2: MODEL AND EMBEDDING INSPECTION")
print("=" * 60)

# 1. Model information
print("\n1. MODEL")
print("Model type:", type(m5_model).__name__)
print("Current device:", next(m5_model.parameters()).device)
print("Training mode:", m5_model.training)

# 2. Checkpoint metadata
checkpoint = torch.load(
    CHECKPOINT,
    map_location="cpu",
    weights_only=False
)

print("\n2. LOCKED CHECKPOINT")
print("Epoch:", checkpoint.get("epoch"))
print("Validation loss:", checkpoint.get("validation_loss"))
print("Species mapping:", checkpoint.get("species_mapping"))

# 3. Embedding archive
print("\n3. SAVED EMBEDDINGS")

with np.load(EMBEDDING_FILE, allow_pickle=False) as data:
    for key in data.files:
        array = data[key]
        print(
            key,
            "| Shape:", array.shape,
            "| Data type:", array.dtype
        )

# 4. Experiment configuration
print("\n4. EXPERIMENT CONFIGURATION")

config_path = M5_DIR / "experiment_config.json"

with open(config_path, "r") as f:
    config = json.load(f)

print(json.dumps(config, indent=2)[:5000])

# 5. Model output structure
print("\n5. MODEL FORWARD METHOD")
print(m5_model.forward)

print("\nInspection completed. No model or checkpoint modified.")

In [ ]:
import numpy as np
import pandas as pd
import json
from pathlib import Path

M5_DIR = Path(
    "/content/drive/MyDrive/Marine_Project/models/"
    "M5_Hybrid_CBAM_MetricLearning_20261122_153321"
)

EMBEDDING_FILE = M5_DIR / "M5_embeddings_train_validation_20261122_155659.npz"

REFERENCE_DIR = M5_DIR / "M6_reference_bank"
REFERENCE_DIR.mkdir(parents=True, exist_ok=True)

species_mapping = {
    0: ("Asparagopsis armata", "Invasive"),
    1: ("Bifurcaria bifurcata", "Native"),
    2: ("Codium spp.", "Native"),
    3: ("Ericaria selaginoides", "Native"),
    4: ("Saccorhiza polyschides", "Native"),
    5: ("Sargassum muticum", "Invasive")
}

with np.load(EMBEDDING_FILE, allow_pickle=False) as data:
    train_embeddings = data["train_embeddings"].copy()
    train_labels = data["train_labels"].copy()

assert train_embeddings.shape == (2345, 256)
assert len(train_labels) == 2345
assert np.isfinite(train_embeddings).all()

prototypes = {}
records = []

for class_id, (species, status) in species_mapping.items():

    class_embeddings = train_embeddings[train_labels == class_id]

    assert len(class_embeddings) > 0, f"No references found for {species}"

    centroid = class_embeddings.mean(axis=0)

    # L2 normalization
    centroid = centroid / (np.linalg.norm(centroid) + 1e-12)

    prototypes[class_id] = centroid.astype(np.float32)

    records.append({
        "class_id": class_id,
        "species": species,
        "ecological_status": status,
        "reference_count": len(class_embeddings),
        "prototype_norm": float(np.linalg.norm(centroid))
    })

prototype_matrix = np.stack(
    [prototypes[i] for i in range(6)]
).astype(np.float32)

# Preserve reference bank
bank_path = REFERENCE_DIR / "M6_species_reference_prototypes.npz"

np.savez_compressed(
    bank_path,
    prototype_matrix=prototype_matrix,
    class_ids=np.arange(6),
    species_names=np.array([species_mapping[i][0] for i in range(6)])
)

summary = pd.DataFrame(records)

summary_path = REFERENCE_DIR / "M6_reference_bank_summary.csv"
summary.to_csv(summary_path, index=False)

print("=" * 60)
print("M6 — STEP 3: SPECIES REFERENCE BANK")
print("=" * 60)

print(summary.to_string(index=False))
print("\nPrototype matrix shape:", prototype_matrix.shape)
print("All prototypes finite:", np.isfinite(prototype_matrix).all())
print("All prototype norms approximately 1:",
      np.allclose(np.linalg.norm(prototype_matrix, axis=1), 1.0, atol=1e-5))

print("\nSaved reference bank:", bank_path)
print("Saved summary:", summary_path)
print("\nM6 Step 3 completed.")

In [ ]:
import torch
import torch.nn.functional as F
import numpy as np
from PIL import Image
from torchvision import transforms
from pathlib import Path

# Device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Species mapping
species_mapping = {
    0: ("Asparagopsis armata", "Invasive"),
    1: ("Bifurcaria bifurcata", "Native"),
    2: ("Codium spp.", "Native"),
    3: ("Ericaria selaginoides", "Native"),
    4: ("Saccorhiza polyschides", "Native"),
    5: ("Sargassum muticum", "Invasive")
}

# Load saved species prototypes
REFERENCE_FILE = (
    "/content/drive/MyDrive/Marine_Project/models/"
    "M5_Hybrid_CBAM_MetricLearning_20261122_153321/"
    "M6_reference_bank/M6_species_reference_prototypes.npz"
)

reference_data = np.load(REFERENCE_FILE, allow_pickle=False)

prototype_matrix = torch.tensor(
    reference_data["prototype_matrix"],
    dtype=torch.float32,
    device=device
)

# Image preprocessing: same validation preprocessing as M5
inference_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

m5_model = m5_model.to(device)
m5_model.eval()

def predict_m6_image(image_path):

    image_path = Path(image_path)

    if not image_path.exists():
        raise FileNotFoundError(f"Image not found: {image_path}")

    # Open image safely and convert to RGB
    with Image.open(image_path) as img:
        original_image = img.convert("RGB")

    input_tensor = inference_transform(original_image).unsqueeze(0).to(device)

    with torch.inference_mode():

        outputs = m5_model(input_tensor)

        logits = outputs["logits"]
        embedding = outputs["embedding"]

        probabilities = F.softmax(logits, dim=1)[0]

        # Cosine similarity with normalized species prototypes
        similarities = F.normalize(embedding, dim=1) @ prototype_matrix.T
        similarities = similarities[0]

        predicted_id = int(torch.argmax(probabilities).item())

        sorted_similarities, sorted_indices = torch.sort(
            similarities,
            descending=True
        )

    species, ecological_status = species_mapping[predicted_id]

    result = {
        "predicted_species": species,
        "ecological_status": ecological_status,
        "predicted_class_id": predicted_id,
        "class_probabilities": {
            species_mapping[i][0]: float(probabilities[i].item())
            for i in range(6)
        },
        "cosine_similarities": {
            species_mapping[i][0]: float(similarities[i].item())
            for i in range(6)
        },
        "nearest_prototype_species": species_mapping[
            int(sorted_indices[0].item())
        ][0],
        "best_similarity": float(sorted_similarities[0].item()),
        "second_best_similarity": float(sorted_similarities[1].item()),
        "similarity_margin": float(
            (sorted_similarities[0] - sorted_similarities[1]).item()
        ),
        "embedding": embedding[0].cpu().numpy(),
        "original_image": original_image
    }

    return result


print("M6 inference function created successfully.")
print("Device:", device)
print("Prototype matrix:", tuple(prototype_matrix.shape))
print("Model mode:", "Evaluation" if not m5_model.training else "Training")
print("Input size: 224 x 224 RGB")
print("M6 image inference is ready.")

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

# Load existing M5 validation manifest
manifest_path = Path(
    "/content/drive/MyDrive/Marine_Project/results/"
    "M5_validation_manifest_20261122_152903.csv"
)

validation_df = pd.read_csv(manifest_path)

# Select one validation image
sample = validation_df.iloc[0]

filename = sample["Filename"]

# Locate image inside selected image ZIP
import zipfile

image_zip_path = "/content/drive/MyDrive/Marine_Project/IMO_M4_selected_images_3199_20261121_181622.zip"

with zipfile.ZipFile(image_zip_path, "r") as archive:
    matches = [
        name for name in archive.namelist()
        if Path(name).name == filename
    ]

    if len(matches) != 1:
        raise RuntimeError(
            f"Expected exactly one image match for {filename}; found {len(matches)}"
        )

    extracted_path = Path("/content/M6_validation_sample.png")

    with archive.open(matches[0]) as source:
        extracted_path.write_bytes(source.read())

# Run M6 inference
result = predict_m6_image(extracted_path)

print("=" * 60)
print("M6 — STEP 5: FIRST IMAGE PREDICTION")
print("=" * 60)

print("Demonstration split: VALIDATION")
print("Filename:", filename)
print("Actual species:", sample["Especies"])
print("Actual ecological status:", sample["Ecological_Status"])

print("\nPREDICTION")
print("Predicted species:", result["predicted_species"])
print("Predicted ecological status:", result["ecological_status"])

print("\nCLASS PROBABILITIES")
for species, score in result["class_probabilities"].items():
    print(f"{species:30s}: {score:.4f}")

print("\nCOSINE SIMILARITIES")
for species, score in result["cosine_similarities"].items():
    print(f"{species:30s}: {score:.4f}")

print("\nNearest prototype:", result["nearest_prototype_species"])
print("Best similarity:", round(result["best_similarity"], 4))
print("Second-best similarity:", round(result["second_best_similarity"], 4))
print("Similarity margin:", round(result["similarity_margin"], 4))

# Display original image
plt.figure(figsize=(7, 6))
plt.imshow(result["original_image"])
plt.title(
    f"M6 Validation Demonstration\n"
    f"Predicted: {result['predicted_species']}"
)
plt.axis("off")
plt.show()

In [ ]:
import torch
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt

from PIL import Image
from pathlib import Path

# Use the existing validation demonstration image
image_path = Path("/content/M6_validation_sample.png")

# Prepare input
with Image.open(image_path) as img:
    original = img.convert("RGB")

input_tensor = inference_transform(original).unsqueeze(0).to(device)

# Storage for feature maps and gradients
activations = {}
gradients = {}

# Target layer: final ResNet18 convolutional block
target_layer = m5_model.backbone[7]

def forward_hook(module, inputs, output):
    activations["value"] = output
    output.register_hook(
        lambda grad: gradients.update({"value": grad})
    )

hook_handle = target_layer.register_forward_hook(forward_hook)

try:
    m5_model.eval()
    m5_model.zero_grad(set_to_none=True)

    # Forward pass with gradient tracking
    outputs = m5_model(input_tensor)

    predicted_class = int(outputs["logits"].argmax(dim=1).item())
    target_score = outputs["logits"][0, predicted_class]

    # Backpropagate predicted class score
    target_score.backward()

    feature_maps = activations["value"].detach()
    feature_gradients = gradients["value"].detach()

    # Grad-CAM calculation
    weights = feature_gradients.mean(dim=(2, 3), keepdim=True)

    cam = torch.sum(weights * feature_maps, dim=1)
    cam = F.relu(cam)

    cam = F.interpolate(
        cam.unsqueeze(1),
        size=(original.height, original.width),
        mode="bilinear",
        align_corners=False
    )[0, 0]

    cam = cam.cpu().numpy()

    # Normalize heatmap
    cam = cam - cam.min()
    cam = cam / (cam.max() + 1e-8)

finally:
    hook_handle.remove()
    m5_model.zero_grad(set_to_none=True)

# Convert heatmap into RGB
heatmap_uint8 = np.uint8(cam * 255)

heatmap_image = Image.fromarray(
    heatmap_uint8
).convert("RGB")

# Apply visualization colormap
heatmap_colored = plt.get_cmap("jet")(
    np.asarray(heatmap_image)[:, :, 0] / 255.0
)[:, :, :3]

heatmap_colored = Image.fromarray(
    np.uint8(heatmap_colored * 255)
)

# Overlay
overlay = Image.blend(
    original,
    heatmap_colored,
    alpha=0.40
)

# Save outputs
OUTPUT_DIR = Path(
    "/content/drive/MyDrive/Marine_Project/models/"
    "M5_Hybrid_CBAM_MetricLearning_20261122_153321/"
    "M6_visual_outputs"
)

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

original.save(OUTPUT_DIR / "M6_original_validation.png")
heatmap_colored.save(OUTPUT_DIR / "M6_gradcam_heatmap.png")
overlay.save(OUTPUT_DIR / "M6_gradcam_overlay.png")

# Display results
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

axes[0].imshow(original)
axes[0].set_title("Original Image")

axes[1].imshow(heatmap_colored)
axes[1].set_title("Grad-CAM Heatmap")

axes[2].imshow(overlay)
axes[2].set_title("Grad-CAM Overlay")

for ax in axes:
    ax.axis("off")

plt.suptitle(
    f"M6 Grad-CAM | Predicted: {species_mapping[predicted_class][0]}",
    fontsize=14
)

plt.tight_layout()
plt.show()

print("Grad-CAM generated successfully.")
print("Target species:", species_mapping[predicted_class][0])
print("Saved outputs:", OUTPUT_DIR)

In [ ]:
import matplotlib.pyplot as plt
from pathlib import Path
import json

OUTPUT_DIR = Path(
    "/content/drive/MyDrive/Marine_Project/models/"
    "M5_Hybrid_CBAM_MetricLearning_20261122_153321/"
    "M6_visual_outputs"
)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Demonstration decision based on the model's predicted species
predicted_species = result["predicted_species"]
predicted_status = result["ecological_status"]

if predicted_status == "Native":
    suggested_action = "PROTECT"
elif predicted_status == "Invasive":
    suggested_action = "REMOVAL CANDIDATE"
else:
    suggested_action = "UNCLASSIFIED"

review_status = "HUMAN VERIFICATION REQUIRED"

# Build annotated visual report
fig = plt.figure(figsize=(16, 10))
grid = fig.add_gridspec(2, 2, height_ratios=[3, 1])

ax1 = fig.add_subplot(grid[0, 0])
ax1.imshow(original)
ax1.set_title("Original Input Image", fontsize=14)
ax1.axis("off")

ax2 = fig.add_subplot(grid[0, 1])
ax2.imshow(overlay)
ax2.set_title("Grad-CAM Attention Overlay", fontsize=14)
ax2.axis("off")

ax3 = fig.add_subplot(grid[1, :])
ax3.axis("off")

report_text = (
    f"PREDICTED SPECIES: {predicted_species}\n"
    f"ECOLOGICAL STATUS: {predicted_status} (Galicia mapping)\n"
    f"SUGGESTED ACTION: {suggested_action}\n"
    f"SPECIES PROBABILITY: {result['class_probabilities'][predicted_species]:.4f}\n"
    f"PROTOTYPE SIMILARITY: {result['best_similarity']:.4f}\n"
    f"REVIEW: {review_status}\n\n"
    "IMPORTANT: Model output is advisory only. Grad-CAM is not a segmentation mask. "
    "No automatic plant removal is authorized."
)

ax3.text(
    0.03, 0.95,
    report_text,
    transform=ax3.transAxes,
    fontsize=12,
    verticalalignment="top",
    family="monospace",
    bbox=dict(boxstyle="round,pad=0.8", facecolor="whitesmoke")
)

fig.suptitle("M6 — Selective Marine Vegetation Decision Report",
             fontsize=18, fontweight="bold")

plt.tight_layout()

report_path = OUTPUT_DIR / "M6_annotated_decision_report.png"
fig.savefig(report_path, dpi=200, bbox_inches="tight")
plt.show()

plt.close(fig)

# Save structured decision record
decision_record = {
    "demonstration_split": "validation",
    "predicted_species": predicted_species,
    "predicted_ecological_status": predicted_status,
    "suggested_action": suggested_action,
    "species_probability": result["class_probabilities"][predicted_species],
    "prototype_similarity": result["best_similarity"],
    "review_status": review_status,
    "automatic_removal": False,
    "segmentation_available": False,
    "gradcam_available": True
}

json_path = OUTPUT_DIR / "M6_decision_record.json"

with open(json_path, "w") as f:
    json.dump(decision_record, f, indent=4)

print("M6 annotated report saved:", report_path)
print("M6 decision record saved:", json_path)
print("Human verification:", review_status)
print("Automatic removal:", False)

In [ ]:
import importlib.util

print("M6 — STEP 8: INTERFACE ENVIRONMENT CHECK")

if importlib.util.find_spec("gradio") is not None:
    import gradio as gr
    print("Gradio available:", gr.__version__)
    print("Ready to build image upload interface.")
else:
    print("Gradio is not installed.")
    print("We will prepare the installation separately.")

In [ ]:
from google.colab import files
from pathlib import Path

sample_path = Path("/content/M6_validation_sample.png")

if sample_path.exists():
    files.download(str(sample_path))
    print("Validation demonstration image download started.")
else:
    print("Sample image not found. We need to recover it from the saved dataset.")

In [ ]:
import gradio as gr
import torch
import torch.nn.functional as F
import numpy as np
from PIL import Image, ImageDraw
from pathlib import Path

# --------------------------------------------------
# M6 IMAGE ANALYSIS
# --------------------------------------------------

def m6_analyze_upload(uploaded_image):

    if uploaded_image is None:
        raise gr.Error("Please upload an image first.")

    original = uploaded_image.convert("RGB")
    input_tensor = inference_transform(original).unsqueeze(0).to(device)

    activations = {}
    gradients = {}

    target_layer = m5_model.backbone[7]

    def forward_hook(module, inputs, output):
        activations["value"] = output
        output.register_hook(
            lambda grad: gradients.update({"value": grad})
        )

    hook_handle = target_layer.register_forward_hook(forward_hook)

    try:
        m5_model.eval()
        m5_model.zero_grad(set_to_none=True)

        with torch.enable_grad():
            outputs = m5_model(input_tensor)

            logits = outputs["logits"]
            embedding = outputs["embedding"]

            probabilities = F.softmax(logits, dim=1)[0]
            predicted_id = int(torch.argmax(probabilities).item())

            similarities = (
                F.normalize(embedding, dim=1) @ prototype_matrix.T
            )[0]

            # Grad-CAM for predicted class
            logits[0, predicted_id].backward()

            feature_maps = activations["value"].detach()
            feature_gradients = gradients["value"].detach()

            weights = feature_gradients.mean(
                dim=(2, 3), keepdim=True
            )

            cam = torch.sum(weights * feature_maps, dim=1)
            cam = F.relu(cam)

            cam = F.interpolate(
                cam.unsqueeze(1),
                size=(original.height, original.width),
                mode="bilinear",
                align_corners=False
            )[0, 0]

            cam = cam.cpu().numpy()

    finally:
        hook_handle.remove()
        m5_model.zero_grad(set_to_none=True)

    # Normalize Grad-CAM
    cam = cam - cam.min()
    cam = cam / (cam.max() + 1e-8)

    heatmap_array = plt.get_cmap("jet")(cam)[:, :, :3]
    heatmap_image = Image.fromarray(
        np.uint8(heatmap_array * 255)
    )

    overlay = Image.blend(original, heatmap_image, alpha=0.40)

    # Prediction details
    species, status = species_mapping[predicted_id]

    sorted_sim, sorted_ids = torch.sort(
        similarities, descending=True
    )

    if status == "Native":
        suggested_action = "PROTECT"
    else:
        suggested_action = "REMOVAL CANDIDATE"

    review_status = "HUMAN VERIFICATION REQUIRED"

    # Create readable annotated report image
    report = Image.new("RGB", (1100, 420), "white")
    draw = ImageDraw.Draw(report)

    lines = [
        "M6 - SELECTIVE MARINE VEGETATION ANALYSIS",
        "",
        f"Predicted species: {species}",
        f"Ecological status: {status} (Galicia mapping)",
        f"Suggested action: {suggested_action}",
        "",
        f"Softmax probability: {probabilities[predicted_id].item():.4f}",
        f"Best prototype similarity: {sorted_sim[0].item():.4f}",
        f"Second similarity: {sorted_sim[1].item():.4f}",
        "",
        review_status,
        "",
        "Advisory output only. Similarity is not calibrated confidence.",
        "Closed-set classifier: unfamiliar species may be misclassified.",
        "Grad-CAM is not a segmentation mask.",
        "Automatic removal is not authorized."
    ]

    y = 20
    for line in lines:
        draw.text((25, y), line, fill="black")
        y += 27

    # Detailed text output
    details = (
        f"Predicted species: {species}\n"
        f"Ecological status: {status}\n"
        f"Suggested action: {suggested_action}\n"
        f"Review: {review_status}\n\n"
        "CLASS PROBABILITIES\n"
    )

    for i in range(6):
        details += (
            f"{species_mapping[i][0]}: "
            f"{probabilities[i].item():.4f}\n"
        )

    details += "\nPROTOTYPE SIMILARITIES\n"

    for i in range(6):
        details += (
            f"{species_mapping[i][0]}: "
            f"{similarities[i].item():.4f}\n"
        )

    details += (
        "\nImportant limitations:\n"
        "- Six-species closed-set classifier.\n"
        "- Softmax scores are not calibrated confidence.\n"
        "- Grad-CAM is not a segmentation mask.\n"
        "- Human verification is mandatory.\n"
        "- No automatic removal."
    )

    return original, heatmap_image, overlay, report, details


# --------------------------------------------------
# M6 GRADIO INTERFACE
# --------------------------------------------------

with gr.Blocks(title="M6 Marine Vegetation Analysis") as m6_app:

    gr.Markdown(
        """
        # M6 — Selective Marine Vegetation Analysis

        Upload a macroalgae image to inspect the M5 hybrid model's
        species prediction, reference similarity and Grad-CAM visualization.

        **Research demonstration only — human verification is mandatory.**
        """
    )

    with gr.Row():
        image_input = gr.Image(
            type="pil",
            label="Upload Marine Vegetation Image"
        )

    analyze_button = gr.Button(
        "Analyze Image",
        variant="primary"
    )

    with gr.Row():
        original_output = gr.Image(label="Original Image")
        heatmap_output = gr.Image(label="Grad-CAM Heatmap")
        overlay_output = gr.Image(label="Grad-CAM Overlay")

    report_output = gr.Image(
        label="Annotated Decision Report"
    )

    details_output = gr.Textbox(
        label="Prediction and Decision Details",
        lines=22
    )

    analyze_button.click(
        fn=m6_analyze_upload,
        inputs=image_input,
        outputs=[
            original_output,
            heatmap_output,
            overlay_output,
            report_output,
            details_output
        ]
    )

    gr.Markdown(
        """
        **Important:** The model recognizes only six trained species.
        Predictions for unfamiliar species may be incorrect.
        This interface does not perform segmentation or physical removal.
        """
    )

print("M6 interface constructed successfully.")

m6_app.launch(share=True, debug=True)

In [ ]:
from pathlib import Path
from datetime import datetime
import shutil
import hashlib
import json

BASE = Path(
    "/content/drive/MyDrive/Marine_Project/models/"
    "M5_Hybrid_CBAM_MetricLearning_20261122_153321"
)

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

BACKUP = BASE / f"M6_backup_before_decision_engine_{timestamp}"
BACKUP.mkdir(parents=True, exist_ok=False)

files_to_preserve = [
    BASE / "LOCKED_M5_best_checkpoint_epoch16.pth",
    BASE / "M6_reference_bank/M6_species_reference_prototypes.npz",
    BASE / "M6_reference_bank/M6_reference_bank_summary.csv",
    BASE / "M6_visual_outputs/M6_original_validation.png",
    BASE / "M6_visual_outputs/M6_gradcam_heatmap.png",
    BASE / "M6_visual_outputs/M6_gradcam_overlay.png",
    BASE / "M6_visual_outputs/M6_annotated_decision_report.png",
    BASE / "M6_visual_outputs/M6_decision_record.json",
    BASE / "validation_evaluation/validation_metrics.json",
    BASE / "validation_evaluation/validation_predictions.csv",
]

manifest = []

for source in files_to_preserve:
    if not source.exists():
        print("MISSING:", source)
        continue

    destination = BACKUP / source.relative_to(BASE)
    destination.parent.mkdir(parents=True, exist_ok=True)

    shutil.copy2(source, destination)

    sha256 = hashlib.sha256(destination.read_bytes()).hexdigest()

    manifest.append({
        "file": str(source.relative_to(BASE)),
        "backup_file": str(destination.relative_to(BACKUP)),
        "size_bytes": destination.stat().st_size,
        "sha256": sha256
    })

record = {
    "backup_created": timestamp,
    "purpose": "Preserve working M6 before decision-engine improvements",
    "files_copied": len(manifest),
    "files": manifest
}

with open(BACKUP / "M6_backup_manifest.json", "w") as f:
    json.dump(record, f, indent=4)

print("\nM6 BACKUP COMPLETE")
print("Backup folder:", BACKUP)
print("Files preserved:", len(manifest))
print("Manifest saved:", (BACKUP / "M6_backup_manifest.json").exists())

In [ ]:
import pandas as pd
import json
from pathlib import Path

BASE = Path(
    "/content/drive/MyDrive/Marine_Project/models/"
    "M5_Hybrid_CBAM_MetricLearning_20261122_153321"
)

prediction_path = BASE / "validation_evaluation/validation_predictions.csv"
metrics_path = BASE / "validation_evaluation/validation_metrics.json"
diagnostic_path = BASE / "M5_embedding_diagnostics_20261122_160000"

print("=" * 65)
print("M6 — STEP 11: UNCERTAINTY AUDIT")
print("=" * 65)

# Validation predictions
df = pd.read_csv(prediction_path)

print("\n1. VALIDATION PREDICTION FILE")
print("Rows:", len(df))
print("Columns:", df.columns.tolist())
print("\nFirst five records:")
display(df.head())

# Validation metrics
with open(metrics_path, "r") as f:
    metrics = json.load(f)

print("\n2. SAVED VALIDATION METRICS")
print(json.dumps(metrics, indent=2))

# Existing embedding diagnostics
print("\n3. EMBEDDING DIAGNOSTIC FILES")

if diagnostic_path.exists():
    for path in sorted(diagnostic_path.iterdir()):
        if path.is_file():
            print(path.name)

    summary_path = diagnostic_path / "M5_embedding_diagnostic_summary.json"

    if summary_path.exists():
        with open(summary_path, "r") as f:
            summary = json.load(f)

        print("\nEmbedding diagnostic summary:")
        print(json.dumps(summary, indent=2))
else:
    print("Diagnostic folder not found.")

print("\nAudit completed.")
print("No model, checkpoint, or validation predictions modified.")
print("Independent M4 test set accessed: False")


In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

BASE = Path(
    "/content/drive/MyDrive/Marine_Project/models/"
    "M5_Hybrid_CBAM_MetricLearning_20261122_153321"
)

pred_path = BASE / "validation_evaluation/validation_predictions.csv"
embed_path = BASE / "M5_embedding_diagnostics_20261122_160000/validation_embedding_predictions.csv"

df = pd.read_csv(pred_path)

prob_cols = [c for c in df.columns if c.startswith("Probability_")]
prob_values = df[prob_cols].to_numpy(dtype=float)

sorted_probs = np.sort(prob_values, axis=1)

df["Top_Probability"] = sorted_probs[:, -1]
df["Second_Probability"] = sorted_probs[:, -2]
df["Probability_Margin"] = (
    df["Top_Probability"] - df["Second_Probability"]
)
df["Prediction_Entropy"] = -np.sum(
    prob_values * np.log(np.clip(prob_values, 1e-12, 1.0)),
    axis=1
)

print("=" * 65)
print("M6 — STEP 12: PREDICTION RELIABILITY ANALYSIS")
print("=" * 65)

print("\n1. CORRECT VS INCORRECT PREDICTIONS")
summary = df.groupby("Correct_Species").agg(
    Images=("Filename", "count"),
    Mean_Top_Probability=("Top_Probability", "mean"),
    Median_Top_Probability=("Top_Probability", "median"),
    Mean_Probability_Margin=("Probability_Margin", "mean"),
    Median_Probability_Margin=("Probability_Margin", "median"),
    Mean_Entropy=("Prediction_Entropy", "mean")
)
display(summary.round(4))

print("\n2. SCORE DISTRIBUTIONS BY PREDICTED SPECIES")
by_species = df.groupby("Predicted_Species").agg(
    Images=("Filename", "count"),
    Correct=("Correct_Species", "sum"),
    Mean_Top_Probability=("Top_Probability", "mean"),
    Mean_Margin=("Probability_Margin", "mean")
)
by_species["Accuracy"] = by_species["Correct"] / by_species["Images"]
display(by_species.round(4))

print("\n3. PROBABILITY SCORE QUANTILES")
quantiles = df.groupby("Correct_Species")[
    ["Top_Probability", "Probability_Margin", "Prediction_Entropy"]
].quantile([0.10, 0.25, 0.50, 0.75, 0.90])
display(quantiles.round(4))

print("\n4. EMBEDDING PREDICTION FILE")
if embed_path.exists():
    emb = pd.read_csv(embed_path)
    print("Rows:", len(emb))
    print("Columns:", emb.columns.tolist())
    display(emb.head(5))

    if "Filename" in emb.columns:
        shared = set(df["Filename"]) & set(emb["Filename"])
        print("Matching filenames:", len(shared), "/", len(df))
else:
    print("Embedding prediction file not found.")

print("\n5. SAFETY")
print("Checkpoint modified: False")
print("Validation predictions modified: False")
print("M4 independent test accessed: False")
print("\nAnalysis complete.")

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

BASE = Path(
    "/content/drive/MyDrive/Marine_Project/models/"
    "M5_Hybrid_CBAM_MetricLearning_20261122_153321"
)

df = pd.read_csv(
    BASE / "validation_evaluation/validation_predictions.csv"
)

prob_cols = [c for c in df.columns if c.startswith("Probability_")]
probs = df[prob_cols].to_numpy(dtype=float)

df["Top_Probability"] = probs.max(axis=1)

thresholds = [0.70, 0.75, 0.78, 0.79, 0.795, 0.80]

results = []

for threshold in thresholds:
    accepted = df[df["Top_Probability"] >= threshold]
    reviewed = df[df["Top_Probability"] < threshold]

    results.append({
        "Threshold": threshold,
        "Accepted": len(accepted),
        "Sent_for_review": len(reviewed),
        "Coverage_percent": round(100 * len(accepted) / len(df), 2),
        "Accuracy_of_accepted_percent": (
            round(100 * accepted["Correct_Species"].mean(), 2)
            if len(accepted) else np.nan
        ),
        "Incorrect_accepted": int((~accepted["Correct_Species"]).sum())
    })

result_df = pd.DataFrame(results)

print("M6 — STEP 13: EXPLORATORY REVIEW-POLICY SIMULATION")
display(result_df)

print("\nInterpretation:")
print("These thresholds are hypothetical, not calibrated.")
print("No threshold is being adopted as a safe decision boundary.")
print("Human verification remains mandatory.")
print("Automatic removal remains disabled.")
print("M4 independent test accessed: False")

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

BASE = Path(
    "/content/drive/MyDrive/Marine_Project/models/"
    "M5_Hybrid_CBAM_MetricLearning_20261122_153321"
)

df = pd.read_csv(
    BASE / "validation_evaluation/validation_predictions.csv"
)

prob_cols = [c for c in df.columns if c.startswith("Probability_")]
df["Top_Probability"] = df[prob_cols].max(axis=1)

# Exploratory threshold only; not a deployment rule
THRESHOLD = 0.795

df["Review_Group"] = np.where(
    df["Top_Probability"] < THRESHOLD,
    "Human review",
    "Retained for analysis"
)

df["Ecological_Error"] = (
    df["True_Ecological_Status"] !=
    df["Predicted_Ecological_Status"]
)

df["Species_Error"] = ~df["Correct_Species"].astype(bool)

print("=" * 65)
print("M6 — STEP 14: ECOLOGICAL SAFETY ANALYSIS")
print("=" * 65)

print("\n1. ECOLOGICAL ERRORS BEFORE FILTERING")
print(pd.crosstab(
    df["True_Ecological_Status"],
    df["Predicted_Ecological_Status"],
    rownames=["Actual"],
    colnames=["Predicted"]
))

print("\n2. ECOLOGICAL ERRORS BY REVIEW GROUP")
for group, part in df.groupby("Review_Group"):
    print(f"\n{group}: {len(part)} images")
    print("Species errors:", int(part["Species_Error"].sum()))
    print("Ecological errors:", int(part["Ecological_Error"].sum()))
    print("Ecological accuracy:",
          round(100 * (1 - part["Ecological_Error"].mean()), 2), "%")

    print(pd.crosstab(
        part["True_Ecological_Status"],
        part["Predicted_Ecological_Status"]
    ))

print("\n3. SPECIES-WISE RETAINED ACCURACY")
retained = df[df["Review_Group"] == "Retained for analysis"]

species_report = retained.groupby("True_Species").agg(
    Retained_images=("Filename", "count"),
    Correct_species=("Correct_Species", "sum"),
    Ecological_errors=("Ecological_Error", "sum")
)

species_report["Retained_species_accuracy"] = (
    species_report["Correct_species"] /
    species_report["Retained_images"]
)

display(species_report.round(4))

print("\n4. SAFETY NOTES")
print("Threshold 0.795 is exploratory only.")
print("No autonomous removal decision is permitted.")
print("No M4 independent test images accessed.")
print("No model or checkpoint modified.")

In [ ]:
import json
from pathlib import Path

# Exploratory threshold from M6 validation analysis
M6_REVIEW_THRESHOLD = 0.795

def m6_decision_engine(result):

    species = result["predicted_species"]
    status = result["ecological_status"]

    probability = float(
        result["class_probabilities"][species]
    )

    # Priority based on exploratory threshold
    if probability < M6_REVIEW_THRESHOLD:
        review_priority = "HIGH"
        review_reason = (
            "Prediction falls below the exploratory "
            "validation-derived review threshold."
        )
    else:
        review_priority = "NORMAL"
        review_reason = (
            "Prediction meets the exploratory threshold, "
            "but human verification remains mandatory."
        )

    # Ecological decision mapping
    if species not in [item[0] for item in species_mapping.values()]:
        recommendation = "UNSUPPORTED SPECIES — HUMAN REVIEW"
        removal_authorized = False

    elif status == "Native":
        recommendation = "PROVISIONAL PROTECT RECOMMENDATION"
        removal_authorized = False

    elif status == "Invasive":
        recommendation = "POTENTIAL REMOVAL CANDIDATE"
        removal_authorized = False
        review_priority = "HIGH"

    else:
        recommendation = "UNCLASSIFIED — HUMAN REVIEW"
        removal_authorized = False
        review_priority = "HIGH"

    decision = {
        "predicted_species": species,
        "predicted_ecological_status": status,
        "ecological_mapping_region": "Galicia, Spain",
        "softmax_probability": probability,
        "probability_threshold": M6_REVIEW_THRESHOLD,
        "threshold_type": "Exploratory, not calibrated",
        "recommendation": recommendation,
        "review_priority": review_priority,
        "review_reason": review_reason,
        "human_verification": "MANDATORY",
        "expert_approval_required": True,
        "automatic_removal_authorized": removal_authorized,
        "segmentation_mask_available": False
    }

    return decision


# Test using the existing validation demonstration
decision = m6_decision_engine(result)

print("=" * 60)
print("M6 — STEP 15: CONSERVATIVE DECISION ENGINE")
print("=" * 60)

print(json.dumps(decision, indent=4))

# Preserve the policy configuration
BASE = Path(
    "/content/drive/MyDrive/Marine_Project/models/"
    "M5_Hybrid_CBAM_MetricLearning_20261122_153321"
)

POLICY_DIR = BASE / "M6_decision_engine"
POLICY_DIR.mkdir(parents=True, exist_ok=True)

policy_path = POLICY_DIR / "M6_decision_policy.json"

with open(policy_path, "w") as f:
    json.dump({
        "review_threshold": M6_REVIEW_THRESHOLD,
        "threshold_status": "Exploratory only",
        "human_verification": "Mandatory for every prediction",
        "automatic_removal": False,
        "ecological_region": "Galicia, Spain",
        "unsupported_species_policy": "Human review",
        "m4_independent_test_accessed": False
    }, f, indent=4)

print("\nDecision policy saved:", policy_path)
print("M5 checkpoint modified: False")
print("Automatic removal: Disabled")

In [ ]:
import gradio as gr
import torch
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image, ImageDraw
from pathlib import Path

def m6_analyze_upload_v2(uploaded_image):

    if uploaded_image is None:
        raise gr.Error("Please upload an image.")

    original = uploaded_image.convert("RGB")
    input_tensor = inference_transform(original).unsqueeze(0).to(device)

    activations = {}
    gradients = {}

    def forward_hook(module, inputs, output):
        activations["value"] = output
        output.register_hook(
            lambda grad: gradients.update({"value": grad})
        )

    hook_handle = m5_model.backbone[7].register_forward_hook(forward_hook)

    try:
        m5_model.eval()
        m5_model.zero_grad(set_to_none=True)

        with torch.enable_grad():
            outputs = m5_model(input_tensor)
            logits = outputs["logits"]
            embedding = outputs["embedding"]

            probabilities = F.softmax(logits, dim=1)[0]
            predicted_id = int(torch.argmax(probabilities).item())

            similarities = (
                F.normalize(embedding, dim=1) @ prototype_matrix.T
            )[0]

            logits[0, predicted_id].backward()

            feature_maps = activations["value"].detach()
            feature_gradients = gradients["value"].detach()

            weights = feature_gradients.mean(dim=(2, 3), keepdim=True)

            cam = torch.sum(weights * feature_maps, dim=1)
            cam = F.relu(cam)

            cam = F.interpolate(
                cam.unsqueeze(1),
                size=(original.height, original.width),
                mode="bilinear",
                align_corners=False
            )[0, 0].cpu().numpy()

    finally:
        hook_handle.remove()
        m5_model.zero_grad(set_to_none=True)

    # Grad-CAM visualization
    cam = cam - cam.min()
    cam = cam / (cam.max() + 1e-8)

    heatmap_array = plt.get_cmap("jet")(cam)[:, :, :3]
    heatmap = Image.fromarray(np.uint8(heatmap_array * 255))
    overlay = Image.blend(original, heatmap, alpha=0.40)

    # Prediction result
    species, status = species_mapping[predicted_id]

    result_v2 = {
        "predicted_species": species,
        "ecological_status": status,
        "class_probabilities": {
            species_mapping[i][0]: float(probabilities[i].item())
            for i in range(6)
        }
    }

    # Apply M6 decision engine
    decision_v2 = m6_decision_engine(result_v2)

    # Create annotated report
    report = Image.new("RGB", (1200, 470), "white")
    draw = ImageDraw.Draw(report)

    lines = [
        "M6 — SELECTIVE MARINE VEGETATION ANALYSIS",
        "",
        f"Predicted species: {species}",
        f"Ecological status: {status} (Galicia mapping)",
        f"Recommendation: {decision_v2['recommendation']}",
        f"Review priority: {decision_v2['review_priority']}",
        f"Softmax probability: {probabilities[predicted_id].item():.4f}",
        f"Best prototype similarity: {similarities.max().item():.4f}",
        "",
        "HUMAN VERIFICATION MANDATORY",
        "Expert approval required.",
        "Automatic removal is disabled.",
        "Grad-CAM is not a segmentation mask.",
        "Predictions for unsupported species may be incorrect."
    ]

    y = 20
    for line in lines:
        draw.text((25, y), line, fill="black")
        y += 32

    # Detailed result
    details = (
        f"PREDICTION\n"
        f"Species: {species}\n"
        f"Ecological status: {status}\n"
        f"Probability: {probabilities[predicted_id].item():.4f}\n"
        f"Prototype similarity: {similarities.max().item():.4f}\n\n"
        f"DECISION\n"
        f"Recommendation: {decision_v2['recommendation']}\n"
        f"Review priority: {decision_v2['review_priority']}\n"
        f"Review reason: {decision_v2['review_reason']}\n\n"
        "CLASS PROBABILITIES\n"
    )

    for i in range(6):
        details += (
            f"{species_mapping[i][0]}: "
            f"{probabilities[i].item():.4f}\n"
        )

    details += "\nPROTOTYPE SIMILARITIES\n"

    for i in range(6):
        details += (
            f"{species_mapping[i][0]}: "
            f"{similarities[i].item():.4f}\n"
        )

    details += (
        "\nSAFETY\n"
        "Human verification: Mandatory\n"
        "Expert approval: Required\n"
        "Automatic removal: Disabled\n"
        "Segmentation mask: Not available\n"
        "Threshold: Exploratory, not calibrated"
    )

    # Save outputs
    output_dir = Path(
        "/content/drive/MyDrive/Marine_Project/models/"
        "M5_Hybrid_CBAM_MetricLearning_20261122_153321/"
        "M6_visual_outputs"
    )
    output_dir.mkdir(parents=True, exist_ok=True)

    original.save(output_dir / "M6_latest_original.png")
    heatmap.save(output_dir / "M6_latest_heatmap.png")
    overlay.save(output_dir / "M6_latest_overlay.png")
    report.save(output_dir / "M6_latest_decision_report.png")

    with open(output_dir / "M6_latest_decision.json", "w") as f:
        import json
        json.dump(decision_v2, f, indent=4)

    return original, heatmap, overlay, report, details


with gr.Blocks(title="M6 Marine Vegetation Analysis v2") as m6_app_v2:

    gr.Markdown(
        """
        # M6 — Selective Marine Vegetation Analysis

        Upload a marine vegetation image for species classification,
        Grad-CAM visualization and conservative ecological decision support.

        **Human verification is mandatory. No automatic removal is authorized.**
        """
    )

    image_input_v2 = gr.Image(type="pil", label="Upload image")

    analyze_button_v2 = gr.Button(
        "Analyze Image",
        variant="primary"
    )

    with gr.Row():
        original_out_v2 = gr.Image(label="Original")
        heatmap_out_v2 = gr.Image(label="Grad-CAM")
        overlay_out_v2 = gr.Image(label="Attention Overlay")

    report_out_v2 = gr.Image(label="Annotated Decision Report")

    details_out_v2 = gr.Textbox(
        label="Prediction and Decision Details",
        lines=24
    )

    analyze_button_v2.click(
        fn=m6_analyze_upload_v2,
        inputs=image_input_v2,
        outputs=[
            original_out_v2,
            heatmap_out_v2,
            overlay_out_v2,
            report_out_v2,
            details_out_v2
        ]
    )

print("M6 v2 interface created.")

m6_app_v2.launch(share=True, debug=True)

In [ ]:
import json
from pathlib import Path
from datetime import datetime

m6_root = Path(
    "/content/drive/MyDrive/Marine_Project/models/"
    "M5_Hybrid_CBAM_MetricLearning_20261122_153321"
)

verification = {
    "milestone": "M6",
    "step": 17,
    "component": "Gradio v2 end-to-end integration",
    "test_type": "Existing validation image demonstration",
    "predicted_species": "Ericaria selaginoides",
    "predicted_ecological_status": "Native",
    "softmax_probability": 0.796318769454956,
    "prototype_similarity": 0.9972,
    "recommendation": "PROVISIONAL PROTECT RECOMMENDATION",
    "review_priority": "NORMAL",
    "human_verification": "MANDATORY",
    "expert_approval_required": True,
    "automatic_removal_authorized": False,
    "segmentation_mask_available": False,
    "gradcam_available": True,
    "six_class_probabilities_available": True,
    "prototype_similarities_available": True,
    "checkpoint_retrained": False,
    "independent_test_accessed": False,
    "threshold_status": "Exploratory, not calibrated",
    "limitations": [
        "The classifier is closed-set and cannot reliably identify unknown species.",
        "The ecological mapping is specific to Galicia, Spain.",
        "The dataset contains intertidal macroalgae imagery, not validated underwater removal imagery.",
        "Grad-CAM is not a segmentation mask or removal boundary.",
        "Prototype similarity and softmax probability are not calibrated confidence.",
        "Human verification and expert approval remain mandatory."
    ],
    "verification_status": "PASSED",
    "verified_at_ist": datetime.now().astimezone().isoformat()
}

verification_dir = m6_root / "M6_final_verification"
verification_dir.mkdir(parents=True, exist_ok=True)

verification_path = (
    verification_dir / "M6_step17_integration_verification.json"
)

with open(verification_path, "w") as f:
    json.dump(verification, f, indent=4)

print("=" * 60)
print("M6 STEP 17 — INTEGRATION VERIFICATION")
print("=" * 60)
print("Status:", verification["verification_status"])
print("Saved:", verification_path)
print("Checkpoint retrained:", verification["checkpoint_retrained"])
print("Independent test accessed:", verification["independent_test_accessed"])
print("Automatic removal:", verification["automatic_removal_authorized"])
print("=" * 60)

In [ ]:
import json
import hashlib
from pathlib import Path
from datetime import datetime

root = Path(
    "/content/drive/MyDrive/Marine_Project/models/"
    "M5_Hybrid_CBAM_MetricLearning_20261122_153321"
)

expected_files = [
    root / "LOCKED_M5_best_checkpoint_epoch16.pth",
    root / "M6_reference_bank/M6_species_reference_prototypes.npz",
    root / "M6_reference_bank/M6_reference_bank_summary.csv",
    root / "M6_decision_engine/M6_decision_policy.json",
    root / "M6_visual_outputs/M6_original_validation.png",
    root / "M6_visual_outputs/M6_gradcam_heatmap.png",
    root / "M6_visual_outputs/M6_gradcam_overlay.png",
    root / "M6_visual_outputs/M6_annotated_decision_report.png",
    root / "M6_visual_outputs/M6_decision_record.json",
    root / "M6_visual_outputs/M6_latest_original.png",
    root / "M6_visual_outputs/M6_latest_heatmap.png",
    root / "M6_visual_outputs/M6_latest_overlay.png",
    root / "M6_visual_outputs/M6_latest_decision_report.png",
    root / "M6_visual_outputs/M6_latest_decision.json",
    root / "M6_final_verification/M6_step17_integration_verification.json",
    root / "training_history.csv",
    root / "experiment_config.json",
    root / "validation_evaluation/validation_predictions.csv",
    root / "validation_evaluation/validation_metrics.json",
    root / "M5_embeddings_train_validation_20261122_155659.npz",
]

def sha256_file(path):
    digest = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

inventory = []

for path in expected_files:
    record = {
        "file": str(path.relative_to(root)),
        "exists": path.is_file()
    }

    if path.is_file():
        record["size_bytes"] = path.stat().st_size
        record["sha256"] = sha256_file(path)

    inventory.append(record)

existing = sum(item["exists"] for item in inventory)
missing = len(inventory) - existing

audit = {
    "milestone": "M6",
    "audit_type": "Final artifact integrity inventory",
    "created_at_ist": datetime.now().astimezone().isoformat(),
    "expected_file_count": len(expected_files),
    "existing_file_count": existing,
    "missing_file_count": missing,
    "files": inventory,
    "checkpoint_retrained": False,
    "M4_independent_test_accessed_during_M6": False,
    "automatic_removal_authorized": False
}

audit_dir = root / "M6_final_verification"
audit_dir.mkdir(parents=True, exist_ok=True)

audit_path = audit_dir / "M6_final_artifact_inventory.json"

with open(audit_path, "w") as f:
    json.dump(audit, f, indent=4)

print("=" * 60)
print("M6 STEP 18 — FINAL ARTIFACT AUDIT")
print("=" * 60)

for item in inventory:
    status = "FOUND" if item["exists"] else "MISSING"
    print(f"{status}: {item['file']}")

print("-" * 60)
print(f"Expected files: {len(expected_files)}")
print(f"Found: {existing}")
print(f"Missing: {missing}")
print("Inventory saved:", audit_path)
print("=" * 60)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
%run /content/Marine_Project_Restoration_Master_M1_M6.py